# FAIR1M ship detection — resolution sweep (Kaggle)

Runs the GPU half of the study: four YOLOv8m detectors on the same 7,706
images and the same 58,982 ship annotations, differing only in image
resolution (100 / 75 / 50 / 25%). Local dataset preparation, annotation
cleaning and preprocessing validation are already complete — this notebook
only trains, evaluates and aggregates, reading the imagery straight from the
attached Kaggle Dataset.

**Before running:**
- Attach the `Fair1m_Ship_Dataset` Kaggle Dataset: Notebook → Add Input →
  Datasets → search "Fair1m_Ship_Dataset".
- Turn on an accelerator: Notebook → Settings → Accelerator → GPU (T4 x2 or
  better).
- Turn Internet ON: Notebook → Settings → Internet. Needed once, to
  `pip install ultralytics` and let it auto-download the COCO-pretrained
  `yolov8m.pt` starting weights.

Then just **Run All**. Steps 0–5 are bootstrap and verification (well under a
minute); step 6 is a cheap GPU sanity run (~1 minute). Step 7 is the training
sweep, and it does **not** try to finish in one run: the full sweep is
12–20 GPU-hours, far more than one Kaggle session or one commit should bet on
completing, so Step 7 does one `TIME_BUDGET_HOURS`-sized slice (default 1.5 h)
and stops itself cleanly — commit (Step 8), then commit this same notebook
again to continue, as many times as it takes. **Every step is safe to
re-run**: finished conditions are skipped, and a condition that was
mid-training resumes from its own checkpoint instead of restarting at epoch 1.
No cell below needs editing.

## Step 0 — Materialize the code package

This notebook is self-contained: the small validated code package (under
200 KB of Python, configs and docs — **not** the multi-GB imagery, which
stays in the attached Kaggle Dataset) is embedded below and written out to
`/kaggle/working/kaggle` once per session. Edit the source under `kaggle/` in
the project repo and regenerate with `scripts/build_kaggle_notebook.py` — do
not hand-edit the cells below.

The ~1.5 MB ship-annotation layer (ground-truth boxes, image manifest) is
**not** embedded here — Kaggle caps a notebook's own source at 1 MB, and that
file alone exceeded it. Step 4 below reconstructs it instead, directly from
the attached dataset's own YOLO labels.

In [ ]:
import base64
from pathlib import Path

PKG = Path("/kaggle/working/kaggle")
_FILES = {
    "src/__init__.py": "IiIiR2VuZXJhdGVkIGNvcHkgb2YgdGhlIHNoYXJlZCByZXNlYXJjaCBtb2R1bGVzLiIiIg0K",
    "src/analysis.py": "IyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgR0VORVJBVEVEIENPUFkgLS0gZG8gbm90IGVkaXQgaGVyZS4NCiMgU291cmNlIG9mIHRydXRoOiA8cHJvamVjdD4vc3JjL2FuYWx5c2lzLnB5DQojIFJlZ2VuZXJhdGUgd2l0aDogcHl0aG9uIHNjcmlwdHMvY3JlYXRlX2NvbGFiX3BhY2thZ2UucHkNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQoiIiINCk9iamVjdC1zaXplIGFuYWx5c2lzIGZvciB0aGUgcmVzb2x1dGlvbiBzdHVkeSAoUGFydHMgRyBhbmQgSCkuDQoNClR3byBpZGVhcyBjYXJyeSB0aGUgd2hvbGUgcmVzb2x1dGlvbiB4IG9iamVjdC1zaXplIGFuYWx5c2lzIGFuZCBhcmUgd29ydGgNCnN0YXRpbmcgdXAgZnJvbnQ6DQoNCjEuIFNpemUgYmlucyBhcmUgYSBwcm9wZXJ0eSBvZiB0aGUgT0JKRUNULCBmaXhlZCBvbmNlIGF0IG9yaWdpbmFsIHJlc29sdXRpb24uDQogICBUaGV5IGFyZSAqbm90KiByZWNvbXB1dGVkIHBlciByZXNvbHV0aW9uLiBJZiBiaW5zIHdlcmUgcmUtZGVyaXZlZCBhdCBlYWNoDQogICByZXNvbHV0aW9uLCBldmVyeSBvYmplY3Qgd291bGQgc2xpZGUgaW50byAic21hbGwiIGF0IHIyNSBhbmQgdGhlDQogICByZXNvbHV0aW9uIHggc2l6ZSB0YWJsZSB3b3VsZCBjb21wYXJlIGRpZmZlcmVudCBwb3B1bGF0aW9ucyBhdCBlYWNoIHJvdywNCiAgIHdoaWNoIHdvdWxkIG1ha2UgaXQgdW5pbnRlcnByZXRhYmxlLiBCaW5uaW5nIGJ5IG9yaWdpbmFsLXJlc29sdXRpb24gYXJlYQ0KICAgbWVhbnMgZWFjaCByb3cgb2YgdGhhdCB0YWJsZSBkZXNjcmliZXMgdGhlIHNhbWUgc2hpcHMsIHNvIGEgZHJvcCBpbg0KICAgYEFQX3NtYWxsYCBpcyBhdHRyaWJ1dGFibGUgdG8gbG9zdCBkZXRhaWwgcmF0aGVyIHRoYW4gdG8gcmVjbGFzc2lmaWNhdGlvbi4NCg0KMi4gRG93bnNhbXBsaW5nIGJ5IGEgbGluZWFyIGZhY3RvciBzIHNjYWxlcyBib3ggYXJlYSBieSBzXjIuIEEgNTAlIHJlc29sdXRpb24NCiAgIHJlZHVjdGlvbiB0aGVyZWZvcmUgcmVtb3ZlcyA3NSUgb2YgYW4gb2JqZWN0J3MgcGl4ZWwgZm9vdHByaW50LCBub3QgNTAlLg0KICAgYGRvd25zYW1wbGVkX3NpemVfdGFibGVgIG1ha2VzIHRoYXQgZXhwbGljaXQuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KIyBDT0NPIG9iamVjdC1zaXplIGNvbnZlbnRpb24sIGluIHBpeGVsc14yIG9mIHRoZSBPUklHSU5BTCBpbWFnZS4NCiMgICBzbWFsbCAgOiBhcmVhIDwgIDMyXjIgPSAgMTAyNA0KIyAgIG1lZGl1bSA6IDEwMjQgPD0gYXJlYSA8IDk2XjIgPSA5MjE2DQojICAgbGFyZ2UgIDogYXJlYSA+PSA5MjE2DQpDT0NPX1NNQUxMX01BWCA9IDMyICoqIDINCkNPQ09fTUVESVVNX01BWCA9IDk2ICoqIDINClNJWkVfQklOUyA9ICgic21hbGwiLCAibWVkaXVtIiwgImxhcmdlIikNCg0KIyBSZXNvbHV0aW9uIGNvbmRpdGlvbnM6IGxhYmVsIC0+IGxpbmVhciBzY2FsZSBmYWN0b3IgYXBwbGllZCB0byB3aWR0aC9oZWlnaHQuDQpSRVNPTFVUSU9OUzogZGljdFtzdHIsIGZsb2F0XSA9IHsNCiAgICAib3JpZ2luYWwiOiAxLjAwLA0KICAgICJyNzUiOiAwLjc1LA0KICAgICJyNTAiOiAwLjUwLA0KICAgICJyMjUiOiAwLjI1LA0KfQ0KDQojIFRoZSAiZG9lcyB0aGlzIG9iamVjdCBzdGlsbCBleGlzdCBhcyBhIGRldGVjdGFibGUgZm9vdHByaW50IiBsYWRkZXIgKFBhcnQgRykuDQpTSVpFX1RIUkVTSE9MRFNfUFggPSAoNCwgOCwgMTYsIDMyLCA2NCkNCg0KDQpkZWYgYXNzaWduX3NpemVfYmluKGFyZWE6IHBkLlNlcmllcywNCiAgICAgICAgICAgICAgICAgICAgc21hbGxfbWF4OiBmbG9hdCA9IENPQ09fU01BTExfTUFYLA0KICAgICAgICAgICAgICAgICAgICBtZWRpdW1fbWF4OiBmbG9hdCA9IENPQ09fTUVESVVNX01BWCkgLT4gcGQuU2VyaWVzOg0KICAgICIiIk1hcCBvcmlnaW5hbC1yZXNvbHV0aW9uIGJveCBhcmVhIChweF4yKSB0byBzbWFsbCAvIG1lZGl1bSAvIGxhcmdlLiIiIg0KICAgIHJldHVybiBwZC5jdXQoDQogICAgICAgIGFyZWEsDQogICAgICAgIGJpbnM9Wy1ucC5pbmYsIHNtYWxsX21heCwgbWVkaXVtX21heCwgbnAuaW5mXSwNCiAgICAgICAgbGFiZWxzPWxpc3QoU0laRV9CSU5TKSwNCiAgICAgICAgcmlnaHQ9RmFsc2UsDQogICAgKS5hc3R5cGUob2JqZWN0KQ0KDQoNCmRlZiBiYm94X2dlb21ldHJ5KGRmOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToNCiAgICAiIiJQZXItb2JqZWN0IGdlb21ldHJ5IHVzZWQgdGhyb3VnaG91dCBQYXJ0IEcuIiIiDQogICAgb3V0ID0gcGQuRGF0YUZyYW1lKGluZGV4PWRmLmluZGV4KQ0KICAgIG91dFsid2lkdGhfcHgiXSA9IGRmWyJidyJdDQogICAgb3V0WyJoZWlnaHRfcHgiXSA9IGRmWyJiaCJdDQogICAgb3V0WyJhcmVhX3B4MiJdID0gZGZbImJhcmVhIl0NCiAgICBvdXRbInJlbGF0aXZlX2FyZWEiXSA9IGRmWyJiYXJlYSJdIC8gKGRmWyJJbWFnZVdpZHRoIl0gKiBkZlsiSW1hZ2VIZWlnaHQiXSkNCiAgICAjIEFzcGVjdCByYXRpbyBpcyB1bmRlZmluZWQgZm9yIHplcm8taGVpZ2h0IGJveGVzOyB0aG9zZSBhcmUgYWxyZWFkeQ0KICAgICMgZXhjbHVkZWQgdXBzdHJlYW0sIGJ1dCBndWFyZCBhbnl3YXkgc28gdGhpcyBpcyBzYWZlIG9uIHJhdyBpbnB1dC4NCiAgICBvdXRbImFzcGVjdF9yYXRpbyJdID0gZGZbImJ3Il0gLyBkZlsiYmgiXS5yZXBsYWNlKDAsIG5wLm5hbikNCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIGRlc2NyaWJlKHNlcmllczogcGQuU2VyaWVzLCBxcz0oMC4wMSwgMC4wNSwgMC4yNSwgMC41LCAwLjc1LCAwLjk1LCAwLjk5KSkgLT4gZGljdDoNCiAgICBzID0gc2VyaWVzLmRyb3BuYSgpDQogICAgZCA9IHsibiI6IGludChsZW4ocykpLCAibWluIjogZmxvYXQocy5taW4oKSksICJtYXgiOiBmbG9hdChzLm1heCgpKSwNCiAgICAgICAgICJtZWFuIjogZmxvYXQocy5tZWFuKCkpLCAic3RkIjogZmxvYXQocy5zdGQoKSl9DQogICAgZm9yIHEgaW4gcXM6DQogICAgICAgIGRbZiJwe2ludChxICogMTAwKX0iXSA9IGZsb2F0KHMucXVhbnRpbGUocSkpDQogICAgcmV0dXJuIGQNCg0KDQpkZWYgdGhyZXNob2xkX2NvdW50cyh3aWR0aDogcGQuU2VyaWVzLCBoZWlnaHQ6IHBkLlNlcmllcywNCiAgICAgICAgICAgICAgICAgICAgIHRocmVzaG9sZHM9U0laRV9USFJFU0hPTERTX1BYKSAtPiBkaWN0Og0KICAgICIiIkNvdW50IG9iamVjdHMgYmVsb3cgZWFjaCBgdCB4IHRgIHB4IGZvb3RwcmludC4NCg0KICAgIFR3byByZWFkaW5ncyBhcmUgcmVwb3J0ZWQgYmVjYXVzZSB0aGV5IGFuc3dlciBkaWZmZXJlbnQgcXVlc3Rpb25zOg0KICAgICAgYnlfYXJlYSAgLS0gYXJlYSA8IHQqdC4gVG90YWwgZm9vdHByaW50LCB0aGUgdXN1YWwgc21hbGwtb2JqZWN0IG1lYXN1cmUuDQogICAgICBieV9zaWRlICAtLSBtYXgod2lkdGgsIGhlaWdodCkgPCB0LiBXaGV0aGVyIHRoZSBvYmplY3QncyAqbG9uZ2VzdCogc2lkZQ0KICAgICAgICAgICAgICAgICAgaGFzIGRyb3BwZWQgYmVsb3cgdCBweCwgaS5lLiB3aGV0aGVyIGl0IGhhcyBlZmZlY3RpdmVseQ0KICAgICAgICAgICAgICAgICAgdmFuaXNoZWQgYWxvbmcgZXZlcnkgYXhpcy4gU3RyaWN0ZXIsIGFuZCB0aGUgbW9yZSBob25lc3QNCiAgICAgICAgICAgICAgICAgIHRlc3Qgb2YgImlzIHRoZXJlIGFueXRoaW5nIGxlZnQgdG8gZGV0ZWN0Ii4NCiAgICAiIiINCiAgICBuID0gbGVuKHdpZHRoKQ0KICAgIGFyZWEgPSB3aWR0aCAqIGhlaWdodA0KICAgIGxvbmdlc3QgPSBucC5tYXhpbXVtKHdpZHRoLCBoZWlnaHQpDQogICAgcmVzID0ge30NCiAgICBmb3IgdCBpbiB0aHJlc2hvbGRzOg0KICAgICAgICBieV9hcmVhID0gaW50KChhcmVhIDwgdCAqIHQpLnN1bSgpKQ0KICAgICAgICBieV9zaWRlID0gaW50KChsb25nZXN0IDwgdCkuc3VtKCkpDQogICAgICAgIHJlc1tmInt0fXh7dH0iXSA9IHsNCiAgICAgICAgICAgICJiZWxvd19ieV9hcmVhIjogYnlfYXJlYSwNCiAgICAgICAgICAgICJiZWxvd19ieV9hcmVhX3BjdCI6IHJvdW5kKGJ5X2FyZWEgLyBuICogMTAwLCAzKSBpZiBuIGVsc2UgMC4wLA0KICAgICAgICAgICAgImJlbG93X2J5X2xvbmdlc3Rfc2lkZSI6IGJ5X3NpZGUsDQogICAgICAgICAgICAiYmVsb3dfYnlfbG9uZ2VzdF9zaWRlX3BjdCI6IHJvdW5kKGJ5X3NpZGUgLyBuICogMTAwLCAzKSBpZiBuIGVsc2UgMC4wLA0KICAgICAgICB9DQogICAgcmV0dXJuIHJlcw0KDQoNCmRlZiBkb3duc2FtcGxlZF9zaXplX3RhYmxlKGRmOiBwZC5EYXRhRnJhbWUsDQogICAgICAgICAgICAgICAgICAgICAgICAgICByZXNvbHV0aW9uczogZGljdFtzdHIsIGZsb2F0XSA9IFJFU09MVVRJT05TKSAtPiBkaWN0Og0KICAgICIiIlBhcnQgRzogaG93IHRoZSBzaGlwIHNpemUgZGlzdHJpYnV0aW9uIG1vdmVzIGFzIHJlc29sdXRpb24gZHJvcHMuDQoNCiAgICBCb3hlcyBhcmUgc2NhbGVkIGFuYWx5dGljYWxseSAodypzLCBoKnMpIHJhdGhlciB0aGFuIGJ5IHJlLW1lYXN1cmluZyB0aGUNCiAgICByZXNpemVkIGltYWdlcnksIGJlY2F1c2UgdGhlIHByZXByb2Nlc3NpbmcgYXBwbGllcyBleGFjdGx5IHRoaXMgbGluZWFyDQogICAgdHJhbnNmb3JtIC0tIHNlZSBzcmMvcHJlcHJvY2Vzc2luZy5weS4NCiAgICAiIiINCiAgICBvdXQgPSB7fQ0KICAgIGZvciBsYWJlbCwgcyBpbiByZXNvbHV0aW9ucy5pdGVtcygpOg0KICAgICAgICB3ID0gZGZbImJ3Il0gKiBzDQogICAgICAgIGggPSBkZlsiYmgiXSAqIHMNCiAgICAgICAgb3V0W2xhYmVsXSA9IHsNCiAgICAgICAgICAgICJzY2FsZSI6IHMsDQogICAgICAgICAgICAid2lkdGhfcHgiOiBkZXNjcmliZSh3KSwNCiAgICAgICAgICAgICJoZWlnaHRfcHgiOiBkZXNjcmliZShoKSwNCiAgICAgICAgICAgICJhcmVhX3B4MiI6IGRlc2NyaWJlKHcgKiBoKSwNCiAgICAgICAgICAgICJiZWxvd190aHJlc2hvbGRzIjogdGhyZXNob2xkX2NvdW50cyh3LCBoKSwNCiAgICAgICAgfQ0KICAgIHJldHVybiBvdXQNCg0KDQpkZWYgc2l6ZV9iaW5fc3Vydml2YWwoZGY6IHBkLkRhdGFGcmFtZSwNCiAgICAgICAgICAgICAgICAgICAgICByZXNvbHV0aW9uczogZGljdFtzdHIsIGZsb2F0XSA9IFJFU09MVVRJT05TLA0KICAgICAgICAgICAgICAgICAgICAgIGZsb29yX3B4OiBpbnQgPSA4KSAtPiBwZC5EYXRhRnJhbWU6DQogICAgIiIiRm9yIGVhY2ggZml4ZWQgb3JpZ2luYWwtcmVzb2x1dGlvbiBzaXplIGJpbiwgdGhlIG1lZGlhbiBmb290cHJpbnQgYW5kDQogICAgdGhlIHNoYXJlIG9mIG9iamVjdHMgd2hvc2UgbG9uZ2VzdCBzaWRlIGZhbGxzIHVuZGVyIGBmbG9vcl9weGAgYXQgZWFjaA0KICAgIHJlc29sdXRpb24uIFRoaXMgaXMgdGhlIHF1YW50aXRhdGl2ZSBmb3JtIG9mIHRoZSBzdHVkeSdzIGNvcmUgaHlwb3RoZXNpczoNCiAgICB0aGF0IHNtYWxsIG9iamVjdHMgZGVncmFkZSBmaXJzdCBhbmQgZmFzdGVzdC4iIiINCiAgICByb3dzID0gW10NCiAgICBmb3IgbGFiZWwsIHMgaW4gcmVzb2x1dGlvbnMuaXRlbXMoKToNCiAgICAgICAgZm9yIGIgaW4gU0laRV9CSU5TOg0KICAgICAgICAgICAgZyA9IGRmW2RmWyJzaXplX2JpbiJdID09IGJdDQogICAgICAgICAgICBpZiBub3QgbGVuKGcpOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICB3LCBoID0gZ1siYnciXSAqIHMsIGdbImJoIl0gKiBzDQogICAgICAgICAgICBsb25nZXN0ID0gbnAubWF4aW11bSh3LCBoKQ0KICAgICAgICAgICAgcm93cy5hcHBlbmQoew0KICAgICAgICAgICAgICAgICJyZXNvbHV0aW9uIjogbGFiZWwsDQogICAgICAgICAgICAgICAgInNjYWxlIjogcywNCiAgICAgICAgICAgICAgICAic2l6ZV9iaW4iOiBiLA0KICAgICAgICAgICAgICAgICJuX29iamVjdHMiOiBsZW4oZyksDQogICAgICAgICAgICAgICAgIm1lZGlhbl93X3B4Ijogcm91bmQoZmxvYXQody5tZWRpYW4oKSksIDEpLA0KICAgICAgICAgICAgICAgICJtZWRpYW5faF9weCI6IHJvdW5kKGZsb2F0KGgubWVkaWFuKCkpLCAxKSwNCiAgICAgICAgICAgICAgICAibWVkaWFuX2FyZWFfcHgyIjogcm91bmQoZmxvYXQoKHcgKiBoKS5tZWRpYW4oKSksIDEpLA0KICAgICAgICAgICAgICAgIGYicGN0X2xvbmdlc3Rfc2lkZV9sdF97Zmxvb3JfcHh9cHgiOiByb3VuZCgNCiAgICAgICAgICAgICAgICAgICAgZmxvYXQoKGxvbmdlc3QgPCBmbG9vcl9weCkubWVhbigpICogMTAwKSwgMiksDQogICAgICAgICAgICAgICAgInBjdF9hcmVhX2x0XzE2cHgyIjogcm91bmQoZmxvYXQoKCh3ICogaCkgPCAxNikubWVhbigpICogMTAwKSwgMiksDQogICAgICAgICAgICB9KQ0KICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykNCg==",
    "src/evaluation.py": "IyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgR0VORVJBVEVEIENPUFkgLS0gZG8gbm90IGVkaXQgaGVyZS4NCiMgU291cmNlIG9mIHRydXRoOiA8cHJvamVjdD4vc3JjL2V2YWx1YXRpb24ucHkNCiMgUmVnZW5lcmF0ZSB3aXRoOiBweXRob24gc2NyaXB0cy9jcmVhdGVfY29sYWJfcGFja2FnZS5weQ0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiIiIg0KRXZhbHVhdGlvbiBmb3Igb25lIHJlc29sdXRpb24gY29uZGl0aW9uLg0KDQpFdmVyeXRoaW5nIGhlcmUgZXhpc3RzIHRvIG1ha2UgZm91ciBkaWZmZXJlbnRseS1zaXplZCBkYXRhc2V0cyBjb21wYXJhYmxlIG9uDQpvbmUgc2NhbGUuIFByZWRpY3Rpb25zIGFyZSBtYWRlIG9uIHRoZSBjb25kaXRpb24ncyBvd24gaW1hZ2VyeSwgdGhlbiBtYXBwZWQNCmJhY2sgaW50byBPUklHSU5BTC1yZXNvbHV0aW9uIHBpeGVsIGNvb3JkaW5hdGVzIGJlZm9yZSBzY29yaW5nOg0KDQogICAgeF9vcmlnaW5hbCA9IHhfcHJlZGljdGVkIC8gKG91dF9kaW0gLyBzcmNfZGltKQ0KDQpJb1UgaXMgaW52YXJpYW50IHVuZGVyIHVuaWZvcm0gc2NhbGluZywgc28gdGhpcyBjaGFuZ2VzIG5vIG1hdGNoIGRlY2lzaW9uLiBXaGF0DQppdCBidXlzIGlzIHRoYXQgZXZlcnkgY29uZGl0aW9uIGlzIHNjb3JlZCBpbiBvbmUgY29vcmRpbmF0ZSBmcmFtZSBhZ2FpbnN0IG9uZQ0KZ3JvdW5kLXRydXRoIHRhYmxlLCB3aXRoIG9iamVjdC1zaXplIGJpbnMgYW5jaG9yZWQgdG8gdGhlIG9yaWdpbmFsIHJlc29sdXRpb24uDQpXaXRob3V0IGl0LCBgQVBfc21hbGxgIGF0IHIyNSB3b3VsZCBiZSBtZWFzdXJlZCBvdmVyIHdoYXRldmVyIGhhcHBlbmVkIHRvIGJlDQpzbWFsbCAqYXQgcjI1KiAtLSBhIGRpZmZlcmVudCBzZXQgb2Ygc2hpcHMgaW4gZXZlcnkgcm93IG9mIHRoZSByZXN1bHRzIHRhYmxlLg0KDQpMYXRlbmN5IGlzIG1lYXN1cmVkIG9uIHRoZSBjb25kaXRpb24ncyByZWFsIGlucHV0IHNpemUsIGJlZm9yZSB0aGF0IG1hcHBpbmcsDQpiZWNhdXNlIHRoZSBwb2ludCBvZiB0aGUgY29zdCBjb21wYXJpc29uIGlzIHdoYXQgdGhlIG1vZGVsIGFjdHVhbGx5IHByb2Nlc3Nlcy4NCiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCB0aW1lDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmZyb20gLm1ldHJpY3MgaW1wb3J0IEFSRUFfUkFOR0VTLCBldmFsdWF0ZSwgcHJmX2F0X3RocmVzaG9sZA0KDQoNCmRlZiBidWlsZF9ncm91bmRfdHJ1dGgoc2hpcHM6IHBkLkRhdGFGcmFtZSwgaW1hZ2VzOiBwZC5EYXRhRnJhbWUsDQogICAgICAgICAgICAgICAgICAgICAgIHNwbGl0OiBzdHIgPSAiVmFsIikgLT4gZGljdDoNCiAgICAiIiJHVCBrZXllZCBieSBpbWFnZSBzdGVtLCBpbiBvcmlnaW5hbC1yZXNvbHV0aW9uIGNvb3JkaW5hdGVzLg0KDQogICAgQmFja2dyb3VuZCBpbWFnZXMgYXJlIGluY2x1ZGVkIHdpdGggemVybyBib3hlcyBvbiBwdXJwb3NlOiBkcm9wcGluZyB0aGVtDQogICAgd291bGQgaGlkZSBldmVyeSBmYWxzZSBwb3NpdGl2ZSB0aGV5IHByb3Zva2UsIHdoaWNoIGlzIGV4YWN0bHkgd2hhdCB0aGV5DQogICAgd2VyZSByZXRhaW5lZCB0byBtZWFzdXJlLg0KICAgICIiIg0KICAgIHNlbCA9IGltYWdlc1tpbWFnZXNbIlNwbGl0Il0gPT0gc3BsaXRdDQogICAgZ3QgPSB7UGF0aChyLmJhc2VuYW1lKS5zdGVtOiB7ImJveGVzIjogbnAuemVyb3MoKDAsIDQpKSwgImFyZWFzIjogbnAuemVyb3MoMCl9DQogICAgICAgICAgZm9yIHIgaW4gc2VsLml0ZXJ0dXBsZXMoKX0NCiAgICBzID0gc2hpcHNbc2hpcHNbIlNwbGl0Il0gPT0gc3BsaXRdDQogICAgZm9yIGltZ19pZCwgZyBpbiBzLmdyb3VwYnkoIkltZ19JRCIpOg0KICAgICAgICBzdGVtID0gUGF0aChnWyJiYXNlbmFtZSJdLmlsb2NbMF0pLnN0ZW0NCiAgICAgICAgZ3Rbc3RlbV0gPSB7DQogICAgICAgICAgICAiYm94ZXMiOiBnW1sieF9taW4iLCAieV9taW4iLCAieF9tYXgiLCAieV9tYXgiXV0udG9fbnVtcHkoZmxvYXQpLA0KICAgICAgICAgICAgIyBUaGUgZml4ZWQgb3JpZ2luYWwtcmVzb2x1dGlvbiBhcmVhIHRoYXQgZHJpdmVzIHRoZSBzaXplIGJpbi4NCiAgICAgICAgICAgICJhcmVhcyI6IGdbImJhcmVhIl0udG9fbnVtcHkoZmxvYXQpLA0KICAgICAgICB9DQogICAgcmV0dXJuIGd0DQoNCg0KZGVmIHByZWRpY3Rfc3BsaXQobW9kZWwsIGRhdGFzZXRfcm9vdDogUGF0aCwgaW1hZ2VzOiBwZC5EYXRhRnJhbWUsDQogICAgICAgICAgICAgICAgICBzcGxpdDogc3RyID0gIlZhbCIsIGNvbmY6IGZsb2F0ID0gMC4wMDEsIGlvdTogZmxvYXQgPSAwLjcsDQogICAgICAgICAgICAgICAgICBtYXhfZGV0OiBpbnQgPSAxMDAwLCBpbWdzejogaW50ID0gMTAyNCwgZGV2aWNlPU5vbmUsDQogICAgICAgICAgICAgICAgICBiYXRjaDogaW50ID0gOCwgdmVyYm9zZTogYm9vbCA9IEZhbHNlKSAtPiB0dXBsZVtkaWN0LCBkaWN0XToNCiAgICAiIiJSdW4gaW5mZXJlbmNlIG92ZXIgb25lIHNwbGl0IGFuZCByZXR1cm4gKGRldGVjdGlvbnMsIHRpbWluZykuDQoNCiAgICBEZXRlY3Rpb25zIGFyZSByZXR1cm5lZCBpbiBPUklHSU5BTC1yZXNvbHV0aW9uIGNvb3JkaW5hdGVzLg0KICAgICIiIg0KICAgIHNwbGl0X2RpciA9IHsiVHJhaW4iOiAidHJhaW4iLCAiVmFsIjogInZhbCJ9W3NwbGl0XQ0KICAgIGltZ19kaXIgPSBkYXRhc2V0X3Jvb3QgLyAiaW1hZ2VzIiAvIHNwbGl0X2Rpcg0KDQogICAgc2VsID0gaW1hZ2VzW2ltYWdlc1siU3BsaXQiXSA9PSBzcGxpdF0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQ0KICAgIHBhdGhzID0gW2ltZ19kaXIgLyBmIntQYXRoKGIpLnN0ZW19LmpwZyIgZm9yIGIgaW4gc2VsWyJiYXNlbmFtZSJdXQ0KICAgIHByZXNlbnQgPSBbcC5leGlzdHMoKSBmb3IgcCBpbiBwYXRoc10NCiAgICBpZiBub3QgYWxsKHByZXNlbnQpOg0KICAgICAgICBtaXNzaW5nID0gW3N0cihwKSBmb3IgcCwgb2sgaW4gemlwKHBhdGhzLCBwcmVzZW50KSBpZiBub3Qgb2tdWzo1XQ0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIntwcmVzZW50LmNvdW50KEZhbHNlKX0gaW1hZ2VzIG1pc3NpbmcsIGUuZy4ge21pc3Npbmd9IikNCg0KICAgIGRldGVjdGlvbnM6IGRpY3Rbc3RyLCBkaWN0XSA9IHt9DQogICAgbl9kZXQgPSAwDQogICAgdF9pbmZlciA9IDAuMA0KDQogICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihwYXRocyksIGJhdGNoKToNCiAgICAgICAgY2h1bmsgPSBwYXRoc1tzdGFydDpzdGFydCArIGJhdGNoXQ0KICAgICAgICByb3dzID0gc2VsLmlsb2Nbc3RhcnQ6c3RhcnQgKyBiYXRjaF0NCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpDQogICAgICAgIHJlc3VsdHMgPSBtb2RlbC5wcmVkaWN0KA0KICAgICAgICAgICAgW3N0cihwKSBmb3IgcCBpbiBjaHVua10sIGltZ3N6PWltZ3N6LCBjb25mPWNvbmYsIGlvdT1pb3UsDQogICAgICAgICAgICBtYXhfZGV0PW1heF9kZXQsIGRldmljZT1kZXZpY2UsIHZlcmJvc2U9dmVyYm9zZSwgc3RyZWFtPUZhbHNlKQ0KICAgICAgICB0X2luZmVyICs9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MA0KDQogICAgICAgIGZvciByZXMsIChfLCByb3cpIGluIHppcChyZXN1bHRzLCByb3dzLml0ZXJyb3dzKCkpOg0KICAgICAgICAgICAgc3RlbSA9IFBhdGgocm93WyJiYXNlbmFtZSJdKS5zdGVtDQogICAgICAgICAgICBiID0gcmVzLmJveGVzDQogICAgICAgICAgICBpZiBiIGlzIE5vbmUgb3IgbGVuKGIpID09IDA6DQogICAgICAgICAgICAgICAgZGV0ZWN0aW9uc1tzdGVtXSA9IHsiYm94ZXMiOiBucC56ZXJvcygoMCwgNCkpLCAic2NvcmVzIjogbnAuemVyb3MoMCl9DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIHh5eHkgPSBiLnh5eHkuY3B1KCkubnVtcHkoKS5hc3R5cGUobnAuZmxvYXQ2NCkNCiAgICAgICAgICAgIHNjb3JlcyA9IGIuY29uZi5jcHUoKS5udW1weSgpLmFzdHlwZShucC5mbG9hdDY0KQ0KDQogICAgICAgICAgICAjIE1hcCBiYWNrIHRvIG9yaWdpbmFsLXJlc29sdXRpb24gY29vcmRpbmF0ZXMgdXNpbmcgdGhlIHJlYWxpemVkDQogICAgICAgICAgICAjIHNjYWxlIG9mIFRISVMgaW1hZ2UsIHJlYWQgZnJvbSB0aGUgcmVuZGVyZWQgcmFzdGVyIHJhdGhlciB0aGFuDQogICAgICAgICAgICAjIGFzc3VtZWQgZnJvbSB0aGUgbm9taW5hbCBmYWN0b3IuDQogICAgICAgICAgICBvdXRfaCwgb3V0X3cgPSByZXMub3JpZ19zaGFwZQ0KICAgICAgICAgICAgc3ggPSBvdXRfdyAvIGZsb2F0KHJvd1siSW1hZ2VXaWR0aCJdKQ0KICAgICAgICAgICAgc3kgPSBvdXRfaCAvIGZsb2F0KHJvd1siSW1hZ2VIZWlnaHQiXSkNCiAgICAgICAgICAgIHh5eHlbOiwgWzAsIDJdXSAvPSBzeA0KICAgICAgICAgICAgeHl4eVs6LCBbMSwgM11dIC89IHN5DQoNCiAgICAgICAgICAgIGRldGVjdGlvbnNbc3RlbV0gPSB7ImJveGVzIjogeHl4eSwgInNjb3JlcyI6IHNjb3Jlc30NCiAgICAgICAgICAgIG5fZGV0ICs9IGxlbihzY29yZXMpDQoNCiAgICB0aW1pbmcgPSB7DQogICAgICAgICJuX2ltYWdlcyI6IGxlbihwYXRocyksDQogICAgICAgICJ0b3RhbF9pbmZlcmVuY2VfcyI6IHJvdW5kKHRfaW5mZXIsIDMpLA0KICAgICAgICAibGF0ZW5jeV9tc19wZXJfaW1hZ2UiOiByb3VuZCh0X2luZmVyIC8gbWF4KGxlbihwYXRocyksIDEpICogMTAwMCwgMyksDQogICAgICAgICJpbWFnZXNfcGVyX3MiOiByb3VuZChsZW4ocGF0aHMpIC8gbWF4KHRfaW5mZXIsIDFlLTkpLCAyKSwNCiAgICAgICAgIm5fZGV0ZWN0aW9ucyI6IG5fZGV0LA0KICAgICAgICAiYmF0Y2giOiBiYXRjaCwNCiAgICAgICAgImltZ3N6IjogaW1nc3osDQogICAgfQ0KICAgIHJldHVybiBkZXRlY3Rpb25zLCB0aW1pbmcNCg0KDQpkZWYgc2NvcmUoZ3Q6IGRpY3QsIGR0OiBkaWN0LCBtYXhfZGV0OiBpbnQgPSAxMDAwLA0KICAgICAgICAgIG9wZXJhdGluZ19jb25mOiBmbG9hdCA9IDAuMjUpIC0+IGRpY3Q6DQogICAgIiIiRnVsbCBtZXRyaWMgYmxvY2sgZm9yIG9uZSBjb25kaXRpb24uIiIiDQogICAgYXAgPSBldmFsdWF0ZShndCwgZHQsIEFSRUFfUkFOR0VTLCBtYXhfZGV0PW1heF9kZXQpDQogICAgcHJmID0gcHJmX2F0X3RocmVzaG9sZChndCwgZHQsIGNvbmY9b3BlcmF0aW5nX2NvbmYsIGlvdV90aHI9MC41LA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2RldD1tYXhfZGV0KQ0KICAgIG91dCA9IHsNCiAgICAgICAgIm1BUDUwIjogYXBbImFsbCJdLmFwNTAsDQogICAgICAgICJtQVA1MF85NSI6IGFwWyJhbGwiXS5hcCwNCiAgICAgICAgIm1BUDc1IjogYXBbImFsbCJdLmFwNzUsDQogICAgICAgICJBUjUwXzk1IjogYXBbImFsbCJdLmFyLA0KICAgICAgICAiQVBfc21hbGwiOiBhcFsic21hbGwiXS5hcCwNCiAgICAgICAgIkFQX21lZGl1bSI6IGFwWyJtZWRpdW0iXS5hcCwNCiAgICAgICAgIkFQX2xhcmdlIjogYXBbImxhcmdlIl0uYXAsDQogICAgICAgICJBUDUwX3NtYWxsIjogYXBbInNtYWxsIl0uYXA1MCwNCiAgICAgICAgIkFQNTBfbWVkaXVtIjogYXBbIm1lZGl1bSJdLmFwNTAsDQogICAgICAgICJBUDUwX2xhcmdlIjogYXBbImxhcmdlIl0uYXA1MCwNCiAgICAgICAgIm5fZ3QiOiB7azogdi5uX2d0IGZvciBrLCB2IGluIGFwLml0ZW1zKCl9LA0KICAgICAgICAibl9kZXRlY3Rpb25zIjogYXBbImFsbCJdLm5fZHQsDQogICAgICAgIGYicHJlY2lzaW9uQHtvcGVyYXRpbmdfY29uZn0iOiBwcmZbInByZWNpc2lvbiJdLA0KICAgICAgICBmInJlY2FsbEB7b3BlcmF0aW5nX2NvbmZ9IjogcHJmWyJyZWNhbGwiXSwNCiAgICAgICAgZiJmMUB7b3BlcmF0aW5nX2NvbmZ9IjogcHJmWyJmMSJdLA0KICAgICAgICAib3BlcmF0aW5nX3BvaW50IjogcHJmLA0KICAgIH0NCiAgICByZXR1cm4gb3V0DQo=",
    "src/metrics.py": "IyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgR0VORVJBVEVEIENPUFkgLS0gZG8gbm90IGVkaXQgaGVyZS4NCiMgU291cmNlIG9mIHRydXRoOiA8cHJvamVjdD4vc3JjL21ldHJpY3MucHkNCiMgUmVnZW5lcmF0ZSB3aXRoOiBweXRob24gc2NyaXB0cy9jcmVhdGVfY29sYWJfcGFja2FnZS5weQ0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiIiIg0KQ09DTy1zdHlsZSBkZXRlY3Rpb24gbWV0cmljcyB3aXRoIGZpeGVkLCBjYWxsZXItc3VwcGxpZWQgb2JqZWN0LXNpemUgYmlucy4NCg0KV2h5IHRoaXMgZXhpc3RzIHJhdGhlciB0aGFuIGEgY2FsbCBpbnRvIHB5Y29jb3Rvb2xzDQotLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NClR3byByZWFzb25zLCBvbmUgcHJhY3RpY2FsIGFuZCBvbmUgbWV0aG9kb2xvZ2ljYWwuDQoNClByYWN0aWNhbDogcHljb2NvdG9vbHMgbmVlZHMgYSBDIHRvb2xjaGFpbiBhbmQgZG9lcyBub3QgaW5zdGFsbCBjbGVhbmx5IG9uIHRoZQ0KV2luZG93cyBtYWNoaW5lIHRoaXMgc3R1ZHkgaXMgcHJlcGFyZWQgb24sIHNvIHRoZSBtZXRyaWMgY291bGQgbm90IGJlIGV4ZXJjaXNlZA0Kb3IgdW5pdC10ZXN0ZWQgbG9jYWxseSBiZWZvcmUgcmVhY2hpbmcgQ29sYWIuDQoNCk1ldGhvZG9sb2dpY2FsOiB0aGUgc2l6ZS1zdHJhdGlmaWVkIG51bWJlcnMgYXJlIHRoZSBjZW50cmVwaWVjZSBvZiB0aGlzIHN0dWR5LA0KYW5kIENPQ08ncyBhcmVhLXJhbmdlIHNlbWFudGljcyAtLSBzcGVjaWZpY2FsbHkgKndoaWNoKiBkZXRlY3Rpb25zIGdldCBpZ25vcmVkDQppbiBhIGJpbiAtLSBkZWNpZGUgd2hhdCBgQVBfc21hbGxgIGFjdHVhbGx5IG1lYW5zLiBUaGF0IGxvZ2ljIHNob3VsZCBiZQ0KcmVhZGFibGUgaW4gdGhpcyByZXBvc2l0b3J5IHJhdGhlciB0aGFuIGluaGVyaXRlZCBmcm9tIGEgYmluYXJ5Lg0KDQpUaGUgaW1wbGVtZW50YXRpb24gZm9sbG93cyBDT0NPZXZhbCBleGFjdGx5OiAxMDEtcG9pbnQgaW50ZXJwb2xhdGVkIHByZWNpc2lvbiwNCklvVSB0aHJlc2hvbGRzIDAuNTA6MC4wNTowLjk1LCBncmVlZHkgc2NvcmUtb3JkZXJlZCBtYXRjaGluZyB3aGVyZSBlYWNoIGdyb3VuZA0KdHJ1dGggaXMgY2xhaW1lZCBhdCBtb3N0IG9uY2UgcGVyIHRocmVzaG9sZCwgYW5kIGFyZWEtcmFuZ2UgZmlsdGVyaW5nIHdpdGgNCmlnbm9yZSBzZW1hbnRpY3MuIGB0ZXN0cy90ZXN0X21ldHJpY3MucHlgIGNoZWNrcyBpdCBhZ2FpbnN0IGhhbmQtY29tcHV0YWJsZQ0KY2FzZXMsIGFuZCBgc2NyaXB0cy9gIGNyb3NzLWNoZWNrcyB0aGUgb3ZlcmFsbCBudW1iZXJzIGFnYWluc3QgdGhlIHZhbHVlDQpVbHRyYWx5dGljcyByZXBvcnRzIGZvciB0aGUgc2FtZSBwcmVkaWN0aW9ucy4NCg0KRXZhbHVhdGluZyBhY3Jvc3MgcmVzb2x1dGlvbnMNCi0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQpBbGwgZ2VvbWV0cnkgaXMgZXhwZWN0ZWQgaW4gKipvcmlnaW5hbC1yZXNvbHV0aW9uIGNvb3JkaW5hdGVzKiouIFByZWRpY3Rpb25zDQptYWRlIG9uIGFuIHIyNSBpbWFnZSBhcmUgZGl2aWRlZCBieSB0aGUgcmVhbGl6ZWQgc2NhbGUgYmVmb3JlIHRoZXkgZ2V0IGhlcmUuDQpJb1UgaXMgaW52YXJpYW50IHVuZGVyIHVuaWZvcm0gc2NhbGluZywgc28gbWF0Y2hpbmcgaXMgaWRlbnRpY2FsIGVpdGhlciB3YXksDQpidXQgd29ya2luZyBpbiBvbmUgY29tbW9uIGZyYW1lIG1lYW5zOg0KDQogICogYSBkZXRlY3Rpb24ncyBvd24gYXJlYSBpcyBjb21wYXJhYmxlIHRvIHRoZSBmaXhlZCBzaXplIGJpbnMsIHNvIENPQ08ncw0KICAgICJpZ25vcmUgdW5tYXRjaGVkIGRldGVjdGlvbnMgb3V0c2lkZSB0aGUgYXJlYSByYW5nZSIgcnVsZSBrZWVwcyBtZWFuaW5nDQogICAgdGhlIHNhbWUgdGhpbmcgYXQgZXZlcnkgcmVzb2x1dGlvbjsNCiAgKiBgQVBfc21hbGxgIGF0IHIyNSBpcyBtZWFzdXJlZCBvdmVyIGV4YWN0bHkgdGhlIHNoaXBzIHRoYXQgd2VyZSBzbWFsbCBhdA0KICAgIG9yaWdpbmFsIHJlc29sdXRpb24sIHdoaWNoIGlzIHRoZSBQYXJ0IEggcmVxdWlyZW1lbnQuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkDQoNCmltcG9ydCBudW1weSBhcyBucA0KDQojIENPQ08gZGVmYXVsdHMuDQpJT1VfVEhSRVNIT0xEUyA9IG5wLnJvdW5kKG5wLmFyYW5nZSgwLjUsIDAuOTYsIDAuMDUpLCAyKQ0KUkVDQUxMX1RIUkVTSE9MRFMgPSBucC5saW5zcGFjZSgwLjAsIDEuMCwgMTAxKQ0KDQojIEFyZWEgcmFuZ2VzLCBpbiBweF4yIG9mIHRoZSBPUklHSU5BTCBpbWFnZS4gImFsbCIgbXVzdCBzdGF5IHVuYm91bmRlZCBzbyB0aGUNCiMgaGVhZGxpbmUgbUFQIGlzIG5vdCBzaWxlbnRseSByZXN0cmljdGVkLg0KQVJFQV9SQU5HRVM6IGRpY3Rbc3RyLCB0dXBsZVtmbG9hdCwgZmxvYXRdXSA9IHsNCiAgICAiYWxsIjogKDAuMCwgZmxvYXQoImluZiIpKSwNCiAgICAic21hbGwiOiAoMC4wLCAxMDI0LjApLCAgICAgICAgICAjIDwgMzJeMg0KICAgICJtZWRpdW0iOiAoMTAyNC4wLCA5MjE2LjApLCAgICAgICMgMzJeMiAuLiA5Nl4yDQogICAgImxhcmdlIjogKDkyMTYuMCwgZmxvYXQoImluZiIpKSwgIyA+PSA5Nl4yDQp9DQoNCg0KZGVmIGlvdV9tYXRyaXgoZHQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5Og0KICAgICIiIlBhaXJ3aXNlIElvVSBiZXR3ZWVuIChOLDQpIGFuZCAoTSw0KSBib3hlcyBpbiB4eXh5LiIiIg0KICAgIGlmIGxlbihkdCkgPT0gMCBvciBsZW4oZ3QpID09IDA6DQogICAgICAgIHJldHVybiBucC56ZXJvcygobGVuKGR0KSwgbGVuKGd0KSksIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgaXgwID0gbnAubWF4aW11bShkdFs6LCBOb25lLCAwXSwgZ3RbTm9uZSwgOiwgMF0pDQogICAgaXkwID0gbnAubWF4aW11bShkdFs6LCBOb25lLCAxXSwgZ3RbTm9uZSwgOiwgMV0pDQogICAgaXgxID0gbnAubWluaW11bShkdFs6LCBOb25lLCAyXSwgZ3RbTm9uZSwgOiwgMl0pDQogICAgaXkxID0gbnAubWluaW11bShkdFs6LCBOb25lLCAzXSwgZ3RbTm9uZSwgOiwgM10pDQogICAgaXcgPSBucC5jbGlwKGl4MSAtIGl4MCwgMCwgTm9uZSkNCiAgICBpaCA9IG5wLmNsaXAoaXkxIC0gaXkwLCAwLCBOb25lKQ0KICAgIGludGVyID0gaXcgKiBpaA0KICAgIGFfZHQgPSBucC5jbGlwKGR0WzosIDJdIC0gZHRbOiwgMF0sIDAsIE5vbmUpICogbnAuY2xpcChkdFs6LCAzXSAtIGR0WzosIDFdLCAwLCBOb25lKQ0KICAgIGFfZ3QgPSBucC5jbGlwKGd0WzosIDJdIC0gZ3RbOiwgMF0sIDAsIE5vbmUpICogbnAuY2xpcChndFs6LCAzXSAtIGd0WzosIDFdLCAwLCBOb25lKQ0KICAgIHVuaW9uID0gYV9kdFs6LCBOb25lXSArIGFfZ3RbTm9uZSwgOl0gLSBpbnRlcg0KICAgIHdpdGggbnAuZXJyc3RhdGUoZGl2aWRlPSJpZ25vcmUiLCBpbnZhbGlkPSJpZ25vcmUiKToNCiAgICAgICAgcmV0dXJuIG5wLndoZXJlKHVuaW9uID4gMCwgaW50ZXIgLyB1bmlvbiwgMC4wKQ0KDQoNCkBkYXRhY2xhc3MNCmNsYXNzIEltYWdlRXZhbDoNCiAgICAiIiJQZXItaW1hZ2UgbWF0Y2hpbmcgb3V0Y29tZSBmb3Igb25lIGFyZWEgcmFuZ2UsIGFsbCBJb1UgdGhyZXNob2xkcy4iIiINCiAgICBkdF9zY29yZXM6IG5wLm5kYXJyYXkgICAgICAgICAgICAgICAgICAgICAgICMgKEQsKQ0KICAgIGR0X21hdGNoZWQ6IG5wLm5kYXJyYXkgICAgICAgICAgICAgICAgICAgICAgIyAoVCwgRCkgYm9vbA0KICAgIGR0X2lnbm9yZTogbnAubmRhcnJheSAgICAgICAgICAgICAgICAgICAgICAgIyAoVCwgRCkgYm9vbA0KICAgIG5fZ3RfY29uc2lkZXJlZDogaW50ICAgICAgICAgICAgICAgICAgICAgICAgIyBub24taWdub3JlZCBHVCBpbiB0aGlzIHJhbmdlDQoNCg0KZGVmIF9ldmFsdWF0ZV9pbWFnZShkdF9ib3hlczogbnAubmRhcnJheSwgZHRfc2NvcmVzOiBucC5uZGFycmF5LA0KICAgICAgICAgICAgICAgICAgICBndF9ib3hlczogbnAubmRhcnJheSwgZ3RfYXJlYXM6IG5wLm5kYXJyYXksDQogICAgICAgICAgICAgICAgICAgIGFyZWFfcmFuZ2U6IHR1cGxlW2Zsb2F0LCBmbG9hdF0sDQogICAgICAgICAgICAgICAgICAgIG1heF9kZXQ6IGludCkgLT4gSW1hZ2VFdmFsOg0KICAgICIiIkNPQ09ldmFsLmV2YWx1YXRlSW1nIGZvciBhIHNpbmdsZSBpbWFnZSwgc2luZ2xlIGNsYXNzLiIiIg0KICAgIGxvLCBoaSA9IGFyZWFfcmFuZ2UNCiAgICBUID0gbGVuKElPVV9USFJFU0hPTERTKQ0KDQogICAgIyBHVCBvdXRzaWRlIHRoZSBhcmVhIHJhbmdlIGlzICppZ25vcmVkKiwgbm90IGRlbGV0ZWQ6IGEgZGV0ZWN0aW9uIGxhbmRpbmcNCiAgICAjIG9uIGl0IG11c3Qgbm90IGJlIGNvdW50ZWQgYXMgYSBmYWxzZSBwb3NpdGl2ZSBmb3IgdGhpcyBiaW4uDQogICAgZ3RfaWdub3JlID0gKGd0X2FyZWFzIDwgbG8pIHwgKGd0X2FyZWFzID49IGhpKSBpZiBsZW4oZ3RfYm94ZXMpIGVsc2UgbnAuemVyb3MoMCwgYm9vbCkNCiAgICAjIENPQ08gc29ydHMgaWdub3JlZCBHVCBsYXN0IHNvIHRoZSBncmVlZHkgbWF0Y2hlciBwcmVmZXJzIGEgcmVhbCBtYXRjaC4NCiAgICBvcmRlciA9IG5wLmFyZ3NvcnQoZ3RfaWdub3JlLCBraW5kPSJzdGFibGUiKQ0KICAgIGd0X2JveGVzLCBndF9pZ25vcmUgPSBndF9ib3hlc1tvcmRlcl0sIGd0X2lnbm9yZVtvcmRlcl0NCg0KICAgICMgRGV0ZWN0aW9ucyBpbiBkZXNjZW5kaW5nIHNjb3JlLCB0cnVuY2F0ZWQgdG8gbWF4X2RldCAoYXMgQ09DTyBkb2VzKS4NCiAgICBkb3JkZXIgPSBucC5hcmdzb3J0KC1kdF9zY29yZXMsIGtpbmQ9InN0YWJsZSIpWzptYXhfZGV0XQ0KICAgIGR0X2JveGVzLCBkdF9zY29yZXMgPSBkdF9ib3hlc1tkb3JkZXJdLCBkdF9zY29yZXNbZG9yZGVyXQ0KICAgIEQsIEcgPSBsZW4oZHRfYm94ZXMpLCBsZW4oZ3RfYm94ZXMpDQoNCiAgICBkdF9tYXRjaGVkID0gbnAuemVyb3MoKFQsIEQpLCBkdHlwZT1ib29sKQ0KICAgIGR0X2lnbm9yZSA9IG5wLnplcm9zKChULCBEKSwgZHR5cGU9Ym9vbCkNCiAgICBpZiBEID09IDA6DQogICAgICAgIHJldHVybiBJbWFnZUV2YWwoZHRfc2NvcmVzLCBkdF9tYXRjaGVkLCBkdF9pZ25vcmUsIGludCgofmd0X2lnbm9yZSkuc3VtKCkpKQ0KDQogICAgaW91cyA9IGlvdV9tYXRyaXgoZHRfYm94ZXMsIGd0X2JveGVzKQ0KICAgIGR0X2FyZWFzID0gKChkdF9ib3hlc1s6LCAyXSAtIGR0X2JveGVzWzosIDBdKQ0KICAgICAgICAgICAgICAgICogKGR0X2JveGVzWzosIDNdIC0gZHRfYm94ZXNbOiwgMV0pKQ0KDQogICAgZm9yIHQsIHRociBpbiBlbnVtZXJhdGUoSU9VX1RIUkVTSE9MRFMpOg0KICAgICAgICBndF90YWtlbiA9IG5wLmZ1bGwoRywgLTEsIGR0eXBlPWludCkNCiAgICAgICAgZm9yIGQgaW4gcmFuZ2UoRCk6DQogICAgICAgICAgICBiZXN0X2lvdSA9IG1pbih0aHIsIDEgLSAxZS0xMCkNCiAgICAgICAgICAgIGJlc3RfZyA9IC0xDQogICAgICAgICAgICBmb3IgZyBpbiByYW5nZShHKToNCiAgICAgICAgICAgICAgICBpZiBndF90YWtlbltnXSA+PSAwIGFuZCBub3QgZ3RfaWdub3JlW2ddOg0KICAgICAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgICAgICMgR1QgaXMgc29ydGVkIGlnbm9yZWQtbGFzdDogb25jZSBhIHJlYWwgbWF0Y2ggZXhpc3RzLCBhbg0KICAgICAgICAgICAgICAgICMgaWdub3JlZCBHVCBjYW4gbmV2ZXIgYmUgcHJlZmVyYWJsZSwgc28gc3RvcC4NCiAgICAgICAgICAgICAgICBpZiBiZXN0X2cgPiAtMSBhbmQgbm90IGd0X2lnbm9yZVtiZXN0X2ddIGFuZCBndF9pZ25vcmVbZ106DQogICAgICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgICAgICAgICAgaWYgaW91c1tkLCBnXSA8IGJlc3RfaW91Og0KICAgICAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgICAgIGJlc3RfaW91ID0gaW91c1tkLCBnXQ0KICAgICAgICAgICAgICAgIGJlc3RfZyA9IGcNCiAgICAgICAgICAgIGlmIGJlc3RfZyA9PSAtMToNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgZ3RfdGFrZW5bYmVzdF9nXSA9IGQNCiAgICAgICAgICAgIGR0X21hdGNoZWRbdCwgZF0gPSBUcnVlDQogICAgICAgICAgICBkdF9pZ25vcmVbdCwgZF0gPSBndF9pZ25vcmVbYmVzdF9nXQ0KDQogICAgIyBBbiB1bm1hdGNoZWQgZGV0ZWN0aW9uIHdob3NlIG93biBmb290cHJpbnQgaXMgb3V0c2lkZSB0aGUgYmluIGlzIG5vdCB0aGlzDQogICAgIyBiaW4ncyBmYWxzZSBwb3NpdGl2ZS4NCiAgICBvdXRzaWRlID0gKGR0X2FyZWFzIDwgbG8pIHwgKGR0X2FyZWFzID49IGhpKQ0KICAgIGR0X2lnbm9yZSB8PSAofmR0X21hdGNoZWQpICYgb3V0c2lkZVtOb25lLCA6XQ0KICAgIHJldHVybiBJbWFnZUV2YWwoZHRfc2NvcmVzLCBkdF9tYXRjaGVkLCBkdF9pZ25vcmUsIGludCgofmd0X2lnbm9yZSkuc3VtKCkpKQ0KDQoNCkBkYXRhY2xhc3MNCmNsYXNzIEFQUmVzdWx0Og0KICAgIGFwOiBmbG9hdCAgICAgICAgICAgICAgICAgICAgICAgICMgbUFQQDUwOjk1DQogICAgYXA1MDogZmxvYXQNCiAgICBhcDc1OiBmbG9hdA0KICAgIGFyOiBmbG9hdCAgICAgICAgICAgICAgICAgICAgICAgICMgbWF4IHJlY2FsbCBhdCBjb25mLT4wLCBhdmVyYWdlZCBvdmVyIElvVQ0KICAgIG5fZ3Q6IGludA0KICAgIG5fZHQ6IGludA0KICAgIHByZWNpc2lvbl9jdXJ2ZTogbnAubmRhcnJheSA9IGZpZWxkKHJlcHI9RmFsc2UsIGRlZmF1bHQ9Tm9uZSkNCiAgICByZWNhbGxfY3VydmU6IG5wLm5kYXJyYXkgPSBmaWVsZChyZXByPUZhbHNlLCBkZWZhdWx0PU5vbmUpDQoNCg0KZGVmIF9hY2N1bXVsYXRlKGV2YWxzOiBsaXN0W0ltYWdlRXZhbF0pIC0+IEFQUmVzdWx0Og0KICAgIG5fZ3QgPSBzdW0oZS5uX2d0X2NvbnNpZGVyZWQgZm9yIGUgaW4gZXZhbHMpDQogICAgc2NvcmVzID0gbnAuY29uY2F0ZW5hdGUoW2UuZHRfc2NvcmVzIGZvciBlIGluIGV2YWxzXSkgaWYgZXZhbHMgZWxzZSBucC56ZXJvcygwKQ0KICAgICMgTm8gZ3JvdW5kIHRydXRoIGluIHRoaXMgYmluOiBBUCBpcyB1bmRlZmluZWQsIG5vdCB6ZXJvLiBDT0NPIHJlcG9ydHMgLTENCiAgICAjIGhlcmU7IG5hbiBpcyB1c2VkIHNvIGl0IGNhbm5vdCBiZSBzaWxlbnRseSBhdmVyYWdlZCBpbnRvIGEgbWVhbi4NCiAgICBpZiBuX2d0ID09IDA6DQogICAgICAgIHJldHVybiBBUFJlc3VsdChmbG9hdCgibmFuIiksIGZsb2F0KCJuYW4iKSwgZmxvYXQoIm5hbiIpLCBmbG9hdCgibmFuIiksDQogICAgICAgICAgICAgICAgICAgICAgICAwLCBpbnQobGVuKHNjb3JlcykpKQ0KICAgICMgR3JvdW5kIHRydXRoIGV4aXN0cyBidXQgbm90aGluZyB3YXMgZGV0ZWN0ZWQ6IEFQIGlzIGEgZ2VudWluZSB6ZXJvLiBUaGlzDQogICAgIyBpcyB0aGUgZXhwZWN0ZWQgcjI1IGZhaWx1cmUgbW9kZSBmb3Igc21hbGwgc2hpcHMsIHNvIGl0IG11c3Qgbm90IGNvbWUNCiAgICAjIGJhY2sgYXMgbmFuIGFuZCB2YW5pc2ggZnJvbSB0aGUgYWdncmVnYXRlLg0KICAgIGlmIGxlbihzY29yZXMpID09IDA6DQogICAgICAgIHJldHVybiBBUFJlc3VsdCgwLjAsIDAuMCwgMC4wLCAwLjAsIG5fZ3QsIDApDQoNCiAgICBtYXRjaGVkID0gbnAuY29uY2F0ZW5hdGUoW2UuZHRfbWF0Y2hlZCBmb3IgZSBpbiBldmFsc10sIGF4aXM9MSkNCiAgICBpZ25vcmVkID0gbnAuY29uY2F0ZW5hdGUoW2UuZHRfaWdub3JlIGZvciBlIGluIGV2YWxzXSwgYXhpcz0xKQ0KDQogICAgb3JkZXIgPSBucC5hcmdzb3J0KC1zY29yZXMsIGtpbmQ9InN0YWJsZSIpDQogICAgbWF0Y2hlZCwgaWdub3JlZCA9IG1hdGNoZWRbOiwgb3JkZXJdLCBpZ25vcmVkWzosIG9yZGVyXQ0KDQogICAgVCA9IGxlbihJT1VfVEhSRVNIT0xEUykNCiAgICBwcmVjaXNpb25zID0gbnAuemVyb3MoKFQsIGxlbihSRUNBTExfVEhSRVNIT0xEUykpKQ0KICAgIHJlY2FsbHMgPSBucC56ZXJvcyhUKQ0KICAgIHByX2N1cnZlcyA9IFtdDQoNCiAgICBmb3IgdCBpbiByYW5nZShUKToNCiAgICAgICAga2VlcCA9IH5pZ25vcmVkW3RdDQogICAgICAgIHRwID0gbnAuY3Vtc3VtKG1hdGNoZWRbdF0gJiBrZWVwKS5hc3R5cGUobnAuZmxvYXQ2NCkNCiAgICAgICAgZnAgPSBucC5jdW1zdW0ofm1hdGNoZWRbdF0gJiBrZWVwKS5hc3R5cGUobnAuZmxvYXQ2NCkNCiAgICAgICAgcmMgPSB0cCAvIG5fZ3QNCiAgICAgICAgcHIgPSB0cCAvIG5wLm1heGltdW0odHAgKyBmcCwgbnAuZmluZm8obnAuZmxvYXQ2NCkuZXBzKQ0KICAgICAgICByZWNhbGxzW3RdID0gcmNbLTFdIGlmIGxlbihyYykgZWxzZSAwLjANCg0KICAgICAgICAjIE1ha2UgcHJlY2lzaW9uIG1vbm90b25pY2FsbHkgbm9uLWluY3JlYXNpbmcgKENPQ08gZG9lcyB0aGlzIGluIHBsYWNlLA0KICAgICAgICAjIHJpZ2h0IHRvIGxlZnQpIGJlZm9yZSBpbnRlcnBvbGF0aW5nIGF0IHRoZSAxMDEgcmVjYWxsIHBvaW50cy4NCiAgICAgICAgcHIgPSBucC5tYXhpbXVtLmFjY3VtdWxhdGUocHJbOjotMV0pWzo6LTFdDQogICAgICAgIGlkeCA9IG5wLnNlYXJjaHNvcnRlZChyYywgUkVDQUxMX1RIUkVTSE9MRFMsIHNpZGU9ImxlZnQiKQ0KICAgICAgICBxID0gbnAuemVyb3MobGVuKFJFQ0FMTF9USFJFU0hPTERTKSkNCiAgICAgICAgdmFsaWQgPSBpZHggPCBsZW4ocHIpDQogICAgICAgIHFbdmFsaWRdID0gcHJbaWR4W3ZhbGlkXV0NCiAgICAgICAgcHJlY2lzaW9uc1t0XSA9IHENCiAgICAgICAgaWYgdCA9PSAwOg0KICAgICAgICAgICAgcHJfY3VydmVzID0gKHEuY29weSgpLCBSRUNBTExfVEhSRVNIT0xEUy5jb3B5KCkpDQoNCiAgICByZXR1cm4gQVBSZXN1bHQoDQogICAgICAgIGFwPWZsb2F0KHByZWNpc2lvbnMubWVhbigpKSwNCiAgICAgICAgYXA1MD1mbG9hdChwcmVjaXNpb25zWzBdLm1lYW4oKSksDQogICAgICAgIGFwNzU9ZmxvYXQocHJlY2lzaW9uc1s1XS5tZWFuKCkpLA0KICAgICAgICBhcj1mbG9hdChyZWNhbGxzLm1lYW4oKSksDQogICAgICAgIG5fZ3Q9bl9ndCwgbl9kdD1pbnQobGVuKHNjb3JlcykpLA0KICAgICAgICBwcmVjaXNpb25fY3VydmU9cHJfY3VydmVzWzBdIGlmIGxlbihwcl9jdXJ2ZXMpIGVsc2UgTm9uZSwNCiAgICAgICAgcmVjYWxsX2N1cnZlPXByX2N1cnZlc1sxXSBpZiBsZW4ocHJfY3VydmVzKSBlbHNlIE5vbmUsDQogICAgKQ0KDQoNCmRlZiBldmFsdWF0ZShndF9ieV9pbWFnZTogZGljdCwgZHRfYnlfaW1hZ2U6IGRpY3QsDQogICAgICAgICAgICAgYXJlYV9yYW5nZXM6IGRpY3Rbc3RyLCB0dXBsZVtmbG9hdCwgZmxvYXRdXSA9IE5vbmUsDQogICAgICAgICAgICAgbWF4X2RldDogaW50ID0gMTAwMCkgLT4gZGljdFtzdHIsIEFQUmVzdWx0XToNCiAgICAiIiJDb21wdXRlIEFQIGZvciBlYWNoIG5hbWVkIGFyZWEgcmFuZ2UuDQoNCiAgICBQYXJhbWV0ZXJzDQogICAgLS0tLS0tLS0tLQ0KICAgIGd0X2J5X2ltYWdlDQogICAgICAgIGltYWdlIGtleSAtPiB7ImJveGVzIjogKE0sNCkgeHl4eSBpbiBPUklHSU5BTC1yZXNvbHV0aW9uIHBpeGVscywNCiAgICAgICAgICAgICAgICAgICAgICAiYXJlYXMiOiAoTSwpIGZpeGVkIG9yaWdpbmFsLXJlc29sdXRpb24gYXJlYXN9Lg0KICAgICAgICBgYXJlYXNgIGlzIHBhc3NlZCBzZXBhcmF0ZWx5IHJhdGhlciB0aGFuIGRlcml2ZWQgZnJvbSBgYm94ZXNgIHNvIHRoZQ0KICAgICAgICBiaW4gYXNzaWdubWVudCBzdGF5cyBhbmNob3JlZCB0byB0aGUgb3JpZ2luYWwgcmVzb2x1dGlvbiBldmVuIGlmIHRoZQ0KICAgICAgICBjYWxsZXIgZXZlciBzdXBwbGllcyBib3hlcyBpbiBhbm90aGVyIGZyYW1lLg0KICAgIGR0X2J5X2ltYWdlDQogICAgICAgIGltYWdlIGtleSAtPiB7ImJveGVzIjogKE4sNCkgeHl4eSwgInNjb3JlcyI6IChOLCl9LiBJbWFnZXMgYWJzZW50IGhlcmUNCiAgICAgICAgYXJlIHRyZWF0ZWQgYXMgaGF2aW5nIHByb2R1Y2VkIG5vIGRldGVjdGlvbnMuDQogICAgIiIiDQogICAgYXJlYV9yYW5nZXMgPSBhcmVhX3JhbmdlcyBvciBBUkVBX1JBTkdFUw0KICAgIG91dCA9IHt9DQogICAga2V5cyA9IHNvcnRlZChzZXQoZ3RfYnlfaW1hZ2UpIHwgc2V0KGR0X2J5X2ltYWdlKSkNCiAgICBmb3IgbmFtZSwgcm5nIGluIGFyZWFfcmFuZ2VzLml0ZW1zKCk6DQogICAgICAgIGV2YWxzID0gW10NCiAgICAgICAgZm9yIGsgaW4ga2V5czoNCiAgICAgICAgICAgIGcgPSBndF9ieV9pbWFnZS5nZXQoaywge30pDQogICAgICAgICAgICBkID0gZHRfYnlfaW1hZ2UuZ2V0KGssIHt9KQ0KICAgICAgICAgICAgZ2IgPSBucC5hc2FycmF5KGcuZ2V0KCJib3hlcyIsIG5wLnplcm9zKCgwLCA0KSkpLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgICAgICAgICAgZ2EgPSBucC5hc2FycmF5KGcuZ2V0KCJhcmVhcyIsIG5wLnplcm9zKGxlbihnYikpKSwgZHR5cGU9bnAuZmxvYXQ2NCkNCiAgICAgICAgICAgIGRiID0gbnAuYXNhcnJheShkLmdldCgiYm94ZXMiLCBucC56ZXJvcygoMCwgNCkpKSwgZHR5cGU9bnAuZmxvYXQ2NCkNCiAgICAgICAgICAgIGRzID0gbnAuYXNhcnJheShkLmdldCgic2NvcmVzIiwgbnAuemVyb3MoMCkpLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgICAgICAgICAgaWYgbGVuKGdiKSA9PSAwIGFuZCBsZW4oZGIpID09IDA6DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIGV2YWxzLmFwcGVuZChfZXZhbHVhdGVfaW1hZ2UoZGIsIGRzLCBnYiwgZ2EsIHJuZywgbWF4X2RldCkpDQogICAgICAgIG91dFtuYW1lXSA9IF9hY2N1bXVsYXRlKGV2YWxzKQ0KICAgIHJldHVybiBvdXQNCg0KDQpkZWYgcHJmX2F0X3RocmVzaG9sZChndF9ieV9pbWFnZTogZGljdCwgZHRfYnlfaW1hZ2U6IGRpY3QsIGNvbmY6IGZsb2F0ID0gMC4yNSwNCiAgICAgICAgICAgICAgICAgICAgIGlvdV90aHI6IGZsb2F0ID0gMC41LCBtYXhfZGV0OiBpbnQgPSAxMDAwKSAtPiBkaWN0Og0KICAgICIiIlByZWNpc2lvbiAvIHJlY2FsbCAvIEYxIGF0IGEgc2luZ2xlIG9wZXJhdGluZyBwb2ludC4NCg0KICAgIEFQIGludGVncmF0ZXMgb3ZlciBldmVyeSB0aHJlc2hvbGQsIHdoaWNoIGlzIHRoZSByaWdodCBzdW1tYXJ5IGJ1dCBpcyBub3QNCiAgICB3aGF0IGEgZGVwbG95ZWQgZGV0ZWN0b3IgZG9lcy4gVGhlc2UgYXJlIHRoZSBudW1iZXJzIHRoYXQgZGVzY3JpYmUgdGhlDQogICAgbW9kZWwgYXQgb25lIHVzYWJsZSBjb25maWRlbmNlIGN1dC4NCiAgICAiIiINCiAgICB0cCA9IGZwID0gMA0KICAgIG5fZ3QgPSAwDQogICAgZm9yIGsgaW4gc29ydGVkKHNldChndF9ieV9pbWFnZSkgfCBzZXQoZHRfYnlfaW1hZ2UpKToNCiAgICAgICAgZyA9IGd0X2J5X2ltYWdlLmdldChrLCB7fSkNCiAgICAgICAgZCA9IGR0X2J5X2ltYWdlLmdldChrLCB7fSkNCiAgICAgICAgZ2IgPSBucC5hc2FycmF5KGcuZ2V0KCJib3hlcyIsIG5wLnplcm9zKCgwLCA0KSkpLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgICAgICBkYiA9IG5wLmFzYXJyYXkoZC5nZXQoImJveGVzIiwgbnAuemVyb3MoKDAsIDQpKSksIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgICAgIGRzID0gbnAuYXNhcnJheShkLmdldCgic2NvcmVzIiwgbnAuemVyb3MoMCkpLCBkdHlwZT1ucC5mbG9hdDY0KQ0KICAgICAgICBrZWVwID0gZHMgPj0gY29uZg0KICAgICAgICBkYiwgZHMgPSBkYltrZWVwXSwgZHNba2VlcF0NCiAgICAgICAgb3JkZXIgPSBucC5hcmdzb3J0KC1kcywga2luZD0ic3RhYmxlIilbOm1heF9kZXRdDQogICAgICAgIGRiID0gZGJbb3JkZXJdDQogICAgICAgIG5fZ3QgKz0gbGVuKGdiKQ0KICAgICAgICBpZiBsZW4oZGIpID09IDA6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBpZiBsZW4oZ2IpID09IDA6DQogICAgICAgICAgICBmcCArPSBsZW4oZGIpDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBpb3VzID0gaW91X21hdHJpeChkYiwgZ2IpDQogICAgICAgIHRha2VuID0gbnAuemVyb3MobGVuKGdiKSwgZHR5cGU9Ym9vbCkNCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKGRiKSk6DQogICAgICAgICAgICBqID0gaW50KG5wLmFyZ21heChucC53aGVyZSh0YWtlbiwgLTEuMCwgaW91c1tpXSkpKQ0KICAgICAgICAgICAgaWYgaW91c1tpLCBqXSA+PSBpb3VfdGhyIGFuZCBub3QgdGFrZW5bal06DQogICAgICAgICAgICAgICAgdGFrZW5bal0gPSBUcnVlDQogICAgICAgICAgICAgICAgdHAgKz0gMQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICBmcCArPSAxDQogICAgZm4gPSBuX2d0IC0gdHANCiAgICBwcmVjaXNpb24gPSB0cCAvICh0cCArIGZwKSBpZiAodHAgKyBmcCkgZWxzZSAwLjANCiAgICByZWNhbGwgPSB0cCAvIG5fZ3QgaWYgbl9ndCBlbHNlIDAuMA0KICAgIGYxID0gMiAqIHByZWNpc2lvbiAqIHJlY2FsbCAvIChwcmVjaXNpb24gKyByZWNhbGwpIGlmIChwcmVjaXNpb24gKyByZWNhbGwpIGVsc2UgMC4wDQogICAgcmV0dXJuIHsiY29uZiI6IGNvbmYsICJpb3UiOiBpb3VfdGhyLCAidHAiOiB0cCwgImZwIjogZnAsICJmbiI6IGZuLA0KICAgICAgICAgICAgIm5fZ3QiOiBuX2d0LCAicHJlY2lzaW9uIjogcHJlY2lzaW9uLCAicmVjYWxsIjogcmVjYWxsLCAiZjEiOiBmMX0NCg0KDQpkZWYgZGVncmFkYXRpb24oYmFzZWxpbmU6IGZsb2F0LCB2YWx1ZTogZmxvYXQpIC0+IGZsb2F0Og0KICAgICIiIlBlcmNlbnRhZ2UgZHJvcCByZWxhdGl2ZSB0byBiYXNlbGluZSAoUGFydCBRKS4gUG9zaXRpdmUgPSB3b3JzZS4iIiINCiAgICBpZiBiYXNlbGluZSBpbiAoMCwgTm9uZSkgb3Igbm90IG5wLmlzZmluaXRlKGJhc2VsaW5lKSBvciBub3QgbnAuaXNmaW5pdGUodmFsdWUpOg0KICAgICAgICByZXR1cm4gZmxvYXQoIm5hbiIpDQogICAgcmV0dXJuIChiYXNlbGluZSAtIHZhbHVlKSAvIGJhc2VsaW5lICogMTAwLjANCg==",
    "src/paths.py": "IiIiCkxvY2F0ZSB0aGUgcHJlcGFyZWQgZGF0YXNldCBhbmQgYW5ub3RhdGlvbiBsYXllciBpbnNpZGUgYSBLYWdnbGUgc2Vzc2lvbi4KClRoZSBLYWdnbGUgRGF0YXNldCAoYEZhaXIxbV9TaGlwX0RhdGFzZXRgKSBpcyBhbHJlYWR5IHVwbG9hZGVkIGFuZCBhdHRhY2hlZCBieQp0aGUgdXNlci4gS2FnZ2xlIG1vdW50cyBpdCByZWFkLW9ubHkgdW5kZXIgL2thZ2dsZS9pbnB1dC88YXV0by1zbHVnPi8uLi4sIGFuZAp0aGUgZXhhY3Qgc2x1ZyBhbmQgdGhlIG5lc3Rpbmcgb2YgdGhlIGZvdXIgcmVzb2x1dGlvbiBmb2xkZXJzIGRlcGVuZCBvbiBob3cKdGhlIGRhdGFzZXQgd2FzIHVwbG9hZGVkIChvYnNlcnZlZCBsYXlvdXQ6CmBzaGlwX2RhdGFzZXRfPGNvbmQ+L2RhdGFzZXRzLzxjb25kPi8uLi5gLCBidXQgdGhpcyBpcyBub3QgYXNzdW1lZCkuIFBhdGgKcmVzb2x1dGlvbiBpcyB0aGVyZWZvcmUgZG9uZSBieSAqc2VhcmNoaW5nKiBmb3IgYGRhdGEueWFtbGAgZmlsZXMgdW5kZXIKL2thZ2dsZS9pbnB1dCBhbmQgY2xhc3NpZnlpbmcgZWFjaCBieSBpdHMgcGF0aCwgcmF0aGVyIHRoYW4gYnkgaGFyZC1jb2RpbmcKYW55IHVzZXJuYW1lLCBzbHVnIG9yIGRpcmVjdG9yeSBkZXB0aC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKaW1wb3J0IHNodXRpbApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClBBQ0tBR0VfUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CgpLQUdHTEVfSU5QVVQgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJGQUlSMU1fS0FHR0xFX0lOUFVUIiwgIi9rYWdnbGUvaW5wdXQiKSkKS0FHR0xFX1dPUktJTkcgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJGQUlSMU1fS0FHR0xFX1dPUktJTkciLCAiL2thZ2dsZS93b3JraW5nIikpCgpDT05ESVRJT05TID0gKCJvcmlnaW5hbCIsICJyNzUiLCAicjUwIiwgInIyNSIpCgojIERpcmVjdG9yeSBuYW1lcyAoY2FzZS1pbnNlbnNpdGl2ZSwgbWF0Y2hlZCBhcyBhIHdob2xlIHBhdGggY29tcG9uZW50KSB0aGF0CiMgaWRlbnRpZnkgZWFjaCBjb25kaXRpb24uIENvdmVycyB0aGUgb2JzZXJ2ZWQgdXBsb2FkIGxheW91dAojIChgc2hpcF9kYXRhc2V0X3IyNS8uLi5gKSwgdGhlIHBsYWluIGNvbmRpdGlvbiBuYW1lLCBhbmQgdGhlIHJ1bi1pZCBmb3JtLAojIHNvIGEgcmUtdXBsb2FkIHdpdGggYSBzbGlnaHRseSBkaWZmZXJlbnQgbGF5b3V0IHN0aWxsIHJlc29sdmVzLgpDT05ESVRJT05fRElSX05BTUVTOiBkaWN0W3N0ciwgc2V0W3N0cl1dID0gewogICAgIm9yaWdpbmFsIjogeyJvcmlnaW5hbCIsICJzaGlwX2RhdGFzZXRfb3JpZ2luYWwiLCAiMDBfYmFzZWxpbmUiLCAiYmFzZWxpbmUifSwKICAgICJyNzUiOiB7InI3NSIsICJzaGlwX2RhdGFzZXRfcjc1IiwgIjAxX3I3NSJ9LAogICAgInI1MCI6IHsicjUwIiwgInNoaXBfZGF0YXNldF9yNTAiLCAiMDJfcjUwIn0sCiAgICAicjI1IjogeyJyMjUiLCAic2hpcF9kYXRhc2V0X3IyNSIsICIwM19yMjUifSwKfQoKIyBNYXJrZXJzIHRoYXQgaWRlbnRpZnkgYSBkaXJlY3RvcnkgYXMgYSAqcmVzdWx0cyogYnVuZGxlICh0aGlzIHN0dWR5J3Mgb3duCiMgb3V0cHV0KSwgdXNlZCB0byBmaW5kIGEgcHJldmlvdXMgcnVuJ3MgS2FnZ2xlIE5vdGVib29rIE91dHB1dCBhdHRhY2hlZCBhcyBhCiMgZnJlc2ggaW5wdXQgc291cmNlIHNvIGEgbmV3IHNlc3Npb24gY2FuIHJlc3VtZSBpbnN0ZWFkIG9mIHJlc3RhcnRpbmcuClJFU1VMVF9NQVJLRVJfRklMRVMgPSAoImV4cGVyaW1lbnRfc3RhdHVzLmpzb24iLCAibWFzdGVyX3Jlc3VsdHMuanNvbiIpClJFU1VMVF9NQVJLRVJfRElSUyA9ICgiMDBfYmFzZWxpbmUiLCAiMDFfcjc1IiwgIjAyX3I1MCIsICIwM19yMjUiKQoKCmRlZiBfY2xhc3NpZnkoZGF0YV95YW1sOiBQYXRoKSAtPiBzdHIgfCBOb25lOgogICAgcGFydHMgPSB7cC5sb3dlcigpIGZvciBwIGluIGRhdGFfeWFtbC5wYXJ0c30KICAgIGZvciBjb25kLCBuYW1lcyBpbiBDT05ESVRJT05fRElSX05BTUVTLml0ZW1zKCk6CiAgICAgICAgaWYgYW55KG4ubG93ZXIoKSBpbiBwYXJ0cyBmb3IgbiBpbiBuYW1lcyk6CiAgICAgICAgICAgIHJldHVybiBjb25kCiAgICByZXR1cm4gTm9uZQoKCmRlZiBmaW5kX2RhdGFzZXRzX3Jvb3QoZXhwbGljaXQ6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKICAgICAgICAgICAgICAgICAgICAgICBzZWFyY2hfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgUGF0aF06CiAgICAiIiJSZXR1cm4ge2NvbmRpdGlvbjogZGF0YXNldF9kaXJ9LCBkaXNjb3ZlcmVkIGJ5IHNjYW5uaW5nIGZvciBkYXRhLnlhbWwuCgogICAgYGRhdGFzZXRfZGlyYCBpcyB0aGUgZGlyZWN0b3J5IHRoYXQgZGlyZWN0bHkgaG9sZHMgYGRhdGEueWFtbGAsIGBpbWFnZXMvYAogICAgYW5kIGBsYWJlbHMvYCBmb3IgdGhhdCBjb25kaXRpb24uIFVubGlrZSB0aGUgbG9jYWwvQ29sYWIgbGF5b3V0LCB0aGlzIGlzCiAgICBOT1QgbmVjZXNzYXJpbHkgYSBjb21tb24gcGFyZW50IGFjcm9zcyBjb25kaXRpb25zIC0tIHRoZSBLYWdnbGUgdXBsb2FkIGNhbgogICAgbmVzdCBlYWNoIGNvbmRpdGlvbiB1bmRlciBpdHMgb3duIHRvcC1sZXZlbCBmb2xkZXIuCiAgICAiIiIKICAgIGlmIGV4cGxpY2l0OgogICAgICAgIHAgPSBQYXRoKGV4cGxpY2l0KQogICAgICAgIGZvdW5kID0ge2M6IHAgLyBjIGZvciBjIGluIENPTkRJVElPTlMgaWYgKHAgLyBjIC8gImRhdGEueWFtbCIpLmV4aXN0cygpfQogICAgICAgIGlmIGxlbihmb3VuZCkgPT0gbGVuKENPTkRJVElPTlMpOgogICAgICAgICAgICByZXR1cm4gZm91bmQKCiAgICByb290ID0gUGF0aChzZWFyY2hfcm9vdCkgaWYgc2VhcmNoX3Jvb3QgZWxzZSBLQUdHTEVfSU5QVVQKICAgIGlmIG5vdCByb290LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIntyb290fSBkb2VzIG5vdCBleGlzdC4gQXR0YWNoIHRoZSAnRmFpcjFtX1NoaXBfRGF0YXNldCcgS2FnZ2xlICIKICAgICAgICAgICAgZiJEYXRhc2V0IHRvIHRoaXMgbm90ZWJvb2sgKE5vdGVib29rIC0+IEFkZCBJbnB1dCkgYW5kIHJlcnVuLiIpCgogICAgZm91bmQ6IGRpY3Rbc3RyLCBQYXRoXSA9IHt9CiAgICBhbGxfeWFtbDogbGlzdFtQYXRoXSA9IFtdCiAgICBmb3IgeWFtbF9wYXRoIGluIHNvcnRlZChyb290LnJnbG9iKCJkYXRhLnlhbWwiKSk6CiAgICAgICAgYWxsX3lhbWwuYXBwZW5kKHlhbWxfcGF0aCkKICAgICAgICBjb25kID0gX2NsYXNzaWZ5KHlhbWxfcGF0aCkKICAgICAgICBpZiBjb25kIGFuZCBjb25kIG5vdCBpbiBmb3VuZDoKICAgICAgICAgICAgZm91bmRbY29uZF0gPSB5YW1sX3BhdGgucGFyZW50CgogICAgbWlzc2luZyA9IFtjIGZvciBjIGluIENPTkRJVElPTlMgaWYgYyBub3QgaW4gZm91bmRdCiAgICBpZiBtaXNzaW5nOgogICAgICAgIHNlZW4gPSAiXG4gICIuam9pbihzdHIocCkgZm9yIHAgaW4gYWxsX3lhbWwpIG9yICIobm9uZSBmb3VuZCkiCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiQ291bGQgbm90IGZpbmQgZGF0YS55YW1sIGZvciBjb25kaXRpb24ocykge21pc3Npbmd9IHVuZGVyIHtyb290fS5cbiIKICAgICAgICAgICAgZiJkYXRhLnlhbWwgZmlsZXMgZm91bmQ6XG4gIHtzZWVufVxuXG4iCiAgICAgICAgICAgIGYiQ29uZmlybSAnRmFpcjFtX1NoaXBfRGF0YXNldCcgaXMgYXR0YWNoZWQgaW4gdGhlIG5vdGVib29rJ3MgRGF0YSAiCiAgICAgICAgICAgIGYicGFuZSBhbmQgY29udGFpbnMgYWxsIGZvdXIgcmVzb2x1dGlvbiBjb25kaXRpb25zIHtDT05ESVRJT05TfS4iKQogICAgcmV0dXJuIGZvdW5kCgoKZGVmIGZpbmRfcmVzdWx0c19yb290KGV4cGxpY2l0OiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IFBhdGg6CiAgICAiIiJXaGVyZSBleHBlcmltZW50IG91dHB1dHMgZ286IGFsd2F5cyB1bmRlciAva2FnZ2xlL3dvcmtpbmcuCgogICAgL2thZ2dsZS93b3JraW5nIGlzIE5PVCBndWFyYW50ZWVkIHRvIHN1cnZpdmUgYSBmcmVzaCBzZXNzaW9uLCBzbyB0aGlzIGRvZXMKICAgIG5vdCB0cnkgdG8gYmUgY2xldmVyIGFib3V0IHBlcnNpc3RlbmNlIC0tIHRoYXQgaXMgaGFuZGxlZCBzZXBhcmF0ZWx5IGJ5CiAgICBgZmluZF9wcmV2aW91c19yZXN1bHRzYCAocmVzdG9yaW5nIGEgcHJpb3IgTm90ZWJvb2sgT3V0cHV0KSBhbmQgYnkgdGhlCiAgICBub3RlYm9vayBpbnN0cnVjdGluZyB0aGUgdXNlciB0byBTYXZlIFZlcnNpb24gYWZ0ZXIgZWFjaCBjb25kaXRpb24uCiAgICAiIiIKICAgIHAgPSBQYXRoKGV4cGxpY2l0KSBpZiBleHBsaWNpdCBlbHNlIEtBR0dMRV9XT1JLSU5HIC8gInJlc3VsdHMiCiAgICBwLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJldHVybiBwCgoKZGVmIGZpbmRfcHJldmlvdXNfcmVzdWx0cyhzZWFyY2hfcm9vdDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBQYXRoIHwgTm9uZToKICAgICIiIkZpbmQgYSBwcmV2aW91c2x5LWF0dGFjaGVkIEthZ2dsZSBOb3RlYm9vayBPdXRwdXQgb2YgVEhJUyBzdHVkeSdzCiAgICByZXN1bHRzLCBpZiB0aGUgdXNlciByZS1hdHRhY2hlZCBvbmUgYWZ0ZXIgYSBzZXNzaW9uIHdhcyBpbnRlcnJ1cHRlZC4KCiAgICBMb29rcyB1bmRlciAva2FnZ2xlL2lucHV0IGZvciBhIGRpcmVjdG9yeSBjb250YWluaW5nIHRoaXMgc3R1ZHkncyBtYXJrZXIKICAgIGZpbGVzL3N1YmRpcmVjdG9yaWVzLCByYXRoZXIgdGhhbiBhc3N1bWluZyBhIGZpeGVkIG5vdGVib29rLW91dHB1dCBzbHVnLgogICAgIiIiCiAgICByb290ID0gUGF0aChzZWFyY2hfcm9vdCkgaWYgc2VhcmNoX3Jvb3QgZWxzZSBLQUdHTEVfSU5QVVQKICAgIGlmIG5vdCByb290LmV4aXN0cygpOgogICAgICAgIHJldHVybiBOb25lCiAgICBmb3IgY2FuZGlkYXRlIGluIHNvcnRlZChyb290Lmdsb2IoIioiKSk6CiAgICAgICAgaWYgbm90IGNhbmRpZGF0ZS5pc19kaXIoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3Igc3ViIGluIFtjYW5kaWRhdGUsICpjYW5kaWRhdGUuZ2xvYigiKiIpXToKICAgICAgICAgICAgaWYgbm90IHN1Yi5pc19kaXIoKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIGFueSgoc3ViIC8gbSkuZXhpc3RzKCkgZm9yIG0gaW4gUkVTVUxUX01BUktFUl9GSUxFUykgb3IgXAogICAgICAgICAgICAgICBhbnkoKHN1YiAvIGQgLyAibWV0cmljcy5qc29uIikuZXhpc3RzKCkgZm9yIGQgaW4gUkVTVUxUX01BUktFUl9ESVJTKToKICAgICAgICAgICAgICAgIHJldHVybiBzdWIKICAgIHJldHVybiBOb25lCgoKZGVmIHJlc3RvcmVfcHJldmlvdXNfcmVzdWx0cyhyZXN1bHRzX3Jvb3Q6IFBhdGgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VhcmNoX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gUGF0aCB8IE5vbmU6CiAgICAiIiJDb3B5IGEgcHJldmlvdXNseS1mb3VuZCByZXN1bHRzIGJ1bmRsZSBpbnRvIHRoZSAoZnJlc2gpIHdvcmtpbmcgcmVzdWx0cwogICAgcm9vdCwgc28gcnVuX2FsbC5weSdzIHNraXAtaWYtbWV0cmljcy5qc29uLWV4aXN0cyBsb2dpYyByZXN1bWVzIGNsZWFubHkuCgogICAgTmV2ZXIgb3ZlcndyaXRlcyBhIHJ1biB0aGF0IGlzIGFscmVhZHkgcHJlc2VudCBsb2NhbGx5IChhIG1pZC1zZXNzaW9uCiAgICByZXJ1biBvZiB0aGlzIGNlbGwgbXVzdCBub3QgY2xvYmJlciBwcm9ncmVzcyBtYWRlIHNpbmNlIHRoZSByZXN0b3JlKS4KICAgICIiIgogICAgcHJldiA9IGZpbmRfcHJldmlvdXNfcmVzdWx0cyhzZWFyY2hfcm9vdCkKICAgIGlmIHByZXYgaXMgTm9uZToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmVzdWx0c19yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGNvcGllZCA9IFtdCiAgICBmb3IgaXRlbSBpbiBzb3J0ZWQocHJldi5pdGVyZGlyKCkpOgogICAgICAgIGRzdCA9IHJlc3VsdHNfcm9vdCAvIGl0ZW0ubmFtZQogICAgICAgIGlmIGRzdC5leGlzdHMoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBpdGVtLmlzX2RpcigpOgogICAgICAgICAgICBzaHV0aWwuY29weXRyZWUoaXRlbSwgZHN0KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNodXRpbC5jb3B5MihpdGVtLCBkc3QpCiAgICAgICAgY29waWVkLmFwcGVuZChpdGVtLm5hbWUpCiAgICByZXR1cm4gcHJldiBpZiBjb3BpZWQgb3IgYW55KHJlc3VsdHNfcm9vdC5pdGVyZGlyKCkpIGVsc2UgTm9uZQoKCmRlZiBhbm5vdGF0aW9uX2RpcigpIC0+IFBhdGg6CiAgICAiIiJUaGUgc2hpcC1vbmx5IGFubm90YXRpb24gbGF5ZXIgc2hpcHBlZCBpbnNpZGUgdGhpcyBwYWNrYWdlLiIiIgogICAgZm9yIHAgaW4gKFBBQ0tBR0VfUk9PVCAvICJkYXRhIiAvICJzaGlwX29ubHkiLCk6CiAgICAgICAgaWYgKHAgLyAic2hpcF9hbm5vdGF0aW9ucy5wYXJxdWV0IikuZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybiBwCiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICBmInNoaXBfYW5ub3RhdGlvbnMucGFycXVldCBub3QgZm91bmQgdW5kZXIge1BBQ0tBR0VfUk9PVH0vZGF0YS9zaGlwX29ubHkiKQoKCmRlZiBlbnN1cmVfd3JpdGFibGVfZGF0YXNldChjb25kOiBzdHIsIHNyY19kaXI6IFBhdGgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB3b3JrX3Jvb3Q6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgICIiIlJldHVybiBhIGRpcmVjdG9yeSBVbHRyYWx5dGljcyBjYW4gdHJhaW4gZnJvbSBmb3IgdGhpcyBjb25kaXRpb24uCgogICAgL2thZ2dsZS9pbnB1dCBpcyBhbHdheXMgcmVhZC1vbmx5LCBidXQgVWx0cmFseXRpY3Mgb25seSBldmVyICp3YW50cyogdG8KICAgIHdyaXRlIGEgcmVkdW5kYW50IGAuY2FjaGVgIGZpbGUgbmV4dCB0byBgbGFiZWxzL2A7IGl0IGNhdGNoZXMgdGhlCiAgICByZXN1bHRpbmcgUGVybWlzc2lvbkVycm9yIGFuZCBkZWdyYWRlcyBncmFjZWZ1bGx5ICh0cmFpbmluZyBzdGlsbCB3b3JrcykKICAgIC0tIHNvIGJ5IGRlZmF1bHQgdGhpcyByZXR1cm5zIGBzcmNfZGlyYCB1bmNoYW5nZWQgYW5kIHJlYWRzIGRpcmVjdGx5IGZyb20KICAgIHRoZSBhdHRhY2hlZCBkYXRhc2V0LiBUaGUgY29zdCBvZiB0aGF0IGRlZ3JhZGF0aW9uIGlzIHJlYWwsIHRob3VnaDogd2l0aAogICAgbm93aGVyZSB0byBwZXJzaXN0IHRoZSBjYWNoZSwgVWx0cmFseXRpY3MgcmUtc2NhbnMgYW5kIHZlcmlmaWVzIGV2ZXJ5CiAgICBpbWFnZS9sYWJlbCBwYWlyIGluIGB0cmFpbjpgL2B2YWw6YCBvbiBldmVyeSBzaW5nbGUgbGF1bmNoIChzZWUKICAgIGBzdGFnZV9zYW5pdHlfc3Vic2V0YCwgd2hpY2ggaXMgd2h5IHRoZSBzYW5pdHkgdGVzdCBzdGFnZXMgYSB0aW55IHN1YnNldAogICAgaW5zdGVhZCBvZiBwb2ludGluZyBhdCB0aGUgZnVsbCBjb25kaXRpb24pLiBUaGlzIGZ1bmN0aW9uIGNvcGllcyAtLSBPTkxZCiAgICB0aGlzIG9uZSBjb25kaXRpb24sIG5ldmVyIGFsbCBmb3VyIC0tIHNvbGVseSBhcyBhbiBleHBsaWNpdCwgb3B0LWluCiAgICBmYWxsYmFjayBpZiBhIHJlYWwgdHJhaW5pbmcgZmFpbHVyZSBzaG93cyB0aGUgcmVhZC1vbmx5IG1vdW50IGlzIG5vdAogICAgdG9sZXJhdGVkIGF0IGFsbC4KICAgICIiIgogICAgd29ya19yb290ID0gd29ya19yb290IG9yIChLQUdHTEVfV09SS0lORyAvICJkYXRhc2V0cyIpCiAgICBkc3QgPSB3b3JrX3Jvb3QgLyBjb25kCiAgICBpZiAoZHN0IC8gImRhdGEueWFtbCIpLmV4aXN0cygpOgogICAgICAgIHJldHVybiBkc3QgICMgYWxyZWFkeSBzdGFnZWQgYnkgYSBwcmV2aW91cyBjYWxsL2NlbGwKICAgIHJldHVybiBzcmNfZGlyCgoKZGVmIHN0YWdlX3Nhbml0eV9zdWJzZXQobGFiZWw6IHN0ciwgZGF0YXNldF9kaXI6IFBhdGgsIG5fdHJhaW46IGludCA9IDMyLAogICAgICAgICAgICAgICAgICAgICAgICBuX3ZhbDogaW50ID0gMTYsIHdvcmtfcm9vdDogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBQYXRoOgogICAgIiIiU3RhZ2UgYSB0aW55LCBzaGlwLWd1YXJhbnRlZWQgc3Vic2V0IGZvciB0aGUgR1BVIHNhbml0eSB0ZXN0IGFuZAogICAgcmV0dXJuIGl0cyByb290IGRpcmVjdG9yeSAoaG9sZGluZyBkYXRhLnlhbWwsIGltYWdlcy8sIGxhYmVscy8pLgoKICAgIFRoZSBzYW5pdHkgdGVzdCdzIHdob2xlIHBvaW50IGlzIHByb3ZpbmcgdGhlIHBsdW1iaW5nIHdvcmtzIGluIGEgY291cGxlCiAgICBvZiBtaW51dGVzLCBub3QgY292ZXJpbmcgdGhlIGRhdGEgLS0gYnV0IGAtLWZyYWN0aW9uYCBkb2VzIE5PVCBtYWtlIHRoYXQKICAgIGNoZWFwOiBVbHRyYWx5dGljcyBzY2FucyBhbmQgdmVyaWZpZXMgRVZFUlkgaW1hZ2UvbGFiZWwgcGFpciBsaXN0ZWQgaW4KICAgIGB0cmFpbjpgL2B2YWw6YCB0byBidWlsZCBpdHMgbGFiZWwgY2FjaGUgKmJlZm9yZSogYGZyYWN0aW9uYCBpcyBldmVyCiAgICBhcHBsaWVkLCBhbmQgKHBlciBgZW5zdXJlX3dyaXRhYmxlX2RhdGFzZXRgKSB0aGF0IGNhY2hlIGNhbiBuZXZlciBiZQogICAgc2F2ZWQgYW5kIHJldXNlZCBvbiB0aGUgcmVhZC1vbmx5IC9rYWdnbGUvaW5wdXQgbW91bnQuIExlZnQgcG9pbnRlZCBhdAogICAgdGhlIGZ1bGwgfjcsNzA2LWltYWdlIGRhdGFzZXQsIGEgIjUlIiBzYW5pdHkgcnVuIHN0aWxsIHBheXMgdGhlIEZVTEwKICAgIHNjYW4gY29zdCAtLSBhY3Jvc3MgYWxsIGZvdXIgY29uZGl0aW9ucywgb24gZXZlcnkgcnVuIC0tIHdoaWNoIGlzIHRoZQogICAgYWN0dWFsIHNvdXJjZSBvZiBTdGVwIDYgYmVpbmcgc2xvdywgbm90IHRoZSBvbmUgZXBvY2ggb2YgdHJhaW5pbmcuCiAgICBDb3B5aW5nIGEgc21hbGwsIGRldGVybWluaXN0aWMsIHNoaXAtY29udGFpbmluZyBzdWJzZXQgKG5vdCB0aGUgd2hvbGUKICAgIGRhdGFzZXQsIGFuZCBvbmx5IGZvciB0aGUgb25lIGNvbmRpdGlvbiBjdXJyZW50bHkgYmVpbmcgc3RhZ2VkKSBtYWtlcwogICAgdGhhdCBzY2FuIHRyaXZpYWwgcmVnYXJkbGVzcyBvZiBtb3VudCBsYXRlbmN5IG9yIGltYWdlIHNpemUuCgogICAgSW1hZ2VzIGFyZSBwaWNrZWQgZnJvbSBgZGF0YS9zaGlwX29ubHkvaW1hZ2VfbWFuaWZlc3QuY3N2YCAoYWxyZWFkeQogICAgYnVuZGxlZCBpbiB0aGlzIHBhY2thZ2UpIGZpbHRlcmVkIHRvIG5vbi1iYWNrZ3JvdW5kIGltYWdlcywgc28gdGhlCiAgICBzdWJzZXQgaXMgZ3VhcmFudGVlZCB0byBjb250YWluIHJlYWwgc2hpcCBhbm5vdGF0aW9ucyAtLSBhbiBlbXB0eS1HVAogICAgc3Vic2V0IHdvdWxkIG1ha2UgU3RlcCA2J3Mgb3duIGBuX2d0ID4gMGAgcGFzcy9mYWlsIGNoZWNrIG1lYW5pbmdsZXNzLgogICAgIiIiCiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgd29ya19yb290ID0gd29ya19yb290IG9yIChLQUdHTEVfV09SS0lORyAvICJrYWdnbGUiIC8gIl9zYW5pdHkiKQogICAgZHN0ID0gd29ya19yb290IC8gbGFiZWwKICAgIGlmIChkc3QgLyAiZGF0YS55YW1sIikuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIGRzdCAgIyBhbHJlYWR5IHN0YWdlZCBieSBhIHByZXZpb3VzIGNlbGwvcnVuCgogICAgaW1hZ2VzID0gcGQucmVhZF9jc3YoYW5ub3RhdGlvbl9kaXIoKSAvICJpbWFnZV9tYW5pZmVzdC5jc3YiKQogICAgZm9yIHNwbGl0X2Rpciwgc3BsaXRfY29sLCBuIGluICgoInRyYWluIiwgIlRyYWluIiwgbl90cmFpbiksICgidmFsIiwgIlZhbCIsIG5fdmFsKSk6CiAgICAgICAgKGRzdCAvICJpbWFnZXMiIC8gc3BsaXRfZGlyKS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgKGRzdCAvICJsYWJlbHMiIC8gc3BsaXRfZGlyKS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgY2FuZCA9IGltYWdlc1soaW1hZ2VzWyJTcGxpdCJdID09IHNwbGl0X2NvbCkgJiAofmltYWdlc1siaXNfYmFja2dyb3VuZCJdKV0KICAgICAgICBmb3IgYmFzZW5hbWUgaW4gY2FuZFsiYmFzZW5hbWUiXS5oZWFkKG4pOgogICAgICAgICAgICBzdGVtID0gUGF0aChiYXNlbmFtZSkuc3RlbQogICAgICAgICAgICBpbWdfc3JjID0gZGF0YXNldF9kaXIgLyAiaW1hZ2VzIiAvIHNwbGl0X2RpciAvIGYie3N0ZW19LmpwZyIKICAgICAgICAgICAgbGJsX3NyYyA9IGRhdGFzZXRfZGlyIC8gImxhYmVscyIgLyBzcGxpdF9kaXIgLyBmIntzdGVtfS50eHQiCiAgICAgICAgICAgIGlmIGltZ19zcmMuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBzaHV0aWwuY29weTIoaW1nX3NyYywgZHN0IC8gImltYWdlcyIgLyBzcGxpdF9kaXIgLyBpbWdfc3JjLm5hbWUpCiAgICAgICAgICAgIGlmIGxibF9zcmMuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBzaHV0aWwuY29weTIobGJsX3NyYywgZHN0IC8gImxhYmVscyIgLyBzcGxpdF9kaXIgLyBsYmxfc3JjLm5hbWUpCgogICAgKGRzdCAvICJkYXRhLnlhbWwiKS53cml0ZV90ZXh0KAogICAgICAgIGYicGF0aDoge2RzdH1cbnRyYWluOiBpbWFnZXMvdHJhaW5cbnZhbDogaW1hZ2VzL3ZhbFxubmM6IDFcbm5hbWVzOlxuICAwOiBzaGlwXG4iLAogICAgICAgIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gZHN0CgoKZGVmIHJlc29sdmVfZGF0YV95YW1sKGxhYmVsOiBzdHIsIGRhdGFzZXRfZGlyOiBQYXRoLAogICAgICAgICAgICAgICAgICAgICAgd29ya19yb290OiBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IFBhdGg6CiAgICAiIiJSZXR1cm4gYSBkYXRhLnlhbWwgVWx0cmFseXRpY3MgY2FuIGFjdHVhbGx5IHVzZSBmb3IgdGhpcyBjb25kaXRpb24uCgogICAgVGhlIGBkYXRhLnlhbWxgIGZpbGVzIGluc2lkZSB0aGUgdXBsb2FkZWQgZGF0YXNldCB3ZXJlIHdyaXR0ZW4gYnkKICAgIGBzY3JpcHRzL2NyZWF0ZV9yZXNvbHV0aW9uX2RhdGFzZXRzLnB5YCBvbiB0aGUgbWFjaGluZSB0aGF0IHByZXBhcmVkCiAgICB0aGVtLCBhbmQgdGhlaXIgYHBhdGg6YCBmaWVsZCBpcyB0aGF0IG1hY2hpbmUncyBvd24gYWJzb2x1dGUgcGF0aCAoZS5nLgogICAgYSBXaW5kb3dzIGRyaXZlIHBhdGgpLiBUaGF0IGlzIGNvcnJlY3QgbG9jYWxseSwgYnV0IG9uIEthZ2dsZSBpdCBpcyBhCiAgICBkZWFkIHBhdGggLS0gYW5kIFVsdHJhbHl0aWNzIGRvZXMgbm90IHJhaXNlIGEgY2xlYXIgZXJyb3IgZm9yIGl0LCBpdAogICAgc2lsZW50bHkgcmUtcmVzb2x2ZXMgdGhlIGJvZ3VzIGBwYXRoOmAgYWdhaW5zdCBpdHMgb3duIGBkYXRhc2V0c19kaXJgCiAgICBzZXR0aW5nIGFuZCBsb29rcyBmb3IgaW1hZ2VzIHNvbWV3aGVyZSBsaWtlCiAgICBgL2thZ2dsZS93b3JraW5nL2RhdGFzZXRzLzxzdGFsZS1wYXRoPi9pbWFnZXMvdmFsYCwgd2hpY2ggZG9lcyBub3QKICAgIGV4aXN0LiBgdHJhaW5gL2B2YWxgL2BuY2AvYG5hbWVzYCBhcmUgdW50b3VjaGVkIGFuZCBjb3JyZWN0ICh0aGV5IGFyZQogICAgcmVsYXRpdmUgdG8gYHBhdGg6YCk7IG9ubHkgYHBhdGg6YCBpdHNlbGYgaXMgc3RhbGUuCgogICAgVGhpcyByZXdyaXRlcyBqdXN0IHRoYXQgZmllbGQgdG8gdGhlIGRpcmVjdG9yeSBgZmluZF9kYXRhc2V0c19yb290KClgCiAgICBhY3R1YWxseSBkaXNjb3ZlcmVkLCBhbmQgd3JpdGVzIHRoZSBjb3JyZWN0ZWQgZmlsZSB1bmRlcgogICAgYC9rYWdnbGUvd29ya2luZ2AgLS0gdGhlIG9yaWdpbmFsIHVuZGVyIGAva2FnZ2xlL2lucHV0YCBpcyByZWFkLW9ubHkgYW5kCiAgICBpcyBuZXZlciBtb2RpZmllZC4KICAgICIiIgogICAgaW1wb3J0IHlhbWwKCiAgICB3b3JrX3Jvb3QgPSB3b3JrX3Jvb3Qgb3IgKEtBR0dMRV9XT1JLSU5HIC8gImthZ2dsZSIgLyAiX2RhdGFfeWFtbCIpCiAgICB3b3JrX3Jvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZCA9IHlhbWwuc2FmZV9sb2FkKChkYXRhc2V0X2RpciAvICJkYXRhLnlhbWwiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBkWyJwYXRoIl0gPSBzdHIoZGF0YXNldF9kaXIpCiAgICBvdXQgPSB3b3JrX3Jvb3QgLyBmIntsYWJlbH0ueWFtbCIKICAgIG91dC53cml0ZV90ZXh0KHlhbWwuc2FmZV9kdW1wKGQsIHNvcnRfa2V5cz1GYWxzZSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gb3V0CgoKZGVmIGlzX2V4cGVyaW1lbnRfY29tcGxldGUob3V0X2RpcjogUGF0aCkgLT4gYm9vbDoKICAgICIiIkEgZmluaXNoZWQsIHNjb3JlZCBydW46IG1ldHJpY3MuanNvbiBoYXMgYmVlbiB3cml0dGVuLiIiIgogICAgcmV0dXJuIChvdXRfZGlyIC8gIm1ldHJpY3MuanNvbiIpLmV4aXN0cygpCgoKZGVmIGZpbmRfcmVzdW1lX2NoZWNrcG9pbnQob3V0X2RpcjogUGF0aCwgcnVuX2lkOiBzdHIpIC0+IFBhdGggfCBOb25lOgogICAgIiIiVGhlIGNoZWNrcG9pbnQgdG8gcmVzdW1lIGZyb20sIGlmIHRoaXMgcnVuIHdhcyBpbnRlcnJ1cHRlZCBtaWQtdHJhaW5pbmcuCgogICAgVWx0cmFseXRpY3Mgd3JpdGVzIGB3ZWlnaHRzL2xhc3QucHRgIGFmdGVyIGV2ZXJ5IGVwb2NoIHVuZGVyCiAgICBgPG91dF9kaXI+L3RyYWluLzxydW5faWQ+L2AuIElmIHRoYXQgZXhpc3RzIGJ1dCB0aGUgcnVuIG5ldmVyIGZpbmlzaGVkCiAgICAobm8gbWV0cmljcy5qc29uKSwgdHJhaW5pbmcgc2hvdWxkIGNvbnRpbnVlIGZyb20gaXQgcmF0aGVyIHRoYW4gcmVzdGFydAogICAgYXQgZXBvY2ggMS4gU2hhcmVkIGJ5IHJ1bl9leHBlcmltZW50LnB5LCBydW5fYWxsLnB5LCBhbmQgdGhlIHNldHVwCiAgICBub3RlYm9vaydzIG93biByZXN1bWUtc3RhdHVzIGRpc3BsYXksIHNvIGFsbCB0aHJlZSBhZ3JlZSBvbiBvbmUKICAgIGRlZmluaXRpb24uCiAgICAiIiIKICAgIGlmIGlzX2V4cGVyaW1lbnRfY29tcGxldGUob3V0X2Rpcik6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGNrcHQgPSBvdXRfZGlyIC8gInRyYWluIiAvIHJ1bl9pZCAvICJ3ZWlnaHRzIiAvICJsYXN0LnB0IgogICAgcmV0dXJuIGNrcHQgaWYgY2twdC5leGlzdHMoKSBlbHNlIE5vbmUKCgpkZWYgc3RhZ2VfY29uZGl0aW9uKGNvbmQ6IHN0ciwgc3JjX2RpcjogUGF0aCwgd29ya19yb290OiBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IFBhdGg6CiAgICAiIiJFeHBsaWNpdGx5IGNvcHkgT05FIGNvbmRpdGlvbidzIGRhdGFzZXQgdG8gL2thZ2dsZS93b3JraW5nLiBGYWxsYmFjawogICAgcGF0aCBvbmx5IC0tIHNlZSBgZW5zdXJlX3dyaXRhYmxlX2RhdGFzZXRgLiBOZXZlciBjb3BpZXMgYWxsIGZvdXIuIiIiCiAgICB3b3JrX3Jvb3QgPSB3b3JrX3Jvb3Qgb3IgKEtBR0dMRV9XT1JLSU5HIC8gImRhdGFzZXRzIikKICAgIGRzdCA9IHdvcmtfcm9vdCAvIGNvbmQKICAgIGlmIChkc3QgLyAiZGF0YS55YW1sIikuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIGRzdAogICAgcHJpbnQoZiIgIHN0YWdpbmcge2NvbmR9OiB7c3JjX2Rpcn0gLT4ge2RzdH0gLi4uIikKICAgIHNodXRpbC5jb3B5dHJlZShzcmNfZGlyLCBkc3QpCiAgICByZXR1cm4gZHN0Cg==",
    "src/training.py": "IyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBBREFQVEVEIENPUFkgZm9yIEthZ2dsZSAtLSBiYXNlZCBvbiA8cHJvamVjdD4vc3JjL3RyYWluaW5nLnB5LgojIERpZmZlcmVuY2UgZnJvbSB0aGUgbG9jYWwvQ29sYWIgdmVyc2lvbjogYGJ1aWxkX3RyYWluX2FyZ3NgL2B0cmFpbl9vbmVgCiMgdGFrZSBhbiBleHBsaWNpdCBgZGF0YV95YW1sYCBwYXRoIHJhdGhlciB0aGFuIGEgYGRhdGFzZXRfcm9vdGAgKyBsYWJlbAojIGNvbnZlbnRpb24sIGJlY2F1c2UgdGhlIEthZ2dsZSB1cGxvYWQgbmVzdHMgZWFjaCBjb25kaXRpb24gdW5kZXIgaXRzIG93bgojIHRvcC1sZXZlbCBmb2xkZXIgKHNlZSBzcmMvcGF0aHMucHkpIGluc3RlYWQgb2Ygb25lIGNvbW1vbiBwYXJlbnQgZGlyZWN0b3J5LgojIE5vIGh5cGVycGFyYW1ldGVyLCBjb250cm9sIG9yIHJlc2VhcmNoLWRlc2lnbiBsb2dpYyBkaWZmZXJzLgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoiIiIKVHJhaW5pbmcgZHJpdmVyIGZvciBvbmUgcmVzb2x1dGlvbiBjb25kaXRpb24uCgpUaGUgd2hvbGUgcG9pbnQgb2YgdGhpcyBtb2R1bGUgaXMgdGhhdCBhIHJ1bidzIGh5cGVycGFyYW1ldGVycyBjYW5ub3QgY29tZQpmcm9tIGFueXdoZXJlIGV4Y2VwdCBjb25maWdzL21hc3Rlci55YW1sLiBgYnVpbGRfdHJhaW5fYXJnc2AgYXNzZW1ibGVzIHRoZQpVbHRyYWx5dGljcyBrZXl3b3JkIHNldCBmcm9tIHRoZSBjb25maWcgYW5kIHRoZSBleHBlcmltZW50J3MgZGVjbGFyZWQKb3ZlcnJpZGVzLCBhbmQgcmVmdXNlcyBhbnkgb3ZlcnJpZGUgdGhhdCBpcyBub3QgcGFydCBvZiB0aGUgZXhwZXJpbWVudCdzCmRlY2xhcmVkIGRpZmZlcmVuY2UuIFRoYXQgaXMgdGhlIG1lY2hhbmljYWwgZW5mb3JjZW1lbnQgb2YgUGFydCBEOiBpdCBpcyBub3QKcG9zc2libGUgdG8gYWNjaWRlbnRhbGx5IGdpdmUgb25lIGNvbmRpdGlvbiBhIGRpZmZlcmVudCBsZWFybmluZyByYXRlLgoKRXZlcnkgcnVuIGFsc28gcmVjb3JkcyB0aGUgZW52aXJvbm1lbnQgaXQgaGFwcGVuZWQgaW4gLS0gR1BVIG1vZGVsLCBWUkFNLApsaWJyYXJ5IHZlcnNpb25zLCB3YWxsLWNsb2NrIC0tIGJlY2F1c2UgInRyYWluaW5nIHRpbWUiIGFuZCAiR1BVIG1lbW9yeSIgYXJlCmRlcGVuZGVudCB2YXJpYWJsZXMgaW4gdGhpcyBzdHVkeSBhbmQgYXJlIG1lYW5pbmdsZXNzIHdpdGhvdXQgdGhlIGhhcmR3YXJlCnRoZXkgd2VyZSBtZWFzdXJlZCBvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCByYW5kb20KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgeWFtbAoKIyBLZXlzIGFuIGV4cGVyaW1lbnQgaXMgYWxsb3dlZCB0byBvdmVycmlkZS4gQW55dGhpbmcgZWxzZSB3b3VsZCBtYWtlIHRoZQojIGNvbmRpdGlvbnMgZGlmZmVyIGluIHNvbWV0aGluZyBvdGhlciB0aGFuIHRoZSBpbmRlcGVuZGVudCB2YXJpYWJsZS4KQUxMT1dFRF9PVkVSUklERVMgPSB7ImRhdGEiLCAibmFtZSIsICJwcm9qZWN0IiwgInNlZWQifQoKIyBFbnZpcm9ubWVudC1sZXZlbCBrbm9icyBgZXh0cmFgIG1heSBzZXQ6IHRoaW5ncyB0aGF0IGNoYW5nZSBob3cgbG9uZy93aGVyZQojIFRISVMgaW52b2NhdGlvbiBydW5zLCBuZXZlciB3aGF0IHRoZSBtb2RlbCBpcyB0cmFpbmVkIG9uLiAidGltZSIgKGhvdXJzKQojIGlzIFVsdHJhbHl0aWNzJyBvd24gd2FsbC1jbG9jayB0cmFpbmluZyBjYXAgLS0gaXQgc3RvcHMgY2xlYW5seSBhZnRlciBhCiMgY29tcGxldGVkIGVwb2NoIG9uY2UgZWxhcHNlZCB0aW1lIGV4Y2VlZHMgaXQsIGxldHRpbmcgYSBzaW5nbGUgY29uZGl0aW9uJ3MKIyA1MC1lcG9jaCBydW4gYmUgc2xpY2VkIGFjcm9zcyBzZXZlcmFsIEthZ2dsZSBzZXNzaW9ucyB0aGF0IGVhY2ggc3RheSB3ZWxsCiMgdW5kZXIgdGhlIHBsYXRmb3JtJ3Mgc2Vzc2lvbiB0aW1lIGxpbWl0LgpFTlZfS0VZUyA9IHsiZGV2aWNlIiwgIndvcmtlcnMiLCAiYmF0Y2giLCAiYW1wIiwgImNhY2hlIiwgInByb2plY3QiLAogICAgICAgICAgICAiZXhpc3Rfb2siLCAicmVzdW1lIiwgImVwb2NocyIsICJmcmFjdGlvbiIsICJwbG90cyIsICJ2YWwiLCAidGltZSJ9CgoKZGVmIGxvYWRfbWFzdGVyKHBhdGg6IHN0ciB8IFBhdGgpIC0+IGRpY3Q6CiAgICB3aXRoIG9wZW4ocGF0aCkgYXMgZjoKICAgICAgICByZXR1cm4geWFtbC5zYWZlX2xvYWQoZikKCgpkZWYgc2V0X3NlZWRzKHNlZWQ6IGludCkgLT4gTm9uZToKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgb3MuZW52aXJvblsiUFlUSE9OSEFTSFNFRUQiXSA9IHN0cihzZWVkKQogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICAgICAgdG9yY2guY3VkYS5tYW51YWxfc2VlZF9hbGwoc2VlZCkKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwYXNzCgoKZGVmIGJ1aWxkX3RyYWluX2FyZ3MoY2ZnOiBkaWN0LCBleHBlcmltZW50OiBzdHIsIHByb2plY3Q6IHN0ciB8IFBhdGgsCiAgICAgICAgICAgICAgICAgICAgIGRhdGFfeWFtbDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgICAgICBleHRyYTogZGljdCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0OgogICAgIiIiQXNzZW1ibGUgdGhlIGV4YWN0IFVsdHJhbHl0aWNzIGt3YXJncyBmb3Igb25lIGV4cGVyaW1lbnQuCgogICAgYGV4dHJhYCBpcyBpbnRlbmRlZCBmb3IgZW52aXJvbm1lbnQtbGV2ZWwga25vYnMgKGRldmljZSwgd29ya2VycywgYmF0Y2ggb24KICAgIGEgc21hbGxlciBHUFUpIC0tIG5ldmVyIGZvciBoeXBlcnBhcmFtZXRlcnMuIEFueXRoaW5nIG91dHNpZGUKICAgIEFMTE9XRURfT1ZFUlJJREVTIHBsdXMgdGhhdCBlbnZpcm9ubWVudCBzZXQgcmFpc2VzLCByYXRoZXIgdGhhbiBzaWxlbnRseQogICAgYnJlYWtpbmcgdGhlIGV4cGVyaW1lbnRhbCBjb250cm9sLgogICAgIiIiCiAgICBleHAgPSBjZmdbImV4cGVyaW1lbnRzIl1bZXhwZXJpbWVudF0KICAgIGFyZ3M6IGRpY3QgPSB7Im1vZGVsIjogY2ZnWyJtb2RlbCJdWyJ3ZWlnaHRzIl0sICJwcmV0cmFpbmVkIjogY2ZnWyJtb2RlbCJdWyJwcmV0cmFpbmVkIl19CiAgICBhcmdzLnVwZGF0ZShjZmdbInRyYWluIl0pCiAgICBhcmdzLnVwZGF0ZShjZmdbImF1Z21lbnRhdGlvbiJdKQogICAgYXJnc1sic2VlZCJdID0gY2ZnWyJyZXByb2R1Y2liaWxpdHkiXVsic2VlZCJdCiAgICBhcmdzWyJkZXRlcm1pbmlzdGljIl0gPSBjZmdbInJlcHJvZHVjaWJpbGl0eSJdWyJkZXRlcm1pbmlzdGljIl0KICAgICMgbWF4X2RldCBsaXZlcyB1bmRlciBgZXZhbHVhdGlvbjpgLCBub3QgYHRyYWluOmAsIGJ1dCBVbHRyYWx5dGljcyBhbHNvCiAgICAjIHVzZXMgaXQgZm9yIHRoZSB2YWxpZGF0aW9uIHBhc3MgaXQgcnVucyBhZnRlciBldmVyeSBlcG9jaCAod2hpY2gKICAgICMgZGVjaWRlcyB3aGljaCBjaGVja3BvaW50IGJlY29tZXMgYmVzdC5wdCkgLS0gd2l0aG91dCB0aGlzLCB0aGF0CiAgICAjIGluLXRyYWluaW5nIHZhbGlkYXRpb24gc2lsZW50bHkgcmV2ZXJ0cyB0byBVbHRyYWx5dGljcycgb3duIGRlZmF1bHQgb2YKICAgICMgMzAwLCBjYXBwaW5nIHJlY2FsbCBvbiB0aGUgZGVuc2VzdCBzY2VuZXMgYW5kIGxldHRpbmcgYSBsb3dlci1yZWNhbGwKICAgICMgZXBvY2ggd2luICJiZXN0IiBwdXJlbHkgZnJvbSBhbiB1bmNvbnRyb2xsZWQgbWF4X2RldCwgbm90IG1vZGVsIHF1YWxpdHkuCiAgICBhcmdzWyJtYXhfZGV0Il0gPSBjZmdbImV2YWx1YXRpb24iXVsibWF4X2RldCJdCgogICAgb3ZlcnJpZGVzID0gZGljdChleHBbIm92ZXJyaWRlcyJdKQogICAgaWYgZGF0YV95YW1sIGlzIG5vdCBOb25lOgogICAgICAgICMgS2FnZ2xlIG5lc3RzIGVhY2ggY29uZGl0aW9uIHVuZGVyIGl0cyBvd24gdG9wLWxldmVsIGlucHV0IGZvbGRlciwgc28KICAgICAgICAjIHRoZSBkYXRhLnlhbWwgcGF0aCBpcyByZXNvbHZlZCBieSBzcmMucGF0aHMuZmluZF9kYXRhc2V0c19yb290KCkgYW5kCiAgICAgICAgIyBwYXNzZWQgaW4gZGlyZWN0bHksIHJhdGhlciB0aGFuIHJlYnVpbHQgZnJvbSBhIGNvbW1vbiByb290ICsgbGFiZWwuCiAgICAgICAgb3ZlcnJpZGVzWyJkYXRhIl0gPSBzdHIoZGF0YV95YW1sKQogICAgYmFkID0gc2V0KG92ZXJyaWRlcykgLSBBTExPV0VEX09WRVJSSURFUwogICAgaWYgYmFkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiZXhwZXJpbWVudCB7ZXhwZXJpbWVudH0gdHJpZXMgdG8gb3ZlcnJpZGUge3NvcnRlZChiYWQpfSwgd2hpY2ggIgogICAgICAgICAgICBmIndvdWxkIG1ha2UgaXQgZGlmZmVyIGZyb20gdGhlIG90aGVyIGNvbmRpdGlvbnMgaW4gc29tZXRoaW5nICIKICAgICAgICAgICAgZiJvdGhlciB0aGFuIHJlc29sdXRpb24uIEFsbG93ZWQ6IHtzb3J0ZWQoQUxMT1dFRF9PVkVSUklERVMpfSIpCiAgICBhcmdzLnVwZGF0ZShvdmVycmlkZXMpCgogICAgaWYgZXh0cmE6CiAgICAgICAgYmFkID0gc2V0KGV4dHJhKSAtIEVOVl9LRVlTCiAgICAgICAgaWYgYmFkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYicmVmdXNpbmcgZW52aXJvbm1lbnQgb3ZlcnJpZGVzIHtzb3J0ZWQoYmFkKX06ICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmInRoZXNlIGFyZSBleHBlcmltZW50YWwgcGFyYW1ldGVycyBhbmQgYmVsb25nIGluICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmImNvbmZpZ3MvbWFzdGVyLnlhbWwiKQogICAgICAgIGFyZ3MudXBkYXRlKGV4dHJhKQoKICAgIGFyZ3NbInByb2plY3QiXSA9IHN0cihwcm9qZWN0KQogICAgYXJncy5zZXRkZWZhdWx0KCJuYW1lIiwgZXhwWyJpZCJdKQogICAgcmV0dXJuIGFyZ3MKCgpkZWYgZW52aXJvbm1lbnRfaW5mbygpIC0+IGRpY3Q6CiAgICBpbmZvID0gewogICAgICAgICJweXRob24iOiBzeXMudmVyc2lvbi5zcGxpdCgpWzBdLAogICAgICAgICJwbGF0Zm9ybSI6IHBsYXRmb3JtLnBsYXRmb3JtKCksCiAgICAgICAgImNwdV9jb3VudCI6IG9zLmNwdV9jb3VudCgpLAogICAgfQogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGluZm9bInRvcmNoIl0gPSB0b3JjaC5fX3ZlcnNpb25fXwogICAgICAgIGluZm9bImN1ZGFfYXZhaWxhYmxlIl0gPSB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgcCA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApCiAgICAgICAgICAgIGluZm9bImdwdV9uYW1lIl0gPSBwLm5hbWUKICAgICAgICAgICAgaW5mb1siZ3B1X3RvdGFsX21lbW9yeV9nYiJdID0gcm91bmQocC50b3RhbF9tZW1vcnkgLyAxZTksIDIpCiAgICAgICAgICAgIGluZm9bImN1ZGFfdmVyc2lvbiJdID0gdG9yY2gudmVyc2lvbi5jdWRhCiAgICAgICAgICAgIGluZm9bImdwdV9jb3VudCJdID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIGluZm9bInRvcmNoIl0gPSBOb25lCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHVsdHJhbHl0aWNzCiAgICAgICAgaW5mb1sidWx0cmFseXRpY3MiXSA9IHVsdHJhbHl0aWNzLl9fdmVyc2lvbl9fCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcGFzcwogICAgdHJ5OgogICAgICAgIGluZm9bIm52aWRpYV9zbWkiXSA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KAogICAgICAgICAgICBbIm52aWRpYS1zbWkiLCAiLS1xdWVyeS1ncHU9bmFtZSxtZW1vcnkudG90YWwsZHJpdmVyX3ZlcnNpb24iLAogICAgICAgICAgICAgIi0tZm9ybWF0PWNzdixub2hlYWRlciJdLCB0ZXh0PVRydWUsIHRpbWVvdXQ9MTUpLnN0cmlwKCkKICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMSAtIGFic2VudCBHUFUgaXMgYSBub3JtYWwgc3RhdGUKICAgICAgICBwYXNzCiAgICByZXR1cm4gaW5mbwoKCmRlZiBwZWFrX2dwdV9tZW1vcnlfZ2IoKSAtPiBmbG9hdCB8IE5vbmU6CiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgcmV0dXJuIHJvdW5kKHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSAvIDFlOSwgMykKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwYXNzCiAgICByZXR1cm4gTm9uZQoKCmRlZiBjb3VudF9jb21wbGV0ZWRfZXBvY2hzKHJ1bl9kaXI6IFBhdGgpIC0+IGludDoKICAgICIiIkVwb2NocyBVbHRyYWx5dGljcyBhY3R1YWxseSBmaW5pc2hlZCwgZnJvbSBpdHMgb3duIHBlci1lcG9jaCBsb2cuCgogICAgUmVhZGluZyBgcmVzdWx0cy5jc3ZgIChvbmUgcm93IHBlciBjb21wbGV0ZWQgZXBvY2gpIHJhdGhlciB0aGFuIGEKICAgIHRyYWluZXIgYXR0cmlidXRlIGF2b2lkcyBkZXBlbmRpbmcgb24gdGhhdCBhdHRyaWJ1dGUncyBleGFjdCBvZmYtYnktb25lCiAgICBzZW1hbnRpY3MsIGFuZCB3b3JrcyBpZGVudGljYWxseSB3aGV0aGVyIHRoaXMgd2FzIGEgZnJlc2ggcnVuIG9yIGEKICAgIHJlc3VtZS4KICAgICIiIgogICAgY3N2X3BhdGggPSBydW5fZGlyIC8gInJlc3VsdHMuY3N2IgogICAgaWYgbm90IGNzdl9wYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiAwCiAgICB3aXRoIG9wZW4oY3N2X3BhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgcmV0dXJuIG1heCgwLCBzdW0oMSBmb3IgXyBpbiBmKSAtIDEpICAjIG1pbnVzIHRoZSBoZWFkZXIgcm93CgoKZGVmIHRyYWluX29uZShjZmc6IGRpY3QsIGV4cGVyaW1lbnQ6IHN0ciwgcHJvamVjdDogUGF0aCwKICAgICAgICAgICAgICBkYXRhX3lhbWw6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKICAgICAgICAgICAgICBleHRyYTogZGljdCB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgIHJlc3VtZV9mcm9tOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6CiAgICAiIiJUcmFpbiBhIHNpbmdsZSBjb25kaXRpb24gYW5kIHJldHVybiBhIHJlY29yZCBvZiB0aGUgcnVuLgoKICAgIGByZXN1bWVfZnJvbWAgY29udGludWVzIGFuIGludGVycnVwdGVkIEthZ2dsZSBzZXNzaW9uIGZyb20gaXRzIG93bgogICAgYGxhc3QucHRgIHJhdGhlciB0aGFuIHJlc3RhcnRpbmcgYXQgZXBvY2ggMSAtLSBVbHRyYWx5dGljcyByZWxvYWRzIHRoYXQKICAgIHJ1bidzIG9yaWdpbmFsIGFyZ3VtZW50cyAoZGF0YSwgaW1nc3osIGh5cGVycGFyYW1ldGVycykgZnJvbSB0aGUKICAgIGBhcmdzLnlhbWxgIGl0IHdyb3RlIG5leHQgdG8gdGhlIGNoZWNrcG9pbnQsIHNvIG5vdGhpbmcgaGVyZSBuZWVkcyB0byBiZQogICAgcmUtc3BlY2lmaWVkIGFuZCBub3RoaW5nIGNhbiBkcmlmdCBmcm9tIHdoYXQgdGhlIHJ1biBzdGFydGVkIHdpdGguCgogICAgYGV4dHJhWyJ0aW1lIl1gIChob3VycyksIGlmIGdpdmVuLCBpcyBhIHdhbGwtY2xvY2sgY2FwIFVsdHJhbHl0aWNzCiAgICBlbmZvcmNlcyBpdHNlbGYgKHN0b3BwaW5nIGNsZWFubHkgYWZ0ZXIgYSBjb21wbGV0ZWQgZXBvY2gpLCBzbyBvbmUKICAgIGludm9jYXRpb24gY2FuIGJlIHNpemVkIHRvIGZpdCBpbnNpZGUgYSBzaW5nbGUgS2FnZ2xlIHNlc3Npb24uIFRoZQogICAgcmVjb3JkJ3MgYGZ1bGx5X3RyYWluZWRgIGZsYWcgdGVsbHMgdGhlIGNhbGxlciB3aGV0aGVyIHRoZSBydW4gYWN0dWFsbHkKICAgIHJlYWNoZWQgdGhlIHN0dWR5J3MgdGFyZ2V0IGVwb2NoIGNvdW50IG9yIHdhcyBjdXQgc2hvcnQgYnkgdGhhdCBjYXAgLS0KICAgIG9ubHkgYSBgZnVsbHlfdHJhaW5lZGAgcnVuIHNob3VsZCBiZSBldmFsdWF0ZWQgYW5kIG1hcmtlZCBjb21wbGV0ZTsKICAgIGEgY3V0LXNob3J0IG9uZSBzaG91bGQganVzdCBzYXZlIGl0cyBjaGVja3BvaW50IGFuZCBleGl0LCB0byBiZSByZXN1bWVkCiAgICBieSBhIGxhdGVyIGludm9jYXRpb24uCiAgICAiIiIKICAgIGZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcGFzcwoKICAgICMgVGhlIGVwb2NoIFRBUkdFVCBmb3IgImRpZCB0aGlzIHJ1biBhY3R1YWxseSBmaW5pc2giIHB1cnBvc2VzIC0tIG11c3QKICAgICMgcmVzcGVjdCBhbiAtLWVwb2NocyBvdmVycmlkZSAoZS5nLiBzYW5pdHkncyBlcG9jaHM9MSksIG5vdCBhbHdheXMgdGhlCiAgICAjIG1hc3RlciBjb25maWcncyA1MCwgb3IgYSBmdWxseS1jb21wbGV0ZWQgMS1lcG9jaCBzYW5pdHkgcnVuIHdvdWxkIGJlCiAgICAjIHdyb25nbHkgZmxhZ2dlZCBhcyBjdXQgc2hvcnQuCiAgICB0YXJnZXRfZXBvY2hzID0gKGV4dHJhIG9yIHt9KS5nZXQoImVwb2NocyIsIGNmZ1sidHJhaW4iXVsiZXBvY2hzIl0pCgogICAgaWYgcmVzdW1lX2Zyb20gaXMgbm90IE5vbmU6CiAgICAgICAgc2V0X3NlZWRzKGNmZ1sicmVwcm9kdWNpYmlsaXR5Il1bInNlZWQiXSkKICAgICAgICBtb2RlbCA9IFlPTE8oc3RyKHJlc3VtZV9mcm9tKSkKICAgICAgICByZXN1bWVfa3dhcmdzID0geyJyZXN1bWUiOiBUcnVlfQogICAgICAgIGlmIGV4dHJhIGFuZCAidGltZSIgaW4gZXh0cmE6CiAgICAgICAgICAgICMgVWx0cmFseXRpY3MgcmVsb2FkcyB0aGUgY2hlY2twb2ludCdzIG9yaWdpbmFsIGFyZ3MgYXMgYSBiYXNlCiAgICAgICAgICAgICMgYW5kIGxheWVycyBhbnkgYXJncyBwYXNzZWQgaGVyZSBvbiB0b3AgLS0gdGhlIHNhbWUgbWVjaGFuaXNtCiAgICAgICAgICAgICMgdXNlZCB0byBleHRlbmQgYSBmaW5pc2hlZCBydW4ncyBlcG9jaCBjb3VudCAtLSBzbyBlYWNoIHJlc3VtZWQKICAgICAgICAgICAgIyBpbnZvY2F0aW9uIGdldHMgaXRzIG93biBmcmVzaCB0aW1lIGJ1ZGdldCByYXRoZXIgdGhhbiByZXVzaW5nCiAgICAgICAgICAgICMgKG9yIGlnbm9yaW5nKSB0aGUgb25lIHJlY29yZGVkIGZyb20gdGhlIHZlcnkgZmlyc3QgaW52b2NhdGlvbi4KICAgICAgICAgICAgcmVzdW1lX2t3YXJnc1sidGltZSJdID0gZXh0cmFbInRpbWUiXQogICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICBtb2RlbC50cmFpbigqKnJlc3VtZV9rd2FyZ3MpCiAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICB1c2VkX2Vwb2NocyA9IHRhcmdldF9lcG9jaHMKICAgICAgICB1c2VkX2JhdGNoID0gY2ZnWyJ0cmFpbiJdWyJiYXRjaCJdCiAgICAgICAgdXNlZF9pbWdzeiA9IGNmZ1sidHJhaW4iXVsiaW1nc3oiXQogICAgICAgIHRyYWluX2FyZ3NfcmVjb3JkID0geyJyZXN1bWVkX2Zyb20iOiBzdHIocmVzdW1lX2Zyb20pLCAqKnJlc3VtZV9rd2FyZ3N9CiAgICBlbHNlOgogICAgICAgIGFyZ3MgPSBidWlsZF90cmFpbl9hcmdzKGNmZywgZXhwZXJpbWVudCwgcHJvamVjdCwgZGF0YV95YW1sLCBleHRyYSkKICAgICAgICBzZXRfc2VlZHMoYXJnc1sic2VlZCJdKQogICAgICAgIHdlaWdodHMgPSBhcmdzLnBvcCgibW9kZWwiKQogICAgICAgIG1vZGVsID0gWU9MTyh3ZWlnaHRzKQogICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICBtb2RlbC50cmFpbigqKmFyZ3MpCiAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICB1c2VkX2Vwb2NocywgdXNlZF9iYXRjaCwgdXNlZF9pbWdzeiA9IGFyZ3NbImVwb2NocyJdLCBhcmdzWyJiYXRjaCJdLCBhcmdzWyJpbWdzeiJdCiAgICAgICAgdHJhaW5fYXJnc19yZWNvcmQgPSB7azogdiBmb3IgaywgdiBpbiBhcmdzLml0ZW1zKCkgaWYgbm90IGNhbGxhYmxlKHYpfQoKICAgIHJ1bl9kaXIgPSBQYXRoKG1vZGVsLnRyYWluZXIuc2F2ZV9kaXIpCiAgICBjb21wbGV0ZWRfZXBvY2hzID0gY291bnRfY29tcGxldGVkX2Vwb2NocyhydW5fZGlyKQogICAgcmVjb3JkID0gewogICAgICAgICJleHBlcmltZW50IjogZXhwZXJpbWVudCwKICAgICAgICAiZXhwZXJpbWVudF9pZCI6IGNmZ1siZXhwZXJpbWVudHMiXVtleHBlcmltZW50XVsiaWQiXSwKICAgICAgICAicmVzb2x1dGlvbl9sYWJlbCI6IGNmZ1siZXhwZXJpbWVudHMiXVtleHBlcmltZW50XVsibGFiZWwiXSwKICAgICAgICAic2NhbGUiOiBjZmdbImV4cGVyaW1lbnRzIl1bZXhwZXJpbWVudF1bInNjYWxlIl0sCiAgICAgICAgInJ1bl9kaXIiOiBzdHIocnVuX2RpciksCiAgICAgICAgIndlaWdodHNfYmVzdCI6IHN0cihydW5fZGlyIC8gIndlaWdodHMiIC8gImJlc3QucHQiKSwKICAgICAgICAid2VpZ2h0c19sYXN0Ijogc3RyKHJ1bl9kaXIgLyAid2VpZ2h0cyIgLyAibGFzdC5wdCIpLAogICAgICAgICJ0cmFpbmluZ190aW1lX3MiOiByb3VuZChlbGFwc2VkLCAxKSwKICAgICAgICAidHJhaW5pbmdfdGltZV9oIjogcm91bmQoZWxhcHNlZCAvIDM2MDAsIDMpLAogICAgICAgICJlcG9jaHMiOiB1c2VkX2Vwb2NocywKICAgICAgICAiYmF0Y2giOiB1c2VkX2JhdGNoLAogICAgICAgICJpbWdzeiI6IHVzZWRfaW1nc3osCiAgICAgICAgInNlZWQiOiBjZmdbInJlcHJvZHVjaWJpbGl0eSJdWyJzZWVkIl0sCiAgICAgICAgInJlc3VtZWQiOiByZXN1bWVfZnJvbSBpcyBub3QgTm9uZSwKICAgICAgICAiY29tcGxldGVkX2Vwb2NocyI6IGNvbXBsZXRlZF9lcG9jaHMsCiAgICAgICAgInRhcmdldF9lcG9jaHMiOiB0YXJnZXRfZXBvY2hzLAogICAgICAgICJmdWxseV90cmFpbmVkIjogY29tcGxldGVkX2Vwb2NocyA+PSB0YXJnZXRfZXBvY2hzLAogICAgICAgICJwZWFrX2dwdV9tZW1vcnlfZ2IiOiBwZWFrX2dwdV9tZW1vcnlfZ2IoKSwKICAgICAgICAiZW52aXJvbm1lbnQiOiBlbnZpcm9ubWVudF9pbmZvKCksCiAgICAgICAgInRyYWluX2FyZ3MiOiB0cmFpbl9hcmdzX3JlY29yZCwKICAgIH0KICAgIChydW5fZGlyIC8gInJ1bl9yZWNvcmQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhyZWNvcmQsIGluZGVudD0yLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gcmVjb3JkCg==",
    "src/visualization.py": "IyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgR0VORVJBVEVEIENPUFkgLS0gZG8gbm90IGVkaXQgaGVyZS4NCiMgU291cmNlIG9mIHRydXRoOiA8cHJvamVjdD4vc3JjL3Zpc3VhbGl6YXRpb24ucHkNCiMgUmVnZW5lcmF0ZSB3aXRoOiBweXRob24gc2NyaXB0cy9jcmVhdGVfY29sYWJfcGFja2FnZS5weQ0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiIiIg0KUmVuZGVyaW5nIGhlbHBlcnMgZm9yIHByZXByb2Nlc3NpbmcgdmFsaWRhdGlvbiBhbmQgcXVhbGl0YXRpdmUgcmVzdWx0cy4NCg0KQm94ZXMgYXJlIGFsd2F5cyBkcmF3biBmcm9tIHRoZSAqbm9ybWFsaXplZCogWU9MTyBsYWJlbHMgcmF0aGVyIHRoYW4gZnJvbQ0KcGl4ZWwgY29vcmRpbmF0ZXMgaGVsZCBpbiBtZW1vcnkuIFRoYXQgd2F5IHRoZSBwaWN0dXJlIHNob3dzIHdoYXQgYSB0cmFpbmluZw0KcnVuIHdvdWxkIGFjdHVhbGx5IGNvbnN1bWU6IGlmIHRoZSB3cml0dGVuIGxhYmVsIGZpbGUgaXMgd3JvbmcsIHRoZSBvdmVybGF5IGlzDQp3cm9uZywgYW5kIHRoZSBlcnJvciBpcyB2aXNpYmxlIGluc3RlYWQgb2YgYmVpbmcgbWFza2VkIGJ5IHJlLWRlcml2aW5nIHRoZSBib3gNCmZyb20gYSBzb3VyY2UgdGhhdCB3YXMgbmV2ZXIgc2F2ZWQuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQoNCkJJTl9DT0xPVVJTID0geyJzbWFsbCI6ICIjZmYyZDU1IiwgIm1lZGl1bSI6ICIjZmZiMDAwIiwgImxhcmdlIjogIiMwMGMyYTgifQ0KREVGQVVMVF9DT0xPVVIgPSAiIzAwZTVmZiINCg0KDQpkZWYgeW9sb190b19waXhlbHMobGluZTogc3RyLCB3OiBpbnQsIGg6IGludCkgLT4gdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdCwgZmxvYXRdOg0KICAgICIiImBjbGFzcyBjeCBjeSBidyBiaGAgKG5vcm1hbGl6ZWQpIC0+ICh4X21pbiwgeV9taW4sIGJveF93LCBib3hfaCkgaW4gcHguIiIiDQogICAgXywgY3gsIGN5LCBidywgYmggPSBsaW5lLnNwbGl0KCkNCiAgICBjeCwgY3ksIGJ3LCBiaCA9IGZsb2F0KGN4KSwgZmxvYXQoY3kpLCBmbG9hdChidyksIGZsb2F0KGJoKQ0KICAgIHJldHVybiAoY3ggLSBidyAvIDIpICogdywgKGN5IC0gYmggLyAyKSAqIGgsIGJ3ICogdywgYmggKiBoDQoNCg0KZGVmIGRyYXdfYm94ZXMoYXgsIGxpbmVzLCB3OiBpbnQsIGg6IGludCwgY29sb3Vycz1Ob25lLA0KICAgICAgICAgICAgICAgbGluZXdpZHRoOiBmbG9hdCA9IDEuMCwgbWluX3Zpc2libGVfcHg6IGZsb2F0ID0gMy4wKSAtPiBOb25lOg0KICAgICIiIk92ZXJsYXkgWU9MTyBsYWJlbCBsaW5lcyBvbnRvIGFuIGF4aXMgYWxyZWFkeSBzaG93aW5nIHRoZSBpbWFnZS4NCg0KICAgIEJveGVzIHRoaW5uZXIgdGhhbiBgbWluX3Zpc2libGVfcHhgIGFyZSBwYWRkZWQgb3V0d2FyZCBmb3IgZHJhd2luZyBvbmx5Lg0KICAgIEF0IHIyNSBhIGdlbnVpbmUgYm94IGNhbiBiZSB3ZWxsIHVuZGVyIG9uZSBwaXhlbCBhY3Jvc3M7IHdpdGhvdXQgdGhpcyBpdA0KICAgIHdvdWxkIHJlbmRlciBhcyBub3RoaW5nIGF0IGFsbCBhbmQgdGhlIGZpZ3VyZSB3b3VsZCBpbXBseSB0aGUgYW5ub3RhdGlvbg0KICAgIHdhcyBsb3N0IHdoZW4gaW4gZmFjdCBpdCBpcyBwcmVzZW50IGFuZCBjb3JyZWN0Lg0KICAgICIiIg0KICAgIGltcG9ydCBtYXRwbG90bGliLnBhdGNoZXMgYXMgbXBhdGNoZXMNCg0KICAgIGZvciBpLCBsaW5lIGluIGVudW1lcmF0ZShsaW5lcyk6DQogICAgICAgIHgsIHksIGJ3LCBiaCA9IHlvbG9fdG9fcGl4ZWxzKGxpbmUsIHcsIGgpDQogICAgICAgIGMgPSBERUZBVUxUX0NPTE9VUiBpZiBjb2xvdXJzIGlzIE5vbmUgZWxzZSBjb2xvdXJzW2ldDQogICAgICAgIGR3ID0gbWF4KDAuMCwgKG1pbl92aXNpYmxlX3B4IC0gYncpIC8gMikNCiAgICAgICAgZGggPSBtYXgoMC4wLCAobWluX3Zpc2libGVfcHggLSBiaCkgLyAyKQ0KICAgICAgICBheC5hZGRfcGF0Y2gobXBhdGNoZXMuUmVjdGFuZ2xlKA0KICAgICAgICAgICAgKHggLSBkdywgeSAtIGRoKSwgYncgKyAyICogZHcsIGJoICsgMiAqIGRoLA0KICAgICAgICAgICAgZmlsbD1GYWxzZSwgZWRnZWNvbG9yPWMsIGxpbmV3aWR0aD1saW5ld2lkdGgpKQ0KDQoNCmRlZiByZXNvbHV0aW9uX3N0cmlwKG91dF9wYXRoOiBQYXRoLCBwYW5lbHM6IGxpc3RbZGljdF0sIHRpdGxlOiBzdHIsDQogICAgICAgICAgICAgICAgICAgICBjb2xvdXJzX2J5X3BhbmVsPU5vbmUpIC0+IE5vbmU6DQogICAgIiIiT25lIHJvdyBvZiBwYW5lbHMgc2hvd2luZyB0aGUgc2FtZSBzY2VuZSBhdCBlYWNoIHJlc29sdXRpb24uDQoNCiAgICBFdmVyeSBwYW5lbCBpcyBkcmF3biBhdCBpdHMgdHJ1ZSBwaXhlbCBzaXplIHJlbGF0aXZlIHRvIHRoZSBvdGhlcnMsIHNvIHRoZQ0KICAgIHNocmlua2luZyByYXN0ZXIgaXMgdmlzaWJsZSByYXRoZXIgdGhhbiBiZWluZyBub3JtYWxpc2VkIGF3YXkgYnkgdGhlDQogICAgZmlndXJlIGxheW91dCAtLSB0aGF0IHNocmlua2FnZSBpcyB0aGUgaW5kZXBlbmRlbnQgdmFyaWFibGUuDQogICAgIiIiDQogICAgaW1wb3J0IG1hdHBsb3RsaWINCiAgICBtYXRwbG90bGliLnVzZSgiQWdnIikNCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0DQoNCiAgICBuID0gbGVuKHBhbmVscykNCiAgICBiYXNlX2ggPSA0LjYNCiAgICB3aWR0aHMgPSBbcFsiaW1hZ2UiXS5zaGFwZVsxXSAvIHBbImltYWdlIl0uc2hhcGVbMF0gZm9yIHAgaW4gcGFuZWxzXQ0KICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygNCiAgICAgICAgMSwgbiwgZmlnc2l6ZT0oYmFzZV9oICogc3VtKHdpZHRocykgKyAwLjYgKiBuLCBiYXNlX2ggKyAwLjkpLA0KICAgICAgICBncmlkc3BlY19rdz17IndpZHRoX3JhdGlvcyI6IHdpZHRoc30pDQogICAgaWYgbiA9PSAxOg0KICAgICAgICBheGVzID0gW2F4ZXNdDQoNCiAgICBmb3IgYXgsIHAgaW4gemlwKG5wLmF0bGVhc3RfMWQoYXhlcyksIHBhbmVscyk6DQogICAgICAgIGltZyA9IHBbImltYWdlIl0NCiAgICAgICAgaCwgdyA9IGltZy5zaGFwZVs6Ml0NCiAgICAgICAgIyBpbnRlcnBvbGF0aW9uPSJuZWFyZXN0IiBzbyBhIGRvd25zYW1wbGVkIHRpbGUgaXMgbm90IHNpbGVudGx5DQogICAgICAgICMgc21vb3RoZWQgYnkgbWF0cGxvdGxpYiBpbnRvIGxvb2tpbmcgYmV0dGVyIHRoYW4gaXQgaXMuDQogICAgICAgIGF4Lmltc2hvdyhpbWcsIGludGVycG9sYXRpb249Im5lYXJlc3QiKQ0KICAgICAgICBkcmF3X2JveGVzKGF4LCBwWyJsaW5lcyJdLCB3LCBoLA0KICAgICAgICAgICAgICAgICAgIGNvbG91cnM9cC5nZXQoImNvbG91cnMiKSwgbGluZXdpZHRoPXAuZ2V0KCJsaW5ld2lkdGgiLCAxLjApKQ0KICAgICAgICBheC5zZXRfdGl0bGUoZiJ7cFsnbGFiZWwnXX1cbnt3fXh7aH0gcHggICh7bGVuKHBbJ2xpbmVzJ10pfSBib3hlcykiLA0KICAgICAgICAgICAgICAgICAgICAgZm9udHNpemU9MTApDQogICAgICAgIGF4LnNldF94dGlja3MoW10pOyBheC5zZXRfeXRpY2tzKFtdKQ0KDQogICAgIyBQYW5lbCB0aXRsZXMgYXJlIHR3byBsaW5lcyB0YWxsLCBzbyB0aGUgc3VwdGl0bGUgbmVlZHMgZXhwbGljaXQgcm9vbTsNCiAgICAjIHRpZ2h0X2xheW91dCBhbG9uZSBsZXRzIHRoZW0gY29sbGlkZSBvbiB3aWRlIHN0cmlwcy4NCiAgICBmaWcuc3VwdGl0bGUodGl0bGUsIGZvbnRzaXplPTExLCB5PTEuMDIpDQogICAgZmlnLnRpZ2h0X2xheW91dChyZWN0PSgwLCAwLCAxLCAwLjk3KSkNCiAgICBvdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIGZpZy5zYXZlZmlnKG91dF9wYXRoLCBkcGk9MTMwLCBiYm94X2luY2hlcz0idGlnaHQiKQ0KICAgIHBsdC5jbG9zZShmaWcpDQoNCg0KZGVmIHpvb21fc3RyaXAob3V0X3BhdGg6IFBhdGgsIHBhbmVsczogbGlzdFtkaWN0XSwgdGl0bGU6IHN0ciwNCiAgICAgICAgICAgICAgIGNlbnRyZV94eV9ub3JtOiB0dXBsZVtmbG9hdCwgZmxvYXRdLCBzcGFuX25vcm06IGZsb2F0KSAtPiBOb25lOg0KICAgICIiIlNhbWUgc2NlbmUsIHNhbWUgKm5vcm1hbGl6ZWQqIHdpbmRvdywgYXQgZWFjaCByZXNvbHV0aW9uLg0KDQogICAgQ3JvcHBpbmcgaW4gbm9ybWFsaXplZCBzcGFjZSBtZWFucyBldmVyeSBwYW5lbCBmcmFtZXMgdGhlIGlkZW50aWNhbCBwYXRjaA0KICAgIG9mIGdyb3VuZCwgc28gdGhlIHBhbmVscyBkaWZmZXIgb25seSBpbiBob3cgbXVjaCBkZXRhaWwgc3Vydml2ZXMgLS0gd2hpY2gNCiAgICBpcyB0aGUgY29tcGFyaXNvbiB0aGUgc3R1ZHkgaXMgbWFraW5nLg0KICAgICIiIg0KICAgIGltcG9ydCBtYXRwbG90bGliDQogICAgbWF0cGxvdGxpYi51c2UoIkFnZyIpDQogICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdA0KDQogICAgbiA9IGxlbihwYW5lbHMpDQogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIG4sIGZpZ3NpemU9KDQuMCAqIG4sIDQuNikpDQogICAgaWYgbiA9PSAxOg0KICAgICAgICBheGVzID0gW2F4ZXNdDQogICAgY3huLCBjeW4gPSBjZW50cmVfeHlfbm9ybQ0KDQogICAgZm9yIGF4LCBwIGluIHppcChucC5hdGxlYXN0XzFkKGF4ZXMpLCBwYW5lbHMpOg0KICAgICAgICBpbWcgPSBwWyJpbWFnZSJdDQogICAgICAgIGgsIHcgPSBpbWcuc2hhcGVbOjJdDQogICAgICAgIGhhbGZfdywgaGFsZl9oID0gc3Bhbl9ub3JtICogdyAvIDIsIHNwYW5fbm9ybSAqIGggLyAyDQogICAgICAgIHgwID0gaW50KG1heCgwLCBtaW4odyAtIDEsIGN4biAqIHcgLSBoYWxmX3cpKSkNCiAgICAgICAgeDEgPSBpbnQobWF4KHgwICsgMSwgbWluKHcsIGN4biAqIHcgKyBoYWxmX3cpKSkNCiAgICAgICAgeTAgPSBpbnQobWF4KDAsIG1pbihoIC0gMSwgY3luICogaCAtIGhhbGZfaCkpKQ0KICAgICAgICB5MSA9IGludChtYXgoeTAgKyAxLCBtaW4oaCwgY3luICogaCArIGhhbGZfaCkpKQ0KICAgICAgICBheC5pbXNob3coaW1nW3kwOnkxLCB4MDp4MV0sIGludGVycG9sYXRpb249Im5lYXJlc3QiKQ0KDQogICAgICAgIGltcG9ydCBtYXRwbG90bGliLnBhdGNoZXMgYXMgbXBhdGNoZXMNCiAgICAgICAgZm9yIGxpbmUgaW4gcFsibGluZXMiXToNCiAgICAgICAgICAgIGJ4LCBieSwgYncsIGJoID0geW9sb190b19waXhlbHMobGluZSwgdywgaCkNCiAgICAgICAgICAgIGF4LmFkZF9wYXRjaChtcGF0Y2hlcy5SZWN0YW5nbGUoDQogICAgICAgICAgICAgICAgKGJ4IC0geDAsIGJ5IC0geTApLCBidywgYmgsIGZpbGw9RmFsc2UsDQogICAgICAgICAgICAgICAgZWRnZWNvbG9yPURFRkFVTFRfQ09MT1VSLCBsaW5ld2lkdGg9MS40KSkNCiAgICAgICAgYXguc2V0X3hsaW0oMCwgeDEgLSB4MCk7IGF4LnNldF95bGltKHkxIC0geTAsIDApDQogICAgICAgIGF4LnNldF90aXRsZShmIntwWydsYWJlbCddfSAgKHt4MSAtIHgwfXh7eTEgLSB5MH0gcHggY3JvcCkiLCBmb250c2l6ZT0xMCkNCiAgICAgICAgYXguc2V0X3h0aWNrcyhbXSk7IGF4LnNldF95dGlja3MoW10pDQoNCiAgICBmaWcuc3VwdGl0bGUodGl0bGUsIGZvbnRzaXplPTExKQ0KICAgIGZpZy50aWdodF9sYXlvdXQoKQ0KICAgIG91dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgZmlnLnNhdmVmaWcob3V0X3BhdGgsIGRwaT0xMzAsIGJib3hfaW5jaGVzPSJ0aWdodCIpDQogICAgcGx0LmNsb3NlKGZpZykNCg==",
}
for _rel, _b64 in _FILES.items():
    _p = PKG / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(base64.b64decode(_b64))
print(f"wrote {len(_FILES)} file(s) into {PKG} (src): {sorted(_FILES)}")

In [ ]:
import base64
from pathlib import Path

PKG = Path("/kaggle/working/kaggle")
_FILES = {
    "scripts/build_annotation_layer.py": "IiIiClJlY29uc3RydWN0IHRoZSBzaGlwLW9ubHkgYW5ub3RhdGlvbiBsYXllciAoaW1hZ2UgbWFuaWZlc3QgKyBncm91bmQtdHJ1dGgKYm94ZXMpIGZyb20gdGhlIGF0dGFjaGVkIEthZ2dsZSBEYXRhc2V0LCBpbnN0ZWFkIG9mIHNoaXBwaW5nIGl0IHByZWNvbXB1dGVkLgoKICAgIHB5dGhvbiBzY3JpcHRzL2J1aWxkX2Fubm90YXRpb25fbGF5ZXIucHkKCldoeSB0aGlzIGV4aXN0czogdGhlIGxvY2FsIHBpcGVsaW5lJ3MgYGRhdGEvc2hpcF9vbmx5L3tpbWFnZV9tYW5pZmVzdC5jc3YsCnNoaXBfYW5ub3RhdGlvbnMucGFycXVldH1gIGlzIH4xLjUgTUIuIEVtYmVkZGluZyBpdCBpbiB0aGUgc2V0dXAgbm90ZWJvb2sKKGFsb25nc2lkZSB0aGUgfjE4MCBLQiBvZiBjb2RlKSBwdXNoZWQgdGhlIG5vdGVib29rJ3Mgb3duIHNvdXJjZSBwYXN0CkthZ2dsZSdzIDEgTUIga2VybmVsLXNvdXJjZSBsaW1pdCAoImtlcm5lbCBzb3VyY2UgbXVzdCBiZSBsZXNzIHRoYW4gMQptZWdhYnl0ZXMiKSwgYW5kIHRoZSBub3RlYm9vayBjb3VsZCBub3QgYmUgc2F2ZWQuCgpUaGUgZml4IGlzIG5vdCB0byBjb21wcmVzcyBoYXJkZXIgLS0gaXQncyB0aGF0IHRoaXMgZmlsZSBkb2Vzbid0IG5lZWQgdG8KdHJhdmVsIHdpdGggdGhlIG5vdGVib29rIGF0IGFsbC4gSXRzIGNvbnRlbnRzIGFyZSBmdWxseSByZWNvdmVyYWJsZSBmcm9tIGRhdGEKYWxyZWFkeSBpbnNpZGUgdGhlIGF0dGFjaGVkIGRhdGFzZXQ6IHRoZSBZT0xPIGxhYmVsIC50eHQgZmlsZXMgQVJFIHRoZSBzb3VyY2UKb2YgdHJ1dGggZm9yIGJveCBnZW9tZXRyeSAobm9ybWFsaXplZCBjb29yZGluYXRlcywgaW52YXJpYW50IHVuZGVyIHJlc2l6ZSwKYWxyZWFkeSB2ZXJpZmllZCBieXRlLWlkZW50aWNhbCBhY3Jvc3MgYWxsIGZvdXIgcmVzb2x1dGlvbiBjb25kaXRpb25zIC0tIHNlZQpTdGVwIDQgb2YgdGhlIG5vdGVib29rKSwgYW5kIGNvbWJpbmVkIHdpdGggZWFjaCBpbWFnZSdzIG93biBwaXhlbCBkaW1lbnNpb25zCihyZWFkIGZyb20gdGhlICdvcmlnaW5hbCcgY29uZGl0aW9uLCB3aGljaCBhbG9uZSBjYXJyaWVzIG5hdGl2ZSByZXNvbHV0aW9uKSwKZXZlcnkgY29sdW1uIHRoZSBwaXBlbGluZSBuZWVkcyBjYW4gYmUgcmVjb21wdXRlZCBleGFjdGx5OgoKICAgIHhfbWluID0gKGN4IC0gdy8yKSAqIEltYWdlV2lkdGggICAoYW5kIHNpbWlsYXJseSB4X21heCwgeV9taW4sIHlfbWF4KQoKVGhpcyByZWFkcyBpbWFnZSBkaW1lbnNpb25zIHZpYSBQSUwncyBsYXp5IGhlYWRlciBwYXJzZSAoYEltYWdlLm9wZW4oLi4uKS5zaXplYApkb2VzIG5vdCBkZWNvZGUgcGl4ZWwgZGF0YSksIHNvIGl0IHN0YXlzIGZhc3QgZXZlbiBmb3IgdGhlIGxhcmdlc3Qgc291cmNlCnRpbGVzICh+MTAsMDAweDksNDcyIHB4KS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpQS0cgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBLRykpCgpmcm9tIHNyYy5hbmFseXNpcyBpbXBvcnQgYXNzaWduX3NpemVfYmluCmZyb20gc3JjLnBhdGhzIGltcG9ydCBmaW5kX2RhdGFzZXRzX3Jvb3QKCgpkZWYgYnVpbGQoZGF0YXNldF9kaXI6IFBhdGgpIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lXToKICAgICIiIlJldHVybiAoaW1hZ2VzLCBzaGlwcykgRGF0YUZyYW1lcyBmcm9tIG9uZSBjb25kaXRpb24ncyBpbWFnZXMrbGFiZWxzLgoKICAgIE11c3QgYmUgcnVuIGFnYWluc3QgdGhlICdvcmlnaW5hbCcgY29uZGl0aW9uOiBpdHMgcGVyLWltYWdlIHBpeGVsCiAgICBkaW1lbnNpb25zIGFyZSB3aGF0ICJvcmlnaW5hbC1yZXNvbHV0aW9uIGdyb3VuZCB0cnV0aCIgbWVhbnMgdGhyb3VnaG91dAogICAgdGhlIHJlc3Qgb2YgdGhlIHBpcGVsaW5lIChldmFsdWF0aW9uIG1hcHMgZXZlcnkgY29uZGl0aW9uJ3MgcHJlZGljdGlvbnMKICAgIGJhY2sgaW50byB0aGlzIHNhbWUgY29vcmRpbmF0ZSBmcmFtZSkuCiAgICAiIiIKICAgIGltYWdlX3Jvd3M6IGxpc3RbZGljdF0gPSBbXQogICAgc2hpcF9yb3dzOiBsaXN0W2RpY3RdID0gW10KICAgIGltZ19pZCA9IDAKICAgIGZvciBzcGxpdF9kaXIsIHNwbGl0X2NvbCBpbiAoKCJ0cmFpbiIsICJUcmFpbiIpLCAoInZhbCIsICJWYWwiKSk6CiAgICAgICAgaW1nX3BhdGhzID0gc29ydGVkKChkYXRhc2V0X2RpciAvICJpbWFnZXMiIC8gc3BsaXRfZGlyKS5nbG9iKCIqLmpwZyIpKQogICAgICAgIGZvciBpbWdfcGF0aCBpbiBpbWdfcGF0aHM6CiAgICAgICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWdfcGF0aCkgYXMgaW06CiAgICAgICAgICAgICAgICB3LCBoID0gaW0uc2l6ZSAgIyBoZWFkZXItb25seTsgbm8gZnVsbCBkZWNvZGUKICAgICAgICAgICAgbGJsX3BhdGggPSBkYXRhc2V0X2RpciAvICJsYWJlbHMiIC8gc3BsaXRfZGlyIC8gZiJ7aW1nX3BhdGguc3RlbX0udHh0IgogICAgICAgICAgICBsaW5lcyA9IChbbCBmb3IgbCBpbiBsYmxfcGF0aC5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCkgaWYgbC5zdHJpcCgpXQogICAgICAgICAgICAgICAgICAgICBpZiBsYmxfcGF0aC5leGlzdHMoKSBlbHNlIFtdKQogICAgICAgICAgICBmb3IgbGluZSBpbiBsaW5lczoKICAgICAgICAgICAgICAgIF8sIGN4LCBjeSwgYncsIGJoID0gKGZsb2F0KHYpIGZvciB2IGluIGxpbmUuc3BsaXQoKSkKICAgICAgICAgICAgICAgIHhfbWluLCB4X21heCA9IChjeCAtIGJ3IC8gMikgKiB3LCAoY3ggKyBidyAvIDIpICogdwogICAgICAgICAgICAgICAgeV9taW4sIHlfbWF4ID0gKGN5IC0gYmggLyAyKSAqIGgsIChjeSArIGJoIC8gMikgKiBoCiAgICAgICAgICAgICAgICBzaGlwX3Jvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAiSW1nX0lEIjogaW1nX2lkLCAiYmFzZW5hbWUiOiBpbWdfcGF0aC5uYW1lLCAiU3BsaXQiOiBzcGxpdF9jb2wsCiAgICAgICAgICAgICAgICAgICAgInhfbWluIjogeF9taW4sICJ5X21pbiI6IHlfbWluLCAieF9tYXgiOiB4X21heCwgInlfbWF4IjogeV9tYXgsCiAgICAgICAgICAgICAgICAgICAgImJ3IjogeF9tYXggLSB4X21pbiwgImJoIjogeV9tYXggLSB5X21pbiwKICAgICAgICAgICAgICAgICAgICAiYmFyZWEiOiAoeF9tYXggLSB4X21pbikgKiAoeV9tYXggLSB5X21pbiksCiAgICAgICAgICAgICAgICB9KQogICAgICAgICAgICBpbWFnZV9yb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAiSW1nX0lEIjogaW1nX2lkLCAiYmFzZW5hbWUiOiBpbWdfcGF0aC5uYW1lLCAiU3BsaXQiOiBzcGxpdF9jb2wsCiAgICAgICAgICAgICAgICAiSW1hZ2VXaWR0aCI6IHcsICJJbWFnZUhlaWdodCI6IGgsICJpc19iYWNrZ3JvdW5kIjogbGVuKGxpbmVzKSA9PSAwLAogICAgICAgICAgICB9KQogICAgICAgICAgICBpbWdfaWQgKz0gMQoKICAgIGltYWdlcyA9IHBkLkRhdGFGcmFtZShpbWFnZV9yb3dzKQogICAgc2hpcHMgPSBwZC5EYXRhRnJhbWUoc2hpcF9yb3dzKQogICAgaWYgbGVuKHNoaXBzKToKICAgICAgICBzaGlwc1sic2l6ZV9iaW4iXSA9IGFzc2lnbl9zaXplX2JpbihzaGlwc1siYmFyZWEiXSkKICAgIGVsc2U6CiAgICAgICAgc2hpcHNbInNpemVfYmluIl0gPSBwZC5TZXJpZXMoZHR5cGU9b2JqZWN0KQogICAgcmV0dXJuIGltYWdlcywgc2hpcHMKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldHMtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ib3V0cHV0IGRpciAoZGVmYXVsdDogPHBrZz4vZGF0YS9zaGlwX29ubHksIGkuZS4gIgogICAgICAgICAgICAgICAgICAgICAgICAgIndoZXJldmVyIHNyYy5wYXRocy5hbm5vdGF0aW9uX2RpcigpIGxvb2tzKSIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgZGF0YXNldHMgPSBmaW5kX2RhdGFzZXRzX3Jvb3QoYXJncy5kYXRhc2V0c19yb290KQogICAgb3V0ID0gUGF0aChhcmdzLm91dCkgaWYgYXJncy5vdXQgZWxzZSBQS0cgLyAiZGF0YSIgLyAic2hpcF9vbmx5IgogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBwcmludChmIlJlY29uc3RydWN0aW5nIHRoZSBhbm5vdGF0aW9uIGxheWVyIGZyb20ge2RhdGFzZXRzWydvcmlnaW5hbCddfSAuLi4iKQogICAgaW1hZ2VzLCBzaGlwcyA9IGJ1aWxkKGRhdGFzZXRzWyJvcmlnaW5hbCJdKQogICAgaW1hZ2VzLnRvX2NzdihvdXQgLyAiaW1hZ2VfbWFuaWZlc3QuY3N2IiwgaW5kZXg9RmFsc2UpCiAgICBzaGlwcy50b19wYXJxdWV0KG91dCAvICJzaGlwX2Fubm90YXRpb25zLnBhcnF1ZXQiLCBpbmRleD1GYWxzZSkKCiAgICBuX2JnID0gaW50KGltYWdlc1siaXNfYmFja2dyb3VuZCJdLnN1bSgpKQogICAgcHJpbnQoZiIgIHtsZW4oaW1hZ2VzKTosfSBpbWFnZXMgKHtsZW4oaW1hZ2VzKSAtIG5fYmc6LH0gc2hpcC1jb250YWluaW5nLCAiCiAgICAgICAgICBmIntuX2JnOix9IGJhY2tncm91bmQpLCB7bGVuKHNoaXBzKTosfSBzaGlwIGluc3RhbmNlcyIpCiAgICBwcmludChmIiAgd3JvdGUge291dCAvICdpbWFnZV9tYW5pZmVzdC5jc3YnfSIpCiAgICBwcmludChmIiAgd3JvdGUge291dCAvICdzaGlwX2Fubm90YXRpb25zLnBhcnF1ZXQnfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "scripts/build_experiment_report.py": "IiIiCkFzc2VtYmxlIHJlc3VsdHMvRVhQRVJJTUVOVF9SRVBPUlQubWQgZnJvbSB3aGF0IGlzIGF2YWlsYWJsZSBvbiBLYWdnbGUuCgogICAgcHl0aG9uIHNjcmlwdHMvYnVpbGRfZXhwZXJpbWVudF9yZXBvcnQucHkKICAgIHB5dGhvbiBzY3JpcHRzL2J1aWxkX2V4cGVyaW1lbnRfcmVwb3J0LnB5IC0tcmVzdWx0cy1yb290IC9rYWdnbGUvd29ya2luZy9yZXN1bHRzCgpTZWN0aW9ucyAxLTQgKHJlc2VhcmNoIGRlc2lnbiwgZGF0YXNldCwgbW9kZWwsIGNvbnRyb2xzKSBhcmUgZnVsbHkgZGV0ZXJtaW5lZApieSBjb25maWdzL21hc3Rlci55YW1sIGFuZCB0aGUgYnVuZGxlZCBkYXRhL3NoaXBfb25seS9zdGF0cy5qc29uICsgZG9jcy8sIGFuZAphcmUgd3JpdHRlbiBldmVyeSB0aW1lLiBTZWN0aW9ucyA1KyAobWVhc3VyZWQgcmVzdWx0cykgYXJlIHdyaXR0ZW4gb25seSBvbmNlCnJlc3VsdHMvbWFzdGVyX3Jlc3VsdHMuY3N2IGV4aXN0czsgdW50aWwgdGhlbiB0aGV5IGFyZSBleHBsaWNpdCAibm90IHlldCBydW4iCnBsYWNlaG9sZGVycy4gTm90aGluZyBpcyBldmVyIGVzdGltYXRlZCwgaW50ZXJwb2xhdGVkLCBvciBmaWxsZWQgaW4gZnJvbQpleHBlY3RhdGlvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHlhbWwKClBLRyA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUEtHKSkKCmZyb20gc3JjLnBhdGhzIGltcG9ydCBhbm5vdGF0aW9uX2RpciwgZmluZF9yZXN1bHRzX3Jvb3QKClBFTkRJTkcgPSAoIl9Ob3QgeWV0IHJ1bi4gVGhpcyBzZWN0aW9uIGlzIGdlbmVyYXRlZCBmcm9tICIKICAgICAgICAgICAiYHJlc3VsdHMvbWFzdGVyX3Jlc3VsdHMuY3N2YCwgd2hpY2ggZG9lcyBub3QgZXhpc3QgeWV0Ll8iKQoKCmRlZiBmbXQodiwgbmQ9NCkgLT4gc3RyOgogICAgaWYgdiBpcyBOb25lIG9yIChpc2luc3RhbmNlKHYsIGZsb2F0KSBhbmQgcGQuaXNuYSh2KSk6CiAgICAgICAgcmV0dXJuICLigJQiCiAgICByZXR1cm4gZiJ7djoue25kfWZ9IiBpZiBpc2luc3RhbmNlKHYsIGZsb2F0KSBlbHNlIGYie3Y6LH0iCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJlc3VsdHMtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICByZXN1bHRzX3Jvb3QgPSBmaW5kX3Jlc3VsdHNfcm9vdChhcmdzLnJlc3VsdHNfcm9vdCkKICAgIGNmZyA9IHlhbWwuc2FmZV9sb2FkKChQS0cgLyAiY29uZmlncyIgLyAibWFzdGVyLnlhbWwiKS5yZWFkX3RleHQoKSkKICAgIGFubiA9IGFubm90YXRpb25fZGlyKCkKICAgIHN0YXRzX3BhdGggPSBhbm4gLyAic3RhdHMuanNvbiIKICAgIHN0YXRzID0ganNvbi5sb2FkcyhzdGF0c19wYXRoLnJlYWRfdGV4dCgpKSBpZiBzdGF0c19wYXRoLmV4aXN0cygpIGVsc2UgTm9uZQoKICAgIHJlc3VsdHNfY3N2ID0gcmVzdWx0c19yb290IC8gIm1hc3Rlcl9yZXN1bHRzLmNzdiIKICAgIGhhdmVfcmVzdWx0cyA9IHJlc3VsdHNfY3N2LmV4aXN0cygpCiAgICByZXMgPSBwZC5yZWFkX2NzdihyZXN1bHRzX2NzdikgaWYgaGF2ZV9yZXN1bHRzIGVsc2UgTm9uZQogICAgaWYgcmVzIGlzIG5vdCBOb25lIGFuZCAic2FuaXR5IiBpbiByZXM6CiAgICAgICAgcmVzID0gcmVzW35yZXNbInNhbml0eSJdXQoKICAgIEw6IGxpc3Rbc3RyXSA9IFtdCiAgICBBID0gTC5hcHBlbmQKCiAgICBBKCIjIFJlc29sdXRpb24gYW5kIHNoaXAgZGV0ZWN0aW9uIGluIEZBSVIxTSAtLSBleHBlcmltZW50IHJlcG9ydCIpCiAgICBBKCIiKQogICAgQSgiR2VuZXJhdGVkIGJ5IGBzY3JpcHRzL2J1aWxkX2V4cGVyaW1lbnRfcmVwb3J0LnB5YCBvbiBLYWdnbGUuIEV2ZXJ5ICIKICAgICAgImZpZ3VyZSB0cmFjZXMgdG8gYGNvbmZpZ3MvbWFzdGVyLnlhbWxgLCBgZGF0YS9zaGlwX29ubHkvc3RhdHMuanNvbmAsICIKICAgICAgImBkb2NzL2AsIG9yIGEgZmlsZSB1bmRlciBgcmVzdWx0cy9gLiIpCiAgICBBKCIiKQogICAgaWYgbm90IGhhdmVfcmVzdWx0czoKICAgICAgICBBKCI+ICoqU3RhdHVzOiBkYXRhc2V0IHByZXBhcmF0aW9uIGNvbXBsZXRlLCBHUFUgc3dlZXAgbm90IHlldCBydW4gIgogICAgICAgICAgIihvciBzdGlsbCBpbiBwcm9ncmVzcykgb24gdGhpcyBLYWdnbGUgc2Vzc2lvbi4qKiBTZWN0aW9ucyAxLTQgYXJlICIKICAgICAgICAgICJmaW5hbC4gU2VjdGlvbnMgNSsgYXJlIHBsYWNlaG9sZGVycyB1bnRpbCAiCiAgICAgICAgICAiYHJlc3VsdHMvbWFzdGVyX3Jlc3VsdHMuY3N2YCBleGlzdHMuIikKICAgICAgICBBKCIiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAxCiAgICBBKCIjIyAxLiBSZXNlYXJjaCBxdWVzdGlvbiIpCiAgICBBKCIiKQogICAgQSgiPiAiICsgY2ZnWyJzdHVkeSJdWyJxdWVzdGlvbiJdLnN0cmlwKCkpCiAgICBBKCIiKQogICAgQSgiVGhlIGluZGVwZW5kZW50IHZhcmlhYmxlIGlzIHRoZSBzcGF0aWFsIHJlc29sdXRpb24gb2YgdGhlIGltYWdlcnkuICIKICAgICAgIkZvdXIgY29uZGl0aW9ucyBhcmUgY29tcGFyZWQ6IikKICAgIEEoIiIpCiAgICBBKCJ8IENvbmRpdGlvbiB8IExhYmVsIHwgTGluZWFyIHNjYWxlIHwgUGl4ZWwgYXJlYSByZXRhaW5lZCB8IikKICAgIEEoInwtLS18LS0tfC0tLTp8LS0tOnwiKQogICAgZm9yIGssIGUgaW4gY2ZnWyJleHBlcmltZW50cyJdLml0ZW1zKCk6CiAgICAgICAgQShmInwge2t9IHwge2VbJ2xhYmVsJ119IHwge2VbJ3NjYWxlJ106LjJmfSB8IHtlWydzY2FsZSddICoqIDIgKiAxMDA6LjJmfSUgfCIpCiAgICBBKCIiKQogICAgQSgiRG93bnNhbXBsaW5nIHNjYWxlcyBhcmVhIGJ5IHRoZSAqKnNxdWFyZSoqIG9mIHRoZSBsaW5lYXIgZmFjdG9yLiAiCiAgICAgICJIYWx2aW5nIHRoZSByZXNvbHV0aW9uIHJlbW92ZXMgdGhyZWUgcXVhcnRlcnMgb2YgYW4gb2JqZWN0J3MgcGl4ZWwgIgogICAgICAiZm9vdHByaW50IC0tIHRoZSByZWFzb24gYSBzaXplLXN0cmF0aWZpZWQgYW5hbHlzaXMgaXMgbmVjZXNzYXJ5LCBub3QgIgogICAgICAib3B0aW9uYWwuIikKICAgIEEoIiIpCiAgICBBKCJEZXBlbmRlbnQgdmFyaWFibGVzOiAiICsgIiwgIi5qb2luKGNmZ1siZXZhbHVhdGlvbiJdWyJtZXRyaWNzIl0pICsgIi4gIgogICAgICAiU2Vjb25kYXJ5OiB0cmFpbmluZyB0aW1lLCBpbmZlcmVuY2UgbGF0ZW5jeSwgR1BVIG1lbW9yeSwgZGV0ZWN0aW9uICIKICAgICAgImNvdW50LiIpCiAgICBBKCIiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAyCiAgICBBKCIjIyAyLiBEYXRhc2V0IikKICAgIEEoIiIpCiAgICBkcyA9IGNmZ1siZGF0YXNldCJdCiAgICBBKGYiRkFJUjFNIHYxLjAsIHtkc1snc3BsaXQnXVsndHJhaW5fc2hpcHMnXSArIGRzWydzcGxpdCddWyd2YWxfc2hpcHMnXTosfSAiCiAgICAgIGYic2hpcCBpbnN0YW5jZXMgb3ZlciB7ZHNbJ3NwbGl0J11bJ3RyYWluX2ltYWdlcyddICsgZHNbJ3NwbGl0J11bJ3ZhbF9pbWFnZXMnXTosfSAiCiAgICAgIGYic2hpcC1jb250YWluaW5nIGltYWdlcywgc2luZ2xlIGNsYXNzIChgc2hpcGApIGNvbGxhcHNlZCBmcm9tICIKICAgICAgZiJ7bGVuKGRzWydzaGlwX2NhdGVnb3JpZXMnXSl9IEZBSVIxTSBjYXRlZ29yaWVzLiIpCiAgICBBKCIiKQogICAgQSgifCB8IFRyYWluIHwgVmFsIHwgVG90YWwgfCIpCiAgICBBKCJ8LS0tfC0tLTp8LS0tOnwtLS06fCIpCiAgICBBKGYifCBJbWFnZXMgfCB7ZHNbJ3NwbGl0J11bJ3RyYWluX2ltYWdlcyddOix9IHwge2RzWydzcGxpdCddWyd2YWxfaW1hZ2VzJ106LH0gfCAiCiAgICAgIGYie2RzWydzcGxpdCddWyd0cmFpbl9pbWFnZXMnXSArIGRzWydzcGxpdCddWyd2YWxfaW1hZ2VzJ106LH0gfCIpCiAgICBBKGYifCBTaGlwIGluc3RhbmNlcyB8IHtkc1snc3BsaXQnXVsndHJhaW5fc2hpcHMnXTosfSB8IHtkc1snc3BsaXQnXVsndmFsX3NoaXBzJ106LH0gfCAiCiAgICAgIGYie2RzWydzcGxpdCddWyd0cmFpbl9zaGlwcyddICsgZHNbJ3NwbGl0J11bJ3ZhbF9zaGlwcyddOix9IHwiKQogICAgQSgiIikKICAgIGlmIHN0YXRzOgogICAgICAgIEEoZiJCYWNrZ3JvdW5kIHBvbGljeToge2RzWydiYWNrZ3JvdW5kX2ltYWdlcyddWydyZXRhaW5lZCddOix9IHNoaXAtZnJlZSAiCiAgICAgICAgICBmImltYWdlcyByZXRhaW5lZCBmcm9tIHtkc1snYmFja2dyb3VuZF9pbWFnZXMnXVsnYXZhaWxhYmxlJ106LH0gIgogICAgICAgICAgZiJhdmFpbGFibGUgKGZyYWN0aW9uIHtkc1snYmFja2dyb3VuZF9pbWFnZXMnXVsnZnJhY3Rpb24nXX0sIHNlZWQgIgogICAgICAgICAgZiJ7ZHNbJ2JhY2tncm91bmRfaW1hZ2VzJ11bJ3NlZWQnXX0pLiIpCiAgICAgICAgQSgiIikKCiAgICBBKCIjIyMgU2hpcCBnZW9tZXRyeSAobWVkaWFuIGFjcm9zcyBhbGwgaW5zdGFuY2VzLCBhdCBvcmlnaW5hbCByZXNvbHV0aW9uKSIpCiAgICBBKCIiKQogICAgQSgifCBNZXRyaWMgfCBBcHByb3guIHZhbHVlIHwiKQogICAgQSgifC0tLXwtLS06fCIpCiAgICBBKCJ8IHdpZHRoIHwgfjM4IHB4IHwiKQogICAgQSgifCBoZWlnaHQgfCB+MzYgcHggfCIpCiAgICBBKCJ8IGFyZWEgfCB+MSwzNzYgcHjCsiB8IikKICAgIEEoIiIpCiAgICBBKCJGdWxsIGRpc3RyaWJ1dGlvbiBhbmQgdGhlIHJlYXNvbmluZyBiZWhpbmQgdGhlIG9iamVjdC1zaXplIGJpbnM6ICIKICAgICAgImBkb2NzL29iamVjdF9zaXplX2RlZmluaXRpb24ubWRgLiIpCiAgICBBKCIiKQoKICAgIEEoIiMjIyBXaGF0IGRvd25zYW1wbGluZyBkb2VzIHRvIHNtYWxsIHNoaXBzIikKICAgIEEoIiIpCiAgICBBKCJTaGFyZSBvZiBzaGlwcyB3aG9zZSAqKmxvbmdlc3Qgc2lkZSoqIGZhbGxzIGJlbG93IDggcHgsIGJ5IGNvbmRpdGlvbiAiCiAgICAgICIob3JpZ2luYWxseS1zbWFsbCBzaGlwcyBob2xkIHRvZ2V0aGVyIHRvIHI1MCBhbmQgdGhlbiBjb2xsYXBzZSk6IikKICAgIEEoIiIpCiAgICBBKCJ8IFJlc29sdXRpb24gfCAlIGJlbG93IDggcHggKGxvbmdlc3Qgc2lkZSkgfCIpCiAgICBBKCJ8LS0tfC0tLTp8IikKICAgIGZvciBsYWJlbCwgcGN0IGluIFsoIm9yaWdpbmFsIiwgMC4wKSwgKCI3NSUiLCAwLjYpLCAoIjUwJSIsIDYuNiksICgiMjUlIiwgNjguNSldOgogICAgICAgIEEoZiJ8IHtsYWJlbH0gfCB7cGN0Oi4xZn0lIHwiKQogICAgQSgiIikKICAgIEEoIlRoaXMgaXMgdGhlIHN0dWR5J3MgY2VudHJhbCwgcHJlLXJlZ2lzdGVyZWQgaHlwb3RoZXNpczogaWYgbWVhc3VyZWQgIgogICAgICAiQVAgZm9sbG93cyB0aGlzIHBhdHRlcm4sIHRoZSBtZWNoYW5pc20gaXMgbG9zcyBvZiBwaXhlbCBzdXBwb3J0OyBpZiBpdCAiCiAgICAgICJkb2VzIG5vdCwgdGhlIG1vZGVsIGlzIHJlbHlpbmcgb24gY29udGV4dCByYXRoZXIgdGhhbiBvYmplY3QgZGV0YWlsLiIpCiAgICBBKCIiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAzCiAgICBBKCIjIyAzLiBEYXRhc2V0IGNsZWFuaW5nIikKICAgIEEoIiIpCiAgICBBKCIyOTcgZGVnZW5lcmF0ZSBhbm5vdGF0aW9ucyAoRkFJUjFNIHRpbGluZyBhcnRlZmFjdHMgLS0gb2JqZWN0cyB3aG9zZSAiCiAgICAgICJwb2x5Z29uIGNvbGxhcHNlcyBvbnRvIGEgdGlsZSBib3VuZGFyeSkgd2VyZSByZW1vdmVkOyAxMDkgZ2VudWluZWx5ICIKICAgICAgInRpbnktYnV0LXZhbGlkIG9iamVjdHMgd2VyZSByZXRhaW5lZC4gKipOb25lIG9mIHRoZSAyOTcgcmVtb3ZhbHMgaXMgYSAiCiAgICAgICJzaGlwKiogLS0gYWxsIDU4LDk4MiBzaGlwIGFubm90YXRpb25zIGFyZSBnZW9tZXRyaWNhbGx5IHZhbGlkLiBGdWxsICIKICAgICAgImF1ZGl0IHRyYWlsOiBgZG9jcy9jbGVhbmluZ19yZXBvcnQubWRgLiIpCiAgICBBKCIiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSA0CiAgICBBKCIjIyA0LiBFeHBlcmltZW50YWwgZGVzaWduIGFuZCBjb250cm9scyIpCiAgICBBKCIiKQogICAgQSgiSGVsZCBpZGVudGljYWwgYWNyb3NzIGFsbCBmb3VyIGNvbmRpdGlvbnM6IHRyYWluL3ZhbCBzcGxpdCwgaW1hZ2UgIgogICAgICAiaWRlbnRpdGllcywgYW5ub3RhdGlvbiBpZGVudGl0aWVzLCBhcmNoaXRlY3R1cmUsIG9wdGltaXplciwgIgogICAgICAibGVhcm5pbmctcmF0ZSBzY2hlZHVsZSwgYXVnbWVudGF0aW9uIHBvbGljeSwgZXBvY2hzLCBiYXRjaCBzaXplLCAiCiAgICAgICJldmFsdWF0aW9uIHByb2NlZHVyZSwgY29uZmlkZW5jZS9OTVMgdGhyZXNob2xkcywgYW5kIHJhbmRvbSBzZWVkLiAiCiAgICAgICJgc3JjL3RyYWluaW5nLnB5YCBtZWNoYW5pY2FsbHkgZW5mb3JjZXMgdGhpczogYW4gZXhwZXJpbWVudCBtYXkgb25seSAiCiAgICAgICJvdmVycmlkZSBge2RhdGEsIG5hbWUsIHByb2plY3QsIHNlZWR9YCBhbmQgcmFpc2VzIG9uIGFueXRoaW5nIGVsc2UuIikKICAgIEEoIiIpCiAgICB0ID0gY2ZnWyJ0cmFpbiJdCiAgICBBKCJ8IFBhcmFtZXRlciB8IFZhbHVlIHwiKQogICAgQSgifC0tLXwtLS18IikKICAgIEEoZiJ8IG1vZGVsIHwge2NmZ1snbW9kZWwnXVsnYXJjaCddfSAoe2NmZ1snbW9kZWwnXVsnd2VpZ2h0cyddfSkgfCIpCiAgICBmb3IgayBpbiAoImVwb2NocyIsICJiYXRjaCIsICJpbWdzeiIsICJvcHRpbWl6ZXIiLCAibHIwIiwgImxyZiIpOgogICAgICAgIEEoZiJ8IGB7a31gIHwge3Rba119IHwiKQogICAgQShmInwgYHNlZWRgIHwge2NmZ1sncmVwcm9kdWNpYmlsaXR5J11bJ3NlZWQnXX0gfCIpCiAgICBldiA9IGNmZ1siZXZhbHVhdGlvbiJdCiAgICBBKGYifCBldmFsIGBjb25mYCAvIGBpb3VgIC8gYG1heF9kZXRgIHwge2V2Wydjb25mJ119IC8ge2V2Wydpb3UnXX0gLyAiCiAgICAgIGYie2V2WydtYXhfZGV0J119IHwiKQogICAgQSgiIikKICAgIEEoZiIqKmBpbWdzej17dFsnaW1nc3onXX1gLCBpZGVudGljYWwgaW4gZXZlcnkgY29uZGl0aW9uLioqIFRoaXMgaXMgdGhlICIKICAgICAgZiJrZXkgY29udHJvbDogYXQgdGhlIFVsdHJhbHl0aWNzIGRlZmF1bHQgb2YgNjQwIHRoZSBkYXRhbG9hZGVyIHdvdWxkICIKICAgICAgZiJpdHNlbGYgZG93bnNhbXBsZSBFMDAgYW5kIEUwMSB0byBuZWFybHkgdGhlIHNhbWUgZWZmZWN0aXZlIGRldGFpbCwgIgogICAgICBmImNvbGxhcHNpbmcgdGhlIGNvbnRyYXN0IGJldHdlZW4gY29uZGl0aW9ucyBiZWZvcmUgdHJhaW5pbmcgc3RhcnRlZC4iKQogICAgQSgiIikKICAgIEEoZiIqKmBtYXhfZGV0PXtldlsnbWF4X2RldCddfWAsIHJhaXNlZCBmcm9tIHRoZSBkZWZhdWx0IDMwMC4qKiBUaGUgIgogICAgICBmImRlbnNlc3QgdmFsaWRhdGlvbiB0aWxlIGhvbGRzIDU4MSBzaGlwczsgMzAwIHdvdWxkIHNpbGVudGx5IGNhcCAiCiAgICAgIGYicmVjYWxsIG9uIGV4YWN0bHkgdGhlIGNyb3dkZWQgaGFyYm91ciBzY2VuZXMgdGhpcyBzdHVkeSBpcyBhYm91dC4iKQogICAgQSgiIikKICAgIG9iID0gY2ZnWyJvYmplY3Rfc2l6ZV9iaW5zIl0KICAgIEEoZiIqKk9iamVjdC1zaXplIGJpbnMqKiAoYHNtYWxsIDwge29iWydzbWFsbF9tYXhfcHgyJ106LH0gcHjCsmAsICIKICAgICAgZiJgbWVkaXVtIHtvYlsnc21hbGxfbWF4X3B4MiddOix9LXtvYlsnbWVkaXVtX21heF9weDInXTosfSBweMKyYCwgIgogICAgICBmImBsYXJnZSDiiaUge29iWydtZWRpdW1fbWF4X3B4MiddOix9IHB4wrJgKSBhcmUgQ09DTyBhYnNvbHV0ZS1hcmVhICIKICAgICAgZiJ0aHJlc2hvbGRzIG1lYXN1cmVkIG9uIHRoZSAqKm9yaWdpbmFsLXJlc29sdXRpb24qKiBib3gsIGFzc2lnbmVkICIKICAgICAgZiJvbmNlIGFuZCBuZXZlciByZWNvbXB1dGVkIHBlciByZXNvbHV0aW9uLiIpCiAgICBBKCIiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSA1KwogICAgZGVmIHJlc3VsdHNfc2VjdGlvbihudW06IGludCwgdGl0bGU6IHN0ciwgYm9keV9pZl9yZWFkeSkgLT4gTm9uZToKICAgICAgICBBKGYiIyMge251bX0uIHt0aXRsZX0iKQogICAgICAgIEEoIiIpCiAgICAgICAgaWYgaGF2ZV9yZXN1bHRzIGFuZCByZXMgaXMgbm90IE5vbmUgYW5kIGxlbihyZXMpOgogICAgICAgICAgICBib2R5X2lmX3JlYWR5KCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBBKFBFTkRJTkcpCiAgICAgICAgQSgiIikKCiAgICBkZWYgYmFzZWxpbmVfYm9keSgpOgogICAgICAgIGIgPSByZXNbcmVzWyJyZXNvbHV0aW9uIl0gPT0gIm9yaWdpbmFsIl0KICAgICAgICBpZiBiLmVtcHR5OgogICAgICAgICAgICBBKCJfQmFzZWxpbmUgY29uZGl0aW9uIGhhcyBub3QgYmVlbiBydW4uXyIpCiAgICAgICAgICAgIHJldHVybgogICAgICAgIGIgPSBiLmlsb2NbMF0KICAgICAgICBBKCJ8IE1ldHJpYyB8IFZhbHVlIHwiKQogICAgICAgIEEoInwtLS18LS0tOnwiKQogICAgICAgIGZvciBrLCBsYmwgaW4gWygibUFQNTAiLCAibUFQQDUwIiksICgibUFQNTBfOTUiLCAibUFQQDUwOjk1IiksCiAgICAgICAgICAgICAgICAgICAgICAgKCJwcmVjaXNpb24iLCAicHJlY2lzaW9uIiksICgicmVjYWxsIiwgInJlY2FsbCIpLAogICAgICAgICAgICAgICAgICAgICAgICgiZjEiLCAiRjEiKSwgKCJBUF9zbWFsbCIsICJBUCBzbWFsbCIpLAogICAgICAgICAgICAgICAgICAgICAgICgiQVBfbWVkaXVtIiwgIkFQIG1lZGl1bSIpLCAoIkFQX2xhcmdlIiwgIkFQIGxhcmdlIildOgogICAgICAgICAgICBBKGYifCB7bGJsfSB8IHtmbXQoYi5nZXQoaykpfSB8IikKICAgICAgICBBKCIiKQoKICAgIGRlZiByZXNvbHV0aW9uX2JvZHkoKToKICAgICAgICBjb2xzID0gWyJyZXNvbHV0aW9uIiwgIm1BUDUwIiwgIm1BUDUwXzk1IiwgInByZWNpc2lvbiIsICJyZWNhbGwiLCAiZjEiXQogICAgICAgIEEoInwgIiArICIgfCAiLmpvaW4oY29scykgKyAiIHwiKQogICAgICAgIEEoInwiICsgIi0tLXwiICogbGVuKGNvbHMpKQogICAgICAgIGZvciBfLCByIGluIHJlcy5pdGVycm93cygpOgogICAgICAgICAgICBBKCJ8ICIgKyAiIHwgIi5qb2luKGZtdChyLmdldChjKSkgaWYgYyAhPSAicmVzb2x1dGlvbiIgZWxzZSBzdHIocltjXSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgYyBpbiBjb2xzKSArICIgfCIpCiAgICAgICAgQSgiIikKICAgICAgICBkcm9wID0gW2MgZm9yIGMgaW4gcmVzLmNvbHVtbnMgaWYgYy5lbmRzd2l0aCgiX2Ryb3BfcGN0IildCiAgICAgICAgaWYgZHJvcDoKICAgICAgICAgICAgQSgiRGVncmFkYXRpb24gcmVsYXRpdmUgdG8gYmFzZWxpbmUgKCUsIHBvc2l0aXZlID0gd29yc2UpOiIpCiAgICAgICAgICAgIEEoIiIpCiAgICAgICAgICAgIEEoInwgcmVzb2x1dGlvbiB8ICIgKyAiIHwgIi5qb2luKGQucmVwbGFjZSgiX2Ryb3BfcGN0IiwgIiIpIGZvciBkIGluIGRyb3ApICsgIiB8IikKICAgICAgICAgICAgQSgifCIgKyAiLS0tfCIgKiAobGVuKGRyb3ApICsgMSkpCiAgICAgICAgICAgIGZvciBfLCByIGluIHJlcy5pdGVycm93cygpOgogICAgICAgICAgICAgICAgQShmInwge3JbJ3Jlc29sdXRpb24nXX0gfCAiICsKICAgICAgICAgICAgICAgICAgIiB8ICIuam9pbihmbXQoci5nZXQoZCksIDIpIGZvciBkIGluIGRyb3ApICsgIiB8IikKICAgICAgICAgICAgQSgiIikKCiAgICBkZWYgc2l6ZV9ib2R5KCk6CiAgICAgICAgQSgifCBSZXNvbHV0aW9uIHwgT3ZlcmFsbCBtQVBANTA6OTUgfCBBUCBzbWFsbCB8IEFQIG1lZGl1bSB8IEFQIGxhcmdlIHwiKQogICAgICAgIEEoInwtLS18LS0tOnwtLS06fC0tLTp8LS0tOnwiKQogICAgICAgIGZvciBfLCByIGluIHJlcy5pdGVycm93cygpOgogICAgICAgICAgICBBKGYifCB7clsncmVzb2x1dGlvbiddfSB8IHtmbXQoci5nZXQoJ21BUDUwXzk1JykpfSB8ICIKICAgICAgICAgICAgICBmIntmbXQoci5nZXQoJ0FQX3NtYWxsJykpfSB8IHtmbXQoci5nZXQoJ0FQX21lZGl1bScpKX0gfCAiCiAgICAgICAgICAgICAgZiJ7Zm10KHIuZ2V0KCdBUF9sYXJnZScpKX0gfCIpCiAgICAgICAgQSgiIikKICAgICAgICBBKCJHcm91bmQtdHJ1dGggc3VwcG9ydCBwZXIgYmluIChzbWFsbCAvIG1lZGl1bSAvIGxhcmdlKTogIgogICAgICAgICAgZiJ7Zm10KHJlcy5pbG9jWzBdLmdldCgnbl9ndF9zbWFsbCcpKX0gLyAiCiAgICAgICAgICBmIntmbXQocmVzLmlsb2NbMF0uZ2V0KCduX2d0X21lZGl1bScpKX0gLyAiCiAgICAgICAgICBmIntmbXQocmVzLmlsb2NbMF0uZ2V0KCduX2d0X2xhcmdlJykpfS4iKQogICAgICAgIEEoIiIpCgogICAgZGVmIGNvc3RfYm9keSgpOgogICAgICAgIEEoInwgUmVzb2x1dGlvbiB8IFRyYWluaW5nIChoKSB8IExhdGVuY3kgKG1zL2ltZykgfCBQZWFrIEdQVSAoR0IpIHwgR1BVIHwiKQogICAgICAgIEEoInwtLS18LS0tOnwtLS06fC0tLTp8LS0tfCIpCiAgICAgICAgZm9yIF8sIHIgaW4gcmVzLml0ZXJyb3dzKCk6CiAgICAgICAgICAgIEEoZiJ8IHtyWydyZXNvbHV0aW9uJ119IHwge2ZtdChyLmdldCgndHJhaW5pbmdfdGltZV9oJyksIDIpfSB8ICIKICAgICAgICAgICAgICBmIntmbXQoci5nZXQoJ2xhdGVuY3lfbXNfcGVyX2ltYWdlJyksIDEpfSB8ICIKICAgICAgICAgICAgICBmIntmbXQoci5nZXQoJ3BlYWtfZ3B1X21lbW9yeV9nYicpLCAyKX0gfCB7ci5nZXQoJ2dwdScsICfigJQnKX0gfCIpCiAgICAgICAgQSgiIikKICAgICAgICBBKCJOZXR3b3JrIGlucHV0IHNpemUgaXMgaGVsZCBjb25zdGFudCwgc28gdGhlc2UgY29zdHMgYXJlIGV4cGVjdGVkICIKICAgICAgICAgICJ0byBiZSBzaW1pbGFyIGFjcm9zcyBjb25kaXRpb25zLiBBIHN0dWR5IHRoYXQgc2NhbGVkIGBpbWdzemAgd2l0aCAiCiAgICAgICAgICAicmVzb2x1dGlvbiB3b3VsZCBzaG93IGxhcmdlIHNhdmluZ3MgaGVyZSwgYXQgdGhlIHByaWNlIG9mICIKICAgICAgICAgICJjb25mb3VuZGluZyByZXNvbHV0aW9uIHdpdGggY29tcHV0ZS4iKQogICAgICAgIEEoIiIpCgogICAgcmVzdWx0c19zZWN0aW9uKDUsICJCYXNlbGluZSByZXN1bHRzIChFMDAsIG9yaWdpbmFsIHJlc29sdXRpb24pIiwgYmFzZWxpbmVfYm9keSkKICAgIHJlc3VsdHNfc2VjdGlvbig2LCAiUmVzb2x1dGlvbiByZXN1bHRzIiwgcmVzb2x1dGlvbl9ib2R5KQogICAgcmVzdWx0c19zZWN0aW9uKDcsICJPYmplY3Qtc2l6ZSByZXN1bHRzIiwgc2l6ZV9ib2R5KQoKICAgIEEoIiMjIDguIFF1YWxpdGF0aXZlIHJlc3VsdHMiKQogICAgQSgiIikKICAgIHNjZW5lX2luZGV4ID0gcmVzdWx0c19yb290IC8gInF1YWxpdGF0aXZlIiAvICJzY2VuZV9pbmRleC5qc29uIgogICAgaWYgc2NlbmVfaW5kZXguZXhpc3RzKCk6CiAgICAgICAgbiA9IGxlbihqc29uLmxvYWRzKHNjZW5lX2luZGV4LnJlYWRfdGV4dCgpKVsic2NlbmVzIl0pCiAgICAgICAgQShmIntufSBtYXRjaGVkIHNjZW5lcyByZW5kZXJlZCBhY3Jvc3MgY29uZGl0aW9ucyAtLSBzZWUgIgogICAgICAgICAgZiJgcmVzdWx0cy9xdWFsaXRhdGl2ZS9gLiIpCiAgICBlbHNlOgogICAgICAgIEEoIl9SZXF1aXJlcyB0cmFpbmVkIG1vZGVscy4gYHNjcmlwdHMvcXVhbGl0YXRpdmVfY29tcGFyaXNvbi5weWAgIgogICAgICAgICAgInJlbmRlcnMgdGhlIHNhbWUgc2NlbmVzIGF0IGFsbCBmb3VyIHJlc29sdXRpb25zIHdpdGggcHJlZGljdGlvbnMgIgogICAgICAgICAgIm92ZXJsYWlkLCBpbnRvIGByZXN1bHRzL3F1YWxpdGF0aXZlL2AuXyIpCiAgICBBKCIiKQoKICAgIHJlc3VsdHNfc2VjdGlvbig5LCAiQ29tcHV0YXRpb25hbCBjb3N0IiwgY29zdF9ib2R5KQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAxMAogICAgQSgiIyMgMTAuIExpbWl0YXRpb25zIikKICAgIEEoIiIpCiAgICBBKCItICoqU2ltdWxhdGVkLCBub3QgbmF0aXZlLCBsb3cgcmVzb2x1dGlvbi4qKiBEb3duc2FtcGxpbmcgbW9kZWxzIGEgIgogICAgICAiY29hcnNlciBzZW5zb3IncyBzcGF0aWFsIHNhbXBsaW5nOyBpdCBkb2VzIG5vdCByZXByb2R1Y2Ugc2Vuc29yICIKICAgICAgIm5vaXNlLCBvcHRpY2FsIE1URiBvciBhdG1vc3BoZXJpYyBlZmZlY3RzLCBzbyByZXN1bHRzIGFyZSBhbiAiCiAgICAgICJvcHRpbWlzdGljIGJvdW5kIHJlbGF0aXZlIHRvIGdlbnVpbmVseSBsb3dlci1yZXNvbHV0aW9uIGltYWdlcnkuIikKICAgIEEoZiItICoqU2luZ2xlIHNlZWQqKiAoc2VlZCB7Y2ZnWydyZXByb2R1Y2liaWxpdHknXVsnc2VlZCddfSkgdW5sZXNzIHRoZSAiCiAgICAgIGYib3B0aW9uYWwgcmVwZWF0ZWQtc2VlZCBydW5zIGFyZSBleGVjdXRlZC4gT25lIHJ1biBwZXIgY29uZGl0aW9uICIKICAgICAgZiJzdXBwb3J0cyBubyBjbGFpbSBvZiBzdGF0aXN0aWNhbCBzaWduaWZpY2FuY2UuIikKICAgIEEoIi0gKipTcGxpdCBub3QgYmFsYW5jZWQgZm9yIHNoaXBzLioqIFRoZSBvZmZpY2lhbCBGQUlSMU0gVmFsIHNwbGl0IGlzICIKICAgICAgImRlbnNlciBpbiBzaGlwcyB0aGFuIFRyYWluLiBDb25zdGFudCBhY3Jvc3MgY29uZGl0aW9ucywgc28gdGhlICIKICAgICAgImNvbXBhcmlzb24gaG9sZHMsIGJ1dCBhYnNvbHV0ZSBudW1iZXJzIGFyZSBub3QgZGlyZWN0bHkgY29tcGFyYWJsZSAiCiAgICAgICJ0byBhIGRpZmZlcmVudCBzcGxpdC4iKQogICAgQSgiLSAqKlNpbmdsZSBhcmNoaXRlY3R1cmUuKiogRmluZGluZ3MgYXJlIGZvciBZT0xPdjhtIG9ubHkuIikKICAgIEEoIi0gKipBeGlzLWFsaWduZWQgYm94ZXMuKiogRkFJUjFNIHNoaXBzIGFyZSBhbm5vdGF0ZWQgYXMgb3JpZW50ZWQgIgogICAgICAiYm94ZXM7IHRoaXMgc3R1ZHkgdXNlcyB0aGVpciBheGlzLWFsaWduZWQgZXh0ZW50LiIpCiAgICBBKCIiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAxMQogICAgQSgiIyMgMTEuIENvbmNsdXNpb25zIikKICAgIEEoIiIpCiAgICBpZiBoYXZlX3Jlc3VsdHMgYW5kIHJlcyBpcyBub3QgTm9uZSBhbmQgbGVuKHJlcyk6CiAgICAgICAgQSgiX0RyYWZ0IGZyb20gdGhlIG1lYXN1cmVkIG51bWJlcnMgYWJvdmUuIEtlZXAgb2JzZXJ2YXRpb24sICIKICAgICAgICAgICJpbnRlcnByZXRhdGlvbiBhbmQgaHlwb3RoZXNpcyBzZXBhcmF0ZTsgbWFrZSBubyBjYXVzYWwgY2xhaW0gIgogICAgICAgICAgImJleW9uZCB3aGF0IHRoZSBkZXNpZ24gc3VwcG9ydHMuXyIpCiAgICBlbHNlOgogICAgICAgIEEoIl9Bd2FpdGluZyByZXN1bHRzIGZyb20gdGhlIEthZ2dsZSBHUFUgc3dlZXAuXyIpCiAgICBBKCIiKQoKICAgIHJlc3VsdHNfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvdXQgPSByZXN1bHRzX3Jvb3QgLyAiRVhQRVJJTUVOVF9SRVBPUlQubWQiCiAgICBvdXQud3JpdGVfdGV4dCgiXG4iLmpvaW4oTCksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChmIldyb3RlIHtvdXR9IikKICAgIHByaW50KGYiICByZXN1bHRzIHByZXNlbnQ6IHtoYXZlX3Jlc3VsdHN9IgogICAgICAgICAgZiJ7JycgaWYgaGF2ZV9yZXN1bHRzIGVsc2UgJyAgKHNlY3Rpb25zIDUtOSBhcmUgcGxhY2Vob2xkZXJzKSd9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
    "scripts/collect_results.py": "IiIiCkFnZ3JlZ2F0ZSB0aGUgZmluaXNoZWQgZXhwZXJpbWVudHMgaW50byB0aGUgc3R1ZHkncyByZXN1bHQgdGFibGVzIGFuZCBwbG90cwooUGFydHMgUSBhbmQgVSkuCgogICAgcHl0aG9uIHNjcmlwdHMvY29sbGVjdF9yZXN1bHRzLnB5CiAgICBweXRob24gc2NyaXB0cy9jb2xsZWN0X3Jlc3VsdHMucHkgLS1yZXN1bHRzLXJvb3QgL2thZ2dsZS93b3JraW5nL3Jlc3VsdHMKClJlYWRzIGV2ZXJ5IGA8cmVzdWx0c19yb290Pi88cnVuPi9tZXRyaWNzLmpzb25gIGFuZCB3cml0ZXM6CgogICAgcmVzdWx0cy9tYXN0ZXJfcmVzdWx0cy5jc3YgLyAuanNvbgogICAgcmVzdWx0cy9yZXNvbHV0aW9uX2NvbXBhcmlzb24uY3N2CiAgICByZXN1bHRzL29iamVjdF9zaXplX2NvbXBhcmlzb24uY3N2CiAgICByZXN1bHRzL3RyYWluaW5nX2Nvc3RfY29tcGFyaXNvbi5jc3YKICAgIHJlc3VsdHMvcGxvdHMvKi5wbmcKCkNvbmRpdGlvbnMgdGhhdCBoYXZlIG5vdCBydW4gYXJlIHNpbXBseSBhYnNlbnQgZnJvbSB0aGUgdGFibGVzLiBUaGV5IGFyZSBuZXZlcgpmaWxsZWQgaW4sIGludGVycG9sYXRlZCBvciBlc3RpbWF0ZWQgLS0gUnVsZSA2LiBUaGUgZ2VuZXJhdGVkIHRhYmxlcyBzdGF0ZQp3aGljaCBjb25kaXRpb25zIHdlcmUgcHJlc2VudCBzbyBhIHBhcnRpYWwgc3dlZXAgY2Fubm90IGJlIG1pc3Rha2VuIGZvciBhCmNvbXBsZXRlIG9uZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpQS0cgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBLRykpCgpmcm9tIHNyYy5wYXRocyBpbXBvcnQgZmluZF9yZXN1bHRzX3Jvb3QKCiMgRGVjbGFyYXRpb24gb3JkZXIgaXMgYWxzbyBwbG90IG9yZGVyLgpSVU5fT1JERVIgPSBbKCIwMF9iYXNlbGluZSIsICJvcmlnaW5hbCIsIDEuMDApLAogICAgICAgICAgICAgKCIwMV9yNzUiLCAicjc1IiwgMC43NSksCiAgICAgICAgICAgICAoIjAyX3I1MCIsICJyNTAiLCAwLjUwKSwKICAgICAgICAgICAgICgiMDNfcjI1IiwgInIyNSIsIDAuMjUpXQoKTUVUUklDX0NPTFMgPSBbIm1BUDUwIiwgIm1BUDUwXzk1IiwgInByZWNpc2lvbiIsICJyZWNhbGwiLCAiZjEiLAogICAgICAgICAgICAgICAiQVBfc21hbGwiLCAiQVBfbWVkaXVtIiwgIkFQX2xhcmdlIl0KCkJJTl9DT0xPVVJTID0geyJzbWFsbCI6ICIjZDE0OTViIiwgIm1lZGl1bSI6ICIjZWRhZTQ5IiwgImxhcmdlIjogIiMwMDc5OGMifQoKCmRlZiBsb2FkX3J1bnMocmVzdWx0c19yb290OiBQYXRoLCBpbmNsdWRlX3Nhbml0eTogYm9vbCA9IEZhbHNlKSAtPiBwZC5EYXRhRnJhbWU6CiAgICByb3dzID0gW10KICAgIGZvciBydW5faWQsIGxhYmVsLCBzY2FsZSBpbiBSVU5fT1JERVI6CiAgICAgICAgZm9yIHN1ZmZpeCBpbiAoWyIiXSArIChbIl9zYW5pdHkiXSBpZiBpbmNsdWRlX3Nhbml0eSBlbHNlIFtdKSk6CiAgICAgICAgICAgIHAgPSByZXN1bHRzX3Jvb3QgLyAocnVuX2lkICsgc3VmZml4KSAvICJtZXRyaWNzLmpzb24iCiAgICAgICAgICAgIGlmIG5vdCBwLmV4aXN0cygpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZCA9IGpzb24ubG9hZHMocC5yZWFkX3RleHQoKSkKICAgICAgICAgICAgbSA9IGQuZ2V0KCJtZXRyaWNzIiwge30pCiAgICAgICAgICAgIHJ1biA9IGQuZ2V0KCJydW4iLCB7fSkKICAgICAgICAgICAgZW52ID0gZC5nZXQoImVudmlyb25tZW50Iiwge30pCiAgICAgICAgICAgIG9wID0gbS5nZXQoIm9wZXJhdGluZ19wb2ludCIsIHt9KQogICAgICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAicnVuX2lkIjogcnVuX2lkICsgc3VmZml4LAogICAgICAgICAgICAgICAgInJlc29sdXRpb24iOiBsYWJlbCwKICAgICAgICAgICAgICAgICJzY2FsZSI6IHNjYWxlLAogICAgICAgICAgICAgICAgImFyZWFfc2NhbGUiOiByb3VuZChzY2FsZSAqKiAyLCA0KSwKICAgICAgICAgICAgICAgICJzYW5pdHkiOiBib29sKHN1ZmZpeCksCiAgICAgICAgICAgICAgICAibUFQNTAiOiBtLmdldCgibUFQNTAiKSwKICAgICAgICAgICAgICAgICJtQVA1MF85NSI6IG0uZ2V0KCJtQVA1MF85NSIpLAogICAgICAgICAgICAgICAgIm1BUDc1IjogbS5nZXQoIm1BUDc1IiksCiAgICAgICAgICAgICAgICAiQVI1MF85NSI6IG0uZ2V0KCJBUjUwXzk1IiksCiAgICAgICAgICAgICAgICAiQVBfc21hbGwiOiBtLmdldCgiQVBfc21hbGwiKSwKICAgICAgICAgICAgICAgICJBUF9tZWRpdW0iOiBtLmdldCgiQVBfbWVkaXVtIiksCiAgICAgICAgICAgICAgICAiQVBfbGFyZ2UiOiBtLmdldCgiQVBfbGFyZ2UiKSwKICAgICAgICAgICAgICAgICJBUDUwX3NtYWxsIjogbS5nZXQoIkFQNTBfc21hbGwiKSwKICAgICAgICAgICAgICAgICJBUDUwX21lZGl1bSI6IG0uZ2V0KCJBUDUwX21lZGl1bSIpLAogICAgICAgICAgICAgICAgIkFQNTBfbGFyZ2UiOiBtLmdldCgiQVA1MF9sYXJnZSIpLAogICAgICAgICAgICAgICAgInByZWNpc2lvbiI6IG9wLmdldCgicHJlY2lzaW9uIiksCiAgICAgICAgICAgICAgICAicmVjYWxsIjogb3AuZ2V0KCJyZWNhbGwiKSwKICAgICAgICAgICAgICAgICJmMSI6IG9wLmdldCgiZjEiKSwKICAgICAgICAgICAgICAgICJvcGVyYXRpbmdfY29uZiI6IG9wLmdldCgiY29uZiIpLAogICAgICAgICAgICAgICAgIm5fZGV0ZWN0aW9ucyI6IG0uZ2V0KCJuX2RldGVjdGlvbnMiKSwKICAgICAgICAgICAgICAgICJuX2d0X2FsbCI6IChtLmdldCgibl9ndCIpIG9yIHt9KS5nZXQoImFsbCIpLAogICAgICAgICAgICAgICAgIm5fZ3Rfc21hbGwiOiAobS5nZXQoIm5fZ3QiKSBvciB7fSkuZ2V0KCJzbWFsbCIpLAogICAgICAgICAgICAgICAgIm5fZ3RfbWVkaXVtIjogKG0uZ2V0KCJuX2d0Iikgb3Ige30pLmdldCgibWVkaXVtIiksCiAgICAgICAgICAgICAgICAibl9ndF9sYXJnZSI6IChtLmdldCgibl9ndCIpIG9yIHt9KS5nZXQoImxhcmdlIiksCiAgICAgICAgICAgICAgICAidHJhaW5pbmdfdGltZV9oIjogcnVuLmdldCgidHJhaW5pbmdfdGltZV9oIiksCiAgICAgICAgICAgICAgICAidHJhaW5pbmdfdGltZV9zIjogcnVuLmdldCgidHJhaW5pbmdfdGltZV9zIiksCiAgICAgICAgICAgICAgICAicGVha19ncHVfbWVtb3J5X2diIjogcnVuLmdldCgicGVha19ncHVfbWVtb3J5X2diIiksCiAgICAgICAgICAgICAgICAibGF0ZW5jeV9tc19wZXJfaW1hZ2UiOiAobS5nZXQoInRpbWluZyIpIG9yIHt9KS5nZXQoImxhdGVuY3lfbXNfcGVyX2ltYWdlIiksCiAgICAgICAgICAgICAgICAiaW1hZ2VzX3Blcl9zIjogKG0uZ2V0KCJ0aW1pbmciKSBvciB7fSkuZ2V0KCJpbWFnZXNfcGVyX3MiKSwKICAgICAgICAgICAgICAgICJlcG9jaHMiOiBydW4uZ2V0KCJlcG9jaHMiKSwKICAgICAgICAgICAgICAgICJiYXRjaCI6IHJ1bi5nZXQoImJhdGNoIiksCiAgICAgICAgICAgICAgICAiaW1nc3oiOiBydW4uZ2V0KCJpbWdzeiIpLAogICAgICAgICAgICAgICAgInNlZWQiOiBydW4uZ2V0KCJzZWVkIiksCiAgICAgICAgICAgICAgICAiZ3B1IjogZW52LmdldCgiZ3B1X25hbWUiKSwKICAgICAgICAgICAgICAgICJncHVfbWVtb3J5X2diIjogZW52LmdldCgiZ3B1X3RvdGFsX21lbW9yeV9nYiIpLAogICAgICAgICAgICAgICAgInVsdHJhbHl0aWNzX21BUDUwIjogKGQuZ2V0KCJ1bHRyYWx5dGljc192YWwiKSBvciB7fSkuZ2V0KCJtQVA1MCIpLAogICAgICAgICAgICAgICAgImNyb3NzX2NoZWNrX21BUDUwX2Fic19kaWZmIjogZC5nZXQoImNyb3NzX2NoZWNrX21BUDUwX2Fic19kaWZmIiksCiAgICAgICAgICAgICAgICAiY3Jvc3NfY2hlY2tfd2FybmluZyI6IGQuZ2V0KCJjcm9zc19jaGVja193YXJuaW5nIiksCiAgICAgICAgICAgIH0pCiAgICBkZiA9IHBkLkRhdGFGcmFtZShyb3dzKQogICAgaWYgbm90IGRmLmVtcHR5OgogICAgICAgIGRmWyJfb3JkIl0gPSBkZlsicmVzb2x1dGlvbiJdLm1hcCh7bDogaSBmb3IgaSwgKF8sIGwsIF8pIGluIGVudW1lcmF0ZShSVU5fT1JERVIpfSkKICAgICAgICBkZiA9IGRmLnNvcnRfdmFsdWVzKFsic2FuaXR5IiwgIl9vcmQiXSkuZHJvcChjb2x1bW5zPSJfb3JkIikucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcmV0dXJuIGRmCgoKZGVmIGFkZF9kZWdyYWRhdGlvbihkZjogcGQuRGF0YUZyYW1lLCBiYXNlbGluZV9sYWJlbDogc3RyID0gIm9yaWdpbmFsIikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUGVyY2VudGFnZSBkcm9wIG9mIGVhY2ggbWV0cmljIHJlbGF0aXZlIHRvIHRoZSBiYXNlbGluZSBjb25kaXRpb24uIiIiCiAgICBvdXQgPSBkZi5jb3B5KCkKICAgIGJhc2UgPSBvdXRbKG91dFsicmVzb2x1dGlvbiJdID09IGJhc2VsaW5lX2xhYmVsKSAmICh+b3V0WyJzYW5pdHkiXSldCiAgICBpZiBiYXNlLmVtcHR5OgogICAgICAgICMgV2l0aG91dCBhIGJhc2VsaW5lIHRoZXJlIGlzIG5vdGhpbmcgdG8gZXhwcmVzcyBhIGRyb3AgYWdhaW5zdC4gTGVhdmluZwogICAgICAgICMgdGhlIGNvbHVtbnMgZW1wdHkgaXMgY29ycmVjdDsgaW52ZW50aW5nIGEgcmVmZXJlbmNlIHdvdWxkIG5vdCBiZS4KICAgICAgICBmb3IgYyBpbiBNRVRSSUNfQ09MUzoKICAgICAgICAgICAgb3V0W2Yie2N9X2Ryb3BfcGN0Il0gPSBucC5uYW4KICAgICAgICByZXR1cm4gb3V0CiAgICBiID0gYmFzZS5pbG9jWzBdCiAgICBmb3IgYyBpbiBNRVRSSUNfQ09MUzoKICAgICAgICBidiA9IGIuZ2V0KGMpCiAgICAgICAgb3V0W2Yie2N9X2Ryb3BfcGN0Il0gPSBvdXRbY10uYXBwbHkoCiAgICAgICAgICAgIGxhbWJkYSB2LCBidj1idjogbnAubmFuIGlmIChidiBpbiAoTm9uZSwgMCkgb3IgcGQuaXNuYShidikgb3IgcGQuaXNuYSh2KSkKICAgICAgICAgICAgZWxzZSByb3VuZCgoYnYgLSB2KSAvIGJ2ICogMTAwLCAyKSkKICAgIHJldHVybiBvdXQKCgpkZWYgbWFrZV9wbG90cyhkZjogcGQuRGF0YUZyYW1lLCBvdXRfZGlyOiBQYXRoKSAtPiBsaXN0W3N0cl06CiAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgbWF0cGxvdGxpYi51c2UoIkFnZyIpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CgogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBkID0gZGZbfmRmWyJzYW5pdHkiXV0uZHJvcG5hKHN1YnNldD1bIm1BUDUwIl0pCiAgICBpZiBkLmVtcHR5OgogICAgICAgIHJldHVybiBbXQogICAgeCA9IGRbInNjYWxlIl0gKiAxMDAKICAgIHdyaXR0ZW4gPSBbXQoKICAgIGRlZiBzaW5nbGUoY29sLCB5bGFiZWwsIHRpdGxlLCBmbmFtZSwgY29sb3VyPSIjMDA3OThjIik6CiAgICAgICAgaWYgZFtjb2xdLmlzbmEoKS5hbGwoKToKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LjQsIDQuNCkpCiAgICAgICAgYXgucGxvdCh4LCBkW2NvbF0sICJvLSIsIGx3PTIuMiwgY29sb3I9Y29sb3VyLCBtYXJrZXJzaXplPTcpCiAgICAgICAgZm9yIHhpLCB5aSBpbiB6aXAoeCwgZFtjb2xdKToKICAgICAgICAgICAgaWYgcGQubm90bmEoeWkpOgogICAgICAgICAgICAgICAgYXguYW5ub3RhdGUoZiJ7eWk6LjNmfSIsICh4aSwgeWkpLCB0ZXh0Y29vcmRzPSJvZmZzZXQgcG9pbnRzIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHh5dGV4dD0oMCwgOCksIGhhPSJjZW50ZXIiLCBmb250c2l6ZT04KQogICAgICAgIGF4LnNldF94bGFiZWwoInJlc29sdXRpb24gKCUgb2Ygb3JpZ2luYWwpIikKICAgICAgICBheC5zZXRfeWxhYmVsKHlsYWJlbCkKICAgICAgICBheC5zZXRfdGl0bGUodGl0bGUpCiAgICAgICAgYXguaW52ZXJ0X3hheGlzKCkKICAgICAgICBheC5ncmlkKGFscGhhPTAuMykKICAgICAgICBheC5zZXRfeWxpbShib3R0b209MCkKICAgICAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgICAgICBmaWcuc2F2ZWZpZyhvdXRfZGlyIC8gZm5hbWUsIGRwaT0xNDApCiAgICAgICAgcGx0LmNsb3NlKGZpZykKICAgICAgICB3cml0dGVuLmFwcGVuZChmbmFtZSkKCiAgICBzaW5nbGUoIm1BUDUwIiwgIm1BUEA1MCIsICJSZXNvbHV0aW9uIHZzIG1BUEA1MCIsICJyZXNvbHV0aW9uX3ZzX21hcC5wbmciKQogICAgc2luZ2xlKCJtQVA1MF85NSIsICJtQVBANTA6OTUiLCAiUmVzb2x1dGlvbiB2cyBtQVBANTA6OTUiLAogICAgICAgICAgICJyZXNvbHV0aW9uX3ZzX21hcDUwOTUucG5nIikKICAgIHNpbmdsZSgicmVjYWxsIiwgInJlY2FsbCIsICJSZXNvbHV0aW9uIHZzIHJlY2FsbCIsICJyZXNvbHV0aW9uX3ZzX3JlY2FsbC5wbmciLAogICAgICAgICAgICIjM2I3ZGQ4IikKICAgIHNpbmdsZSgicHJlY2lzaW9uIiwgInByZWNpc2lvbiIsICJSZXNvbHV0aW9uIHZzIHByZWNpc2lvbiIsCiAgICAgICAgICAgInJlc29sdXRpb25fdnNfcHJlY2lzaW9uLnBuZyIsICIjOGU0NGFkIikKICAgIHNpbmdsZSgiQVBfc21hbGwiLCAiQVAgKHNtYWxsKSIsICJSZXNvbHV0aW9uIHZzIHNtYWxsLW9iamVjdCBBUCIsCiAgICAgICAgICAgInJlc29sdXRpb25fdnNfc21hbGxfYXAucG5nIiwgQklOX0NPTE9VUlNbInNtYWxsIl0pCiAgICBzaW5nbGUoIkFQX21lZGl1bSIsICJBUCAobWVkaXVtKSIsICJSZXNvbHV0aW9uIHZzIG1lZGl1bS1vYmplY3QgQVAiLAogICAgICAgICAgICJyZXNvbHV0aW9uX3ZzX21lZGl1bV9hcC5wbmciLCBCSU5fQ09MT1VSU1sibWVkaXVtIl0pCiAgICBzaW5nbGUoIkFQX2xhcmdlIiwgIkFQIChsYXJnZSkiLCAiUmVzb2x1dGlvbiB2cyBsYXJnZS1vYmplY3QgQVAiLAogICAgICAgICAgICJyZXNvbHV0aW9uX3ZzX2xhcmdlX2FwLnBuZyIsIEJJTl9DT0xPVVJTWyJsYXJnZSJdKQoKICAgICMgQ29tYmluZWQgc2l6ZSB2aWV3OiB0aGUgc3R1ZHkncyBoZWFkbGluZSBmaWd1cmUuCiAgICBpZiBub3QgZFtbIkFQX3NtYWxsIiwgIkFQX21lZGl1bSIsICJBUF9sYXJnZSJdXS5pc25hKCkuYWxsKCkuYWxsKCk6CiAgICAgICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEyLjUsIDQuNikpCiAgICAgICAgYXggPSBheGVzWzBdCiAgICAgICAgZm9yIGIgaW4gKCJzbWFsbCIsICJtZWRpdW0iLCAibGFyZ2UiKToKICAgICAgICAgICAgYXgucGxvdCh4LCBkW2YiQVBfe2J9Il0sICJvLSIsIGx3PTIuMiwgbGFiZWw9YiwgY29sb3I9QklOX0NPTE9VUlNbYl0pCiAgICAgICAgYXgucGxvdCh4LCBkWyJtQVA1MF85NSJdLCAiay0tIiwgbHc9MS42LCBsYWJlbD0ib3ZlcmFsbCIsIGFscGhhPTAuNykKICAgICAgICBheC5zZXRfeGxhYmVsKCJyZXNvbHV0aW9uICglIG9mIG9yaWdpbmFsKSIpCiAgICAgICAgYXguc2V0X3lsYWJlbCgiQVBANTA6OTUiKQogICAgICAgIGF4LnNldF90aXRsZSgiQVAgYnkgZml4ZWQgb3JpZ2luYWwtcmVzb2x1dGlvbiBzaXplIGJpbiIpCiAgICAgICAgYXguaW52ZXJ0X3hheGlzKCk7IGF4LmdyaWQoYWxwaGE9MC4zKTsgYXgubGVnZW5kKGZvbnRzaXplPTkpCiAgICAgICAgYXguc2V0X3lsaW0oYm90dG9tPTApCgogICAgICAgIGF4ID0gYXhlc1sxXQogICAgICAgIGZvciBiIGluICgic21hbGwiLCAibWVkaXVtIiwgImxhcmdlIik6CiAgICAgICAgICAgIGNvbCA9IGYiQVBfe2J9X2Ryb3BfcGN0IgogICAgICAgICAgICBpZiBjb2wgaW4gZDoKICAgICAgICAgICAgICAgIGF4LnBsb3QoeCwgZFtjb2xdLCAiby0iLCBsdz0yLjIsIGxhYmVsPWIsIGNvbG9yPUJJTl9DT0xPVVJTW2JdKQogICAgICAgIGlmICJtQVA1MF85NV9kcm9wX3BjdCIgaW4gZDoKICAgICAgICAgICAgYXgucGxvdCh4LCBkWyJtQVA1MF85NV9kcm9wX3BjdCJdLCAiay0tIiwgbHc9MS42LCBsYWJlbD0ib3ZlcmFsbCIsCiAgICAgICAgICAgICAgICAgICAgYWxwaGE9MC43KQogICAgICAgIGF4LmF4aGxpbmUoMCwgY29sb3I9ImsiLCBsdz0wLjgpCiAgICAgICAgYXguc2V0X3hsYWJlbCgicmVzb2x1dGlvbiAoJSBvZiBvcmlnaW5hbCkiKQogICAgICAgIGF4LnNldF95bGFiZWwoIiUgZHJvcCB2cyBiYXNlbGluZSIpCiAgICAgICAgYXguc2V0X3RpdGxlKCJEZWdyYWRhdGlvbiByZWxhdGl2ZSB0byBvcmlnaW5hbCByZXNvbHV0aW9uIikKICAgICAgICBheC5pbnZlcnRfeGF4aXMoKTsgYXguZ3JpZChhbHBoYT0wLjMpOyBheC5sZWdlbmQoZm9udHNpemU9OSkKICAgICAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgICAgICBmaWcuc2F2ZWZpZyhvdXRfZGlyIC8gInJlc29sdXRpb25feF9vYmplY3Rfc2l6ZS5wbmciLCBkcGk9MTQwKQogICAgICAgIHBsdC5jbG9zZShmaWcpCiAgICAgICAgd3JpdHRlbi5hcHBlbmQoInJlc29sdXRpb25feF9vYmplY3Rfc2l6ZS5wbmciKQoKICAgICMgQ29tcHV0YXRpb25hbCBjb3N0LgogICAgY29zdCA9IGQuZHJvcG5hKHN1YnNldD1bInRyYWluaW5nX3RpbWVfaCJdKQogICAgaWYgbm90IGNvc3QuZW1wdHk6CiAgICAgICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDMsIGZpZ3NpemU9KDE0LCA0LjIpKQogICAgICAgIGZvciBheCwgY29sLCBsYmwgaW4gWwogICAgICAgICAgICAgICAgKGF4ZXNbMF0sICJ0cmFpbmluZ190aW1lX2giLCAidHJhaW5pbmcgdGltZSAoaCkiKSwKICAgICAgICAgICAgICAgIChheGVzWzFdLCAibGF0ZW5jeV9tc19wZXJfaW1hZ2UiLCAiaW5mZXJlbmNlIGxhdGVuY3kgKG1zL2ltZykiKSwKICAgICAgICAgICAgICAgIChheGVzWzJdLCAicGVha19ncHVfbWVtb3J5X2diIiwgInBlYWsgR1BVIG1lbW9yeSAoR0IpIildOgogICAgICAgICAgICBpZiBjb2wgaW4gY29zdCBhbmQgbm90IGNvc3RbY29sXS5pc25hKCkuYWxsKCk6CiAgICAgICAgICAgICAgICBheC5iYXIoY29zdFsicmVzb2x1dGlvbiJdLCBjb3N0W2NvbF0sIGNvbG9yPSIjM2I3ZGQ4IikKICAgICAgICAgICAgICAgIGZvciBpLCB2IGluIGVudW1lcmF0ZShjb3N0W2NvbF0pOgogICAgICAgICAgICAgICAgICAgIGlmIHBkLm5vdG5hKHYpOgogICAgICAgICAgICAgICAgICAgICAgICBheC50ZXh0KGksIHYsIGYie3Y6LjJmfSIsIGhhPSJjZW50ZXIiLCB2YT0iYm90dG9tIiwgZm9udHNpemU9OCkKICAgICAgICAgICAgYXguc2V0X3lsYWJlbChsYmwpCiAgICAgICAgICAgIGF4LnNldF90aXRsZShsYmwpCiAgICAgICAgZmlnLnN1cHRpdGxlKCJDb21wdXRhdGlvbmFsIGNvc3QgYnkgY29uZGl0aW9uICIKICAgICAgICAgICAgICAgICAgICAgIihuZXR3b3JrIGlucHV0IHNpemUgaGVsZCBjb25zdGFudCBhdCBpbWdzej0xMDI0KSIsCiAgICAgICAgICAgICAgICAgICAgIGZvbnRzaXplPTEwKQogICAgICAgIGZpZy50aWdodF9sYXlvdXQocmVjdD0oMCwgMCwgMSwgMC45NCkpCiAgICAgICAgZmlnLnNhdmVmaWcob3V0X2RpciAvICJ0cmFpbmluZ19jb3N0LnBuZyIsIGRwaT0xNDApCiAgICAgICAgcGx0LmNsb3NlKGZpZykKICAgICAgICB3cml0dGVuLmFwcGVuZCgidHJhaW5pbmdfY29zdC5wbmciKQoKICAgIHJldHVybiB3cml0dGVuCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJlc3VsdHMtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PU5vbmUsIGhlbHA9Im91dHB1dCBkaXJlY3RvcnkgKGRlZmF1bHQ6IDxwa2c+L3Jlc3VsdHMpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pbmNsdWRlLXNhbml0eSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgcmVzdWx0c19yb290ID0gZmluZF9yZXN1bHRzX3Jvb3QoYXJncy5yZXN1bHRzX3Jvb3QpCiAgICBvdXRfZGlyID0gUGF0aChhcmdzLm91dCkgaWYgYXJncy5vdXQgZWxzZSBQS0cgLyAicmVzdWx0cyIKICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGRmID0gbG9hZF9ydW5zKHJlc3VsdHNfcm9vdCwgYXJncy5pbmNsdWRlX3Nhbml0eSkKICAgIGlmIGRmLmVtcHR5OgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJObyBtZXRyaWNzLmpzb24gZm91bmQgdW5kZXIge3Jlc3VsdHNfcm9vdH0uICIKICAgICAgICAgICAgICAgICAgICAgICAgIGYiUnVuIHNjcmlwdHMvcnVuX2FsbC5weSBmaXJzdC4iKQoKICAgIGRmID0gYWRkX2RlZ3JhZGF0aW9uKGRmKQogICAgcHJlc2VudCA9IFtyIGZvciByIGluIGRmLmxvY1t+ZGZbInNhbml0eSJdLCAicmVzb2x1dGlvbiJdXQogICAgbWlzc2luZyA9IFtsIGZvciBfLCBsLCBfIGluIFJVTl9PUkRFUiBpZiBsIG5vdCBpbiBwcmVzZW50XQoKICAgIGRmLnRvX2NzdihvdXRfZGlyIC8gIm1hc3Rlcl9yZXN1bHRzLmNzdiIsIGluZGV4PUZhbHNlKQogICAgKG91dF9kaXIgLyAibWFzdGVyX3Jlc3VsdHMuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyh7CiAgICAgICAgICAgICJjb25kaXRpb25zX3ByZXNlbnQiOiBwcmVzZW50LAogICAgICAgICAgICAiY29uZGl0aW9uc19taXNzaW5nIjogbWlzc2luZywKICAgICAgICAgICAgImNvbXBsZXRlX3N3ZWVwIjogbm90IG1pc3NpbmcsCiAgICAgICAgICAgICJiYXNlbGluZSI6ICJvcmlnaW5hbCIgaWYgIm9yaWdpbmFsIiBpbiBwcmVzZW50IGVsc2UgTm9uZSwKICAgICAgICAgICAgInNpemVfYmlucyI6IHsibWV0aG9kIjogIkNPQ08gYXJlYSB0aHJlc2hvbGRzIG9uIE9SSUdJTkFMLXJlc29sdXRpb24gYm94ZXMiLAogICAgICAgICAgICAgICAgICAgICAgICAgICJzbWFsbCI6ICI8IDEwMjQgcHheMiIsICJtZWRpdW0iOiAiMTAyNC05MjE2IHB4XjIiLAogICAgICAgICAgICAgICAgICAgICAgICAgICJsYXJnZSI6ICI+PSA5MjE2IHB4XjIiLAogICAgICAgICAgICAgICAgICAgICAgICAgICJmaXhlZF9hY3Jvc3NfcmVzb2x1dGlvbnMiOiBUcnVlfSwKICAgICAgICAgICAgInJ1bnMiOiBqc29uLmxvYWRzKGRmLnRvX2pzb24ob3JpZW50PSJyZWNvcmRzIikpLAogICAgICAgIH0sIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICAjIFBhcnQgUSB0YWJsZS4KICAgIGNtcF9jb2xzID0gWyJyZXNvbHV0aW9uIiwgInNjYWxlIiwgImFyZWFfc2NhbGUiLCAibUFQNTAiLCAibUFQNTBfOTUiLAogICAgICAgICAgICAgICAgInByZWNpc2lvbiIsICJyZWNhbGwiLCAiZjEiLAogICAgICAgICAgICAgICAgIm1BUDUwX2Ryb3BfcGN0IiwgIm1BUDUwXzk1X2Ryb3BfcGN0IiwKICAgICAgICAgICAgICAgICJyZWNhbGxfZHJvcF9wY3QiLCAicHJlY2lzaW9uX2Ryb3BfcGN0Il0KICAgIGRmLmxvY1t+ZGZbInNhbml0eSJdLCBbYyBmb3IgYyBpbiBjbXBfY29scyBpZiBjIGluIGRmXV0udG9fY3N2KAogICAgICAgIG91dF9kaXIgLyAicmVzb2x1dGlvbl9jb21wYXJpc29uLmNzdiIsIGluZGV4PUZhbHNlKQoKICAgIHNpemVfY29scyA9IFsicmVzb2x1dGlvbiIsICJzY2FsZSIsICJtQVA1MF85NSIsCiAgICAgICAgICAgICAgICAgIkFQX3NtYWxsIiwgIkFQX21lZGl1bSIsICJBUF9sYXJnZSIsCiAgICAgICAgICAgICAgICAgIkFQX3NtYWxsX2Ryb3BfcGN0IiwgIkFQX21lZGl1bV9kcm9wX3BjdCIsICJBUF9sYXJnZV9kcm9wX3BjdCIsCiAgICAgICAgICAgICAgICAgIm5fZ3Rfc21hbGwiLCAibl9ndF9tZWRpdW0iLCAibl9ndF9sYXJnZSJdCiAgICBkZi5sb2NbfmRmWyJzYW5pdHkiXSwgW2MgZm9yIGMgaW4gc2l6ZV9jb2xzIGlmIGMgaW4gZGZdXS50b19jc3YoCiAgICAgICAgb3V0X2RpciAvICJvYmplY3Rfc2l6ZV9jb21wYXJpc29uLmNzdiIsIGluZGV4PUZhbHNlKQoKICAgIGNvc3RfY29scyA9IFsicmVzb2x1dGlvbiIsICJzY2FsZSIsICJ0cmFpbmluZ190aW1lX2giLCAibGF0ZW5jeV9tc19wZXJfaW1hZ2UiLAogICAgICAgICAgICAgICAgICJpbWFnZXNfcGVyX3MiLCAicGVha19ncHVfbWVtb3J5X2diIiwgImdwdSIsICJncHVfbWVtb3J5X2diIiwKICAgICAgICAgICAgICAgICAiZXBvY2hzIiwgImJhdGNoIiwgImltZ3N6IiwgInNlZWQiXQogICAgZGYubG9jW35kZlsic2FuaXR5Il0sIFtjIGZvciBjIGluIGNvc3RfY29scyBpZiBjIGluIGRmXV0udG9fY3N2KAogICAgICAgIG91dF9kaXIgLyAidHJhaW5pbmdfY29zdF9jb21wYXJpc29uLmNzdiIsIGluZGV4PUZhbHNlKQoKICAgIHBsb3RzID0gbWFrZV9wbG90cyhkZiwgb3V0X2RpciAvICJwbG90cyIpCgogICAgcHJpbnQoZiJDb25kaXRpb25zIGZvdW5kIDoge3ByZXNlbnQgb3IgJ25vbmUnfSIpCiAgICBpZiBtaXNzaW5nOgogICAgICAgIHByaW50KGYiQ29uZGl0aW9ucyBNSVNTSU5HOiB7bWlzc2luZ30gIDwtIHRhYmxlcyBhcmUgYSBwYXJ0aWFsIHN3ZWVwIikKICAgIHdhcm4gPSBkZltkZlsiY3Jvc3NfY2hlY2tfd2FybmluZyJdLm5vdG5hKCldCiAgICBpZiBub3Qgd2Fybi5lbXB0eToKICAgICAgICBwcmludCgiXG4hISBtZXRyaWMgY3Jvc3MtY2hlY2sgZGlzYWdyZWVtZW50OiIpCiAgICAgICAgZm9yIF8sIHIgaW4gd2Fybi5pdGVycm93cygpOgogICAgICAgICAgICBwcmludChmIiAgIHtyWydydW5faWQnXX06IHtyWydjcm9zc19jaGVja193YXJuaW5nJ119IikKCiAgICBzaG93ID0gWyJyZXNvbHV0aW9uIiwgIm1BUDUwIiwgIm1BUDUwXzk1IiwgIkFQX3NtYWxsIiwgIkFQX21lZGl1bSIsCiAgICAgICAgICAgICJBUF9sYXJnZSIsICJyZWNhbGwiLCAicHJlY2lzaW9uIl0KICAgIHByaW50KCJcbiIgKyBkZi5sb2NbfmRmWyJzYW5pdHkiXSwgW2MgZm9yIGMgaW4gc2hvdyBpZiBjIGluIGRmXV0KICAgICAgICAgIC50b19zdHJpbmcoaW5kZXg9RmFsc2UsIGZsb2F0X2Zvcm1hdD1sYW1iZGEgdjogZiJ7djouNGZ9IikpCgogICAgcHJpbnQoZiJcbldyb3RlIHtsZW4ocGxvdHMpfSBwbG90KHMpIGFuZCA1IHRhYmxlKHMpIHRvIHtvdXRfZGlyfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "scripts/evaluate.py": "IiIiClNjb3JlIGEgdHJhaW5lZCBjaGVja3BvaW50LCB3aXRoIHNpemUtc3RyYXRpZmllZCBBUCBvbiB0aGUgZml4ZWQKb3JpZ2luYWwtcmVzb2x1dGlvbiBiaW5zLgoKICAgICMgc2NvcmUgYSBjb25kaXRpb24ncyBvd24gbW9kZWwgb24gaXRzIG93biBkYXRhCiAgICBweXRob24gc2NyaXB0cy9ldmFsdWF0ZS5weSAtLWNvbmZpZyBjb25maWdzL3I1MC55YW1sIC0td2VpZ2h0cyAuLi4vYmVzdC5wdAoKICAgICMgc2NvcmUgdGhlIFNBTUUgbW9kZWwgb24gYSBkaWZmZXJlbnQgcmVzb2x1dGlvbiAoY3Jvc3MtcmVzb2x1dGlvbiB0cmFuc2ZlcikKICAgIHB5dGhvbiBzY3JpcHRzL2V2YWx1YXRlLnB5IC0tY29uZmlnIGNvbmZpZ3MvcjUwLnlhbWwgLS13ZWlnaHRzIC4uLi9iZXN0LnB0IFwKICAgICAgICAtLWV2YWwtb24gb3JpZ2luYWwgLS10YWcgcjUwX21vZGVsX29uX29yaWdpbmFsCgpUaGUgYC0tZXZhbC1vbmAgZm9ybSBpcyBhIHNlcGFyYXRlIHF1ZXN0aW9uIGZyb20gdGhlIHByaW1hcnkgc3dlZXA6IGl0IGFza3MKd2hldGhlciBhIG1vZGVsIHRyYWluZWQgYXQgb25lIHJlc29sdXRpb24gc3RpbGwgd29ya3MgYXQgYW5vdGhlciwgcmF0aGVyIHRoYW4Kd2hhdCByZXNvbHV0aW9uIGNvc3RzIHdoZW4geW91IHRyYWluIGFuZCB0ZXN0IGNvbnNpc3RlbnRseS4gS2VlcCBpdHMgb3V0cHV0IG91dApvZiB0aGUgbWFpbiByZXN1bHRzIHRhYmxlIC0tIGAtLXRhZ2Agd3JpdGVzIGl0IHRvIGl0cyBvd24gZGlyZWN0b3J5IHNvIGl0CmNhbm5vdCBiZSBwaWNrZWQgdXAgYnkgY29sbGVjdF9yZXN1bHRzLnB5IGJ5IGFjY2lkZW50LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgeWFtbAoKUEtHID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQS0cpKQoKZnJvbSBzcmMuZXZhbHVhdGlvbiBpbXBvcnQgYnVpbGRfZ3JvdW5kX3RydXRoLCBwcmVkaWN0X3NwbGl0LCBzY29yZQpmcm9tIHNyYy5wYXRocyBpbXBvcnQgYW5ub3RhdGlvbl9kaXIsIGZpbmRfZGF0YXNldHNfcm9vdCwgZmluZF9yZXN1bHRzX3Jvb3QKZnJvbSBzcmMudHJhaW5pbmcgaW1wb3J0IGVudmlyb25tZW50X2luZm8sIGxvYWRfbWFzdGVyCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS13ZWlnaHRzIiwgcmVxdWlyZWQ9VHJ1ZSwgdHlwZT1QYXRoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWV2YWwtb24iLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0icmVzb2x1dGlvbiBsYWJlbCB0byBldmFsdWF0ZSBhZ2FpbnN0ICIKICAgICAgICAgICAgICAgICAgICAgICAgICIoZGVmYXVsdDogdGhlIGNvbmZpZydzIG93biBjb25kaXRpb24pIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGRlZmF1bHQ9IlZhbCIsIGNob2ljZXM9WyJWYWwiLCAiVHJhaW4iXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YWciLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ib3V0cHV0IHN1YmRpcmVjdG9yeSBuYW1lOyByZXF1aXJlZCB3aGVuIC0tZXZhbC1vbiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiZGlmZmVycyBmcm9tIHRoZSBjb25maWcncyBjb25kaXRpb24iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGFzZXRzLXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVzdWx0cy1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZiIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zYXZlLXByZWRpY3Rpb25zIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBzdHViID0geWFtbC5zYWZlX2xvYWQoYXJncy5jb25maWcucmVhZF90ZXh0KCkpCiAgICBjZmcgPSBsb2FkX21hc3RlcihhcmdzLmNvbmZpZy5wYXJlbnQgLyBzdHViLmdldCgiaW5oZXJpdHMiLCAibWFzdGVyLnlhbWwiKSkKICAgIGV4cGVyaW1lbnQgPSBzdHViWyJleHBlcmltZW50Il0KICAgIGV4cCA9IGNmZ1siZXhwZXJpbWVudHMiXVtleHBlcmltZW50XQogICAgZXYgPSBjZmdbImV2YWx1YXRpb24iXQoKICAgIGV2YWxfb24gPSBhcmdzLmV2YWxfb24gb3IgZXhwWyJsYWJlbCJdCiAgICBjcm9zcyA9IGV2YWxfb24gIT0gZXhwWyJsYWJlbCJdCiAgICBpZiBjcm9zcyBhbmQgbm90IGFyZ3MudGFnOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoCiAgICAgICAgICAgIGYiLS1ldmFsLW9uIHtldmFsX29ufSBkaWZmZXJzIGZyb20gdGhpcyBjb25maWcncyBjb25kaXRpb24gIgogICAgICAgICAgICBmIih7ZXhwWydsYWJlbCddfSkuIFBhc3MgLS10YWcgc28gdGhlIHJlc3VsdCBsYW5kcyBpbiBpdHMgb3duICIKICAgICAgICAgICAgZiJkaXJlY3RvcnkgYW5kIGNhbm5vdCBiZSBtaXN0YWtlbiBmb3IgYSBwcmltYXJ5LXN3ZWVwIG51bWJlci4iKQoKICAgIGRhdGFzZXRzID0gZmluZF9kYXRhc2V0c19yb290KGFyZ3MuZGF0YXNldHNfcm9vdCkKICAgIGRhdGFzZXRfZGlyID0gZGF0YXNldHNbZXZhbF9vbl0KICAgIHJlc3VsdHNfcm9vdCA9IGZpbmRfcmVzdWx0c19yb290KGFyZ3MucmVzdWx0c19yb290KQogICAgb3V0X2RpciA9IHJlc3VsdHNfcm9vdCAvIChhcmdzLnRhZyBvciBleHBbImlkIl0pCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICBtb2RlbCA9IFlPTE8oc3RyKGFyZ3Mud2VpZ2h0cykpCgogICAgYW5uID0gYW5ub3RhdGlvbl9kaXIoKQogICAgc2hpcHMgPSBwZC5yZWFkX3BhcnF1ZXQoYW5uIC8gInNoaXBfYW5ub3RhdGlvbnMucGFycXVldCIpCiAgICBpbWFnZXMgPSBwZC5yZWFkX2Nzdihhbm4gLyAiaW1hZ2VfbWFuaWZlc3QuY3N2IikKCiAgICBwcmludChmIk1vZGVsICAgICAgOiB7YXJncy53ZWlnaHRzfSIpCiAgICBwcmludChmIlRyYWluZWQgb24gOiB7ZXhwWydsYWJlbCddfSIpCiAgICBwcmludChmIkV2YWx1YXRpbmcgOiB7ZXZhbF9vbn0gICh7YXJncy5zcGxpdH0gc3BsaXQpICBkYXRhc2V0OiB7ZGF0YXNldF9kaXJ9IikKICAgIHByaW50KGYiaW1nc3ogICAgICA6IHtjZmdbJ3RyYWluJ11bJ2ltZ3N6J119ICAgbWF4X2RldDoge2V2WydtYXhfZGV0J119IikKCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBkdCwgdGltaW5nID0gcHJlZGljdF9zcGxpdCgKICAgICAgICBtb2RlbCwgZGF0YXNldF9kaXIsIGltYWdlcywgc3BsaXQ9YXJncy5zcGxpdCwKICAgICAgICBjb25mPWFyZ3MuY29uZiBpZiBhcmdzLmNvbmYgaXMgbm90IE5vbmUgZWxzZSBldlsiY29uZiJdLAogICAgICAgIGlvdT1ldlsiaW91Il0sIG1heF9kZXQ9ZXZbIm1heF9kZXQiXSwgaW1nc3o9Y2ZnWyJ0cmFpbiJdWyJpbWdzeiJdLAogICAgICAgIGRldmljZT1hcmdzLmRldmljZSwgYmF0Y2g9YXJncy5iYXRjaCBvciBjZmdbInRyYWluIl1bImJhdGNoIl0pCiAgICBndCA9IGJ1aWxkX2dyb3VuZF90cnV0aChzaGlwcywgaW1hZ2VzLCBzcGxpdD1hcmdzLnNwbGl0KQogICAgbSA9IHNjb3JlKGd0LCBkdCwgbWF4X2RldD1ldlsibWF4X2RldCJdLAogICAgICAgICAgICAgIG9wZXJhdGluZ19jb25mPWV2WyJvcGVyYXRpbmdfcG9pbnRfY29uZiJdKQogICAgbVsidGltaW5nIl0gPSB0aW1pbmcKICAgIG1bImV2YWxfd2FsbF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAxKQoKICAgIHBheWxvYWQgPSB7CiAgICAgICAgImV4cGVyaW1lbnQiOiBleHBlcmltZW50LAogICAgICAgICJ0cmFpbmVkX29uIjogZXhwWyJsYWJlbCJdLAogICAgICAgICJldmFsdWF0ZWRfb24iOiBldmFsX29uLAogICAgICAgICJjcm9zc19yZXNvbHV0aW9uIjogY3Jvc3MsCiAgICAgICAgInNwbGl0IjogYXJncy5zcGxpdCwKICAgICAgICAid2VpZ2h0cyI6IHN0cihhcmdzLndlaWdodHMpLAogICAgICAgICJtZXRyaWNzIjogbSwKICAgICAgICAiZW52aXJvbm1lbnQiOiBlbnZpcm9ubWVudF9pbmZvKCksCiAgICB9CiAgICBuYW1lID0gIm1ldHJpY3MuanNvbiIgaWYgbm90IGNyb3NzIGVsc2UgZiJtZXRyaWNzX29uX3tldmFsX29ufS5qc29uIgogICAgKG91dF9kaXIgLyBuYW1lKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocGF5bG9hZCwgaW5kZW50PTIsIGRlZmF1bHQ9c3RyKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiKQoKICAgIGlmIGFyZ3Muc2F2ZV9wcmVkaWN0aW9uczoKICAgICAgICByb3dzID0gW10KICAgICAgICBmb3Igc3RlbSwgZCBpbiBkdC5pdGVtcygpOgogICAgICAgICAgICBmb3IgKHgwLCB5MCwgeDEsIHkxKSwgcyBpbiB6aXAoZFsiYm94ZXMiXSwgZFsic2NvcmVzIl0pOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJpbWFnZSI6IHN0ZW0sICJ4X21pbiI6IHgwLCAieV9taW4iOiB5MCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAieF9tYXgiOiB4MSwgInlfbWF4IjogeTEsICJzY29yZSI6IHN9KQogICAgICAgIHByZWRfZGlyID0gb3V0X2RpciAvICJwcmVkaWN0aW9ucyIKICAgICAgICBwcmVkX2Rpci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgICAgICMgUHJlZGljdGlvbnMgYXJlIHN0b3JlZCBpbiBPUklHSU5BTC1yZXNvbHV0aW9uIGNvb3JkaW5hdGVzIHNvIHRoZXkgY2FuCiAgICAgICAgIyBiZSBvdmVybGFpZCBvbiBhbnkgY29uZGl0aW9uJ3MgaW1hZ2VyeSB3aXRob3V0IGZ1cnRoZXIgY29udmVyc2lvbi4KICAgICAgICBwZC5EYXRhRnJhbWUocm93cykudG9fY3N2KAogICAgICAgICAgICBwcmVkX2RpciAvIGYicHJlZGljdGlvbnNfe2V2YWxfb259X3thcmdzLnNwbGl0fS5jc3YiLCBpbmRleD1GYWxzZSkKICAgICAgICBwcmludChmIiAgc2F2ZWQge2xlbihyb3dzKTosfSBwcmVkaWN0aW9ucyAiCiAgICAgICAgICAgICAgZiIob3JpZ2luYWwtcmVzb2x1dGlvbiBjb29yZGluYXRlcykiKQoKICAgIHByaW50KGYiXG4gIG1BUDUwICAgICAge21bJ21BUDUwJ106LjRmfSIpCiAgICBwcmludChmIiAgbUFQNTAtOTUgICB7bVsnbUFQNTBfOTUnXTouNGZ9IikKICAgIHByaW50KGYiICBBUF9zbWFsbCAgIHttWydBUF9zbWFsbCddOi40Zn0gICAobl9ndCB7bVsnbl9ndCddWydzbWFsbCddOix9KSIpCiAgICBwcmludChmIiAgQVBfbWVkaXVtICB7bVsnQVBfbWVkaXVtJ106LjRmfSAgIChuX2d0IHttWyduX2d0J11bJ21lZGl1bSddOix9KSIpCiAgICBwcmludChmIiAgQVBfbGFyZ2UgICB7bVsnQVBfbGFyZ2UnXTouNGZ9ICAgKG5fZ3Qge21bJ25fZ3QnXVsnbGFyZ2UnXTosfSkiKQogICAgcHJpbnQoZiIgIFAvUi9GMSBAIHtldlsnb3BlcmF0aW5nX3BvaW50X2NvbmYnXX06ICIKICAgICAgICAgIGYie21bJ29wZXJhdGluZ19wb2ludCddWydwcmVjaXNpb24nXTouNGZ9IC8gIgogICAgICAgICAgZiJ7bVsnb3BlcmF0aW5nX3BvaW50J11bJ3JlY2FsbCddOi40Zn0gLyAiCiAgICAgICAgICBmInttWydvcGVyYXRpbmdfcG9pbnQnXVsnZjEnXTouNGZ9IikKICAgIHByaW50KGYiXG5Xcm90ZSB7b3V0X2RpciAvIG5hbWV9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
    "scripts/qualitative_comparison.py": "IiIiClBhcnQgUiAtLSBxdWFsaXRhdGl2ZSBwcmVkaWN0aW9uIGNvbXBhcmlzb25zIGFjcm9zcyByZXNvbHV0aW9ucy4KCiAgICBweXRob24gc2NyaXB0cy9xdWFsaXRhdGl2ZV9jb21wYXJpc29uLnB5IC0tcmVzdWx0cy1yb290IDxkaXI+CgpGb3IgZWFjaCBzZWxlY3RlZCBzY2VuZSwgcmVuZGVycyBvbmUgcm93IG9mIHBhbmVsczogdGhlIHNhbWUgdGlsZSBhdCBvcmlnaW5hbCwKcjc1LCByNTAgYW5kIHIyNSwgZWFjaCB3aXRoIHRoYXQgY29uZGl0aW9uJ3Mgb3duIG1vZGVsJ3MgcHJlZGljdGlvbnMgb3ZlcmxhaWQKYWdhaW5zdCB0aGUgc2hhcmVkIGdyb3VuZCB0cnV0aC4KClRoZSBzY2VuZXMgYXJlIGNob3NlbiBvbmNlIGFuZCByZXVzZWQgZm9yIGV2ZXJ5IGNvbmRpdGlvbi4gUGlja2luZyB3aGF0ZXZlcgpsb29rcyBnb29kIHBlciBjb25kaXRpb24gd291bGQgcHJvZHVjZSBhIGZpZ3VyZSB0aGF0IGFyZ3VlcyBmb3IgYSBjb25jbHVzaW9uCnJhdGhlciB0aGFuIHNob3dpbmcgb25lOyBob2xkaW5nIHRoZSBzY2VuZXMgZml4ZWQgbWVhbnMgdGhlIHBhbmVscyBkaWZmZXIgb25seQppbiB3aGF0IHRoZSBtb2RlbCBmb3VuZC4KClNjZW5lIGNhdGVnb3JpZXMgKFBhcnQgUik6IHNtYWxsIHNoaXBzLCBtZWRpdW0gc2hpcHMsIGxhcmdlIHNoaXBzLCBkZW5zZQpzY2VuZXMsIGlzb2xhdGVkIHNoaXBzLCBkaWZmaWN1bHQgYmFja2dyb3VuZHMuIFNlbGVjdGlvbiBpcyBkZXRlcm1pbmlzdGljCmdpdmVuIGEgc2VlZCwgYW5kIHRoZSBjaG9zZW4gc2NlbmUgaWRzIGFyZSB3cml0dGVuIGFsb25nc2lkZSB0aGUgZmlndXJlcyBzbwp0aGUgc2FtZSBzZXQgY2FuIGJlIHJlZ2VuZXJhdGVkIG9yIGNoYWxsZW5nZWQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKUEtHID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQS0cpKQoKZnJvbSBzcmMucGF0aHMgaW1wb3J0IGFubm90YXRpb25fZGlyLCBmaW5kX2RhdGFzZXRzX3Jvb3QsIGZpbmRfcmVzdWx0c19yb290CgpDT05ESVRJT05TID0gWygib3JpZ2luYWwiLCAiMDBfYmFzZWxpbmUiLCAxLjAwKSwgKCJyNzUiLCAiMDFfcjc1IiwgMC43NSksCiAgICAgICAgICAgICAgKCJyNTAiLCAiMDJfcjUwIiwgMC41MCksICgicjI1IiwgIjAzX3IyNSIsIDAuMjUpXQoKR1RfQ09MT1VSID0gIiMwMGU1ZmYiClRQX0NPTE9VUiA9ICIjMjhkMTdjIgpGUF9DT0xPVVIgPSAiI2ZmM2IzMCIKCgpkZWYgc2VsZWN0X3NjZW5lcyhzaGlwczogcGQuRGF0YUZyYW1lLCBpbWFnZXM6IHBkLkRhdGFGcmFtZSwgc2VlZDogaW50LAogICAgICAgICAgICAgICAgICBwZXJfY2F0ZWdvcnk6IGludCA9IDIpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlBpY2sgYSBmaXhlZCwgcmV1c2FibGUgc2V0IG9mIHZhbCBzY2VuZXMgc3Bhbm5pbmcgdGhlIFBhcnQgUiBjYXNlcy4iIiIKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgdmFsID0gaW1hZ2VzWyhpbWFnZXNbIlNwbGl0Il0gPT0gIlZhbCIpICYgKH5pbWFnZXNbImlzX2JhY2tncm91bmQiXSldLmNvcHkoKQogICAgcyA9IHNoaXBzW3NoaXBzWyJTcGxpdCJdID09ICJWYWwiXQogICAgYWdnID0gcy5ncm91cGJ5KCJJbWdfSUQiKS5hZ2coCiAgICAgICAgbj0oImJhcmVhIiwgInNpemUiKSwgbWluX2FyZWE9KCJiYXJlYSIsICJtaW4iKSwKICAgICAgICBtYXhfYXJlYT0oImJhcmVhIiwgIm1heCIpLCBtZWRfYXJlYT0oImJhcmVhIiwgIm1lZGlhbiIpLAogICAgICAgIG5fc21hbGw9KCJzaXplX2JpbiIsIGxhbWJkYSB4OiAoeCA9PSAic21hbGwiKS5zdW0oKSksCiAgICAgICAgbl9sYXJnZT0oInNpemVfYmluIiwgbGFtYmRhIHg6ICh4ID09ICJsYXJnZSIpLnN1bSgpKSkKICAgIHZhbCA9IHZhbC5qb2luKGFnZywgb249IkltZ19JRCIpLmRyb3BuYShzdWJzZXQ9WyJuIl0pCgogICAgcGlja2VkOiBsaXN0W3R1cGxlW3N0ciwgaW50XV0gPSBbXQogICAgdGFrZW46IHNldFtpbnRdID0gc2V0KCkKCiAgICBkZWYgdGFrZShtYXNrLCBjYXRlZ29yeTogc3RyLCBrOiBpbnQgPSBwZXJfY2F0ZWdvcnkpIC0+IE5vbmU6CiAgICAgICAgY2FuZCA9IHZhbFttYXNrICYgfnZhbFsiSW1nX0lEIl0uaXNpbih0YWtlbildCiAgICAgICAgaWYgY2FuZC5lbXB0eToKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgaWR4ID0gcm5nLmNob2ljZShjYW5kLmluZGV4LnRvX251bXB5KCksIHNpemU9bWluKGssIGxlbihjYW5kKSksIHJlcGxhY2U9RmFsc2UpCiAgICAgICAgZm9yIGkgaW4gaWR4OgogICAgICAgICAgICBwaWNrZWQuYXBwZW5kKChjYXRlZ29yeSwgaW50KHZhbC5sb2NbaSwgIkltZ19JRCJdKSkpCiAgICAgICAgICAgIHRha2VuLmFkZChpbnQodmFsLmxvY1tpLCAiSW1nX0lEIl0pKQoKICAgIHRha2UoKHZhbFsibl9zbWFsbCJdID49IDMpICYgKHZhbFsibiJdIDw9IDEyKSwgInNtYWxsX3NoaXBzIikKICAgIHRha2UoKHZhbFsibWVkX2FyZWEiXS5iZXR3ZWVuKDEwMjQsIDkyMTYpKSAmICh2YWxbIm4iXS5iZXR3ZWVuKDIsIDEwKSksCiAgICAgICAgICJtZWRpdW1fc2hpcHMiKQogICAgdGFrZSh2YWxbIm5fbGFyZ2UiXSA+PSAxLCAibGFyZ2Vfc2hpcHMiKQogICAgdGFrZSh2YWxbIm4iXSA+PSAyNSwgImRlbnNlX3NjZW5lIikKICAgIHRha2UodmFsWyJuIl0gPT0gMSwgImlzb2xhdGVkX3NoaXAiKQogICAgIyBIYXJib3VycyBhbmQgbW9vcmVkIGNsdXN0ZXJzOiBtYW55IHNoaXBzIHBhY2tlZCBpbnRvIGEgc21hbGwgZm9vdHByaW50LAogICAgIyB3aGVyZSB0aGUgc3Vycm91bmRpbmdzIGxvb2sgbXVjaCBsaWtlIHRoZSB0YXJnZXRzLgogICAgdGFrZSgodmFsWyJuIl0gPj0gOCkgJiAodmFsWyJtZWRfYXJlYSJdIDwgMjAwMCksICJkaWZmaWN1bHRfYmFja2dyb3VuZCIpCgogICAgb3V0ID0gdmFsW3ZhbFsiSW1nX0lEIl0uaXNpbih0YWtlbildLmNvcHkoKQogICAgb3V0WyJzY2VuZV9jYXRlZ29yeSJdID0gb3V0WyJJbWdfSUQiXS5tYXAoZGljdCgoaSwgYykgZm9yIGMsIGkgaW4gcGlja2VkKSkKICAgIHJldHVybiBvdXQuc29ydF92YWx1ZXMoWyJzY2VuZV9jYXRlZ29yeSIsICJJbWdfSUQiXSkKCgpkZWYgbWF0Y2gocHJlZDogbnAubmRhcnJheSwgZ3Q6IG5wLm5kYXJyYXksIHRocjogZmxvYXQgPSAwLjUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJHcmVlZHkgSW9VIG1hdGNoaW5nOyByZXR1cm5zIGEgYm9vbCBhcnJheSBvZiB3aGljaCBwcmVkaWN0aW9ucyBoaXQuIiIiCiAgICBmcm9tIHNyYy5tZXRyaWNzIGltcG9ydCBpb3VfbWF0cml4CiAgICBpZiBsZW4ocHJlZCkgPT0gMDoKICAgICAgICByZXR1cm4gbnAuemVyb3MoMCwgZHR5cGU9Ym9vbCkKICAgIGlmIGxlbihndCkgPT0gMDoKICAgICAgICByZXR1cm4gbnAuemVyb3MobGVuKHByZWQpLCBkdHlwZT1ib29sKQogICAgaW91cyA9IGlvdV9tYXRyaXgocHJlZCwgZ3QpCiAgICB0YWtlbiA9IG5wLnplcm9zKGxlbihndCksIGR0eXBlPWJvb2wpCiAgICBoaXQgPSBucC56ZXJvcyhsZW4ocHJlZCksIGR0eXBlPWJvb2wpCiAgICBmb3IgaSBpbiByYW5nZShsZW4ocHJlZCkpOgogICAgICAgIGogPSBpbnQobnAuYXJnbWF4KG5wLndoZXJlKHRha2VuLCAtMS4wLCBpb3VzW2ldKSkpCiAgICAgICAgaWYgaW91c1tpLCBqXSA+PSB0aHIgYW5kIG5vdCB0YWtlbltqXToKICAgICAgICAgICAgdGFrZW5bal0gPSBoaXRbaV0gPSBUcnVlCiAgICByZXR1cm4gaGl0CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJlc3VsdHMtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0cy1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb25mIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjI1KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wZXItY2F0ZWdvcnkiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgbWF0cGxvdGxpYi51c2UoIkFnZyIpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5wYXRjaGVzIGFzIG1wYXRjaGVzCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKICAgIGZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCiAgICByZXN1bHRzX3Jvb3QgPSBmaW5kX3Jlc3VsdHNfcm9vdChhcmdzLnJlc3VsdHNfcm9vdCkKICAgIGRhdGFzZXRzID0gZmluZF9kYXRhc2V0c19yb290KGFyZ3MuZGF0YXNldHNfcm9vdCkKICAgIG91dF9kaXIgPSBQYXRoKGFyZ3Mub3V0KSBpZiBhcmdzLm91dCBlbHNlIHJlc3VsdHNfcm9vdCAvICJxdWFsaXRhdGl2ZSIKICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGFubiA9IGFubm90YXRpb25fZGlyKCkKICAgIHNoaXBzID0gcGQucmVhZF9wYXJxdWV0KGFubiAvICJzaGlwX2Fubm90YXRpb25zLnBhcnF1ZXQiKQogICAgaW1hZ2VzID0gcGQucmVhZF9jc3YoYW5uIC8gImltYWdlX21hbmlmZXN0LmNzdiIpCgogICAgbW9kZWxzID0ge30KICAgIGZvciBsYWJlbCwgcnVuX2lkLCBfIGluIENPTkRJVElPTlM6CiAgICAgICAgdyA9IHJlc3VsdHNfcm9vdCAvIHJ1bl9pZCAvICJiZXN0LnB0IgogICAgICAgIGlmIHcuZXhpc3RzKCk6CiAgICAgICAgICAgIG1vZGVsc1tsYWJlbF0gPSBZT0xPKHN0cih3KSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludChmIiAgbm8gY2hlY2twb2ludCBmb3Ige2xhYmVsfSAoe3d9KTsgaXQgd2lsbCBiZSBza2lwcGVkIikKICAgIGlmIG5vdCBtb2RlbHM6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIk5vIHRyYWluZWQgY2hlY2twb2ludHMgdW5kZXIge3Jlc3VsdHNfcm9vdH0uIikKCiAgICBzY2VuZXMgPSBzZWxlY3Rfc2NlbmVzKHNoaXBzLCBpbWFnZXMsIGFyZ3Muc2VlZCwgYXJncy5wZXJfY2F0ZWdvcnkpCiAgICBwcmludChmIntsZW4oc2NlbmVzKX0gc2NlbmUocykgYWNyb3NzICIKICAgICAgICAgIGYie3NjZW5lc1snc2NlbmVfY2F0ZWdvcnknXS5udW5pcXVlKCl9IGNhdGVnb3JpZXMsICIKICAgICAgICAgIGYie2xlbihtb2RlbHMpfSBjb25kaXRpb24ocykgd2l0aCBjaGVja3BvaW50c1xuIikKCiAgICBpbmRleCA9IFtdCiAgICBmb3IgciBpbiBzY2VuZXMuaXRlcnR1cGxlcygpOgogICAgICAgIGd0ID0gc2hpcHNbc2hpcHNbIkltZ19JRCJdID09IHIuSW1nX0lEXQogICAgICAgIGd0X2JveGVzID0gZ3RbWyJ4X21pbiIsICJ5X21pbiIsICJ4X21heCIsICJ5X21heCJdXS50b19udW1weShmbG9hdCkKICAgICAgICBzdGVtID0gUGF0aChyLmJhc2VuYW1lKS5zdGVtCgogICAgICAgIGF2YWlsID0gWyhsLCBpLCBzKSBmb3IgbCwgaSwgcyBpbiBDT05ESVRJT05TIGlmIGwgaW4gbW9kZWxzXQogICAgICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygxLCBsZW4oYXZhaWwpLCBmaWdzaXplPSg0LjYgKiBsZW4oYXZhaWwpLCA1LjIpKQogICAgICAgIGF4ZXMgPSBucC5hdGxlYXN0XzFkKGF4ZXMpCiAgICAgICAgcm93ID0geyJJbWdfSUQiOiBpbnQoci5JbWdfSUQpLCAiaW1hZ2UiOiByLmJhc2VuYW1lLAogICAgICAgICAgICAgICAic2NlbmVfY2F0ZWdvcnkiOiByLnNjZW5lX2NhdGVnb3J5LCAibl9ndCI6IGludChsZW4oZ3RfYm94ZXMpKSwKICAgICAgICAgICAgICAgInBlcl9jb25kaXRpb24iOiB7fX0KCiAgICAgICAgZm9yIGF4LCAobGFiZWwsIF8sIHNjYWxlKSBpbiB6aXAoYXhlcywgYXZhaWwpOgogICAgICAgICAgICBwID0gZGF0YXNldHNbbGFiZWxdIC8gImltYWdlcyIgLyAidmFsIiAvIGYie3N0ZW19LmpwZyIKICAgICAgICAgICAgaW1nID0gbnAuYXNhcnJheShJbWFnZS5vcGVuKHApLmNvbnZlcnQoIlJHQiIpKQogICAgICAgICAgICBoLCB3ID0gaW1nLnNoYXBlWzoyXQogICAgICAgICAgICBzeCwgc3kgPSB3IC8gci5JbWFnZVdpZHRoLCBoIC8gci5JbWFnZUhlaWdodAoKICAgICAgICAgICAgcmVzID0gbW9kZWxzW2xhYmVsXS5wcmVkaWN0KHN0cihwKSwgaW1nc3o9MTAyNCwgY29uZj1hcmdzLmNvbmYsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpb3U9MC43LCBtYXhfZGV0PTEwMDAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZXZpY2U9YXJncy5kZXZpY2UsIHZlcmJvc2U9RmFsc2UpWzBdCiAgICAgICAgICAgIGlmIHJlcy5ib3hlcyBpcyBub3QgTm9uZSBhbmQgbGVuKHJlcy5ib3hlcyk6CiAgICAgICAgICAgICAgICBwcmVkID0gcmVzLmJveGVzLnh5eHkuY3B1KCkubnVtcHkoKS5hc3R5cGUoZmxvYXQpCiAgICAgICAgICAgICAgICBzY29yZXMgPSByZXMuYm94ZXMuY29uZi5jcHUoKS5udW1weSgpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBwcmVkLCBzY29yZXMgPSBucC56ZXJvcygoMCwgNCkpLCBucC56ZXJvcygwKQoKICAgICAgICAgICAgIyBNYXRjaCBpbiBvcmlnaW5hbC1yZXNvbHV0aW9uIHNwYWNlIHNvIHRoZSBzYW1lIElvVSBjcml0ZXJpb24KICAgICAgICAgICAgIyBhcHBsaWVzIGF0IGV2ZXJ5IGNvbmRpdGlvbi4KICAgICAgICAgICAgcHJlZF9vcmlnID0gcHJlZC5jb3B5KCkKICAgICAgICAgICAgaWYgbGVuKHByZWRfb3JpZyk6CiAgICAgICAgICAgICAgICBwcmVkX29yaWdbOiwgWzAsIDJdXSAvPSBzeAogICAgICAgICAgICAgICAgcHJlZF9vcmlnWzosIFsxLCAzXV0gLz0gc3kKICAgICAgICAgICAgaGl0ID0gbWF0Y2gocHJlZF9vcmlnLCBndF9ib3hlcykKCiAgICAgICAgICAgIGF4Lmltc2hvdyhpbWcsIGludGVycG9sYXRpb249Im5lYXJlc3QiKQogICAgICAgICAgICBmb3IgYiBpbiBndF9ib3hlczoKICAgICAgICAgICAgICAgIGF4LmFkZF9wYXRjaChtcGF0Y2hlcy5SZWN0YW5nbGUoCiAgICAgICAgICAgICAgICAgICAgKGJbMF0gKiBzeCwgYlsxXSAqIHN5KSwgKGJbMl0gLSBiWzBdKSAqIHN4LCAoYlszXSAtIGJbMV0pICogc3ksCiAgICAgICAgICAgICAgICAgICAgZmlsbD1GYWxzZSwgZWRnZWNvbG9yPUdUX0NPTE9VUiwgbGluZXdpZHRoPTEuMCwgbGluZXN0eWxlPSI6IikpCiAgICAgICAgICAgIGZvciBiLCBvayBpbiB6aXAocHJlZCwgaGl0KToKICAgICAgICAgICAgICAgIGF4LmFkZF9wYXRjaChtcGF0Y2hlcy5SZWN0YW5nbGUoCiAgICAgICAgICAgICAgICAgICAgKGJbMF0sIGJbMV0pLCBiWzJdIC0gYlswXSwgYlszXSAtIGJbMV0sIGZpbGw9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgZWRnZWNvbG9yPVRQX0NPTE9VUiBpZiBvayBlbHNlIEZQX0NPTE9VUiwgbGluZXdpZHRoPTEuMykpCgogICAgICAgICAgICB0cCwgZnAgPSBpbnQoaGl0LnN1bSgpKSwgaW50KCh+aGl0KS5zdW0oKSkKICAgICAgICAgICAgZm4gPSBsZW4oZ3RfYm94ZXMpIC0gdHAKICAgICAgICAgICAgYXguc2V0X3RpdGxlKGYie2xhYmVsfSAoe3d9eHtofSlcblRQIHt0cH0gIEZQIHtmcH0gIEZOIHtmbn0iLCBmb250c2l6ZT0xMCkKICAgICAgICAgICAgYXguc2V0X3h0aWNrcyhbXSk7IGF4LnNldF95dGlja3MoW10pCiAgICAgICAgICAgIHJvd1sicGVyX2NvbmRpdGlvbiJdW2xhYmVsXSA9IHsidHAiOiB0cCwgImZwIjogZnAsICJmbiI6IGZuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm5fcHJlZCI6IGludChsZW4ocHJlZCkpfQoKICAgICAgICBmaWcuc3VwdGl0bGUoCiAgICAgICAgICAgIGYie3Iuc2NlbmVfY2F0ZWdvcnkucmVwbGFjZSgnXycsICcgJyl9IOKAlCB7ci5iYXNlbmFtZX0g4oCUICIKICAgICAgICAgICAgZiJ7bGVuKGd0X2JveGVzKX0gc2hpcHMgICAiCiAgICAgICAgICAgIGYiKGRvdHRlZCBjeWFuID0gZ3JvdW5kIHRydXRoLCBncmVlbiA9IG1hdGNoZWQsIHJlZCA9IGZhbHNlIHBvc2l0aXZlOyAiCiAgICAgICAgICAgIGYiY29uZiDiiaUge2FyZ3MuY29uZn0pIiwgZm9udHNpemU9MTEsIHk9MS4wMikKICAgICAgICBmaWcudGlnaHRfbGF5b3V0KHJlY3Q9KDAsIDAsIDEsIDAuOTYpKQogICAgICAgIG5hbWUgPSBmIntyLnNjZW5lX2NhdGVnb3J5fV9fe3N0ZW19LnBuZyIKICAgICAgICBmaWcuc2F2ZWZpZyhvdXRfZGlyIC8gbmFtZSwgZHBpPTEzMCwgYmJveF9pbmNoZXM9InRpZ2h0IikKICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIHJvd1siZmlndXJlIl0gPSBuYW1lCiAgICAgICAgaW5kZXguYXBwZW5kKHJvdykKICAgICAgICBwcmludChmIiAge25hbWV9ICAgIiArICIgICIuam9pbigKICAgICAgICAgICAgZiJ7bH06e3Jvd1sncGVyX2NvbmRpdGlvbiddW2xdWyd0cCddfS97cm93WyduX2d0J119IgogICAgICAgICAgICBmb3IgbCwgXywgXyBpbiBhdmFpbCkpCgogICAgKG91dF9kaXIgLyAic2NlbmVfaW5kZXguanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyh7ImNvbmZfdGhyZXNob2xkIjogYXJncy5jb25mLCAic2VlZCI6IGFyZ3Muc2VlZCwKICAgICAgICAgICAgICAgICAgICAiY29uZGl0aW9ucyI6IFtsIGZvciBsLCBfLCBfIGluIENPTkRJVElPTlMgaWYgbCBpbiBtb2RlbHNdLAogICAgICAgICAgICAgICAgICAgICJub3RlIjogIlNjZW5lcyBhcmUgZml4ZWQgYWNyb3NzIGNvbmRpdGlvbnM7IFRQL0ZQL0ZOIGFyZSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAibWF0Y2hlZCBhdCBJb1UgMC41IGluIG9yaWdpbmFsLXJlc29sdXRpb24gIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgImNvb3JkaW5hdGVzLiIsCiAgICAgICAgICAgICAgICAgICAgInNjZW5lcyI6IGluZGV4fSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcHJpbnQoZiJcbldyb3RlIHtsZW4oaW5kZXgpfSBmaWd1cmUocykgdG8ge291dF9kaXJ9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
    "scripts/run_all.py": "IiIiClJ1biB0aGUgRTAwLUUwMyBzd2VlcCBvbiBLYWdnbGUsIG9uZSBjb25kaXRpb24gYWZ0ZXIgYW5vdGhlci4KCiAgICBweXRob24gc2NyaXB0cy9ydW5fYWxsLnB5CiAgICBweXRob24gc2NyaXB0cy9ydW5fYWxsLnB5IC0tb3JkZXIgY2hlYXBlc3QtZmlyc3QKICAgIHB5dGhvbiBzY3JpcHRzL3J1bl9hbGwucHkgLS1zYW5pdHkgLS1lcG9jaHMgMQogICAgcHl0aG9uIHNjcmlwdHMvcnVuX2FsbC5weSAtLXRpbWUtYnVkZ2V0IDEuNQoKRWFjaCBjb25kaXRpb24gaXMgYW4gaW5kZXBlbmRlbnQgc3VicHJvY2Vzcy4gVGhhdCBpcyBkZWxpYmVyYXRlOiBhIGNyYXNoZWQgb3IKT09NLWtpbGxlZCBydW4gY2Fubm90IHRha2UgdGhlIHJlc3Qgb2YgdGhlIHN3ZWVwIHdpdGggaXQsIGFuZCBDVURBIG1lbW9yeSBpcwpyZXR1cm5lZCBjbGVhbmx5IGJldHdlZW4gY29uZGl0aW9ucyBpbnN0ZWFkIG9mIGFjY3VtdWxhdGluZy4KCkNvbmRpdGlvbnMgdGhhdCBhbHJlYWR5IGhhdmUgYSBtZXRyaWNzLmpzb24gYXJlIHNraXBwZWQsIHNvIHJlLXJ1bm5pbmcgYWZ0ZXIgYQpLYWdnbGUgc2Vzc2lvbiBpbnRlcnJ1cHRpb24gcmVzdW1lcyB0aGUgc3dlZXAgcmF0aGVyIHRoYW4gcmVzdGFydGluZyBpdC4gQQpjb25kaXRpb24gdGhhdCBoYXMgYSBgbGFzdC5wdGAgYnV0IG5vIGBtZXRyaWNzLmpzb25gIHJlc3VtZXMgdHJhaW5pbmcgZnJvbQp0aGF0IGNoZWNrcG9pbnQgKHNlZSBydW5fZXhwZXJpbWVudC5weSAvIHNyYy90cmFpbmluZy5weSkgaW5zdGVhZCBvZiBzdGFydGluZwphdCBlcG9jaCAxLgoKLS10aW1lLWJ1ZGdldCBIT1VSUyBjYXBzIHRoZSBUT1RBTCB3YWxsLWNsb2NrIHRpbWUgb2YgdGhpcyBvbmUgaW52b2NhdGlvbiwKYWNyb3NzIGhvd2V2ZXIgbWFueSBjb25kaXRpb25zIGZpdCAtLSB0aGUgaW50ZW5kZWQgd2F5IHRvIHJ1biB0aGlzIG9uIEthZ2dsZSwKd2hlcmUgYSBmdWxsIHN3ZWVwICh+MTItMjAgR1BVLWhvdXJzKSBkb2VzIG5vdCBmaXQgaW4gb25lIHNlc3Npb24uIEJlZm9yZSBlYWNoCmNvbmRpdGlvbiwgdGhlIHJlbWFpbmluZyBidWRnZXQgaXMgcGFzc2VkIHRocm91Z2ggYXMgdGhhdCBjb25kaXRpb24ncyBvd24KLS10aW1lOyBvbmNlIHJlbWFpbmluZyB0aW1lIGRyb3BzIGJlbG93IGEgdXNlZnVsIG1pbmltdW0sIHRoaXMgc3RvcHMgKGxlYXZpbmcKd2hhdGV2ZXIncyBsZWZ0IGZvciB0aGUgbmV4dCBpbnZvY2F0aW9uKSByYXRoZXIgdGhhbiBzdGFydGluZyBhIGNvbmRpdGlvbiBpdApjYW4ndCBtYWtlIG1lYW5pbmdmdWwgcHJvZ3Jlc3Mgb24uIFJlLXJ1bm5pbmcgdGhlIFNBTUUgY29tbWFuZCBsYXRlciByZXN1bWVzCmV4YWN0bHkgd2hlcmUgdGhpcyBzdG9wcGVkOiBhbHJlYWR5LWZpbmlzaGVkIGNvbmRpdGlvbnMgYXJlIHNraXBwZWQsIHRoZQppbi1wcm9ncmVzcyBvbmUgcmVzdW1lcyBmcm9tIGl0cyBvd24gY2hlY2twb2ludC4gVGhpcyBpcyBtZWFudCB0byBiZSBjb21taXRlZAooU2F2ZSBWZXJzaW9uIC0+IFNhdmUgJiBSdW4gQWxsKSByZXBlYXRlZGx5LCBlYWNoIGNvbW1pdCBkb2luZyBvbmUgYnVkZ2V0J3MKd29ydGggb2Ygd29yaywgdW50aWwgZXhwZXJpbWVudF9zdGF0dXMuanNvbiBzaG93cyBldmVyeXRoaW5nIGRvbmUuCgpCZWZvcmUgc3RhcnRpbmcsIHRoaXMgYWxzbyBsb29rcyBmb3IgYSBwcmV2aW91c2x5LWF0dGFjaGVkIEthZ2dsZSBOb3RlYm9vawpPdXRwdXQgb2YgdGhpcyBzdHVkeSdzIHJlc3VsdHMgdW5kZXIgL2thZ2dsZS9pbnB1dCAoc2VlCnNyYy9wYXRocy5yZXN0b3JlX3ByZXZpb3VzX3Jlc3VsdHMpIGFuZCByZXN0b3JlcyBpdCBpbnRvIHRoZSByZXN1bHRzIHJvb3QsIHNvCmEgZnJlc2ggS2FnZ2xlIHNlc3Npb24gdGhhdCByZWF0dGFjaGVzIGl0cyBvd24gcHJpb3Igb3V0cHV0IHJlc3VtZXMgaW5zdGVhZCBvZgpzdGFydGluZyBvdmVyLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUEtHID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQS0cpKQoKZnJvbSBzcmMucGF0aHMgaW1wb3J0IChmaW5kX3Jlc3VtZV9jaGVja3BvaW50LCBmaW5kX3Jlc3VsdHNfcm9vdCwKICAgICAgICAgICAgICAgICAgICAgICBpc19leHBlcmltZW50X2NvbXBsZXRlLCByZXN0b3JlX3ByZXZpb3VzX3Jlc3VsdHMpCgojIERlY2xhcmF0aW9uIG9yZGVyID0gdGhlIG9yZGVyIHJlc3VsdHMgYXJlIHJlcG9ydGVkIGluLgpDT05ESVRJT05TID0gWwogICAgKCJiYXNlbGluZSIsICIwMF9iYXNlbGluZSIpLAogICAgKCJyNzUiLCAiMDFfcjc1IiksCiAgICAoInI1MCIsICIwMl9yNTAiKSwKICAgICgicjI1IiwgIjAzX3IyNSIpLApdCgojIEJlbG93IHRoaXMgbXVjaCByZW1haW5pbmcgYnVkZ2V0LCBkb24ndCBzdGFydCAob3IgcmVzdW1lKSBhIGNvbmRpdGlvbiAtLQojIHRoZXJlIGlzbid0IGVub3VnaCB0aW1lIGxlZnQgdG8gbWFrZSBtZWFuaW5nZnVsIHByb2dyZXNzLCBzbyBpdCdzIGJldHRlciB0bwojIHN0b3AgY2xlYW5seSBhbmQgbGVhdmUgaXQgd2hvbGUgZm9yIHRoZSBuZXh0IGludm9jYXRpb24uCk1JTl9VU0VGVUxfU0xJQ0VfSE9VUlMgPSAwLjEgICMgNiBtaW51dGVzCgoKZGVmIHJlbWFpbmluZ19ob3VycyhkZWFkbGluZTogZmxvYXQpIC0+IGZsb2F0OgogICAgcmV0dXJuIChkZWFkbGluZSAtIHRpbWUudGltZSgpKSAvIDM2MDAKCgpkZWYgd3JpdGVfc3RhdHVzKHJlc3VsdHNfcm9vdDogUGF0aCwgc3VtbWFyeTogbGlzdFt0dXBsZVtzdHIsIHN0ciwgZmxvYXRdXSkgLT4gTm9uZToKICAgIHN0YXR1cyA9IHtuYW1lOiB7InN0YXR1cyI6IHMsICJtaW51dGVzIjogcm91bmQobSwgMSl9IGZvciBuYW1lLCBzLCBtIGluIHN1bW1hcnl9CiAgICAocmVzdWx0c19yb290IC8gImV4cGVyaW1lbnRfc3RhdHVzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMoc3RhdHVzLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW9yZGVyIiwgY2hvaWNlcz1bImRlY2xhcmVkIiwgImNoZWFwZXN0LWZpcnN0Il0sCiAgICAgICAgICAgICAgICAgICAgZGVmYXVsdD0iZGVjbGFyZWQiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImNoZWFwZXN0LWZpcnN0IHJ1bnMgcjI1IC0+IG9yaWdpbmFsLCBzbyBhIHRydW5jYXRlZCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAic2Vzc2lvbiBzdGlsbCB5aWVsZHMgYSBwYXJ0aWFsIHRyZW5kIGxpbmUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJlc3VsdHMtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0cy1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1mcmFjdGlvbiIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zYW5pdHkiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWZvcmNlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zdGFnZS10by13b3JraW5nIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10aW1lLWJ1ZGdldCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJ0b3RhbCB3YWxsLWNsb2NrIGhvdXJzIGZvciB0aGlzIHdob2xlIGludm9jYXRpb247ICIKICAgICAgICAgICAgICAgICAgICAgICAgICJzZWUgbW9kdWxlIGRvY3N0cmluZy4gSWdub3JlZCBmb3IgLS1zYW5pdHkgKGFscmVhZHkgIgogICAgICAgICAgICAgICAgICAgICAgICAgImZhc3QvZml4ZWQtZXBvY2gpLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbm8tcmVzdG9yZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ic2tpcCBzZWFyY2hpbmcgL2thZ2dsZS9pbnB1dCBmb3IgYSBwcmV2aW91cyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiTm90ZWJvb2sgT3V0cHV0IHRvIHJlc3VtZSBmcm9tIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBvcmRlciA9IENPTkRJVElPTlMgaWYgYXJncy5vcmRlciA9PSAiZGVjbGFyZWQiIGVsc2UgbGlzdChyZXZlcnNlZChDT05ESVRJT05TKSkKICAgIHJlc3VsdHNfcm9vdCA9IGZpbmRfcmVzdWx0c19yb290KGFyZ3MucmVzdWx0c19yb290KQoKICAgIGlmIG5vdCBhcmdzLm5vX3Jlc3RvcmUgYW5kIG5vdCBhcmdzLnNhbml0eToKICAgICAgICBwcmV2ID0gcmVzdG9yZV9wcmV2aW91c19yZXN1bHRzKHJlc3VsdHNfcm9vdCkKICAgICAgICBpZiBwcmV2OgogICAgICAgICAgICBwcmludChmIlJlc3RvcmVkIHByZXZpb3VzIHJlc3VsdHMgZnJvbSB7cHJldn0gLT4ge3Jlc3VsdHNfcm9vdH1cbiIpCgogICAgcGFzc3Rocm91Z2g6IGxpc3Rbc3RyXSA9IFtdCiAgICBmb3IgZmxhZywgdmFsIGluIFsoIi0tZGF0YXNldHMtcm9vdCIsIGFyZ3MuZGF0YXNldHNfcm9vdCksCiAgICAgICAgICAgICAgICAgICAgICAoIi0tcmVzdWx0cy1yb290Iiwgc3RyKHJlc3VsdHNfcm9vdCkpLAogICAgICAgICAgICAgICAgICAgICAgKCItLWRldmljZSIsIGFyZ3MuZGV2aWNlKSwgKCItLWJhdGNoIiwgYXJncy5iYXRjaCksCiAgICAgICAgICAgICAgICAgICAgICAoIi0td29ya2VycyIsIGFyZ3Mud29ya2VycyksICgiLS1lcG9jaHMiLCBhcmdzLmVwb2NocyksCiAgICAgICAgICAgICAgICAgICAgICAoIi0tZnJhY3Rpb24iLCBhcmdzLmZyYWN0aW9uKV06CiAgICAgICAgaWYgdmFsIGlzIG5vdCBOb25lOgogICAgICAgICAgICBwYXNzdGhyb3VnaCArPSBbZmxhZywgc3RyKHZhbCldCiAgICBpZiBhcmdzLnNhbml0eToKICAgICAgICBwYXNzdGhyb3VnaC5hcHBlbmQoIi0tc2FuaXR5IikKICAgIGlmIGFyZ3MuZm9yY2U6CiAgICAgICAgcGFzc3Rocm91Z2guYXBwZW5kKCItLWZvcmNlIikKICAgIGlmIGFyZ3Muc3RhZ2VfdG9fd29ya2luZzoKICAgICAgICBwYXNzdGhyb3VnaC5hcHBlbmQoIi0tc3RhZ2UtdG8td29ya2luZyIpCgogICAgZGVhZGxpbmUgPSAodGltZS50aW1lKCkgKyBhcmdzLnRpbWVfYnVkZ2V0ICogMzYwMAogICAgICAgICAgICAgICBpZiBhcmdzLnRpbWVfYnVkZ2V0IGFuZCBub3QgYXJncy5zYW5pdHkgZWxzZSBOb25lKQoKICAgIHByaW50KGYiU3dlZXAgb3JkZXI6IHsnIC0+ICcuam9pbihjWzBdIGZvciBjIGluIG9yZGVyKX0iKQogICAgcHJpbnQoZiJSZXN1bHRzIHJvb3Q6IHtyZXN1bHRzX3Jvb3R9IikKICAgIGlmIGRlYWRsaW5lOgogICAgICAgIHByaW50KGYiVGltZSBidWRnZXQ6IHthcmdzLnRpbWVfYnVkZ2V0Oi4yZn1oIGZvciB0aGlzIGludm9jYXRpb24iKQogICAgcHJpbnQoKQoKICAgIHN1bW1hcnkgPSBbXQogICAgZm9yIGNmZ19uYW1lLCBydW5faWQgaW4gb3JkZXI6CiAgICAgICAgbmFtZSA9IHJ1bl9pZCArICgiX3Nhbml0eSIgaWYgYXJncy5zYW5pdHkgZWxzZSAiIikKICAgICAgICBvdXRfZGlyID0gcmVzdWx0c19yb290IC8gbmFtZQogICAgICAgIGlmIGlzX2V4cGVyaW1lbnRfY29tcGxldGUob3V0X2RpcikgYW5kIG5vdCBhcmdzLmZvcmNlOgogICAgICAgICAgICBwcmludChmIltza2lwXSB7bmFtZX0gYWxyZWFkeSBoYXMgbWV0cmljcy5qc29uIikKICAgICAgICAgICAgc3VtbWFyeS5hcHBlbmQoKG5hbWUsICJza2lwcGVkIChhbHJlYWR5IGNvbXBsZXRlKSIsIDAuMCkpCiAgICAgICAgICAgIHdyaXRlX3N0YXR1cyhyZXN1bHRzX3Jvb3QsIHN1bW1hcnkpCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHRoaXNfcGFzc3Rocm91Z2ggPSBsaXN0KHBhc3N0aHJvdWdoKQogICAgICAgIGlmIGRlYWRsaW5lIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZW1haW5pbmcgPSByZW1haW5pbmdfaG91cnMoZGVhZGxpbmUpCiAgICAgICAgICAgIGlmIHJlbWFpbmluZyA8PSBNSU5fVVNFRlVMX1NMSUNFX0hPVVJTOgogICAgICAgICAgICAgICAgcHJpbnQoZiJbc3RvcF0gdGltZSBidWRnZXQgZXhoYXVzdGVkICh7bWF4KHJlbWFpbmluZywgMCkgKiA2MDouMGZ9ICIKICAgICAgICAgICAgICAgICAgICAgIGYibWluIGxlZnQsIGJlbG93IHRoZSB7TUlOX1VTRUZVTF9TTElDRV9IT1VSUyAqIDYwOi4wZn0tbWluICIKICAgICAgICAgICAgICAgICAgICAgIGYibWluaW11bSkgLS0gbGVhdmluZyB7bmFtZX0gb253YXJkIGZvciB0aGUgbmV4dCBydW4iKQogICAgICAgICAgICAgICAgZm9yIHJlc3RfY2ZnLCByZXN0X2lkIGluIG9yZGVyW29yZGVyLmluZGV4KChjZmdfbmFtZSwgcnVuX2lkKSk6XToKICAgICAgICAgICAgICAgICAgICByZXN0X25hbWUgPSByZXN0X2lkICsgKCJfc2FuaXR5IiBpZiBhcmdzLnNhbml0eSBlbHNlICIiKQogICAgICAgICAgICAgICAgICAgIHN1bW1hcnkuYXBwZW5kKChyZXN0X25hbWUsICJub3QgYXR0ZW1wdGVkICh0aW1lIGJ1ZGdldCkiLCAwLjApKQogICAgICAgICAgICAgICAgd3JpdGVfc3RhdHVzKHJlc3VsdHNfcm9vdCwgc3VtbWFyeSkKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIHByaW50KGYiW2J1ZGdldF0ge3JlbWFpbmluZzouMmZ9aCByZW1haW5pbmcgLT4gZ2l2aW5nIHtuYW1lfSB1cCB0byB0aGF0IikKICAgICAgICAgICAgdGhpc19wYXNzdGhyb3VnaCArPSBbIi0tdGltZSIsIGYie3JlbWFpbmluZzouM2Z9Il0KCiAgICAgICAgcmVzdW1lX2NrcHQgPSBOb25lIGlmIGFyZ3MuZm9yY2UgZWxzZSBmaW5kX3Jlc3VtZV9jaGVja3BvaW50KG91dF9kaXIsIHJ1bl9pZCkKICAgICAgICBpZiByZXN1bWVfY2twdDoKICAgICAgICAgICAgcHJpbnQoZiJbcmVzdW1lXSB7bmFtZX0gaGFzIGEgY2hlY2twb2ludCBhdCB7cmVzdW1lX2NrcHR9OyBjb250aW51aW5nICIKICAgICAgICAgICAgICAgICAgZiJmcm9tIHRoZXJlIHJhdGhlciB0aGFuIGVwb2NoIDEuIikKCiAgICAgICAgY21kID0gW3N5cy5leGVjdXRhYmxlLCBzdHIoUEtHIC8gInNjcmlwdHMiIC8gInJ1bl9leHBlcmltZW50LnB5IiksCiAgICAgICAgICAgICAgICItLWNvbmZpZyIsIHN0cihQS0cgLyAiY29uZmlncyIgLyBmIntjZmdfbmFtZX0ueWFtbCIpXSArIHRoaXNfcGFzc3Rocm91Z2gKICAgICAgICBwcmludChmIlxueycjJyAqIDcwfVxuIyB7bmFtZX1cbiMgeycgJy5qb2luKGNtZCl9XG57JyMnICogNzB9IiwgZmx1c2g9VHJ1ZSkKCiAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgIHJjID0gc3VicHJvY2Vzcy5jYWxsKGNtZCkKICAgICAgICBlbCA9ICh0aW1lLnRpbWUoKSAtIHQwKSAvIDYwCiAgICAgICAgaWYgcmMgIT0gMDoKICAgICAgICAgICAgc3RhdHVzID0gZiJGQUlMRUQgKHJjPXtyY30pIgogICAgICAgIGVsaWYgaXNfZXhwZXJpbWVudF9jb21wbGV0ZShvdXRfZGlyKToKICAgICAgICAgICAgc3RhdHVzID0gIm9rIgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHN0YXR1cyA9ICJpbiBwcm9ncmVzcyAodGltZSBidWRnZXQ7IHJlc3VtZSBuZXh0IHJ1bikiCiAgICAgICAgc3VtbWFyeS5hcHBlbmQoKG5hbWUsIHN0YXR1cywgZWwpKQogICAgICAgIHdyaXRlX3N0YXR1cyhyZXN1bHRzX3Jvb3QsIHN1bW1hcnkpCiAgICAgICAgaWYgcmMgIT0gMDoKICAgICAgICAgICAgIyBLZWVwIGdvaW5nOiB0aHJlZSBnb29kIGNvbmRpdGlvbnMgYmVhdCBub25lLCBhbmQgYSBwYXJ0aWFsIHN3ZWVwCiAgICAgICAgICAgICMgaXMgc3RpbGwgcmVwb3J0YWJsZSBhcyBsb25nIGFzIHRoZSBnYXAgaXMgc3RhdGVkLgogICAgICAgICAgICBwcmludChmIlxuISEge25hbWV9IGV4aXRlZCB3aXRoIHtyY307IGNvbnRpbnVpbmcgd2l0aCB0aGUgcmVzdC5cbiIpCiAgICAgICAgZWxpZiBzdGF0dXMuc3RhcnRzd2l0aCgiaW4gcHJvZ3Jlc3MiKToKICAgICAgICAgICAgIyBUaGlzIGludm9jYXRpb24ncyBidWRnZXQgaXMgbm93IGNlcnRhaW5seSBzcGVudCAodGhlIGNvbmRpdGlvbgogICAgICAgICAgICAjIGl0c2VsZiB1c2VkIHVwIGl0cyBhbGxvdHRlZCAtLXRpbWUpOyBubyBwb2ludCB0cnlpbmcgdGhlIG5leHQKICAgICAgICAgICAgIyBjb25kaXRpb24gd2l0aCB3aGF0ZXZlciBzbGl2ZXIgaXMgbm9taW5hbGx5IGxlZnQuCiAgICAgICAgICAgIGZvciByZXN0X2NmZywgcmVzdF9pZCBpbiBvcmRlcltvcmRlci5pbmRleCgoY2ZnX25hbWUsIHJ1bl9pZCkpICsgMTpdOgogICAgICAgICAgICAgICAgcmVzdF9uYW1lID0gcmVzdF9pZCArICgiX3Nhbml0eSIgaWYgYXJncy5zYW5pdHkgZWxzZSAiIikKICAgICAgICAgICAgICAgIHN1bW1hcnkuYXBwZW5kKChyZXN0X25hbWUsICJub3QgYXR0ZW1wdGVkICh0aW1lIGJ1ZGdldCkiLCAwLjApKQogICAgICAgICAgICB3cml0ZV9zdGF0dXMocmVzdWx0c19yb290LCBzdW1tYXJ5KQogICAgICAgICAgICBicmVhawoKICAgIHByaW50KGYiXG57Jz0nICogNzB9XG5TV0VFUCBTVU1NQVJZXG57Jz0nICogNzB9IikKICAgIGZvciBuYW1lLCBzdGF0dXMsIGVsIGluIHN1bW1hcnk6CiAgICAgICAgcHJpbnQoZiIgIHtuYW1lOjwyMH0ge3N0YXR1czo8MzJ9IHtlbDo2LjFmfSBtaW4iKQogICAgZmFpbGVkID0gW3MgZm9yIHMgaW4gc3VtbWFyeSBpZiBzWzFdLnN0YXJ0c3dpdGgoIkZBSUxFRCIpXQogICAgZG9uZSA9IFtzIGZvciBzIGluIHN1bW1hcnkgaWYgc1sxXSA9PSAib2siIG9yIHNbMV0uc3RhcnRzd2l0aCgic2tpcHBlZCIpXQogICAgcHJpbnQoZiJcbntsZW4oZG9uZSl9L3tsZW4oQ09ORElUSU9OUyl9IGNvbmRpdGlvbihzKSBjb21wbGV0ZS4iKQogICAgcmVtYWluaW5nX3dvcmsgPSBbcyBmb3IgcyBpbiBzdW1tYXJ5IGlmIG5vdCAoc1sxXSA9PSAib2siIG9yIHNbMV0uc3RhcnRzd2l0aCgic2tpcHBlZCIpKV0KICAgIGlmIHJlbWFpbmluZ193b3JrIGFuZCBub3QgZmFpbGVkOgogICAgICAgIHByaW50KCJOb3QgZmluaXNoZWQgeWV0IC0tIHJlLXJ1biB0aGlzIHNhbWUgY29tbWFuZCAob3IgY29tbWl0IHRoaXMgIgogICAgICAgICAgICAgICJub3RlYm9vayBhZ2FpbikgdG8gY29udGludWUgZnJvbSBoZXJlLiIpCiAgICBlbGlmIGxlbihkb25lKSA9PSBsZW4oQ09ORElUSU9OUyk6CiAgICAgICAgcHJpbnQoZiJcbk5leHQ6IHB5dGhvbiBzY3JpcHRzL2NvbGxlY3RfcmVzdWx0cy5weSIpCiAgICBwcmludCgiXG5SZW1lbWJlciB0byBTYXZlIFZlcnNpb24gKFNhdmUgJiBSdW4gQWxsKSBzbyB0aGlzIHByb2dyZXNzICIKICAgICAgICAgICJwZXJzaXN0cyBhcyBhIEthZ2dsZSBOb3RlYm9vayBPdXRwdXQgLS0gL2thZ2dsZS93b3JraW5nIGlzIG5vdCAiCiAgICAgICAgICAiZ3VhcmFudGVlZCB0byBzdXJ2aXZlIGEgZnJlc2ggc2Vzc2lvbi4iKQogICAgaWYgZmFpbGVkOgogICAgICAgIHN5cy5leGl0KDEpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "scripts/run_experiment.py": "IiIiClRyYWluIGFuZCBldmFsdWF0ZSBPTkUgcmVzb2x1dGlvbiBjb25kaXRpb24gZW5kIHRvIGVuZCwgb24gS2FnZ2xlLgoKICAgIHB5dGhvbiBzY3JpcHRzL3J1bl9leHBlcmltZW50LnB5IC0tY29uZmlnIGNvbmZpZ3MvYmFzZWxpbmUueWFtbAogICAgcHl0aG9uIHNjcmlwdHMvcnVuX2V4cGVyaW1lbnQucHkgLS1jb25maWcgY29uZmlncy9yMjUueWFtbCAtLWVwb2NocyAxIC0tc2FuaXR5CiAgICBweXRob24gc2NyaXB0cy9ydW5fZXhwZXJpbWVudC5weSAtLWNvbmZpZyBjb25maWdzL2Jhc2VsaW5lLnlhbWwgLS10aW1lIDEuNQoKV3JpdGVzIGV2ZXJ5dGhpbmcgZm9yIHRoZSBjb25kaXRpb24gaW50byBpdHMgb3duIGRpcmVjdG9yeSB1bmRlciB0aGUgcmVzdWx0cwpyb290IChkZWZhdWx0IC9rYWdnbGUvd29ya2luZy9yZXN1bHRzKSwgYW5kIHJlZnVzZXMgdG8gdG91Y2ggYSBkaXJlY3RvcnkgdGhhdAphbHJlYWR5IGhvbGRzIGEgZmluaXNoZWQgcnVuIC0tIGEgcmUtcnVuIGFmdGVyIGFuIHVubm90aWNlZCBLYWdnbGUgc2Vzc2lvbgp0aW1lb3V0IGlzIHRoZSByZWFsaXN0aWMgd2F5IHRoYXQgaGFwcGVucy4gVXNlIC0tZm9yY2UgdG8gb3ZlcnJpZGUKZGVsaWJlcmF0ZWx5LgoKUmVzdW1hYmlsaXR5OiBpZiB0aGUgcnVuJ3Mgb3duIGBsYXN0LnB0YCBleGlzdHMgYnV0IGBtZXRyaWNzLmpzb25gIGRvZXMgbm90LAp0cmFpbmluZyByZXN1bWVzIGZyb20gdGhhdCBjaGVja3BvaW50IChzYW1lIGVwb2NoLCBvcHRpbWl6ZXIgc3RhdGUsIFJORykKaW5zdGVhZCBvZiByZXN0YXJ0aW5nIGF0IGVwb2NoIDEgLS0gc2VlIHNyYy90cmFpbmluZy5weTp0cmFpbl9vbmUuCgotLXNhbml0eSBzdGFnZXMgYSBzbWFsbCwgc2hpcC1ndWFyYW50ZWVkIDMyKzE2LWltYWdlIHN1YnNldCBpbnN0ZWFkIG9mCmFwcGx5aW5nIC0tZnJhY3Rpb24gdG8gdGhlIGZ1bGwgY29uZGl0aW9uIC0tIHNlZSBzcmMvcGF0aHMuc3RhZ2Vfc2FuaXR5X3N1YnNldApmb3Igd2h5IC0tZnJhY3Rpb24gYWxvbmUgZG9lcyBub3QgYWN0dWFsbHkgbWFrZSBhIHNhbml0eSBydW4gY2hlYXAgb24gS2FnZ2xlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc2h1dGlsCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB5YW1sCgpQS0cgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBLRykpCgpmcm9tIHNyYy5ldmFsdWF0aW9uIGltcG9ydCBidWlsZF9ncm91bmRfdHJ1dGgsIHByZWRpY3Rfc3BsaXQsIHNjb3JlCmZyb20gc3JjLnBhdGhzIGltcG9ydCAoYW5ub3RhdGlvbl9kaXIsIGVuc3VyZV93cml0YWJsZV9kYXRhc2V0LCBmaW5kX2RhdGFzZXRzX3Jvb3QsCiAgICAgICAgICAgICAgICAgICAgICAgZmluZF9yZXN1bWVfY2hlY2twb2ludCwgZmluZF9yZXN1bHRzX3Jvb3QsIGlzX2V4cGVyaW1lbnRfY29tcGxldGUsCiAgICAgICAgICAgICAgICAgICAgICAgcmVzb2x2ZV9kYXRhX3lhbWwsIHN0YWdlX3Nhbml0eV9zdWJzZXQpCmZyb20gc3JjLnRyYWluaW5nIGltcG9ydCBlbnZpcm9ubWVudF9pbmZvLCBsb2FkX21hc3Rlciwgc2V0X3NlZWRzLCB0cmFpbl9vbmUKCgpkZWYgbG9hZF9leHBlcmltZW50X2NvbmZpZyhjb25maWdfcGF0aDogUGF0aCkgLT4gdHVwbGVbZGljdCwgc3RyXToKICAgIHN0dWIgPSB5YW1sLnNhZmVfbG9hZChjb25maWdfcGF0aC5yZWFkX3RleHQoKSkKICAgIG1hc3Rlcl9wYXRoID0gY29uZmlnX3BhdGgucGFyZW50IC8gc3R1Yi5nZXQoImluaGVyaXRzIiwgIm1hc3Rlci55YW1sIikKICAgIGNmZyA9IGxvYWRfbWFzdGVyKG1hc3Rlcl9wYXRoKQogICAgcmV0dXJuIGNmZywgc3R1YlsiZXhwZXJpbWVudCJdCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0cy1yb290IiwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImV4cGxpY2l0IGNvbW1vbiByb290IGhvbGRpbmcgb3JpZ2luYWwvcjc1L3I1MC9yMjUgIgogICAgICAgICAgICAgICAgICAgICAgICAgInN1YmZvbGRlcnM7IGRlZmF1bHQ6IGF1dG8tZGlzY292ZXIgdW5kZXIgL2thZ2dsZS9pbnB1dCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVzdWx0cy1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iZW52aXJvbm1lbnQgb3ZlcnJpZGUgb25seTsga2VlcCBpdCB0aGUgc2FtZSBmb3IgYWxsICIKICAgICAgICAgICAgICAgICAgICAgICAgICJmb3VyIGNvbmRpdGlvbnMgb3IgdGhlIGNvbXBhcmlzb24gaXMgaW52YWxpZCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJzYW5pdHkgcnVucyBvbmx5OyB0aGUgc3R1ZHkgdmFsdWUgbGl2ZXMgaW4gbWFzdGVyLnlhbWwiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWZyYWN0aW9uIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InNhbml0eSBydW5zIG9ubHk6IHRyYWluIG9uIGEgZnJhY3Rpb24gb2YgdGhlIGRhdGEiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRpbWUiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0id2FsbC1jbG9jayB0cmFpbmluZyBjYXAgaW4gaG91cnMgZm9yIFRISVMgaW52b2NhdGlvbiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiKFVsdHJhbHl0aWNzJyBvd24gdGltZT0gYXJnKS4gU3RvcHMgY2xlYW5seSBhZnRlciBhICIKICAgICAgICAgICAgICAgICAgICAgICAgICJjb21wbGV0ZWQgZXBvY2g7IGlmIHRoZSBzdHVkeSdzIHRhcmdldCBlcG9jaCBjb3VudCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiaXNuJ3QgcmVhY2hlZCB5ZXQsIHRoaXMgZXhpdHMgYWZ0ZXIgc2F2aW5nIHRoZSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiY2hlY2twb2ludCBXSVRIT1VUIGV2YWx1YXRpbmcgb3Igd3JpdGluZyBtZXRyaWNzLmpzb24sICIKICAgICAgICAgICAgICAgICAgICAgICAgICJzbyB0aGUgbmV4dCBpbnZvY2F0aW9uIHJlc3VtZXMgdHJhaW5pbmcgcmF0aGVyIHRoYW4gIgogICAgICAgICAgICAgICAgICAgICAgICAgInRyZWF0aW5nIHRoZSBjb25kaXRpb24gYXMgZmluaXNoZWQuIFNpemVkIHRvIGZpdCBvbmUgIgogICAgICAgICAgICAgICAgICAgICAgICAgImludm9jYXRpb24gaW5zaWRlIGEgc2luZ2xlIEthZ2dsZSBzZXNzaW9uLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2FuaXR5IiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJtYXJrIG91dHB1dHMgYXMgYSB0aHJvd2F3YXkgc2FuaXR5IHJ1biIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZm9yY2UiLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9Im92ZXJ3cml0ZSBhbiBleGlzdGluZyBmaW5pc2hlZCBydW4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXN0YWdlLXRvLXdvcmtpbmciLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImNvcHkgdGhpcyBjb25kaXRpb24ncyBkYXRhc2V0IHRvIC9rYWdnbGUvd29ya2luZyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiYmVmb3JlIHRyYWluaW5nIChmYWxsYmFjayBvbmx5OyBkZWZhdWx0IHJlYWRzICIKICAgICAgICAgICAgICAgICAgICAgICAgICJkaXJlY3RseSBmcm9tIHRoZSByZWFkLW9ubHkgL2thZ2dsZS9pbnB1dCBtb3VudCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNraXAtZXZhbCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgY2ZnLCBleHBlcmltZW50ID0gbG9hZF9leHBlcmltZW50X2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIGV4cCA9IGNmZ1siZXhwZXJpbWVudHMiXVtleHBlcmltZW50XQogICAgZGF0YXNldHMgPSBmaW5kX2RhdGFzZXRzX3Jvb3QoYXJncy5kYXRhc2V0c19yb290KQogICAgZGF0YXNldF9kaXIgPSBkYXRhc2V0c1tleHBbImxhYmVsIl1dCiAgICBpZiBhcmdzLnN0YWdlX3RvX3dvcmtpbmc6CiAgICAgICAgZnJvbSBzcmMucGF0aHMgaW1wb3J0IHN0YWdlX2NvbmRpdGlvbgogICAgICAgIGRhdGFzZXRfZGlyID0gc3RhZ2VfY29uZGl0aW9uKGV4cFsibGFiZWwiXSwgZGF0YXNldF9kaXIpCiAgICByZXN1bHRzX3Jvb3QgPSBmaW5kX3Jlc3VsdHNfcm9vdChhcmdzLnJlc3VsdHNfcm9vdCkKCiAgICBpZiBhcmdzLnNhbml0eToKICAgICAgICAjIEEgc2FuaXR5IHJ1bidzIHdob2xlIHBvaW50IGlzIHByb3ZpbmcgdGhlIHBsdW1iaW5nIHdvcmtzIGluIGEKICAgICAgICAjIGNvdXBsZSBvZiBtaW51dGVzLiBQb2ludGluZyBVbHRyYWx5dGljcyBhdCB0aGUgZnVsbCB+Nyw3MDYtaW1hZ2UKICAgICAgICAjIGNvbmRpdGlvbiB3b3VsZCBub3QgYWN0dWFsbHkgYmUgY2hlYXA6IGl0IHNjYW5zIGFuZCB2ZXJpZmllcyBldmVyeQogICAgICAgICMgaW1hZ2UvbGFiZWwgcGFpciB0byBidWlsZCBpdHMgbGFiZWwgY2FjaGUgQkVGT1JFIC0tZnJhY3Rpb24gaXMKICAgICAgICAjIGFwcGxpZWQsIGFuZCB0aGF0IGNhY2hlIGNhbiBuZXZlciBiZSBzYXZlZCBvbiB0aGUgcmVhZC1vbmx5CiAgICAgICAgIyAva2FnZ2xlL2lucHV0IG1vdW50IC0tIHNvIGEgIjUlIiBydW4gc3RpbGwgcGF5cyB0aGUgZnVsbCBzY2FuIGNvc3QsCiAgICAgICAgIyBvbiBldmVyeSBsYXVuY2gsIGFjcm9zcyBhbGwgZm91ciBjb25kaXRpb25zLiBTdGFnZSBhIHNtYWxsLAogICAgICAgICMgc2hpcC1ndWFyYW50ZWVkIHN1YnNldCBpbnN0ZWFkOyBib3RoIHRyYWluaW5nIGFuZCBldmFsdWF0aW9uIHJlYWQKICAgICAgICAjIGZyb20gaXQuCiAgICAgICAgZXZhbF9kaXIgPSBzdGFnZV9zYW5pdHlfc3Vic2V0KGV4cFsibGFiZWwiXSwgZGF0YXNldF9kaXIpCiAgICAgICAgZGF0YV95YW1sID0gZXZhbF9kaXIgLyAiZGF0YS55YW1sIgogICAgZWxzZToKICAgICAgICAjIFRoZSBkYXRhLnlhbWwgdW5kZXIgZGF0YXNldF9kaXIgYmFrZXMgaW4gdGhlIEFCU09MVVRFIHBhdGggb2YgdGhlCiAgICAgICAgIyBtYWNoaW5lIHRoYXQgcHJlcGFyZWQgaXQgKGUuZy4gYSBXaW5kb3dzIGRyaXZlIHBhdGgpIGFzIGl0cyBgcGF0aDpgCiAgICAgICAgIyBmaWVsZC4gVGhhdCBpcyBzdGFsZSBoZXJlLCBhbmQgVWx0cmFseXRpY3MgbWlzLXJlc29sdmVzIGl0IHNpbGVudGx5CiAgICAgICAgIyByYXRoZXIgdGhhbiByYWlzaW5nIGEgY2xlYXIgZXJyb3IgLS0gc28gYSBjb3JyZWN0ZWQgY29weSAocGF0aDoKICAgICAgICAjIHJld3JpdHRlbiB0byB0aGUgYWN0dWFsbHktZGlzY292ZXJlZCBkYXRhc2V0X2RpcjsgdHJhaW4vdmFsL25jL25hbWVzCiAgICAgICAgIyB1bnRvdWNoZWQpIGlzIHdyaXR0ZW4gdW5kZXIgL2thZ2dsZS93b3JraW5nIGFuZCB1c2VkIGluc3RlYWQuCiAgICAgICAgZXZhbF9kaXIgPSBkYXRhc2V0X2RpcgogICAgICAgIGRhdGFfeWFtbCA9IHJlc29sdmVfZGF0YV95YW1sKGV4cFsibGFiZWwiXSwgZGF0YXNldF9kaXIpCgogICAgcnVuX25hbWUgPSBleHBbImlkIl0gKyAoIl9zYW5pdHkiIGlmIGFyZ3Muc2FuaXR5IGVsc2UgIiIpCiAgICBvdXRfZGlyID0gcmVzdWx0c19yb290IC8gcnVuX25hbWUKICAgIGlmIGlzX2V4cGVyaW1lbnRfY29tcGxldGUob3V0X2RpcikgYW5kIG5vdCBhcmdzLmZvcmNlOgogICAgICAgIHByaW50KGYiW3NraXBdIHtvdXRfZGlyIC8gJ21ldHJpY3MuanNvbid9IGFscmVhZHkgZXhpc3RzIC0tIHtydW5fbmFtZX0gIgogICAgICAgICAgICAgIGYiaGFzIGFscmVhZHkgY29tcGxldGVkLiBQYXNzIC0tZm9yY2UgdG8gb3ZlcndyaXRlLiIpCiAgICAgICAgcmV0dXJuCiAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICAjIFJlc3VtZSBzdXBwb3J0OiBVbHRyYWx5dGljcyB3cml0ZXMgd2VpZ2h0cy9sYXN0LnB0IGV2ZXJ5IGVwb2NoLiBJZiBpdAogICAgIyBleGlzdHMgd2l0aG91dCBhIGZpbmlzaGVkIG1ldHJpY3MuanNvbiwgdGhpcyBpcyBhIHJlc3VtZSwgbm90IGEgZnJlc2gKICAgICMgc3RhcnQuCiAgICB0cmFpbl9wcm9qZWN0ID0gb3V0X2RpciAvICJ0cmFpbiIKICAgIHJlc3VtZV9mcm9tID0gTm9uZSBpZiBhcmdzLmZvcmNlIGVsc2UgZmluZF9yZXN1bWVfY2hlY2twb2ludChvdXRfZGlyLCBleHBbImlkIl0pCgogICAgZXh0cmEgPSB7azogdiBmb3IgaywgdiBpbiB7CiAgICAgICAgImRldmljZSI6IGFyZ3MuZGV2aWNlLCAiYmF0Y2giOiBhcmdzLmJhdGNoLCAid29ya2VycyI6IGFyZ3Mud29ya2VycywKICAgICAgICAiZXBvY2hzIjogYXJncy5lcG9jaHMsICJmcmFjdGlvbiI6IGFyZ3MuZnJhY3Rpb24sICJ0aW1lIjogYXJncy50aW1lLAogICAgfS5pdGVtcygpIGlmIHYgaXMgbm90IE5vbmV9CgogICAgcHJpbnQoIj0iICogNzApCiAgICBwcmludChmIntleHBlcmltZW50fSAge2V4cFsnbGFiZWwnXX0gIChzY2FsZSB7ZXhwWydzY2FsZSddfSkiKQogICAgcHJpbnQoZiIgIGRhdGFzZXQgIDoge2RhdGFzZXRfZGlyfSIpCiAgICBpZiBhcmdzLnNhbml0eToKICAgICAgICBuX3RyID0gbGVuKGxpc3QoKGV2YWxfZGlyIC8gImltYWdlcyIgLyAidHJhaW4iKS5nbG9iKCIqLmpwZyIpKSkKICAgICAgICBuX3ZhID0gbGVuKGxpc3QoKGV2YWxfZGlyIC8gImltYWdlcyIgLyAidmFsIikuZ2xvYigiKi5qcGciKSkpCiAgICAgICAgcHJpbnQoZiIgIGRhdGEueWFtbDoge2RhdGFfeWFtbH0gIChzdGFnZWQge25fdHJ9K3tuX3ZhfS1pbWFnZSBzYW5pdHkgIgogICAgICAgICAgICAgIGYic3Vic2V0LCBub3QgdGhlIGZ1bGwge2V4cFsnbGFiZWwnXX0gY29uZGl0aW9uKSIpCiAgICBlbHNlOgogICAgICAgIHByaW50KGYiICBkYXRhLnlhbWw6IHtkYXRhX3lhbWx9ICAocGF0aDogY29ycmVjdGVkIGZyb20gc291cmNlKSIpCiAgICBwcmludChmIiAgcmVzdWx0cyAgOiB7b3V0X2Rpcn0iKQogICAgcHJpbnQoZiIgIGltZ3N6ICAgIDoge2NmZ1sndHJhaW4nXVsnaW1nc3onXX0gICBlcG9jaHM6ICIKICAgICAgICAgIGYie2V4dHJhLmdldCgnZXBvY2hzJywgY2ZnWyd0cmFpbiddWydlcG9jaHMnXSl9ICAgIgogICAgICAgICAgZiJiYXRjaDoge2V4dHJhLmdldCgnYmF0Y2gnLCBjZmdbJ3RyYWluJ11bJ2JhdGNoJ10pfSIpCiAgICBpZiByZXN1bWVfZnJvbToKICAgICAgICBwcmludChmIiAgUkVTVU1JTkcgZnJvbSB7cmVzdW1lX2Zyb219IikKICAgIGlmIGV4dHJhOgogICAgICAgIHByaW50KGYiICBlbnYgb3ZlcnJpZGVzOiB7ZXh0cmF9IikKICAgIHByaW50KCI9IiAqIDcwKQoKICAgIHNldF9zZWVkcyhjZmdbInJlcHJvZHVjaWJpbGl0eSJdWyJzZWVkIl0pCgogICAgIyBUaGUgcmVzb2x2ZWQgY29uZmlnIGlzIHdyaXR0ZW4gQkVGT1JFIHRyYWluaW5nLCBzbyBhbiBpbnRlcnJ1cHRlZCBydW4KICAgICMgc3RpbGwgbGVhdmVzIGEgcmVjb3JkIG9mIGV4YWN0bHkgd2hhdCBpdCB3YXMgZ29pbmcgdG8gZG8uCiAgICAob3V0X2RpciAvICJjb25maWcueWFtbCIpLndyaXRlX3RleHQoCiAgICAgICAgeWFtbC5zYWZlX2R1bXAoeyJleHBlcmltZW50IjogZXhwZXJpbWVudCwgInJlc29sdmVkX2Zyb20iOiBzdHIoYXJncy5jb25maWcpLAogICAgICAgICAgICAgICAgICAgICAgICAic2FuaXR5IjogYXJncy5zYW5pdHksICJlbnZfb3ZlcnJpZGVzIjogZXh0cmEsCiAgICAgICAgICAgICAgICAgICAgICAgICJkYXRhc2V0X2RpciI6IHN0cihkYXRhc2V0X2RpciksICJtYXN0ZXIiOiBjZmd9LAogICAgICAgICAgICAgICAgICAgICAgIHNvcnRfa2V5cz1GYWxzZSksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgcmVjb3JkID0gdHJhaW5fb25lKGNmZywgZXhwZXJpbWVudCwgcHJvamVjdD10cmFpbl9wcm9qZWN0LAogICAgICAgICAgICAgICAgICAgICAgIGRhdGFfeWFtbD1kYXRhX3lhbWwsIGV4dHJhPWV4dHJhLAogICAgICAgICAgICAgICAgICAgICAgIHJlc3VtZV9mcm9tPXJlc3VtZV9mcm9tKQogICAgcnVuX2RpciA9IFBhdGgocmVjb3JkWyJydW5fZGlyIl0pCgogICAgIyBVbHRyYWx5dGljcycgb3duIHBlci1lcG9jaCBsb2cgaXMgdGhlIHRyYWluaW5nIGN1cnZlOyBrZWVwIGl0IG5leHQgdG8gdGhlCiAgICAjIG1ldHJpY3MgcmF0aGVyIHRoYW4gYnVyaWVkIGluIHRoZSB0cmFpbmVyIGRpcmVjdG9yeS4KICAgIGNzdl9zcmMgPSBydW5fZGlyIC8gInJlc3VsdHMuY3N2IgogICAgaWYgY3N2X3NyYy5leGlzdHMoKToKICAgICAgICBzaHV0aWwuY29weTIoY3N2X3NyYywgb3V0X2RpciAvICJ0cmFpbmluZ19sb2cuY3N2IikKICAgIGJlc3QgPSBydW5fZGlyIC8gIndlaWdodHMiIC8gImJlc3QucHQiCiAgICBpZiBiZXN0LmV4aXN0cygpOgogICAgICAgIHNodXRpbC5jb3B5MihiZXN0LCBvdXRfZGlyIC8gImJlc3QucHQiKQogICAgbGFzdCA9IHJ1bl9kaXIgLyAid2VpZ2h0cyIgLyAibGFzdC5wdCIKICAgIGlmIGxhc3QuZXhpc3RzKCk6CiAgICAgICAgc2h1dGlsLmNvcHkyKGxhc3QsIG91dF9kaXIgLyAibGFzdC5wdCIpCiAgICBjdXJ2ZXMgPSBvdXRfZGlyIC8gImN1cnZlcyIKICAgIGN1cnZlcy5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZm9yIHBuZyBpbiBydW5fZGlyLmdsb2IoIioucG5nIik6CiAgICAgICAgc2h1dGlsLmNvcHkyKHBuZywgY3VydmVzIC8gcG5nLm5hbWUpCgogICAgaWYgbm90IHJlY29yZFsiZnVsbHlfdHJhaW5lZCJdOgogICAgICAgICMgLS10aW1lIGN1dCB0aGlzIGludm9jYXRpb24gc2hvcnQgYmVmb3JlIHRoZSBzdHVkeSdzIHRhcmdldCBlcG9jaAogICAgICAgICMgY291bnQuIFRoZSBjaGVja3BvaW50IGFib3ZlIGlzIGFscmVhZHkgc2F2ZWQgKHRoYXQncyB3aGF0IG1ha2VzCiAgICAgICAgIyB0aGUgbmV4dCBpbnZvY2F0aW9uJ3MgcmVzdW1lIHBvc3NpYmxlKSwgYnV0IHRoaXMgcnVuIGlzIE5PVAogICAgICAgICMgZmluaXNoZWQ6IGV2YWx1YXRpbmcgYSBwYXJ0aWFsbHktdHJhaW5lZCBtb2RlbCBhbmQgd3JpdGluZwogICAgICAgICMgbWV0cmljcy5qc29uIHdvdWxkIG1ha2UgcnVuX2FsbC5weSAvIGNvbGxlY3RfcmVzdWx0cy5weSB0cmVhdCB0aGlzCiAgICAgICAgIyBjb25kaXRpb24gYXMgZG9uZS4gRXhpdCBoZXJlIGluc3RlYWQsIHNvIHRoZSBuZXh0IGludm9jYXRpb24ncwogICAgICAgICMgZmluZF9yZXN1bWVfY2hlY2twb2ludCgpIHBpY2tzIGl0IGJhY2sgdXAgYXV0b21hdGljYWxseS4KICAgICAgICBwcmludChmIlxuVGltZSBidWRnZXQgcmVhY2hlZDoge3JlY29yZFsnY29tcGxldGVkX2Vwb2NocyddfS8iCiAgICAgICAgICAgICAgZiJ7cmVjb3JkWyd0YXJnZXRfZXBvY2hzJ119IGVwb2NocyBkb25lIGZvciB7ZXhwWydsYWJlbCddfS4gIgogICAgICAgICAgICAgIGYiU2F2ZWQge291dF9kaXIgLyAnbGFzdC5wdCd9OyBOT1QgZXZhbHVhdGluZyB5ZXQuIikKICAgICAgICBwcmludCgiUmUtcnVuIChzYW1lIGNvbW1hbmQsIG9yIHZpYSBydW5fYWxsLnB5KSB0byBjb250aW51ZSB0cmFpbmluZyAiCiAgICAgICAgICAgICAgInRoaXMgY29uZGl0aW9uIGZyb20gaGVyZS4iKQogICAgICAgIHJldHVybgoKICAgIG1ldHJpY3MgPSB7ImV4cGVyaW1lbnQiOiBleHBlcmltZW50LCAicnVuIjogcmVjb3JkfQoKICAgIGlmIG5vdCBhcmdzLnNraXBfZXZhbDoKICAgICAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPCiAgICAgICAgcHJpbnQoIlxuRXZhbHVhdGluZyBvbiB0aGUgdmFsIHNwbGl0LCBpbiBvcmlnaW5hbC1yZXNvbHV0aW9uIGNvb3JkaW5hdGVzIC4uLiIpCiAgICAgICAgYW5uID0gYW5ub3RhdGlvbl9kaXIoKQogICAgICAgIHNoaXBzID0gcGQucmVhZF9wYXJxdWV0KGFubiAvICJzaGlwX2Fubm90YXRpb25zLnBhcnF1ZXQiKQogICAgICAgIGltYWdlcyA9IHBkLnJlYWRfY3N2KGFubiAvICJpbWFnZV9tYW5pZmVzdC5jc3YiKQogICAgICAgIGlmIGFyZ3Muc2FuaXR5OgogICAgICAgICAgICAjIFRoZSBhbm5vdGF0aW9uIHRhYmxlcyBjb3ZlciB0aGUgZnVsbCAyLDc3Ni1pbWFnZSB2YWwgc3BsaXQsIGJ1dAogICAgICAgICAgICAjIGV2YWxfZGlyIG9ubHkgaG9sZHMgdGhlIHN0YWdlZCBoYW5kZnVsIC0tIHByZWRpY3Rfc3BsaXQoKSBoYXJkCiAgICAgICAgICAgICMgLWZhaWxzIG9uIGFueSBsaXN0ZWQgaW1hZ2UgaXQgY2FuJ3QgZmluZCBvbiBkaXNrLCBzbyBib3RoCiAgICAgICAgICAgICMgdGFibGVzIGFyZSBuYXJyb3dlZCB0byBleGFjdGx5IHdoYXQgd2FzIHN0YWdlZCAoZ3JvdW5kIHRydXRoCiAgICAgICAgICAgICMgYW5kIHByZWRpY3Rpb25zIHRoZW4gY292ZXIgdGhlIHNhbWUgaW1hZ2VzLCBub3QgYSBzdWJzZXQgdnMuCiAgICAgICAgICAgICMgdGhlIGZ1bGwgc3BsaXQpLgogICAgICAgICAgICBzdGVtcyA9IHtwLnN0ZW0gZm9yIHAgaW4gKGV2YWxfZGlyIC8gImltYWdlcyIgLyAidmFsIikuZ2xvYigiKi5qcGciKX0KICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzW2ltYWdlc1siYmFzZW5hbWUiXS5hcHBseShsYW1iZGEgYjogUGF0aChiKS5zdGVtIGluIHN0ZW1zKV0KICAgICAgICAgICAgc2hpcHMgPSBzaGlwc1tzaGlwc1siYmFzZW5hbWUiXS5hcHBseShsYW1iZGEgYjogUGF0aChiKS5zdGVtIGluIHN0ZW1zKV0KCiAgICAgICAgbW9kZWwgPSBZT0xPKHN0cihiZXN0IGlmIGJlc3QuZXhpc3RzKCkgZWxzZSBydW5fZGlyIC8gIndlaWdodHMiIC8gImxhc3QucHQiKSkKICAgICAgICBldiA9IGNmZ1siZXZhbHVhdGlvbiJdCiAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgIGR0LCB0aW1pbmcgPSBwcmVkaWN0X3NwbGl0KAogICAgICAgICAgICBtb2RlbCwgZXZhbF9kaXIsIGltYWdlcywgc3BsaXQ9IlZhbCIsCiAgICAgICAgICAgIGNvbmY9ZXZbImNvbmYiXSwgaW91PWV2WyJpb3UiXSwgbWF4X2RldD1ldlsibWF4X2RldCJdLAogICAgICAgICAgICBpbWdzej1jZmdbInRyYWluIl1bImltZ3N6Il0sIGRldmljZT1hcmdzLmRldmljZSwKICAgICAgICAgICAgYmF0Y2g9ZXh0cmEuZ2V0KCJiYXRjaCIsIGNmZ1sidHJhaW4iXVsiYmF0Y2giXSkpCiAgICAgICAgZ3QgPSBidWlsZF9ncm91bmRfdHJ1dGgoc2hpcHMsIGltYWdlcywgc3BsaXQ9IlZhbCIpCiAgICAgICAgbSA9IHNjb3JlKGd0LCBkdCwgbWF4X2RldD1ldlsibWF4X2RldCJdLAogICAgICAgICAgICAgICAgICBvcGVyYXRpbmdfY29uZj1ldlsib3BlcmF0aW5nX3BvaW50X2NvbmYiXSkKICAgICAgICBtWyJ0aW1pbmciXSA9IHRpbWluZwogICAgICAgIG1bImV2YWxfd2FsbF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAxKQogICAgICAgIG1ldHJpY3NbIm1ldHJpY3MiXSA9IG0KCiAgICAgICAgIyBVbHRyYWx5dGljcycgb3duIHZhbCgpIGFzIGFuIGluZGVwZW5kZW50IGNyb3NzLWNoZWNrIG9mIHRoZSBoZWFkbGluZQogICAgICAgICMgbnVtYmVycy4gQSBsYXJnZSBkaXNhZ3JlZW1lbnQgbWVhbnMgb25lIG9mIHRoZSB0d28gaXMgd3JvbmcgYW5kIHRoZQogICAgICAgICMgcmVzdWx0cyBzaG91bGQgbm90IGJlIHRydXN0ZWQgdW50aWwgaXQgaXMgZXhwbGFpbmVkLgogICAgICAgIHRyeToKICAgICAgICAgICAgdiA9IG1vZGVsLnZhbChkYXRhPXN0cihkYXRhX3lhbWwpLAogICAgICAgICAgICAgICAgICAgICAgICAgIGltZ3N6PWNmZ1sidHJhaW4iXVsiaW1nc3oiXSwgY29uZj1ldlsiY29uZiJdLAogICAgICAgICAgICAgICAgICAgICAgICAgIGlvdT1ldlsiaW91Il0sIG1heF9kZXQ9ZXZbIm1heF9kZXQiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICBkZXZpY2U9YXJncy5kZXZpY2UsIHZlcmJvc2U9RmFsc2UsIHBsb3RzPUZhbHNlKQogICAgICAgICAgICBtZXRyaWNzWyJ1bHRyYWx5dGljc192YWwiXSA9IHsKICAgICAgICAgICAgICAgICJtQVA1MCI6IGZsb2F0KHYuYm94Lm1hcDUwKSwgIm1BUDUwXzk1IjogZmxvYXQodi5ib3gubWFwKSwKICAgICAgICAgICAgICAgICJwcmVjaXNpb24iOiBmbG9hdCh2LmJveC5tcCksICJyZWNhbGwiOiBmbG9hdCh2LmJveC5tciksCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZDUwID0gYWJzKHYuYm94Lm1hcDUwIC0gbVsibUFQNTAiXSkKICAgICAgICAgICAgbWV0cmljc1siY3Jvc3NfY2hlY2tfbUFQNTBfYWJzX2RpZmYiXSA9IHJvdW5kKGZsb2F0KGQ1MCksIDQpCiAgICAgICAgICAgIGlmIGQ1MCA+IDAuMDI6CiAgICAgICAgICAgICAgICBtZXRyaWNzWyJjcm9zc19jaGVja193YXJuaW5nIl0gPSAoCiAgICAgICAgICAgICAgICAgICAgZiJjdXN0b20gbUFQNTAge21bJ21BUDUwJ106LjRmfSB2cyB1bHRyYWx5dGljcyAiCiAgICAgICAgICAgICAgICAgICAgZiJ7di5ib3gubWFwNTA6LjRmfSBkaWZmZXIgYnkge2Q1MDouNGZ9OyBpbnZlc3RpZ2F0ZSBiZWZvcmUgIgogICAgICAgICAgICAgICAgICAgIGYicmVwb3J0aW5nIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICBtZXRyaWNzWyJ1bHRyYWx5dGljc192YWxfZXJyb3IiXSA9IHN0cihlKQoKICAgICAgICBwcmludChmIlxuICBtQVA1MCAgICAgIHttWydtQVA1MCddOi40Zn0iKQogICAgICAgIHByaW50KGYiICBtQVA1MC05NSAgIHttWydtQVA1MF85NSddOi40Zn0iKQogICAgICAgIHByaW50KGYiICBBUF9zbWFsbCAgIHttWydBUF9zbWFsbCddOi40Zn0gICAobl9ndCB7bVsnbl9ndCddWydzbWFsbCddOix9KSIpCiAgICAgICAgcHJpbnQoZiIgIEFQX21lZGl1bSAge21bJ0FQX21lZGl1bSddOi40Zn0gICAobl9ndCB7bVsnbl9ndCddWydtZWRpdW0nXTosfSkiKQogICAgICAgIHByaW50KGYiICBBUF9sYXJnZSAgIHttWydBUF9sYXJnZSddOi40Zn0gICAobl9ndCB7bVsnbl9ndCddWydsYXJnZSddOix9KSIpCiAgICAgICAgcHJpbnQoZiIgIGxhdGVuY3kgICAge3RpbWluZ1snbGF0ZW5jeV9tc19wZXJfaW1hZ2UnXTouMWZ9IG1zL2ltZyIpCgogICAgbWV0cmljc1siZW52aXJvbm1lbnQiXSA9IGVudmlyb25tZW50X2luZm8oKQogICAgKG91dF9kaXIgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChmIlxuV3JvdGUge291dF9kaXIgLyAnbWV0cmljcy5qc29uJ30iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
    "scripts/verify_kaggle_setup.py": "...",
}
for _rel, _b64 in _FILES.items():
    _p = PKG / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(base64.b64decode(_b64))
print(f"wrote {len(_FILES)} file(s) into {PKG} (scripts): {sorted(_FILES)}")

In [ ]:
import base64
from pathlib import Path

PKG = Path("/kaggle/working/kaggle")
_FILES = {
    "configs/baseline.yaml": "IyBFeHBlcmltZW50IEUwMCAtLSBCYXNlbGluZS4gRnVsbCBuYXRpdmUgRkFJUjFNIHJlc29sdXRpb24uDQojDQojIFRoaXMgZmlsZSBpbnRlbnRpb25hbGx5IGNvbnRhaW5zIGFsbW9zdCBub3RoaW5nLiBFdmVyeSBoeXBlcnBhcmFtZXRlciBjb21lcw0KIyBmcm9tIG1hc3Rlci55YW1sOyB0aGUgT05MWSB0aGluZyBhbiBleHBlcmltZW50IGlzIHBlcm1pdHRlZCB0byBjaGFuZ2UgaXMNCiMgd2hpY2ggcmVzb2x1dGlvbiBvZiB0aGUgZGF0YXNldCBpdCByZWFkcy4gc3JjL3RyYWluaW5nLnB5IGVuZm9yY2VzIHRoYXQgLS0NCiMgYWRkaW5nLCBzYXksIGBscjBgIGhlcmUgcmFpc2VzIHJhdGhlciB0aGFuIHNpbGVudGx5IGJyZWFraW5nIHRoZSBjb250cm9sLg0KDQppbmhlcml0czogbWFzdGVyLnlhbWwNCmV4cGVyaW1lbnQ6IEUwMA0KDQojIEZvciByZWZlcmVuY2Ugb25seTsgcmVhZCBmcm9tIG1hc3Rlci55YW1sIGF0IHJ1bnRpbWUuDQpfcmVzb2x1dGlvbjoNCiAgbGFiZWw6IG9yaWdpbmFsDQogIHNjYWxlOiAxLjANCiAgcnVuX2lkOiAwMF9iYXNlbGluZQ0KICBhcmVhX3NjYWxlOiAxDQo=",
    "configs/master.yaml": "IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIE1BU1RFUiBSRVBST0RVQ0lCSUxJVFkgQ09ORklHICAoUGFydCBWKQojCiMgU2luZ2xlIHNvdXJjZSBvZiB0cnV0aCBmb3IgdGhlIEZBSVIxTSByZXNvbHV0aW9uIHN0dWR5LiBFdmVyeSBleHBlcmltZW50CiMgaW5oZXJpdHMgdGhpcyBmaWxlIGFuZCBvdmVycmlkZXMgT05MWSB0aGUga2V5cyBsaXN0ZWQgdW5kZXIKIyBgZXhwZXJpbWVudHMuPGlkPi5vdmVycmlkZXNgLCB3aGljaCBpbiB0aGUgcHJpbWFyeSBzd2VlcCBpcyBub3RoaW5nIGJ1dCB0aGUKIyBkYXRhc2V0IHBhdGggYW5kIHRoZSBydW4gbmFtZS4KIwojIElmIGEgdmFsdWUgaXMgbm90IGluIHRoaXMgZmlsZSwgaXQgaXMgbm90IGNvbnRyb2xsZWQuIERvIG5vdCBwYXNzIHRyYWluaW5nCiMgaHlwZXJwYXJhbWV0ZXJzIG9uIHRoZSBjb21tYW5kIGxpbmUuCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCnN0dWR5OgogIG5hbWU6IGZhaXIxbV9zaGlwX3Jlc29sdXRpb24KICBxdWVzdGlvbjogPgogICAgSG93IGRvZXMgcmVkdWNpbmcgdGhlIHNwYXRpYWwgcmVzb2x1dGlvbiBvZiBzYXRlbGxpdGUgaW1hZ2VyeSBhZmZlY3Qgc2hpcAogICAgZGV0ZWN0aW9uIHBlcmZvcm1hbmNlLCBhbmQgZG9lcyB0aGUgZWZmZWN0IGRlcGVuZCBvbiBvYmplY3Qgc2l6ZT8KICBpbmRlcGVuZGVudF92YXJpYWJsZTogaW1hZ2Vfc3BhdGlhbF9yZXNvbHV0aW9uCiAgcHJpbWFyeV90YXNrOiBzaW5nbGVfY2xhc3Nfc2hpcF9kZXRlY3Rpb24KICBjcmVhdGVkX3dpdGg6CiAgICB1bHRyYWx5dGljczogIjguNC42MSIKICAgIHRvcmNoOiAiMi45LjEiCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRGF0YXNldAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRhdGFzZXQ6CiAgc291cmNlOiBGQUlSMU0gdjEuMCAoR2FvZmVuIENoYWxsZW5nZSBleHBvcnQpCiAgZGVyaXZlZF9mcm9tOiBkYXRhL3NoaXBfb25seS9zaGlwX2Fubm90YXRpb25zLnBhcnF1ZXQKICBjbGFzc2VzOgogICAgMDogc2hpcAogIG5jOiAxCiAgIyBBbGwgOSBGQUlSMU0gc2hpcCBjYXRlZ29yaWVzIGNvbGxhcHNlIHRvIGNsYXNzIDAuCiAgc2hpcF9jYXRlZ29yaWVzOgogICAgLSBEcnkgQ2FyZ28gU2hpcAogICAgLSBNb3RvcmJvYXQKICAgIC0gRmlzaGluZyBCb2F0CiAgICAtIG90aGVyLXNoaXAKICAgIC0gRW5naW5lZXJpbmcgU2hpcAogICAgLSBMaXF1aWQgQ2FyZ28gU2hpcAogICAgLSBQYXNzZW5nZXIgU2hpcAogICAgLSBUdWdib2F0CiAgICAtIFdhcnNoaXAKICBzcGxpdDoKICAgIG1ldGhvZDogb2ZmaWNpYWxfRkFJUjFNX3RyYWluX3ZhbAogICAgIyBJZGVudGljYWwgaW1hZ2UgYW5kIGFubm90YXRpb24gaWRlbnRpdGllcyBpbiBldmVyeSBjb25kaXRpb24uIEVuZm9yY2VkIGJ5CiAgICAjIGNvbnN0cnVjdGlvbjogYWxsIGZvdXIgZGF0YXNldHMgYXJlIGdlbmVyYXRlZCBmcm9tIG9uZSBpbWFnZSBtYW5pZmVzdC4KICAgIG1hbmlmZXN0OiBkYXRhL3NoaXBfb25seS9pbWFnZV9tYW5pZmVzdC5jc3YKICAgIHRyYWluX2ltYWdlczogNDkzMAogICAgdmFsX2ltYWdlczogMjc3NgogICAgdHJhaW5fc2hpcHM6IDMxNDYyCiAgICB2YWxfc2hpcHM6IDI3NTIwCiAgYmFja2dyb3VuZF9pbWFnZXM6CiAgICBwb2xpY3k6IHNhbXBsZWRfZnJhY3Rpb25fb2Zfc2hpcF9mcmVlX2ltYWdlcwogICAgZnJhY3Rpb246IDAuMTAKICAgIHNlZWQ6IDQyCiAgICByZXRhaW5lZDogNzcxCiAgICBhdmFpbGFibGU6IDE3ODQwCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgT2JqZWN0LXNpemUgYmlucyAgKFBhcnQgSCkgIC0tIHNlZSBhbmFseXNpcy9vYmplY3Rfc2l6ZV9kZWZpbml0aW9uLm1kCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0Kb2JqZWN0X3NpemVfYmluczoKICBtZXRob2Q6IGNvY29fYWJzb2x1dGVfcGl4ZWxfYXJlYQogIG1lYXN1cmVkX29uOiBPUklHSU5BTF9yZXNvbHV0aW9uX2Fubm90YXRpb24KICAjIEFzc2lnbmVkIG9uY2UsIHBlciBvYmplY3QsIGFuZCBuZXZlciByZWNvbXB1dGVkIHBlciByZXNvbHV0aW9uLiBSZWNvbXB1dGluZwogICMgd291bGQgc2xpZGUgdGhlIHdob2xlIHBvcHVsYXRpb24gaW50byBgc21hbGxgIGF0IHIyNSBhbmQgbWFrZSBlYWNoIHJvdyBvZgogICMgdGhlIHJlc29sdXRpb24geCBzaXplIHRhYmxlIGRlc2NyaWJlIGEgZGlmZmVyZW50IHNldCBvZiBzaGlwcy4KICBmaXhlZF9hY3Jvc3NfcmVzb2x1dGlvbnM6IHRydWUKICBzbWFsbF9tYXhfcHgyOiAxMDI0ICAgICAgIyAzMl4yCiAgbWVkaXVtX21heF9weDI6IDkyMTYgICAgICMgOTZeMgogIGNvdW50czoge3NtYWxsOiAyNDk5OCwgbWVkaXVtOiAyNzc3NCwgbGFyZ2U6IDYyMTB9CiAgc2Vjb25kYXJ5X3RlcnRpbGVfdGhyZXNob2xkc19weDI6IFs2ODIuMCwgMjc2MC4wXQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFJlc29sdXRpb24gcHJlcHJvY2Vzc2luZyAgKFBhcnQgSSkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpwcmVwcm9jZXNzaW5nOgogIHJlc2FtcGxlOiBnYXVzc2lhbl9wcmVmaWx0ZXJfdGhlbl9pbnRlcl9hcmVhCiAgcHJlZmlsdGVyX3NpZ21hOiAiKDEvc2NhbGUgLSAxKSAvIDIiCiAgYm94X3RyYW5zZm9ybTogcmVhbGl6ZWRfc2NhbGUgICAjIG91dF9kaW0gLyBzcmNfZGltLCBOT1QgdGhlIG5vbWluYWwgZmFjdG9yCiAgcHJlc2VydmVfYXNwZWN0X3JhdGlvOiB0cnVlCiAgZm9yY2VfY29tbW9uX3NoYXBlOiBmYWxzZQogIGRyb3Bfc21hbGxfYm94ZXM6IGZhbHNlICAgICAgICAgIyBuZXZlcjsgd291bGQgY2hhbmdlIHRoZSBHVCBzZXQgcGVyIGNvbmRpdGlvbgogIGpwZWc6IHtxdWFsaXR5OiA5NSwgc3Vic2FtcGxpbmc6IDB9CiAgYXBwbGllZF91bmlmb3JtbHlfdG9fYWxsX2NvbmRpdGlvbnM6IHRydWUKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBNb2RlbAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCm1vZGVsOgogIGFyY2g6IHlvbG92OG0KICB3ZWlnaHRzOiB5b2xvdjhtLnB0ICAgICAgIyBDT0NPLXByZXRyYWluZWQsIGlkZW50aWNhbCBzdGFydGluZyBwb2ludCBldmVyeXdoZXJlCiAgcHJldHJhaW5lZDogdHJ1ZQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFRyYWluaW5nIC0tIHBhc3NlZCB2ZXJiYXRpbSB0byB1bHRyYWx5dGljcyBZT0xPLnRyYWluKCkKIwojIEFueXRoaW5nIGxlZnQgYXQgYW4gVWx0cmFseXRpY3MgZGVmYXVsdCBpcyBzdGlsbCB3cml0dGVuIG91dCBleHBsaWNpdGx5LCBzbwojIHRoYXQgYSBmdXR1cmUgdmVyc2lvbiBjaGFuZ2luZyBpdHMgZGVmYXVsdHMgY2Fubm90IHNpbGVudGx5IGNoYW5nZSB0aGlzIHN0dWR5LgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCnRyYWluOgogIGVwb2NoczogNTAKICAjIEZpeGVkLCBub3QgYGF1dG9gLiBBIGJhdGNoIHNpemUgY2hvc2VuIHBlciBjb25kaXRpb24gd291bGQgdmFyeSB0aGUKICAjIGVmZmVjdGl2ZSBvcHRpbWlzYXRpb24gdHJhamVjdG9yeSBhbmQgY29uZm91bmQgdGhlIHJlc29sdXRpb24gdmFyaWFibGUuCiAgYmF0Y2g6IDgKICAjIFRIRSBrZXkgY29udHJvbC4gQ29uc3RhbnQgbmV0d29yayBpbnB1dCBhY3Jvc3MgRTAwLUUwMywgc28gdGhlIG9ubHkgdGhpbmcKICAjIHRoYXQgY2hhbmdlcyBpcyBob3cgbXVjaCByZWFsIGRldGFpbCB0aGUgaW1hZ2VyeSBjYXJyaWVzLgogICMKICAjIE11c3QgYmUgPj0gdGhlIGxhcmdlc3QgY29tbW9uIHNvdXJjZSBkaW1lbnNpb24gKDEwMDAgcHgpLiBBdCB0aGUgVWx0cmFseXRpY3MKICAjIGRlZmF1bHQgb2YgNjQwIHRoZSBkYXRhbG9hZGVyIHdvdWxkIGl0c2VsZiBkb3duc2FtcGxlIEUwMCAoMTAwMCAtPiA2NDApIGFuZAogICMgRTAxICg3NTAgLT4gNjQwKSB0byBuZWFybHkgdGhlIHNhbWUgZWZmZWN0aXZlIGRldGFpbCwgY29sbGFwc2luZyB0aGUKICAjIGNvbnRyYXN0IGJldHdlZW4gdGhlIGZpcnN0IHR3byBjb25kaXRpb25zIGJlZm9yZSB0cmFpbmluZyBzdGFydHMuCiAgaW1nc3o6IDEwMjQKICByZWN0OiBmYWxzZQogIHNpbmdsZV9jbHM6IHRydWUKICAjIEVhcmx5IHN0b3BwaW5nIGRpc2FibGVkOiBzdG9wcGluZyBhdCBhIGRpZmZlcmVudCBlcG9jaCBwZXIgY29uZGl0aW9uIHdvdWxkCiAgIyBiZSBhbiB1bmNvbnRyb2xsZWQgZGlmZmVyZW5jZSBpbiB0cmFpbmluZyBsZW5ndGguCiAgcGF0aWVuY2U6IDEwMDAwMAogIG9wdGltaXplcjogU0dEICAgICAgICAgICAjIHBpbm5lZDsgYGF1dG9gIGNhbiByZXNvbHZlIGRpZmZlcmVudGx5IHBlciBydW4KICBscjA6IDAuMDEKICBscmY6IDAuMDEKICBjb3NfbHI6IGZhbHNlCiAgbW9tZW50dW06IDAuOTM3CiAgd2VpZ2h0X2RlY2F5OiAwLjAwMDUKICB3YXJtdXBfZXBvY2hzOiAzLjAKICB3YXJtdXBfbW9tZW50dW06IDAuOAogIHdhcm11cF9iaWFzX2xyOiAwLjEKICBuYnM6IDY0CiAgYm94OiA3LjUKICBjbHM6IDAuNQogIGRmbDogMS41CiAgYW1wOiB0cnVlCiAgY2FjaGU6IGZhbHNlCiAgd29ya2VyczogOAogIHZhbDogdHJ1ZQogIHBsb3RzOiB0cnVlCiAgc2F2ZV9wZXJpb2Q6IC0xCiAgZnJhY3Rpb246IDEuMAogIG11bHRpX3NjYWxlOiAwLjAgICAgICAgICAjIG9mZjogd291bGQgcmVpbnRyb2R1Y2Ugc2NhbGUgdmFyaWF0aW9uIHBlciBiYXRjaAoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEF1Z21lbnRhdGlvbiAtLSBpZGVudGljYWwgaW4gZXZlcnkgY29uZGl0aW9uIChQYXJ0IEQpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KYXVnbWVudGF0aW9uOgogIGhzdl9oOiAwLjAxNQogIGhzdl9zOiAwLjcKICBoc3ZfdjogMC40CiAgIyBSb3RhdGlvbiBzdGF5cyBvZmYuIE92ZXJoZWFkIGltYWdlcnkgaGFzIG5vIGNhbm9uaWNhbCBvcmllbnRhdGlvbiwgYnV0CiAgIyByb3RhdGluZyBhbiBheGlzLWFsaWduZWQgYm94IGluZmxhdGVzIGl0IChhIDQ1LWRlZ3JlZSByb3RhdGVkIHNoaXAgZ2V0cyBhCiAgIyBtdWNoIGxvb3NlciBBQUJCKSwgd2hpY2ggd291bGQgZGVncmFkZSBsYWJlbCBxdWFsaXR5LiBBeGlzLWFsaWduZWQgZmxpcHMKICAjIGFjaGlldmUgdGhlIHNhbWUgb3JpZW50YXRpb24gaW52YXJpYW5jZSB3aXRoIGV4YWN0IGJveCBnZW9tZXRyeS4KICBkZWdyZWVzOiAwLjAKICB0cmFuc2xhdGU6IDAuMQogIHNjYWxlOiAwLjUKICBzaGVhcjogMC4wCiAgcGVyc3BlY3RpdmU6IDAuMAogIGZsaXB1ZDogMC41ICAgICAgICAgICAgICAjIHJhaXNlZCBmcm9tIHRoZSAwLjAgZGVmYXVsdDogdmFsaWQgZm9yIG5hZGlyIGltYWdlcnkKICBmbGlwbHI6IDAuNQogIGJncjogMC4wCiAgbW9zYWljOiAxLjAKICBjbG9zZV9tb3NhaWM6IDEwCiAgbWl4dXA6IDAuMAogIGN1dG1peDogMC4wCiAgY29weV9wYXN0ZTogMC4wCiAgZXJhc2luZzogMC40CgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRXZhbHVhdGlvbgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmV2YWx1YXRpb246CiAgIyAwLjAwMSBpcyB0aGUgc3RhbmRhcmQgbUFQIHN3ZWVwIHRocmVzaG9sZDsgYSBoaWdoZXIgdmFsdWUgd291bGQgdHJ1bmNhdGUgdGhlCiAgIyBwcmVjaXNpb24tcmVjYWxsIGN1cnZlIGFuZCBpbmZsYXRlIEFQLgogIGNvbmY6IDAuMDAxCiAgaW91OiAwLjcgICAgICAgICAgICAgICAgICMgTk1TIElvVQogICMgUmFpc2VkIGZyb20gdGhlIFVsdHJhbHl0aWNzIGRlZmF1bHQgb2YgMzAwLiBUaGUgZGVuc2VzdCBGQUlSMU0gdmFsIHRpbGUKICAjIGhvbGRzIDU4MSBzaGlwcywgc28gMzAwIHdvdWxkIHNpbGVudGx5IGNhcCByZWNhbGwgb24gZXhhY3RseSB0aGUgY3Jvd2RlZAogICMgaGFyYm91ciBzY2VuZXMgdGhpcyBzdHVkeSBjYXJlcyBtb3N0IGFib3V0LgogIG1heF9kZXQ6IDEwMDAKICAjIFJlcG9ydGVkIHNlcGFyYXRlbHksIGZvciB0aGUgb3BlcmF0aW5nLXBvaW50IG1ldHJpY3MgKFAgLyBSIC8gRjEpLgogIG9wZXJhdGluZ19wb2ludF9jb25mOiAwLjI1CiAgbWV0cmljczoKICAgIC0gbUFQNTAKICAgIC0gbUFQNTAtOTUKICAgIC0gcHJlY2lzaW9uCiAgICAtIHJlY2FsbAogICAgLSBmMQogICAgLSBBUF9zbWFsbAogICAgLSBBUF9tZWRpdW0KICAgIC0gQVBfbGFyZ2UKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBSZXByb2R1Y2liaWxpdHkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpyZXByb2R1Y2liaWxpdHk6CiAgc2VlZDogNDIKICBkZXRlcm1pbmlzdGljOiB0cnVlCiAgIyBQYXJ0IFM6IGFkZGl0aW9uYWwgc2VlZHMgb25seSBpZiBHUFUgYnVkZ2V0IGFsbG93cy4gSWYgbm90IHJ1biwgdGhlIHN0dWR5CiAgIyBtdXN0IGJlIHJlcG9ydGVkIGFzIHNpbmdsZS1zZWVkIHdpdGggbm8gc2lnbmlmaWNhbmNlIGNsYWltcy4KICBhZGRpdGlvbmFsX3NlZWRzOiBbMTIzLCA0NTZdCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRXhwZXJpbWVudHMgLS0gdGhlIE9OTFkgaW50ZW5kZWQgZGlmZmVyZW5jZXMgYmV0d2VlbiBjb25kaXRpb25zCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZXhwZXJpbWVudHM6CiAgRTAwOgogICAgaWQ6ICIwMF9iYXNlbGluZSIKICAgIGxhYmVsOiBvcmlnaW5hbAogICAgc2NhbGU6IDEuMDAKICAgIG92ZXJyaWRlczoge2RhdGE6IGRhdGFzZXRzL29yaWdpbmFsL2RhdGEueWFtbCwgbmFtZTogIjAwX2Jhc2VsaW5lIn0KICBFMDE6CiAgICBpZDogIjAxX3I3NSIKICAgIGxhYmVsOiByNzUKICAgIHNjYWxlOiAwLjc1CiAgICBvdmVycmlkZXM6IHtkYXRhOiBkYXRhc2V0cy9yNzUvZGF0YS55YW1sLCBuYW1lOiAiMDFfcjc1In0KICBFMDI6CiAgICBpZDogIjAyX3I1MCIKICAgIGxhYmVsOiByNTAKICAgIHNjYWxlOiAwLjUwCiAgICBvdmVycmlkZXM6IHtkYXRhOiBkYXRhc2V0cy9yNTAvZGF0YS55YW1sLCBuYW1lOiAiMDJfcjUwIn0KICBFMDM6CiAgICBpZDogIjAzX3IyNSIKICAgIGxhYmVsOiByMjUKICAgIHNjYWxlOiAwLjI1CiAgICBvdmVycmlkZXM6IHtkYXRhOiBkYXRhc2V0cy9yMjUvZGF0YS55YW1sLCBuYW1lOiAiMDNfcjI1In0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBPcHRpb25hbCBmb2xsb3ctb24gc3R1ZGllcy4gS2VwdCBzdHJpY3RseSBzZXBhcmF0ZSBmcm9tIHRoZSBwcmltYXJ5IHN3ZWVwLgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCm9wdGlvbmFsOgogIHJlcGVhdGVkX3NlZWRzOiAgICAgICAgICAgICMgUGFydCBTCiAgICBlbmFibGVkOiBmYWxzZQogICAgY29uZGl0aW9uczogW0UwMCwgRTAyLCBFMDNdCiAgICBzZWVkczogWzQyLCAxMjMsIDQ1Nl0KICBmaW5lX2dyYWluZWRfc2hpcHM6ICAgICAgICAjIFBhcnQgVAogICAgZW5hYmxlZDogZmFsc2UKICAgIG5jOiA5CiAgICBub3RlOiA+CiAgICAgIFNlcGFyYXRlIGV4cGVyaW1lbnQuIE11c3QgbmV2ZXIgYmUgbWVyZ2VkIGludG8gdGhlIHByaW1hcnkgc2luZ2xlLWNsYXNzCiAgICAgIHJlc3VsdHMgdGFibGUuCg==",
    "configs/r25.yaml": "IyBFeHBlcmltZW50IEUwMyAtLSAyNSUgbGluZWFyIHJlc29sdXRpb24uIEJveCBhcmVhIGZhbGxzIHRvIDYuMjUlIG9mIG9yaWdpbmFsLg0KIw0KIyBUaGlzIGZpbGUgaW50ZW50aW9uYWxseSBjb250YWlucyBhbG1vc3Qgbm90aGluZy4gRXZlcnkgaHlwZXJwYXJhbWV0ZXIgY29tZXMNCiMgZnJvbSBtYXN0ZXIueWFtbDsgdGhlIE9OTFkgdGhpbmcgYW4gZXhwZXJpbWVudCBpcyBwZXJtaXR0ZWQgdG8gY2hhbmdlIGlzDQojIHdoaWNoIHJlc29sdXRpb24gb2YgdGhlIGRhdGFzZXQgaXQgcmVhZHMuIHNyYy90cmFpbmluZy5weSBlbmZvcmNlcyB0aGF0IC0tDQojIGFkZGluZywgc2F5LCBgbHIwYCBoZXJlIHJhaXNlcyByYXRoZXIgdGhhbiBzaWxlbnRseSBicmVha2luZyB0aGUgY29udHJvbC4NCg0KaW5oZXJpdHM6IG1hc3Rlci55YW1sDQpleHBlcmltZW50OiBFMDMNCg0KIyBGb3IgcmVmZXJlbmNlIG9ubHk7IHJlYWQgZnJvbSBtYXN0ZXIueWFtbCBhdCBydW50aW1lLg0KX3Jlc29sdXRpb246DQogIGxhYmVsOiByMjUNCiAgc2NhbGU6IDAuMjUNCiAgcnVuX2lkOiAwM19yMjUNCiAgYXJlYV9zY2FsZTogMC4wNjI1DQo=",
    "configs/r50.yaml": "IyBFeHBlcmltZW50IEUwMiAtLSA1MCUgbGluZWFyIHJlc29sdXRpb24uIEJveCBhcmVhIGZhbGxzIHRvIDI1JSBvZiBvcmlnaW5hbC4NCiMNCiMgVGhpcyBmaWxlIGludGVudGlvbmFsbHkgY29udGFpbnMgYWxtb3N0IG5vdGhpbmcuIEV2ZXJ5IGh5cGVycGFyYW1ldGVyIGNvbWVzDQojIGZyb20gbWFzdGVyLnlhbWw7IHRoZSBPTkxZIHRoaW5nIGFuIGV4cGVyaW1lbnQgaXMgcGVybWl0dGVkIHRvIGNoYW5nZSBpcw0KIyB3aGljaCByZXNvbHV0aW9uIG9mIHRoZSBkYXRhc2V0IGl0IHJlYWRzLiBzcmMvdHJhaW5pbmcucHkgZW5mb3JjZXMgdGhhdCAtLQ0KIyBhZGRpbmcsIHNheSwgYGxyMGAgaGVyZSByYWlzZXMgcmF0aGVyIHRoYW4gc2lsZW50bHkgYnJlYWtpbmcgdGhlIGNvbnRyb2wuDQoNCmluaGVyaXRzOiBtYXN0ZXIueWFtbA0KZXhwZXJpbWVudDogRTAyDQoNCiMgRm9yIHJlZmVyZW5jZSBvbmx5OyByZWFkIGZyb20gbWFzdGVyLnlhbWwgYXQgcnVudGltZS4NCl9yZXNvbHV0aW9uOg0KICBsYWJlbDogcjUwDQogIHNjYWxlOiAwLjUNCiAgcnVuX2lkOiAwMl9yNTANCiAgYXJlYV9zY2FsZTogMC4yNQ0K",
    "configs/r75.yaml": "IyBFeHBlcmltZW50IEUwMSAtLSA3NSUgbGluZWFyIHJlc29sdXRpb24uIEJveCBhcmVhIGZhbGxzIHRvIDU2LjI1JSBvZiBvcmlnaW5hbC4NCiMNCiMgVGhpcyBmaWxlIGludGVudGlvbmFsbHkgY29udGFpbnMgYWxtb3N0IG5vdGhpbmcuIEV2ZXJ5IGh5cGVycGFyYW1ldGVyIGNvbWVzDQojIGZyb20gbWFzdGVyLnlhbWw7IHRoZSBPTkxZIHRoaW5nIGFuIGV4cGVyaW1lbnQgaXMgcGVybWl0dGVkIHRvIGNoYW5nZSBpcw0KIyB3aGljaCByZXNvbHV0aW9uIG9mIHRoZSBkYXRhc2V0IGl0IHJlYWRzLiBzcmMvdHJhaW5pbmcucHkgZW5mb3JjZXMgdGhhdCAtLQ0KIyBhZGRpbmcsIHNheSwgYGxyMGAgaGVyZSByYWlzZXMgcmF0aGVyIHRoYW4gc2lsZW50bHkgYnJlYWtpbmcgdGhlIGNvbnRyb2wuDQoNCmluaGVyaXRzOiBtYXN0ZXIueWFtbA0KZXhwZXJpbWVudDogRTAxDQoNCiMgRm9yIHJlZmVyZW5jZSBvbmx5OyByZWFkIGZyb20gbWFzdGVyLnlhbWwgYXQgcnVudGltZS4NCl9yZXNvbHV0aW9uOg0KICBsYWJlbDogcjc1DQogIHNjYWxlOiAwLjc1DQogIHJ1bl9pZDogMDFfcjc1DQogIGFyZWFfc2NhbGU6IDAuNTYyNQ0K",
}
for _rel, _b64 in _FILES.items():
    _p = PKG / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(base64.b64decode(_b64))
print(f"wrote {len(_FILES)} file(s) into {PKG} (configs): {sorted(_FILES)}")

In [ ]:
import base64
from pathlib import Path

PKG = Path("/kaggle/working/kaggle")
_FILES = {
    "docs/cleaning_report.md": "IyBGQUlSMU0gYW5ub3RhdGlvbiB2YWxpZGF0aW9uIGFuZCBjbGVhbmluZyByZXBvcnQNCg0KR2VuZXJhdGVkIGJ5IGBzY3JpcHRzL3ZhbGlkYXRlX2Fubm90YXRpb25zLnB5YC4gU291cmNlIG9mIHRydXRoOiBgRGF0YXNldC9sYWJlbHMucGFycXVldGAgKHJlYWQtb25seTsgbmV2ZXIgbW9kaWZpZWQpLg0KDQojIyAxLiBDb29yZGluYXRlIGNvbnZlbnRpb24gKHZlcmlmaWVkLCBub3QgYXNzdW1lZCkNCg0KYGxhYmVscy5wYXJxdWV0YCBjYXJyaWVzIGJvdGggdGhlIEZBSVIxTSBvcmllbnRlZCBib3VuZGluZyBib3ggKE9CQiwgZm91ciBjb3JuZXJzIGB4MCx5MC4ueDMseTNgKSBhbmQgYSBwcmVjb21wdXRlZCBheGlzLWFsaWduZWQgYm94IChgeF9taW4seV9taW4seF9tYXgseV9tYXhgKS4gQ2hlY2tlZCBvdmVyIGFsbCA1OTQsNzc3IHJvd3M6DQoNCi0gVGhlIEFBQkIgZXF1YWxzIHRoZSBPQkIgY29ybmVyIGV4dGVudCAqKmNsaXBwZWQqKiB0byBgWzAsIEltYWdlV2lkdGhdIHggWzAsIEltYWdlSGVpZ2h0XWAgZm9yICoqMTAwLjAwJSoqIG9mIHJvd3MuDQotIEl0IGVxdWFscyB0aGUgKnVuY2xpcHBlZCogZXh0ZW50IGZvciBvbmx5IH45OS42JSBvZiByb3dzLCBpLmUuIGNsaXBwaW5nIGlzIGdlbnVpbmVseSBhcHBsaWVkIGFuZCBpcyB0aGUgZGlmZmVyZW5jZS4NCg0KQ29uc2VxdWVuY2U6IHRoZSBQYXJ0IEYgY2hlY2tzIGZvciAqbmVnYXRpdmUgY29vcmRpbmF0ZXMqIGFuZCAqYm94ZXMgb3V0c2lkZSBpbWFnZSBib3VuZGFyaWVzKiByZXR1cm4gKiowIG9uIHRoZSBzdG9yZWQgQUFCQiBieSBjb25zdHJ1Y3Rpb24qKi4gUmVwb3J0aW5nIG9ubHkgdGhhdCBudW1iZXIgd291bGQgYmUgbWlzbGVhZGluZywgc28gYm90aCBjaGVja3MgYXJlIGFsc28gcnVuIGFnYWluc3QgdGhlIHVuZGVybHlpbmcgT0JCLCB3aGVyZSB0aGUgcmVhbCBvdXQtb2YtcmFzdGVyIGdlb21ldHJ5IGxpdmVzLg0KDQojIyAyLiBQYXJ0IEYgY291bnRlcnMgKGFsbCAzNyBjYXRlZ29yaWVzKQ0KDQp8IENoZWNrIHwgQ291bnQgfCAlIG9mIGFsbCBhbm5vdGF0aW9ucyB8DQp8LS0tfC0tLTp8LS0tOnwNCnwgVG90YWwgYW5ub3RhdGlvbnMgfCA1OTQsNzc3IHwgMTAwLjAwMDAlIHwNCnwgVmFsaWQgKHVzYWJsZSkgYm94ZXMgfCA1OTQsNDgwIHwgOTkuOTUwMSUgfA0KfCBaZXJvLXdpZHRoIGJveGVzIHwgMjcxIHwgMC4wNDU2JSB8DQp8IFplcm8taGVpZ2h0IGJveGVzIHwgMjUxIHwgMC4wNDIyJSB8DQp8IFplcm8tYXJlYSBib3hlcyB8IDI5NyB8IDAuMDQ5OSUgfA0KfCBOZWdhdGl2ZS1jb29yZGluYXRlIGJveGVzIChzdG9yZWQgQUFCQikgfCAwIHwgMC4wMDAwJSB8DQp8IEJveGVzIG91dHNpZGUgaW1hZ2UgYm91bmRhcmllcyAoc3RvcmVkIEFBQkIpIHwgMCB8IDAuMDAwMCUgfA0KfCBOZWdhdGl2ZS1jb29yZGluYXRlIHBvbHlnb25zIChzb3VyY2UgT0JCKSB8IDQsNjAwIHwgMC43NzM0JSB8DQp8IE91dC1vZi1ib3VuZHMgcG9seWdvbnMgKHNvdXJjZSBPQkIpIHwgNCw0NjkgfCAwLjc1MTQlIHwNCnwgUG9seWdvbnMgY2xpcHBlZCBieSB0aGUgdGlsZSBlZGdlIChzb3VyY2UgT0JCKSB8IDgsODc0IHwgMS40OTIwJSB8DQp8IFBvbHlnb25zIGVudGlyZWx5IG91dHNpZGUgdGhlIHJhc3RlciB8IDI5MCB8IDAuMDQ4OCUgfA0KfCBFeHRyZW1lbHkgc21hbGwgYm94ZXMgKGFyZWEgPCAxNiBweF4yKSB8IDQwNiB8IDAuMDY4MyUgfA0KDQojIyAzLiBJbnZlc3RpZ2F0aW9uIG9mIHRoZSA0MDYgJ3RpbnkgYm94ZXMnDQoNClRoZSBkYXRhc2V0IHJlcG9ydCdzIGB0aW55X2JveGVzOiA0MDZgIGNvdW50cyBhbm5vdGF0aW9ucyB3aXRoIEFBQkIgYXJlYSA8IDE2IHB4XjIuIFRoYXQgZmlndXJlIGNvbmZsYXRlcyB0d28gY29tcGxldGVseSBkaWZmZXJlbnQgcG9wdWxhdGlvbnMsIHdoaWNoIGlzIHdoeSBpdCBoYWQgdG8gYmUgc3BsaXQgYmVmb3JlIGFueSBjbGVhbmluZyBkZWNpc2lvbjoNCg0KLSAqKjI5NyBhcmUgemVyby1hcmVhKiogKGRlZ2VuZXJhdGUgZ2VvbWV0cnkpLg0KLSAqKjEwOSBhcmUgbm9uLWRlZ2VuZXJhdGUqKiA0LWNvcm5lciBwb2x5Z29ucyB0aGF0IGFyZSBzaW1wbHkgdmVyeSBzbWFsbC4NCg0KIyMjIDNhLiBUaGUgemVyby1hcmVhIGJveGVzIGFyZSBzb3VyY2UtYW5ub3RhdGlvbiBkZWZlY3RzLCBub3QgY29udmVyc2lvbiBlcnJvcnMNCg0KQ291bnRpbmcgKmRpc3RpbmN0KiB2ZXJ0aWNlcyBpbiBlYWNoIHNvdXJjZSBPQkIgc2VwYXJhdGVzIHRoZSBjYXVzZXMgY2xlYW5seToNCg0KfCBSb290IGNhdXNlIHwgQ291bnQgfCBEaWFnbm9zaXMgfA0KfC0tLXwtLS06fC0tLXwNCnwgYHBvaW50X3BvbHlnb25gIHwgMjI1IHwgQWxsIGZvdXIgT0JCIGNvcm5lcnMgaWRlbnRpY2FsOiBzb3VyY2UgcG9seWdvbiBjb2xsYXBzZWQgdG8gYSBzaW5nbGUgcG9pbnQsIHNvIHRoZSBvYmplY3QgaGFzIG5vIGV4dGVudCBpbiBlaXRoZXIgYXhpcy4gfA0KfCBgbGluZV9wb2x5Z29uYCB8IDY5IHwgT0JCIGhhcyBvbmx5IHR3byBkaXN0aW5jdCBjb3JuZXJzOiBhIHplcm8tYXJlYSBsaW5lIHNlZ21lbnQsIG5vdCBhIHF1YWRyaWxhdGVyYWwuIHwNCnwgYHNsaXZlcl9wb2x5Z29uYCB8IDIgfCBPQkIgaGFzIHRocmVlIGRpc3RpbmN0IGNvcm5lcnMgYW5kIGNvbGxhcHNlcyB0byB6ZXJvIHdpZHRoIG9yIGhlaWdodCBhZnRlciBjbGlwcGluZy4gfA0KfCBgb3V0c2lkZV9pbWFnZWAgfCAxIHwgUG9seWdvbiBsaWVzIGVudGlyZWx5IG91dHNpZGUgdGhlIGltYWdlIHJhc3RlcjsgY2xpcHBpbmcgbGVhdmVzIG5vIHBpeGVscy4gfA0KDQpUaHJlZSBpbmRlcGVuZGVudCBsaW5lcyBvZiBldmlkZW5jZSBydWxlIG91dCBhIHBhcnNpbmcgb3IgY29udmVyc2lvbiBlcnJvciBvbiBvdXIgc2lkZSwgYW5kIHBvaW50IGluc3RlYWQgYXQgdGhlIEZBSVIxTSB0aWxpbmcgcGlwZWxpbmU6DQoNCjEuIFRoZSBkZWZlY3QgaXMgcHJlc2VudCBpbiB0aGUgKipzb3VyY2UgcG9seWdvbiBpdHNlbGYqKiAtIGUuZy4gYHRfMTYzODUuanBnYCBMYWJlbF9JRCA4OTYgc3RvcmVzIGAoLTg2LDQzKWAgZm91ciB0aW1lcyBvdmVyLiBUaGVyZSBpcyBubyBhcml0aG1ldGljIHRoYXQgcmVjb3ZlcnMgYW4gZXh0ZW50IGZyb20gYSBzaW5nbGUgcmVwZWF0ZWQgcG9pbnQsIHNvIG5vdGhpbmcgd2FzIGxvc3QgaW4gdGhlIEFBQkIgZGVyaXZhdGlvbi4NCjIuICoqMjkwIG9mIHRoZSAyOTcgdW51c2FibGUgcG9seWdvbnMgKDk4JSkgc2l0IG9uIG9yIGJleW9uZCBhIHRpbGUgYm91bmRhcnkuKiogRkFJUjFNIGN1dHMgYSBsYXJnZSBzY2VuZSBpbnRvIHRpbGVzOyBhbiBvYmplY3QgbHlpbmcgYWxtb3N0IGVudGlyZWx5IG91dHNpZGUgYSBnaXZlbiB0aWxlIHN0aWxsIHJlY2VpdmVzIGFuIGFubm90YXRpb24gcmVjb3JkIGluIHRoYXQgdGlsZSwgYW5kIGl0cyBwb2x5Z29uIGNvbGxhcHNlcyBvbnRvIChvciBwYXN0KSB0aGUgZWRnZS4gT25seSA3IGRlZ2VuZXJhdGUgcG9seWdvbnMgc2l0IHN0cmljdGx5IGluc2lkZSBhIHRpbGUuIEEgcmFuZG9tIGxhYmVsbGluZyBvciBwYXJzaW5nIGZhdWx0IHdvdWxkIG5vdCBjb25jZW50cmF0ZSBvbiB0aGUgcmFzdGVyIGJvcmRlciBsaWtlIHRoaXMuDQozLiBUaGUgY29sbGFwc2UgaXMgKipjbHVzdGVyZWQsIG5vdCBzY2F0dGVyZWQqKjogMyBpbWFnZShzKSBhY2NvdW50IGZvciAxMjAgb2YgdGhlIDIyNSBwb2ludCBwb2x5Z29ucywgYW5kIHdpdGhpbiBlYWNoIG9mIHRob3NlIGltYWdlcyBldmVyeSBjb2xsYXBzZWQgb2JqZWN0IHNoYXJlcyBvbmUgaWRlbnRpY2FsIGNvb3JkaW5hdGUgLSBjb25zaXN0ZW50IHdpdGggYSB3aG9sZSBibG9jayBvZiBuZWlnaGJvdXJpbmctdGlsZSBvYmplY3RzIGJlaW5nIHByb2plY3RlZCB0byB0aGUgc2FtZSBvZmYtdGlsZSBwb3NpdGlvbi4NCg0KVmlzdWFsIGNvbmZpcm1hdGlvbjogYGV2aWRlbmNlL3BvaW50X3BvbHlnb25fX3RfMTYzODVfXzkwNS5wbmdgIHNob3dzIHRoZSBjbGlwcGVkIGxvY2F0aW9uIG9mIGEgc3VwcG9zZWQgKlNtYWxsIENhciogZmFsbGluZyBvbiB1bmJyb2tlbiB0cmVlIGNhbm9weSAtIHRoZXJlIGlzIG5vIHZlaGljbGUgdGhlcmUuIGBldmlkZW5jZS9saW5lX3BvbHlnb25fX3RfMjk2X18zOTI4Mi5wbmdgIHNob3dzIGEgKlZhbiogd2hvc2UgcG9seWdvbiBgKDMzNiwwKS0oMzQ1LDApYCBpcyBhIDkgcHggbGluZSBseWluZyBleGFjdGx5IGFsb25nIHRoZSB0b3AgdGlsZSBlZGdlOiB0aGUgdmVoaWNsZSBpdHNlbGYgaXMgaW4gdGhlIG5laWdoYm91cmluZyB0aWxlLg0KDQp8IERlZmVjdCB8IFRvdGFsIHwgT24vb3ZlciBhIHRpbGUgZWRnZSB8IFN0cmljdGx5IGluc2lkZSBhIHRpbGUgfA0KfC0tLXwtLS06fC0tLTp8LS0tOnwNCnwgYHBvaW50X3BvbHlnb25gIHwgMjI1IHwgMjE5IHwgNiB8DQp8IGBsaW5lX3BvbHlnb25gIHwgNjkgfCA2OCB8IDEgfA0KfCBgc2xpdmVyX3BvbHlnb25gIHwgMiB8IDIgfCAwIHwNCnwgYG91dHNpZGVfaW1hZ2VgIHwgMSB8IDEgfCAwIHwNCg0KV29yc3QtYWZmZWN0ZWQgaW1hZ2VzOg0KDQp8IEltYWdlIHwgU3BsaXQgfCBVbnVzYWJsZSBvYmplY3RzIHwNCnwtLS18LS0tfC0tLTp8DQp8IGB0XzE2Mzg1LmpwZ2AgfCBUcmFpbiB8IDYwIHwNCnwgYHRfODY5My5qcGdgIHwgVHJhaW4gfCA0MyB8DQp8IGB0XzM2MTAuanBnYCB8IFRyYWluIHwgMTcgfA0KfCBgdF8yMTcuanBnYCB8IFRyYWluIHwgNCB8DQp8IGB0XzE2MDg4LmpwZ2AgfCBUcmFpbiB8IDQgfA0KfCBgdF8xMzc3LmpwZ2AgfCBUcmFpbiB8IDMgfA0KfCBgdF81NTUuanBnYCB8IFRyYWluIHwgMyB8DQp8IGB0XzExNTAuanBnYCB8IFRyYWluIHwgMiB8DQoNCiMjIyAzYi4gVGhlIHJlbWFpbmluZyB0aW55IGJveGVzIGFyZSB2YWxpZCBleHRyZW1lbHkgc21hbGwgb2JqZWN0cw0KDQpUaGUgMTA5IG5vbi1kZWdlbmVyYXRlIHRpbnkgYm94ZXMgaGF2ZSBmb3VyIGRpc3RpbmN0IGNvcm5lcnMgYW5kIGEgcG9zaXRpdmUgZXh0ZW50LiBUaGVpciBjYXRlZ29yeSBtaXggaXMgZG9taW5hdGVkIGJ5IHRoZSBjbGFzc2VzIHRoYXQgYXJlIGdlbnVpbmVseSBhIGZldyBwaXhlbHMgYWNyb3NzIGF0IH4wLjggbSBHU0Q6DQoNCnwgQ2F0ZWdvcnkgfCBDb3VudCB8DQp8LS0tfC0tLTp8DQp8IFZhbiB8IDQ3IHwNCnwgU21hbGwgQ2FyIHwgMzEgfA0KfCBDYXJnbyBUcnVjayB8IDExIHwNCnwgRHVtcCBUcnVjayB8IDggfA0KfCBJbnRlcnNlY3Rpb24gfCA1IHwNCnwgTW90b3Jib2F0IHwgMiB8DQp8IFRyYWlsZXIgfCAyIHwNCnwgRmlzaGluZyBCb2F0IHwgMSB8DQp8IEZvb3RiYWxsIEZpZWxkIHwgMSB8DQp8IEJ1cyB8IDEgfA0KDQpNZWRpYW4gc2l6ZSBvZiB0aGlzIGdyb3VwOiAyIHggMyBweCwgYXJlYSA2IHB4XjIuIFRoZXNlIGFyZSAqKnJldGFpbmVkKiogLSB0aGV5IGFyZSByZWFsIG9iamVjdHMsIGFuZCBkaXNjYXJkaW5nIHRoZSBzbWFsbGVzdCB0cnVlIHBvc2l0aXZlcyB3b3VsZCBiaWFzIHRoZSB2ZXJ5IHNtYWxsLW9iamVjdCBhbmFseXNpcyB0aGlzIHN0dWR5IGlzIGFib3V0Lg0KDQojIyA0LiBJbXBhY3Qgb24gdGhlIHByaW1hcnkgKHNoaXAtb25seSkgZXhwZXJpbWVudA0KDQotIFNoaXAgYW5ub3RhdGlvbnMgKDkgRkFJUjFNIHNoaXAgY2F0ZWdvcmllcyk6ICoqNTgsOTgyKioNCi0gU2hpcCBhbm5vdGF0aW9ucyB0aGF0IGFyZSBkZWdlbmVyYXRlL3VudXNhYmxlOiAqKjAqKg0KLSBTaGlwIGFubm90YXRpb25zIHdpdGggYXJlYSA8IDE2IHB4XjIgYnV0IHZhbGlkOiAqKjMqKg0KDQoqKk5vIHNoaXAgYW5ub3RhdGlvbiBpcyBhZmZlY3RlZCBieSBhbnkgb2YgdGhlIGRlZ2VuZXJhdGUtZ2VvbWV0cnkgZGVmZWN0cy4qKiBFdmVyeSBvbmUgb2YgdGhlIHVudXNhYmxlIG9iamVjdHMgYmVsb25ncyB0byBhIFZlaGljbGUsIFJvYWQsIENvdXJ0IG9yIEFpcnBsYW5lIGNhdGVnb3J5LiBUaGUgc2hpcC1vbmx5IGRhdGFzZXQgdXNlZCBmb3IgRTAwLUUwMyB0aGVyZWZvcmUgcmVxdWlyZXMgbm8gZ2VvbWV0cmljIGNsZWFuaW5nIGF0IGFsbCwgYW5kIG5vIHNoaXAgaW5zdGFuY2UgaXMgcmVtb3ZlZC4NCg0KLSBTcGxpdCBsb2NhdGlvbiBvZiB0aGUgdW51c2FibGUgb2JqZWN0czogYHsnVHJhaW4nOiAyOTd9YC4NCg0KIyMgNS4gUmVtb3ZhbCBkZWNpc2lvbnMNCg0KfCBEZWZlY3QgfCBDb3VudCB8IEFjdGlvbiB8IFJlYXNvbiB8DQp8LS0tfC0tLTp8LS0tfC0tLXwNCnwgYHBvaW50X3BvbHlnb25gIHwgMjI1IHwgKipSZW1vdmVkKiogfCBBbGwgZm91ciBPQkIgY29ybmVycyBpZGVudGljYWw6IHNvdXJjZSBwb2x5Z29uIGNvbGxhcHNlZCB0byBhIHNpbmdsZSBwb2ludCwgc28gdGhlIG9iamVjdCBoYXMgbm8gZXh0ZW50IGluIGVpdGhlciBheGlzLiBObyByZWNvdmVyYWJsZSBleHRlbnQsIHNvIHRoZSBib3ggY2Fubm90IHN1cHBseSBhIHJlZ3Jlc3Npb24gdGFyZ2V0LiB8DQp8IGBsaW5lX3BvbHlnb25gIHwgNjkgfCAqKlJlbW92ZWQqKiB8IE9CQiBoYXMgb25seSB0d28gZGlzdGluY3QgY29ybmVyczogYSB6ZXJvLWFyZWEgbGluZSBzZWdtZW50LCBub3QgYSBxdWFkcmlsYXRlcmFsLiBObyByZWNvdmVyYWJsZSBleHRlbnQsIHNvIHRoZSBib3ggY2Fubm90IHN1cHBseSBhIHJlZ3Jlc3Npb24gdGFyZ2V0LiB8DQp8IGBzbGl2ZXJfcG9seWdvbmAgfCAyIHwgKipSZW1vdmVkKiogfCBPQkIgaGFzIHRocmVlIGRpc3RpbmN0IGNvcm5lcnMgYW5kIGNvbGxhcHNlcyB0byB6ZXJvIHdpZHRoIG9yIGhlaWdodCBhZnRlciBjbGlwcGluZy4gTm8gcmVjb3ZlcmFibGUgZXh0ZW50LCBzbyB0aGUgYm94IGNhbm5vdCBzdXBwbHkgYSByZWdyZXNzaW9uIHRhcmdldC4gfA0KfCBgb3V0c2lkZV9pbWFnZWAgfCAxIHwgKipSZW1vdmVkKiogfCBQb2x5Z29uIGxpZXMgZW50aXJlbHkgb3V0c2lkZSB0aGUgaW1hZ2UgcmFzdGVyOyBjbGlwcGluZyBsZWF2ZXMgbm8gcGl4ZWxzLiBObyByZWNvdmVyYWJsZSBleHRlbnQsIHNvIHRoZSBib3ggY2Fubm90IHN1cHBseSBhIHJlZ3Jlc3Npb24gdGFyZ2V0LiB8DQp8IGB0aW55X2J1dF92YWxpZGAgfCAxMDkgfCAqKlJldGFpbmVkKiogfCBOb24tZGVnZW5lcmF0ZSByZWFsIG9iamVjdHM7IHJlbW92aW5nIHRoZW0gd291bGQgYmlhcyBzbWFsbC1vYmplY3QgYW5hbHlzaXMuIHwNCnwgYG9rYCB8IDU5NCwzNzEgfCAqKlJldGFpbmVkKiogfCBQYXNzZXMgYWxsIGNoZWNrcy4gfA0KDQpBY2NvdW50aW5nOg0KDQpgYGB0ZXh0DQpvcmlnaW5hbCBhbm5vdGF0aW9ucyA6IDU5NCw3NzcNCmludmFsaWQgYW5ub3RhdGlvbnMgIDogMjk3DQpyZW1vdmVkIGFubm90YXRpb25zICA6IDI5Nw0KcmV0YWluZWQgYW5ub3RhdGlvbnMgOiA1OTQsNDgwDQpyZW1vdmVkIHNoaXAgb2JqZWN0cyA6IDANCmBgYA0KDQpFdmVyeSByZW1vdmVkIHJvdyBpcyBsaXN0ZWQgaW5kaXZpZHVhbGx5IGluIGByZW1vdmFsX2xvZy5jc3ZgOyBldmVyeSBmbGFnZ2VkIHJvdyAocmVtb3ZlZCBvciByZXRhaW5lZCkgaXMgaW4gYGZsYWdnZWRfYW5ub3RhdGlvbnMuY3N2YC4gTm90aGluZyBpcyBkcm9wcGVkIHNpbGVudGx5Lg0KDQojIyA2LiBOb24tZGVmZWN0cywgZXhwbGljaXRseQ0KDQotICoqOCw4NzQgcG9seWdvbnMgY3Jvc3MgYSB0aWxlIGVkZ2UuKiogVGhlc2UgYXJlIGNvcnJlY3QgYW5ub3RhdGlvbnMgb2Ygb2JqZWN0cyB0aGF0IGdlbnVpbmVseSBzdHJhZGRsZSB0aGUgdGlsZSBib3VuZGFyeTsgRkFJUjFNIHRpbGVzIGEgbGFyZ2VyIHNjZW5lLiBUaGVpciBzdG9yZWQgQUFCQiBpcyBhbHJlYWR5IGNvcnJlY3RseSBjbGlwcGVkIHRvIHRoZSByYXN0ZXIsIGFuZCB0aGV5IGFyZSBrZXB0Lg0KLSAqKlplcm8gbmVnYXRpdmUgLyBvdXQtb2YtYm91bmRzIHN0b3JlZCBBQUJCIGNvb3JkaW5hdGVzLioqIE5vdCBiZWNhdXNlIHRoZSBjaGVjayBpcyB0cml2aWFsbHkgc2F0aXNmaWVkLCBidXQgYmVjYXVzZSBjbGlwcGluZyBhbHJlYWR5IGhhcHBlbmVkIHVwc3RyZWFtLiBSZS12ZXJpZmllZCByYXRoZXIgdGhhbiBhc3N1bWVkIChzZWN0aW9uIDEpLg0KDQojIyA3LiBWaXN1YWwgZXZpZGVuY2UNCg0KYGV2aWRlbmNlL2AgaG9sZHMgMTEgY3JvcHMgY2VudHJlZCBvbiBzYW1wbGVkIGRlZmVjdGl2ZSBib3hlcyAocmVkIHggPSB0aGUgZGVmZWN0aXZlIGFubm90YXRpb24sIGdyZWVuID0gdmFsaWQgbmVpZ2hib3VycyBpbiB0aGUgc2FtZSB0aWxlLCBmb3Igc2NhbGUpLg0KDQotIGBsaW5lX3BvbHlnb25gIC0gYGV2aWRlbmNlL2xpbmVfcG9seWdvbl9fdF8xODhfXzEyMDA3My5wbmdgIChTbWFsbCBDYXIpDQotIGBsaW5lX3BvbHlnb25gIC0gYGV2aWRlbmNlL2xpbmVfcG9seWdvbl9fdF83MDZfXzYwOTgucG5nYCAoSW50ZXJzZWN0aW9uKQ0KLSBgbGluZV9wb2x5Z29uYCAtIGBldmlkZW5jZS9saW5lX3BvbHlnb25fX3RfNjU2X18yNjE1ODIucG5nYCAoU21hbGwgQ2FyKQ0KLSBgbGluZV9wb2x5Z29uYCAtIGBldmlkZW5jZS9saW5lX3BvbHlnb25fX3RfMjk2X18zOTI4Mi5wbmdgIChWYW4pDQotIGBvdXRzaWRlX2ltYWdlYCAtIGBldmlkZW5jZS9vdXRzaWRlX2ltYWdlX190Xzg1X184ODQ5OC5wbmdgIChWYW4pDQotIGBwb2ludF9wb2x5Z29uYCAtIGBldmlkZW5jZS9wb2ludF9wb2x5Z29uX190XzE2Mzg1X185MDUucG5nYCAoU21hbGwgQ2FyKQ0KLSBgcG9pbnRfcG9seWdvbmAgLSBgZXZpZGVuY2UvcG9pbnRfcG9seWdvbl9fdF80MzZfXzIwNDA0MS5wbmdgIChTbWFsbCBDYXIpDQotIGBwb2ludF9wb2x5Z29uYCAtIGBldmlkZW5jZS9wb2ludF9wb2x5Z29uX190XzY4MV9fNTQyMzMucG5nYCAoVmFuKQ0KLSBgcG9pbnRfcG9seWdvbmAgLSBgZXZpZGVuY2UvcG9pbnRfcG9seWdvbl9fdF8xODZfXzMxMjAyOC5wbmdgIChWYW4pDQotIGBzbGl2ZXJfcG9seWdvbmAgLSBgZXZpZGVuY2Uvc2xpdmVyX3BvbHlnb25fX3RfMjY0X18yOTAxODYucG5nYCAoU21hbGwgQ2FyKQ0KLSBgc2xpdmVyX3BvbHlnb25gIC0gYGV2aWRlbmNlL3NsaXZlcl9wb2x5Z29uX190XzI3X18yMTEwMDcucG5nYCAoQnVzKQ0KDQojIyA4LiBWZXJkaWN0DQoNClNhZmUgdG8gcHJvY2VlZC4gMjk3IG9mIDU5NCw3NzcgYW5ub3RhdGlvbnMgKDAuMDUwJSkgYXJlIHJlbW92ZWQsIGFsbCBmb3IgZG9jdW1lbnRlZCBnZW9tZXRyaWMgcmVhc29ucywgYWxsIGluIHRoZSBUcmFpbiBzcGxpdCwgYW5kICoqbm9uZSBvZiB0aGVtIHNoaXBzKiouIFRoZSBwcmltYXJ5IHNpbmdsZS1jbGFzcyBzaGlwIGV4cGVyaW1lbnQgaW5oZXJpdHMgYSBmdWxseSBjbGVhbiBhbm5vdGF0aW9uIHNldC4NCg==",
    "docs/object_size_definition.md": "IyBPYmplY3Qtc2l6ZSBiaW4gZGVmaW5pdGlvbg0KDQpHZW5lcmF0ZWQgYnkgYHNjcmlwdHMvYW5hbHl6ZV9vYmplY3Rfc2l6ZXMucHlgLiBUaGlzIGZpeGVzIHRoZSBgc21hbGxgIC8gYG1lZGl1bWAgLyBgbGFyZ2VgIGdyb3VwaW5nIHVzZWQgZm9yIGV2ZXJ5IGBBUF9zbWFsbGAgLyBgQVBfbWVkaXVtYCAvIGBBUF9sYXJnZWAgbnVtYmVyIGluIHRoZSBzdHVkeS4NCg0KIyMgRGVjaXNpb24NCg0KKipQcmltYXJ5OiBDT0NPIGFic29sdXRlIHBpeGVsLWFyZWEgdGhyZXNob2xkcywgbWVhc3VyZWQgb24gdGhlIE9SSUdJTkFMLXJlc29sdXRpb24gYW5ub3RhdGlvbi4qKg0KDQpgYGB0ZXh0DQpzbWFsbCAgIGFyZWEgPCAgMzJeMiAgPSAgMSwwMjQgcHheMg0KbWVkaXVtICAxLDAyNCA8PSBhcmVhIDwgOTZeMiA9IDksMjE2IHB4XjINCmxhcmdlICAgYXJlYSA+PSA5LDIxNiBweF4yDQpgYGANCg0KIyMgV2h5IHRoZXNlIHRocmVzaG9sZHMNCg0KMS4gKipDb21wYXJhYmlsaXR5LioqIFRoZXkgYXJlIHRoZSB0aHJlc2hvbGRzIGJlaGluZCBldmVyeSBwdWJsaXNoZWQgYEFQX1NgL2BBUF9NYC9gQVBfTGAgbnVtYmVyIGluIHRoZSBkZXRlY3Rpb24gbGl0ZXJhdHVyZS4gSW52ZW50aW5nIHByb2plY3Qtc3BlY2lmaWMgY3V0IHBvaW50cyB3b3VsZCBtYWtlIHRoaXMgc3R1ZHkncyBzbWFsbC1vYmplY3QgcmVzdWx0IGltcG9zc2libGUgdG8gcGxhY2UgYWdhaW5zdCBwcmlvciB3b3JrLCB3aGljaCBpcyB0aGUgbWFpbiB0aGluZyBhIHJlc29sdXRpb24gc3R1ZHkgaXMgcmVhZCBmb3IuDQoNCjIuICoqVGhleSBhcmUgbm90IGRlZ2VuZXJhdGUgb24gdGhpcyBkYXRhLioqIEFwcGxpZWQgdG8gdGhlIDU4LDk4MiBjbGVhbiBGQUlSMU0gc2hpcCBib3hlcyB0aGV5IHNwbGl0IHRoZSBwb3B1bGF0aW9uIDQyLjQlIC8gNDcuMSUgLyAxMC41JToNCg0KfCBCaW4gfCBUaHJlc2hvbGQgKHB4wrIpIHwgU2hpcHMgfCBTaGFyZSB8DQp8LS0tfC0tLXwtLS06fC0tLTp8DQp8IHNtYWxsIHwgPCAxLDAyNCB8IDI0LDk5OCB8IDQyLjQlIHwNCnwgbWVkaXVtIHwgMSwwMjQg4oCTIDksMjE2IHwgMjcsNzc0IHwgNDcuMSUgfA0KfCBsYXJnZSB8IOKJpSA5LDIxNiB8IDYsMjEwIHwgMTAuNSUgfA0KDQogICBFdmVyeSBiaW4gaG9sZHMgdGhvdXNhbmRzIG9mIGluc3RhbmNlcywgc28gcGVyLWJpbiBBUCBpcyBlc3RpbWFibGUgcmF0aGVyIHRoYW4gbm9pc2UuIEEgdGhyZXNob2xkIHNldCB0aGF0IGR1bXBlZCA5NSUgb2Ygc2hpcHMgaW50byBvbmUgYmluIHdvdWxkIGhhdmUgaGFkIHRvIGJlIHJlamVjdGVkIGluIGZhdm91ciBvZiBxdWFudGlsZXM7IHRoYXQgaXMgbm90IHRoZSBjYXNlIGhlcmUuDQoNCjMuICoqVGhleSBsaW5lIHVwIHdpdGggdGhlIHNlbnNvci4qKiBBdCB0aGUgRkFJUjFNIG1lZGlhbiBHU0Qgb2YgfjAuODEgbSwgdGhlIDMyIHB4IGJvdW5kYXJ5IGlzIGFib3V0IDI2IG0gb24gdGhlIGdyb3VuZCBhbmQgdGhlIDk2IHB4IGJvdW5kYXJ5IGFib3V0IDc4IG0uIFRob3NlIGFyZSBjbG9zZSB0byB0aGUgcmVhbCB0cmFuc2l0aW9ucyBiZXR3ZWVuIHNtYWxsIGNyYWZ0IChtb3RvcmJvYXRzLCBmaXNoaW5nIGJvYXRzKSwgbWlkLXNpemUgd29ya2luZyB2ZXNzZWxzLCBhbmQgbGFyZ2UgY2FyZ28gc2hpcHMg4oCUIHNvIHRoZSBiaW5zIGNhcnJ5IHBoeXNpY2FsIG1lYW5pbmcsIG5vdCBqdXN0IHBpeGVsIGJvb2trZWVwaW5nLg0KDQo0LiAqKlRoZXkgbWF0Y2ggdGhlIGRldGVjdG9yJ3Mgb3duIHNjYWxlIHN0cnVjdHVyZS4qKiBZT0xPdjggcHJlZGljdHMgb24gUDMvUDQvUDUgZmVhdHVyZSBtYXBzIGF0IHN0cmlkZXMgOC8xNi8zMi4gQSAzMiBweCBvYmplY3Qgc3BhbnMgfjQgY2VsbHMgYXQgUDMgYW5kIGEgOTYgcHggb2JqZWN0IH4zIGNlbGxzIGF0IFA1LCBzbyB0aGUgYmlucyByb3VnaGx5IGNvcnJlc3BvbmQgdG8gd2hpY2ggaGVhZCBpcyByZXNwb25zaWJsZSBmb3IgdGhlIG9iamVjdC4NCg0KIyMgU2Vjb25kYXJ5OiBkYXRhc2V0IHRlcnRpbGVzIChzZW5zaXRpdml0eSBjaGVjaykNCg0KRXF1YWwtY291bnQgdGVydGlsZXMgb2YgdGhlIHNhbWUgcG9wdWxhdGlvbiBmYWxsIGF0ICoqNjgyIHB4wrIqKiBhbmQgKioyLDc2MCBweMKyKiogKOKJiDI2w5cyNiBweCBhbmQg4omINTPDlzUzIHB4KS4gQm90aCBiaW5uaW5ncyBhcmUgcmVjb3JkZWQgcGVyIG9iamVjdCBpbiB0aGUgc2hpcCBkYXRhc2V0IHNvIHJlc3VsdHMgY2FuIGJlIHJlcG9ydGVkIGVpdGhlciB3YXkuIFRoZSB0ZXJ0aWxlIHNwbGl0IGd1YXJhbnRlZXMgZXF1YWwgc3VwcG9ydCBwZXIgYmluLCB3aGljaCB0aGUgQ09DTyBzcGxpdCBkb2VzIG5vdDsgaWYgdGhlIHR3byBkaXNhZ3JlZSwgdGhhdCBkaXNhZ3JlZW1lbnQgaXMgaXRzZWxmIGEgZmluZGluZyBhbmQgbXVzdCBiZSByZXBvcnRlZCByYXRoZXIgdGhhbiByZXNvbHZlZCBieSBwaWNraW5nIHRoZSBuaWNlciBudW1iZXIuDQoNCiMjIFRoZSBiaW5zIGFyZSBmaXhlZCBhdCBvcmlnaW5hbCByZXNvbHV0aW9uIGFuZCBuZXZlciByZWNvbXB1dGVkDQoNClRoaXMgaXMgdGhlIGxvYWQtYmVhcmluZyBtZXRob2RvbG9naWNhbCBjaG9pY2Ugb2YgdGhlIHdob2xlIHJlc29sdXRpb24gw5cgb2JqZWN0LXNpemUgYW5hbHlzaXMuDQoNCkVhY2ggc2hpcCBpcyBhc3NpZ25lZCBpdHMgYmluICoqb25jZSoqLCBmcm9tIGl0cyBvcmlnaW5hbC1yZXNvbHV0aW9uIGFyZWEsIGFuZCBrZWVwcyB0aGF0IGxhYmVsIGluIGV2ZXJ5IGV4cGVyaW1lbnQgRTAw4oCTRTAzLiBCaW5zIGFyZSBhdHRhY2hlZCB0byB0aGUgb2JqZWN0IGlkZW50aXR5IGAoSW1nX0lELCBMYWJlbF9JRClgLCBub3QgcmVjb21wdXRlZCBmcm9tIHRoZSByZXNpemVkIGJveGVzLg0KDQpUaGUgYWx0ZXJuYXRpdmUg4oCUIHJlLWRlcml2aW5nIGJpbnMgYXQgZWFjaCByZXNvbHV0aW9uIOKAlCB3b3VsZCBiZSB3cm9uZy4gRG93bnNhbXBsaW5nIHNjYWxlcyBhcmVhIGJ5IGBzwrJgLCBzbyBhdCByMjUgYW4gb2JqZWN0J3MgYXJlYSBmYWxscyB0byAxLzE2IG9mIGl0cyBvcmlnaW5hbCB2YWx1ZSBhbmQgYWxtb3N0IHRoZSBlbnRpcmUgcG9wdWxhdGlvbiBzbGlkZXMgaW50byBgc21hbGxgLiBFYWNoIHJvdyBvZiB0aGUgcmVzb2x1dGlvbiDDlyBzaXplIHRhYmxlIHdvdWxkIHRoZW4gZGVzY3JpYmUgYSBkaWZmZXJlbnQgc2V0IG9mIHNoaXBzLCBhbmQgYSBjaGFuZ2UgaW4gYEFQX3NtYWxsYCBkb3duIHRoZSBjb2x1bW4gY291bGQgbm90IGJlIHNlcGFyYXRlZCBmcm9tIHJlY2xhc3NpZmljYXRpb24uIEZpeGluZyB0aGUgYmlucyBrZWVwcyBldmVyeSByb3cgYWJvdXQgKip0aGUgc2FtZSBzaGlwcyoqLCBzbyBhIGRyb3AgaXMgYXR0cmlidXRhYmxlIHRvIGxvc3QgaW1hZ2UgZGV0YWlsLg0KDQojIyBXaGF0IGhhcHBlbnMgdG8gZWFjaCBiaW4gYXMgcmVzb2x1dGlvbiBmYWxscw0KDQpBcmVhIHNjYWxlcyB3aXRoIHRoZSAqc3F1YXJlKiBvZiB0aGUgbGluZWFyIGZhY3RvcjogcjUwIGtlZXBzIGhhbGYgdGhlIGxpbmVhciBzaXplIGJ1dCBvbmx5ICoqMjUlKiogb2YgdGhlIHBpeGVsIGZvb3RwcmludDsgcjI1IGtlZXBzICoqNi4yNSUqKi4NCg0KfCBSZXNvbHV0aW9uIHwgQmluIHwgU2hpcHMgfCBNZWRpYW4gYm94IChweCkgfCBNZWRpYW4gYXJlYSAocHjCsikgfCAlIHdpdGggbG9uZ2VzdCBzaWRlIDwgOCBweCB8DQp8LS0tfC0tLXwtLS06fC0tLXwtLS06fC0tLTp8DQp8IG9yaWdpbmFsIHwgc21hbGwgfCAyNCw5OTggfCAyMiDDlyAxOSB8IDQyNCB8IDAuMCUgfA0KfCBvcmlnaW5hbCB8IG1lZGl1bSB8IDI3LDc3NCB8IDU1IMOXIDUwIHwgMiw2ODAgfCAwLjAlIHwNCnwgb3JpZ2luYWwgfCBsYXJnZSB8IDYsMjEwIHwgMTQ2IMOXIDEyMyB8IDE2LDExNCB8IDAuMCUgfA0KfCByNzUgfCBzbWFsbCB8IDI0LDk5OCB8IDE2IMOXIDE0IHwgMjM4IHwgMC42JSB8DQp8IHI3NSB8IG1lZGl1bSB8IDI3LDc3NCB8IDQxIMOXIDM4IHwgMSw1MDggfCAwLjAlIHwNCnwgcjc1IHwgbGFyZ2UgfCA2LDIxMCB8IDExMCDDlyA5MiB8IDksMDY0IHwgMC4wJSB8DQp8IHI1MCB8IHNtYWxsIHwgMjQsOTk4IHwgMTEgw5cgMTAgfCAxMDYgfCA2LjUlIHwNCnwgcjUwIHwgbWVkaXVtIHwgMjcsNzc0IHwgMjggw5cgMjUgfCA2NzAgfCAwLjAlIHwNCnwgcjUwIHwgbGFyZ2UgfCA2LDIxMCB8IDczIMOXIDYyIHwgNCwwMjggfCAwLjAlIHwNCnwgcjI1IHwgc21hbGwgfCAyNCw5OTggfCA2IMOXIDUgfCAyNiB8IDY4LjUlIHwNCnwgcjI1IHwgbWVkaXVtIHwgMjcsNzc0IHwgMTQgw5cgMTIgfCAxNjggfCAwLjAlIHwNCnwgcjI1IHwgbGFyZ2UgfCA2LDIxMCB8IDM2IMOXIDMxIHwgMSwwMDcgfCAwLjAlIHwNCg0KIyMgUmVwb3J0aW5nIHJlcXVpcmVtZW50DQoNCkFueSB0YWJsZSBvciBwbG90IG9mIGBBUF9zbWFsbGAgLyBgQVBfbWVkaXVtYCAvIGBBUF9sYXJnZWAgaW4gdGhpcyBzdHVkeSBtdXN0IHN0YXRlIHRoZSB0aHJlc2hvbGRzIGlubGluZSAoYHNtYWxsID0gYXJlYSA8IDEwMjQgcHjCsiBhdCBvcmlnaW5hbCByZXNvbHV0aW9uYCksIGJlY2F1c2UgdGhlIG51bWJlcnMgYXJlIG1lYW5pbmdsZXNzIHdpdGhvdXQgdGhlbS4NCg==",
}
for _rel, _b64 in _FILES.items():
    _p = PKG / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(base64.b64decode(_b64))
print(f"wrote {len(_FILES)} file(s) into {PKG} (docs): {sorted(_FILES)}")

In [ ]:
import base64
from pathlib import Path

PKG = Path("/kaggle/working/kaggle")
_FILES = {
    "data/ship_only/README.md": "IyBTaGlwLW9ubHkgYW5ub3RhdGlvbiBsYXllcg0KDQpHZW5lcmF0ZWQgYnkgYHNjcmlwdHMvY3JlYXRlX3NoaXBfZGF0YXNldC5weWAuIERlcml2ZWQgZnJvbSBgRGF0YXNldC9sYWJlbHMucGFycXVldGA7IHRoZSBvcmlnaW5hbCBkYXRhc2V0IGlzIHVudG91Y2hlZC4NCg0KIyMgQ29udGVudHMNCg0KfCBGaWxlIHwgUm93cyB8IERlc2NyaXB0aW9uIHwNCnwtLS18LS0tOnwtLS18DQp8IGBzaGlwX2Fubm90YXRpb25zLnBhcnF1ZXRgIHwgNTgsOTgyIHwgb25lIHJvdyBwZXIgc2hpcCBvYmplY3QsIG9yaWdpbmFsLXJlc29sdXRpb24gcGl4ZWwgY29vcmRpbmF0ZXMgfA0KfCBgaW1hZ2VfbWFuaWZlc3QuY3N2YCB8IDcsNzA2IHwgb25lIHJvdyBwZXIgaW1hZ2UgaW4gdGhlIGRlcml2ZWQgZGF0YXNldCwgcG9zaXRpdmVzIGFuZCByZXRhaW5lZCBuZWdhdGl2ZXMgfA0KfCBgc3RhdHMuanNvbmAgfCAtIHwgY291bnRzIHVzZWQgYnkgZG93bnN0cmVhbSBzY3JpcHRzIHwNCg0KIyMgQ2xhc3MgbWFwcGluZw0KDQpBbGwgOSBGQUlSMU0gc2hpcCBjYXRlZ29yaWVzIGNvbGxhcHNlIHRvIGEgc2luZ2xlIGNsYXNzIGZvciB0aGUgcHJpbWFyeSBleHBlcmltZW50Og0KDQpgYGB0ZXh0DQpjbGFzc19pZCAgID0gMA0KY2xhc3NfbmFtZSA9IHNoaXANCmBgYA0KDQp8IEZBSVIxTSBjYXRlZ29yeSB8IE9iamVjdHMgfCBmaW5lX2NsYXNzX2lkIChQYXJ0IFQgb25seSkgfA0KfC0tLXwtLS06fC0tLTp8DQp8IERyeSBDYXJnbyBTaGlwIHwgMTcsNDc0IHwgMCB8DQp8IEVuZ2luZWVyaW5nIFNoaXAgfCAzLDg5NyB8IDEgfA0KfCBGaXNoaW5nIEJvYXQgfCA5LDAzMSB8IDIgfA0KfCBMaXF1aWQgQ2FyZ28gU2hpcCB8IDMsODgzIHwgMyB8DQp8IE1vdG9yYm9hdCB8IDE1LDQ0NSB8IDQgfA0KfCBQYXNzZW5nZXIgU2hpcCB8IDEsOTI0IHwgNSB8DQp8IFR1Z2JvYXQgfCAxLDg5NyB8IDYgfA0KfCBXYXJzaGlwIHwgMSwxNzggfCA3IHwNCnwgb3RoZXItc2hpcCB8IDQsMjUzIHwgOCB8DQoNCmBmaW5lX2NsYXNzX2lkYCBpcyBjYXJyaWVkIG9uIGV2ZXJ5IHJvdyBidXQgaXMgKipub3QqKiB1c2VkIGJ5IHRoZSBwcmltYXJ5IGV4cGVyaW1lbnQuIEl0IGV4aXN0cyBzbyB0aGUgb3B0aW9uYWwgZmluZS1ncmFpbmVkIHN0dWR5IChQYXJ0IFQpIGNhbiBiZSBidWlsdCBmcm9tIHRoaXMgc2FtZSBsYXllciB3aXRob3V0IHJlLWRlcml2aW5nIHRoZSBzcGxpdCBvciB0aGUgZ2VvbWV0cnkuDQoNCiMjIEltYWdlIHNlbGVjdGlvbiBhbmQgYmFja2dyb3VuZCBwb2xpY3kNCg0KLSBJbWFnZXMgY29udGFpbmluZyBhdCBsZWFzdCBvbmUgc2hpcDogKio2LDkzNSoqDQotIFNoaXAtZnJlZSBpbWFnZXMgYXZhaWxhYmxlIGluIEZBSVIxTTogKioxNyw4NDAqKg0KLSBTaGlwLWZyZWUgaW1hZ2VzIHJldGFpbmVkOiAqKjc3MSoqIChgLS1iYWNrZ3JvdW5kLWZyYWN0aW9uIDAuMWAsIHNlZWQgNDIpDQotIFRvdGFsIGltYWdlcyBpbiB0aGUgZGVyaXZlZCBkYXRhc2V0OiAqKjcsNzA2KioNCg0KIyMjIFdoeSBrZWVwIGFueSBuZWdhdGl2ZXMgYXQgYWxsDQoNCkEgZGV0ZWN0b3IgdHJhaW5lZCBvbmx5IG9uIGltYWdlcyB0aGF0IGFyZSBndWFyYW50ZWVkIHRvIGNvbnRhaW4gYSBzaGlwIGxlYXJucyBhIGJpYXNlZCBwcmlvciBhbmQgcHJvZHVjZXMgZmFsc2UgcG9zaXRpdmVzIG9uIGVtcHR5IHdhdGVyIGFuZCBvbiBsYW5kIGNsdXR0ZXIuIFJldGFpbmluZyBhIHNsaWNlIG9mIHNoaXAtZnJlZSBpbWFnZXJ5IGdpdmVzIHRoZSBtb2RlbCBleHBsaWNpdCBldmlkZW5jZSBhYm91dCB3aGF0IGlzICpub3QqIGEgc2hpcCwgd2hpY2ggaXMgd2hhdCBrZWVwcyBwcmVjaXNpb24gbWVhbmluZ2Z1bC4NCg0KIyMjIFdoeSBub3Qga2VlcCBhbGwgb2YgdGhlbQ0KDQpSZXRhaW5pbmcgYWxsIDE3LDg0MCB3b3VsZCBtYWtlIHRoZSBkYXRhc2V0IDIuNnggbGFyZ2VyIGZvciB0aGUgcmVzb2x1dGlvbiBzd2VlcCwgbXVsdGlwbHkgdGhlIENvbGFiIHVwbG9hZCBieSByb3VnaGx5IHRoZSBzYW1lIGZhY3RvciwgYW5kIHB1c2ggdGhlIHBvc2l0aXZlOm5lZ2F0aXZlIGltYWdlIHJhdGlvIHRvIGFib3V0IDE6Mi42LiBNb3N0IG9mIHRoYXQgaW1hZ2VyeSBpcyB1cmJhbiwgYWlycG9ydCBvciBmYXJtbGFuZCBzY2VuZXJ5IHdpdGggbm8gd2F0ZXIgaW4gaXQsIHNvIHRoZSBtYXJnaW5hbCBuZWdhdGl2ZSB0ZWFjaGVzIHZlcnkgbGl0dGxlIGJleW9uZCB0aGUgZmlyc3QgZmV3IGh1bmRyZWQuDQoNClRoZSBjaG9zZW4gMTAlIHNpdHMgaW4gdGhlIHJhbmdlIFVsdHJhbHl0aWNzIHJlY29tbWVuZHMgZm9yIGJhY2tncm91bmQgaW1hZ2VyeSBhbmQga2VlcHMgdGhlIHN3ZWVwIGFmZm9yZGFibGUuIFRoZSB2YWx1ZSBpcyBhIGZsYWcsIG5vdCBhIGNvbnN0YW50OiBgLS1iYWNrZ3JvdW5kLWZyYWN0aW9uIDAuMGAgcmVwcm9kdWNlcyBhIHBvc2l0aXZlcy1vbmx5IGRhdGFzZXQgYW5kIGAxLjBgIHJldGFpbnMgZXZlcnkgbmVnYXRpdmUuDQoNCiMjIyBTZWxlY3Rpb24gaXMgZml4ZWQgYWNyb3NzIGFsbCBmb3VyIHJlc29sdXRpb25zDQoNClNhbXBsaW5nIGlzIHNlZWRlZCAoYG51bXB5LmRlZmF1bHRfcm5nKDQyKWApIGFuZCBzdHJhdGlmaWVkIGJ5IHNwbGl0LiBgaW1hZ2VfbWFuaWZlc3QuY3N2YCBpcyB3cml0dGVuIG9uY2UgYW5kIGlzIHRoZSBzaW5nbGUgaW5wdXQgdG8gZXZlcnkgcmVzb2x1dGlvbiBjb25kaXRpb24sIHNvIEUwMC1FMDMgc2VlIGJ5dGUtaWRlbnRpY2FsIGltYWdlIGFuZCBhbm5vdGF0aW9uIGlkZW50aXRpZXMuIFRoaXMgaXMgdGhlIFBhcnQgRCBjb250cm9sOyBpdCBpcyBlbmZvcmNlZCBieSBjb25zdHJ1Y3Rpb24gcmF0aGVyIHRoYW4gYnkgY29udmVudGlvbi4NCg0KIyMjIENhdmVhdA0KDQpUaGVzZSBuZWdhdGl2ZXMgYXJlICplYXN5KiBuZWdhdGl2ZXMgLS0gcHJlZG9taW5hbnRseSBsYW5kIHNjZW5lcyByYXRoZXIgdGhhbiBvcGVuIHdhdGVyIG9yIGhhcmJvdXJzIHdpdGhvdXQgdmVzc2Vscy4gVGhleSB3aWxsIHN1cHByZXNzIG9idmlvdXMgZmFsc2UgcG9zaXRpdmVzIGJ1dCBzaG91bGQgbm90IGJlIHJlYWQgYXMgYSBoYXJkLW5lZ2F0aXZlIG1pbmluZyBzdHJhdGVneS4NCg0KIyMgU3BsaXQNCg0KVGhlIG9mZmljaWFsIEZBSVIxTSBUcmFpbi9WYWwgc3BsaXQgaXMgdXNlZCB1bmNoYW5nZWQuDQoNCnwgU3BsaXQgfCBJbWFnZXMgfCBTaGlwcyB8IEJhY2tncm91bmQgaW1hZ2VzIHwNCnwtLS18LS0tOnwtLS06fC0tLTp8DQp8IFRyYWluIHwgNCw5MzAgfCAzMSw0NjIgfCA1MjIgfA0KfCBWYWwgfCAyLDc3NiB8IDI3LDUyMCB8IDI0OSB8DQoNCioqTm90ZSBmb3IgdGhlIHdyaXRlLXVwOioqIHRoZSBvZmZpY2lhbCBzcGxpdCBpcyBub3QgYmFsYW5jZWQgZm9yIHNoaXBzLiA0NyUgb2YgYWxsIHNoaXAgaW5zdGFuY2VzIGZhbGwgaW4gVmFsLCBldmVuIHRob3VnaCBWYWwgaG9sZHMgb25seSAzNiUgb2YgdGhlIGltYWdlcyAtLSBWYWwgdGlsZXMgYXJlIG1hcmtlZGx5IGRlbnNlciBpbiBzaGlwcy4gVGhhdCBpcyBhIHByb3BlcnR5IG9mIEZBSVIxTSwgbm90IG9mIHRoaXMgcGlwZWxpbmUuIFRoZSBzcGxpdCBpcyBrZXB0IGFzLWlzIGJlY2F1c2UgaXQgaXMgdGhlIHB1Ymxpc2hlZCBvbmUgYW5kIGJlY2F1c2UgdGhlIHJlc29sdXRpb24gY29tcGFyaXNvbiBvbmx5IHJlcXVpcmVzIHRoYXQgdGhlIHNwbGl0IGJlICppZGVudGljYWwqIGFjcm9zcyBFMDAtRTAzLCB3aGljaCBpdCBpcy4gSXQgc2hvdWxkIGJlIHN0YXRlZCBhcyBhIGxpbWl0YXRpb24gcmF0aGVyIHRoYW4gc2lsZW50bHkgcmUtc3BsaXQuDQoNCiMjIE9iamVjdC1zaXplIGJpbnMNCg0KQXNzaWduZWQgb25jZSBoZXJlLCBmcm9tIG9yaWdpbmFsLXJlc29sdXRpb24gYXJlYSwgYW5kIGNhcnJpZWQgdW5jaGFuZ2VkIGludG8gZXZlcnkgcmVzb2x1dGlvbiBjb25kaXRpb24uIFNlZSBgYW5hbHlzaXMvb2JqZWN0X3NpemVfZGVmaW5pdGlvbi5tZGAuDQoNCnwgQmluIHwgVGhyZXNob2xkIChweMKyLCBvcmlnaW5hbCkgfCBTaGlwcyB8IFNoYXJlIHwNCnwtLS18LS0tfC0tLTp8LS0tOnwNCnwgc21hbGwgfCA8IDEsMDI0IHwgMjQsOTk4IHwgNDIuNCUgfA0KfCBtZWRpdW0gfCAxLDAyNCDigJMgOSwyMTYgfCAyNyw3NzQgfCA0Ny4xJSB8DQp8IGxhcmdlIHwg4omlIDksMjE2IHwgNiwyMTAgfCAxMC41JSB8DQoNCiMjIEFubm90YXRpb24gY2xlYW5pbmcNCg0KUm93cyBmbGFnZ2VkIHVudXNhYmxlIGJ5IGBzY3JpcHRzL3ZhbGlkYXRlX2Fubm90YXRpb25zLnB5YCBhcmUgZXhjbHVkZWQuIEZvciBzaGlwcyB0aGF0IGV4Y2x1c2lvbiBpcyBlbXB0eSAtLSBub25lIG9mIHRoZSAyOTcgZGVnZW5lcmF0ZSBGQUlSMU0gcG9seWdvbnMgaXMgYSBzaGlwIC0tIHNvIHRoaXMgbGF5ZXIgY29udGFpbnMgZXZlcnkgc2hpcCBhbm5vdGF0aW9uIGluIHRoZSBkYXRhc2V0LiBTZWUgYGRhdGEvYW5ub3RhdGlvbl92YWxpZGF0aW9uL2NsZWFuaW5nX3JlcG9ydC5tZGAuDQo=",
    "data/ship_only/stats.json": "ew0KICAic2hpcF9jYXRlZ29yaWVzIjogWw0KICAgICJEcnkgQ2FyZ28gU2hpcCIsDQogICAgIk1vdG9yYm9hdCIsDQogICAgIkZpc2hpbmcgQm9hdCIsDQogICAgIm90aGVyLXNoaXAiLA0KICAgICJFbmdpbmVlcmluZyBTaGlwIiwNCiAgICAiTGlxdWlkIENhcmdvIFNoaXAiLA0KICAgICJQYXNzZW5nZXIgU2hpcCIsDQogICAgIlR1Z2JvYXQiLA0KICAgICJXYXJzaGlwIg0KICBdLA0KICAiY2xhc3NfbWFwcGluZyI6IHsNCiAgICAiY2xhc3NfaWQiOiAwLA0KICAgICJjbGFzc19uYW1lIjogInNoaXAiDQogIH0sDQogICJuX3NoaXBfYW5ub3RhdGlvbnMiOiA1ODk4MiwNCiAgIm5fc2hpcF9hbm5vdGF0aW9uc19kcm9wcGVkX2FzX3VudXNhYmxlIjogMCwNCiAgIm5faW1hZ2VzX3RvdGFsIjogNzcwNiwNCiAgIm5faW1hZ2VzX3dpdGhfc2hpcHMiOiA2OTM1LA0KICAibl9iYWNrZ3JvdW5kX2ltYWdlc19yZXRhaW5lZCI6IDc3MSwNCiAgIm5fYmFja2dyb3VuZF9pbWFnZXNfYXZhaWxhYmxlIjogMTc4NDAsDQogICJiYWNrZ3JvdW5kX2ZyYWN0aW9uX3JlcXVlc3RlZCI6IDAuMSwNCiAgImJhY2tncm91bmRfZnJhY3Rpb25fcmVhbGl6ZWQiOiAwLjEwMDEsDQogICJzZWVkIjogNDIsDQogICJieV9zcGxpdCI6IHsNCiAgICAiVHJhaW4iOiB7DQogICAgICAiaW1hZ2VzIjogNDkzMCwNCiAgICAgICJiYWNrZ3JvdW5kX2ltYWdlcyI6IDUyMiwNCiAgICAgICJzaGlwcyI6IDMxNDYyDQogICAgfSwNCiAgICAiVmFsIjogew0KICAgICAgImltYWdlcyI6IDI3NzYsDQogICAgICAiYmFja2dyb3VuZF9pbWFnZXMiOiAyNDksDQogICAgICAic2hpcHMiOiAyNzUyMA0KICAgIH0NCiAgfSwNCiAgInNpemVfYmlucyI6IHsNCiAgICAibWV0aG9kIjogIkNPQ08gcGl4ZWwtYXJlYSB0aHJlc2hvbGRzIG9uIG9yaWdpbmFsLXJlc29sdXRpb24gYm94ZXMiLA0KICAgICJzbWFsbF9tYXhfcHgyIjogMTAyNCwNCiAgICAibWVkaXVtX21heF9weDIiOiA5MjE2LA0KICAgICJ0ZXJ0aWxlX3RocmVzaG9sZHNfcHgyIjogWw0KICAgICAgNjgyLjAsDQogICAgICAyNzYwLjANCiAgICBdLA0KICAgICJjb3VudHMiOiB7DQogICAgICAic21hbGwiOiAyNDk5OCwNCiAgICAgICJtZWRpdW0iOiAyNzc3NCwNCiAgICAgICJsYXJnZSI6IDYyMTANCiAgICB9DQogIH0NCn0=",
}
for _rel, _b64 in _FILES.items():
    _p = PKG / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_bytes(base64.b64decode(_b64))
print(f"wrote {len(_FILES)} file(s) into {PKG} (data/ship_only): {sorted(_FILES)}")

In [ ]:
import sys
from pathlib import Path

PKG = Path("/kaggle/working/kaggle")
if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
print(f"code package ready at {PKG}")

## Step 1 — Environment

In [ ]:
import os
import platform
import shutil
import sys

import torch

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"CPU cores    : {os.cpu_count()}  (a sensible default for Step 7's "
      f"WORKERS -- master.yaml's dataloader worker count is pinned higher, "
      f"for a bigger machine)")
print(f"PyTorch      : {torch.__version__}")
try:
    import ultralytics
    print(f"Ultralytics  : {ultralytics.__version__}")
except ImportError:
    print("Ultralytics  : not installed yet (installed in Step 2)")

print(f"CUDA avail.  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version : {torch.version.cuda}")
    print(f"GPU count    : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}        : {p.name}  ({p.total_memory / 1e9:.1f} GB)")
else:
    print("No GPU detected. Notebook -> Settings -> Accelerator -> GPU, then "
          "Factory reset & rerun. Training at imgsz=1024 is not viable on CPU.")

free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk (/kaggle/working): {free_gb:.1f} GB")

## Step 2 — Install dependencies

Kaggle's GPU image already ships a working `torch` matched to its CUDA
driver; it is deliberately **not** reinstalled here, since replacing it is
the single most common way to break an otherwise-working Kaggle GPU image.
Only `ultralytics` (pinned to the version this study was built against) and a
couple of small, harmless libraries are installed or upgraded.

In [ ]:
!pip install -q "ultralytics==8.4.61" pyyaml pyarrow

import ultralytics
print("ultralytics", ultralytics.__version__)

## Step 3 — Locate the attached Kaggle dataset

`Fair1m_Ship_Dataset` is searched for dynamically under `/kaggle/input` —
this notebook never hard-codes a username or dataset slug, since both are
assigned by Kaggle at upload time. If any of the four resolution conditions
is missing, this stops here rather than failing confusingly deep inside
training.

In [ ]:
from src.paths import CONDITIONS, find_datasets_root

print("=" * 56)
print("FAIR1M SHIP DATASET")
print("=" * 56)
try:
    DATASETS = find_datasets_root()
except FileNotFoundError as e:
    print(f"\nERROR: {e}")
    raise SystemExit(
        "\nSTOPPING: attach 'Fair1m_Ship_Dataset' via Notebook -> Add Input, "
        "then Run All again.")

print(f"\nDataset found under:")
for p in sorted({str(p) for p in DATASETS.values()}):
    print(f"  {p}")
print()
for cond in CONDITIONS:
    label = {"original": "E00", "r75": "E01", "r50": "E02", "r25": "E03"}[cond]
    print(f"{label} ({cond:<8}): FOUND  -> {DATASETS[cond]}")

## Step 4 — Reconstruct the annotation layer, then validate dataset structure

The ~1.5 MB ground-truth annotation layer (image manifest + per-ship boxes in
original-resolution coordinates) isn't embedded in this notebook — see Step 0
— so it's rebuilt here from the attached dataset's own `original`-condition
YOLO labels plus each image's native pixel size. This is lossless: the label
files ARE the source of truth for box geometry (normalized coordinates,
invariant under resize), so reconstructing from them reproduces the same
ground truth the local pipeline validated.

Then confirms every condition has `images/{train,val}` and `labels/{train,val}`,
that `data.yaml` declares exactly one class (`ship`), and — the strongest
check — that the label files are **byte-identical across all four
resolutions**. Normalized YOLO coordinates are invariant under a uniform
resize, so any difference here means the annotation identities are not
shared between conditions and the comparison would be invalid.

In [ ]:
!python {PKG}/scripts/build_annotation_layer.py

In [ ]:
import hashlib

import pandas as pd
import yaml

from src.paths import annotation_dir

images = pd.read_csv(annotation_dir() / "image_manifest.csv")
ships = pd.read_parquet(annotation_dir() / "ship_annotations.parquet")
print(f"annotation layer: {len(images):,} images, {len(ships):,} ship instances\n")

ok = True
ref_digest = None
for cond in ("original", "r75", "r50", "r25"):
    root = DATASETS[cond]
    d = yaml.safe_load((root / "data.yaml").read_text())
    names = list(d["names"].values()) if isinstance(d["names"], dict) else d["names"]
    n_img = len(list((root / "images").rglob("*.jpg")))
    n_lbl = len(list((root / "labels").rglob("*.txt")))
    boxes = sum(sum(1 for l in p.read_text().splitlines() if l.strip())
                for p in (root / "labels").rglob("*.txt"))
    h = hashlib.sha256()
    for p in sorted((root / "labels").rglob("*.txt")):
        h.update(p.read_bytes())
    digest = h.hexdigest()
    if ref_digest is None:
        ref_digest = digest
    same = digest == ref_digest
    class_ok = d.get("nc") == 1 and names == ["ship"]
    good = n_img == len(images) and n_lbl == len(images) and boxes == len(ships) and same and class_ok
    ok &= good
    print(f"  {cond:<9} images {n_img:>6,}  labels {n_lbl:>6,}  boxes {boxes:>7,}  "
          f"nc=1/ship: {class_ok}  labels identical: {same}   {'OK' if good else 'PROBLEM'}")

print("\n" + ("ALL CHECKS PASSED" if ok else "PROBLEMS FOUND -- do not train"))
assert ok

## Step 5 — Paths, resume status, and local pipeline tests

Sets the results root to `/kaggle/working/results` (kept separate from the
`/kaggle/working/kaggle` code package) and looks for a **previously-attached
Notebook Output** of this study under `/kaggle/input` (from an earlier
interrupted session) to restore before continuing — see
`src/paths.restore_previous_results`. `/kaggle/working` alone is **not**
guaranteed to survive a fresh session, which is why this study's own outputs
double as a re-attachable input.

Then runs the local self-test suite (`scripts/verify_kaggle_setup.py`) —
structural and logic checks that need neither a GPU nor the dataset, plus the
dataset checks now that Steps 3/4 confirmed the attachment.

In [ ]:
from src.paths import find_results_root, restore_previous_results

RESULTS_ROOT = find_results_root()
restored_from = restore_previous_results(RESULTS_ROOT)
if restored_from:
    print(f"Restored previous results from {restored_from} -> {RESULTS_ROOT}")
else:
    print(f"No previous results found to restore; starting fresh at {RESULTS_ROOT}")

print()
run_ids = [("E00", "00_baseline"), ("E01", "01_r75"), ("E02", "02_r50"), ("E03", "03_r25")]
for label, run_id in run_ids:
    out_dir = RESULTS_ROOT / run_id
    if (out_dir / "metrics.json").exists():
        state = "COMPLETE"
    elif (out_dir / "train" / run_id / "weights" / "last.pt").exists():
        state = "RESUME (checkpoint found)"
    else:
        state = "NOT STARTED"
    print(f"  {label} {run_id:<12} {state}")

In [ ]:
!python {PKG}/scripts/verify_kaggle_setup.py

## Step 6 — GPU sanity test (required before the full sweep)

One epoch on a small, curated, ship-guaranteed subset (32 train / 16 val
images per condition), at the study's real `imgsz=1024` / `max_det=1000`.
Proves the plumbing on this GPU before committing hours to it: CUDA works,
the model initializes, images and labels load, training starts, loss is
produced, validation runs, predictions are generated, a checkpoint is saved,
and metrics come out finite.

This is a small staged subset rather than a `--fraction` of the full
condition on purpose: Ultralytics scans and verifies every image/label pair
in `train:`/`val:` to build its label cache *before* `fraction` is ever
applied, and that cache can't be saved and reused on the read-only
`/kaggle/input` mount -- so a "5%" run pointed at the full ~7,706-image
condition would still pay the full scan cost, on every launch, across all
four conditions, which is what made this step slow. See
`src/paths.stage_sanity_subset`.

**These numbers are not results.** They go to `*_sanity` directories and are
excluded from the aggregated tables. If the assertion below fails, **stop**
— do not proceed to Step 7.

Runs once: if a previous commit's restored Output already has passing sanity
results, this is skipped (no `--force`) rather than repeated every commit.

In [ ]:
!python {PKG}/scripts/run_all.py --sanity --epochs 1 --results-root {RESULTS_ROOT}

In [ ]:
import json

ok = True
for _, run_id in run_ids:
    p = RESULTS_ROOT / f"{run_id}_sanity" / "metrics.json"
    if not p.exists():
        print(f"{run_id}_sanity: NO metrics.json -- FAIL"); ok = False
        continue
    m = json.loads(p.read_text()).get("metrics", {})
    n_gt = m.get("n_gt", {}).get("all", 0)
    has_ckpt = (RESULTS_ROOT / f"{run_id}_sanity" / "best.pt").exists()
    good = n_gt > 0 and has_ckpt
    print(f"{run_id}_sanity   mAP50 {m.get('mAP50', float('nan')):.4f}   "
          f"detections {m.get('n_detections', 0):,}   n_gt {n_gt:,}   "
          f"checkpoint saved: {has_ckpt}")
    ok &= good

print("\n" + ("SANITY TEST PASSED -- safe to run the full sweep (Step 7)"
              if ok else "SANITY TEST FAILED -- investigate before proceeding"))
assert ok, "Do not begin the full sweep: the sanity test did not pass."

## Step 7 — Time-budgeted training slice: E00 → E01 → E02 → E03

Uses the **full** 7,706-image, 58,982-ship dataset (not the sanity subset).
The full sweep is roughly 3–5 GPU-hours per condition on a T4 (12–20 h total)
— far more than fits in one Kaggle session or one commit you'd want to bet on
completing. This is the expected cost of the design (`imgsz=1024`, 50 epochs,
early stopping deliberately disabled — see `configs/master.yaml`), not a sign
anything is wrong.

**This cell is meant to be committed (Save Version → Save & Run All)
repeatedly**, not run once. Each run does up to `TIME_BUDGET_HOURS` of work —
resuming whatever condition is in progress, then starting the next one with
whatever time is left — and stops itself cleanly before the budget runs out,
rather than risking Kaggle killing it mid-epoch. Pick a budget comfortably
under your session limit (1–1.5 h leaves plenty of margin for Steps 0–6 and
any slow epoch). Re-commit this same notebook until the status cell below
reports all four conditions complete.

`BATCH` and `WORKERS` are **environment** knobs, not experimental ones — see
`src/training.ALLOWED_OVERRIDES` / `ENV_KEYS`, which mechanically forbid
overriding anything that would actually change the comparison (`imgsz`,
`epochs`, augmentation, LR, etc. cannot be touched this way). Both are fixed
once and used identically for all four conditions. Two safe ways to speed
individual runs up if they're GPU- or CPU-bound:

- **`BATCH`** — raise it if `nvidia-smi` (Step 1) shows headroom below the
  GPU's VRAM (T4: 15 GB; YOLOv8m at imgsz=1024/batch=8 typically uses well
  under that). A larger batch improves GPU utilization. If it OOMs, Ultralytics
  errors clearly — just lower it back down and rerun (already-finished
  conditions are skipped, so nothing is lost).
- **`WORKERS`** — `configs/master.yaml` pins 8 dataloader workers, tuned for
  a many-core machine. Kaggle GPU sessions typically have far fewer CPU cores;
  oversubscribing workers can add contention overhead rather than help. Try
  matching it to the actual core count (`!nproc` or the Step 1 output).

In [ ]:
BATCH = 8              # keep identical for all four conditions; see configs/master.yaml
WORKERS = 4            # match this to the CPU cores Kaggle actually gave this session
TIME_BUDGET_HOURS = 1.5   # this run stops cleanly at or before this many hours

!python {PKG}/scripts/run_all.py --results-root {RESULTS_ROOT} \
    --batch {BATCH} --workers {WORKERS} --time-budget {TIME_BUDGET_HOURS}

In [ ]:
import json

status_path = RESULTS_ROOT / "experiment_status.json"
status = json.loads(status_path.read_text()) if status_path.exists() else {}
done = [k for k, v in status.items() if v["status"] == "ok" or v["status"].startswith("skipped")]

print("=" * 56)
print("PROGRESS")
print("=" * 56)
for run_id in [r for _, r in run_ids]:
    s = status.get(run_id, {}).get("status", "not started")
    print(f"  {run_id:<14} {s}")

if len(done) == len(run_ids):
    print("\nALL FOUR CONDITIONS COMPLETE -- proceed to Step 8.")
else:
    print(f"\n{len(done)}/{len(run_ids)} complete. NOT finished yet.")
    print("Proceed to Step 8 to persist this progress, then come back and "
          "commit this notebook again (Step 7 onward) to continue -- it will "
          "resume exactly where this run stopped.")

## Step 8 — Persist progress as a Kaggle Notebook Output

`/kaggle/working` is not guaranteed to survive a fresh session. After every
Step 7 run (finished or not), use Kaggle's **Save Version → Save & Run All
(Commit)** to persist everything currently in `/kaggle/working` as this
notebook's Output. This is the mechanism this whole workflow depends on:
Kaggle's "Save Version" always re-executes the notebook fresh in a new
container rather than snapshotting a live session, so a run's progress is
IN THE OUTPUT SAVED HERE or it doesn't exist for next time.

To resume, whether in a brand-new session or the next commit: **Add Input →
Notebook Output → this notebook's latest version** (in addition to
`Fair1m_Ship_Dataset`), enable GPU, and Run All — Step 5 finds and restores
that output automatically, and Step 7 picks up exactly where it left off
(skips finished conditions, resumes the in-progress one, starts the next).

**The loop, in short:** commit → check the Step 7 status cell → if not all
four are done, reattach this notebook's own latest Output and commit again →
repeat until done → Step 9 onward.

In [ ]:
print("Currently in", RESULTS_ROOT, ":")
for p in sorted(RESULTS_ROOT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(RESULTS_ROOT))

## Step 9 — Collect results (tables + plots)

Builds the resolution × object-size table, degradation percentages, and
plots. A condition that has not run is reported as **missing**, never filled
in or estimated.

In [ ]:
!python {PKG}/scripts/collect_results.py --results-root {RESULTS_ROOT} --out {RESULTS_ROOT}

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(RESULTS_ROOT / "master_results.csv")
cols = ["resolution", "mAP50", "mAP50_95", "AP_small", "AP_medium", "AP_large",
        "recall", "precision"]
print("PERFORMANCE (size bins fixed at ORIGINAL resolution:")
print("  small < 1024 px^2, medium 1024-9216, large >= 9216)\n")
display(df[[c for c in cols if c in df]].round(4))

drop = [c for c in df.columns if c.endswith("_drop_pct")]
if drop:
    print("\nDEGRADATION vs baseline (%):")
    display(df[["resolution"] + drop].round(2))

for name in ("resolution_x_object_size.png", "resolution_vs_map.png",
             "training_cost.png"):
    p = RESULTS_ROOT / "plots" / name
    if p.exists():
        display(Image(filename=str(p)))

## Step 10 — Qualitative comparisons

Renders the same validation scenes (small ships, medium ships, large ships,
dense harbours, isolated ships, difficult backgrounds) across all four
resolutions with each condition's own model's predictions overlaid on the
shared ground truth.

In [ ]:
!python {PKG}/scripts/qualitative_comparison.py --results-root {RESULTS_ROOT} \
    --out {RESULTS_ROOT}/qualitative

## Step 11 — Experiment report

In [ ]:
!python {PKG}/scripts/build_experiment_report.py --results-root {RESULTS_ROOT}

## Step 12 — Package a compact results archive

One zip with metrics, configs, logs, plots, qualitative results, the report,
and the best/last checkpoints for each condition. **Does not include the
image datasets** — those already live in the attached Kaggle Dataset, so
duplicating 6+ GB here would be pure waste.

In [ ]:
import zipfile
from pathlib import Path

archive_path = Path("/kaggle/working/fair1m_ship_resolution_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(RESULTS_ROOT.parent)))
    for cfg in (PKG / "configs").glob("*.yaml"):
        zf.write(cfg, arcname=f"configs/{cfg.name}")

size_mb = archive_path.stat().st_size / 1e6
print(f"Wrote {archive_path}  ({size_mb:.1f} MB)")
print("This archive, plus the Notebook Output from Step 8, is everything "
      "needed to preserve the experimental results without duplicating the "
      "attached dataset.")

## Optional follow-ons

Only after the primary sweep is complete and its results are recorded.

- **Repeated seeds.** One run per condition supports no significance claim.
  If GPU budget allows, repeat E00/E02/E03 with seeds 123 and 456 and report
  mean ± sd. If not, state plainly that the study is single-seed.
- **Cross-resolution transfer.** `scripts/evaluate.py --eval-on <cond>
  --tag ...` scores a model trained at one resolution on another. A different
  question from the primary sweep; keep it separately tagged so
  `collect_results.py` cannot mistake it for a primary-sweep number.

In [ ]:
# Repeated seeds -- uncomment to run. Costs roughly 2x the primary sweep.
# for seed in (123, 456):
#     for cfg in ("baseline", "r50", "r25"):
#         !python {PKG}/scripts/run_experiment.py --config {PKG}/configs/{cfg}.yaml \
#             --results-root {RESULTS_ROOT}_seed{seed} --batch {BATCH}